# Cross-trait and multi-polytranscriptional risk score analysis of PD in AMP-PD: Polygenic score analyses

**Project**: Cross-trait and multi-polytranscriptomic score analysis of Parkinson's disease identifies novel associations and improves prediction

**Date last updated**: July 2026 

 # Initial set-up 

## Loading Python libraries

In [ ]:
# Use the os package to interact with the environment
import os
import sys
import subprocess
import json

# Bring in Pandas for Dataframe functionality
import pandas as pd
from functools import reduce

# Bring some visualization functionality 
import seaborn as sns  

# numpy for basics
import numpy as np

# Use StringIO for working with file contents
from io import StringIO

# Enable IPython to display matplotlib graphs
import matplotlib.pyplot as plt
%matplotlib inline

# Enable interaction with the FireCloud API
from firecloud import api as fapi

# Import the iPython HTML rendering for displaying links to Google Cloud Console
from IPython.core.display import display, HTML

# Import urllib modules for building URLs to Google Cloud Console
import urllib.parse

# BigQuery for querying data
from google.cloud import bigquery

#Import Sys
import sys as sys

## Defining shell functions

In [ ]:
# Utility routine for printing a shell command before executing it
def shell_do(command):
    
    print(f'Executing: {command}', file=sys.stderr)
    !$command
    
def shell_return(command):
    print(f'Executing: {command}', file=sys.stderr)
    output = !$command
    return '\n'.join(output)

# Utility routine for printing a query before executing it
def bq_query(query):
    print(f'Executing: {query}', file=sys.stderr)
    return pd.read_gbq(query, project_id=BILLING_PROJECT_ID, dialect='standard')

# Utility routine for display a message and a link
def display_html_link(description, link_text, url):
    html = f'''
    <p>
    </p>
    <p>
    {description}
    <a target=_blank href="{url}">{link_text}</a>.
    </p>
    '''

    display(HTML(html))

# Utility routines for reading files from Google Cloud Storage
def gcs_read_file(path):
    """Return the contents of a file in GCS"""
    contents = !gsutil -u {BILLING_PROJECT_ID} cat {path}
    return '\n'.join(contents)
    
def gcs_read_csv(path, sep=None):
    """Return a DataFrame from the contents of a delimited file in GCS"""
    return pd.read_csv(StringIO(gcs_read_file(path)), sep=sep, engine='python')

# Utility routine for displaying a message and link to Cloud Console
def link_to_cloud_console_gcs(description, link_text, gcs_path):
    url = '{}?{}'.format(
        os.path.join('https://console.cloud.google.com/storage/browser',
                     gcs_path.replace("gs://","")),
        urllib.parse.urlencode({'userProject': BILLING_PROJECT_ID}))

    display_html_link(description, link_text, url)

## Set paths

In [ ]:
# Set up billing project and data path variables
BILLING_PROJECT_ID = os.environ['GOOGLE_PROJECT']
WORKSPACE_NAMESPACE = os.environ['WORKSPACE_NAMESPACE']
WORKSPACE_NAME = os.environ['WORKSPACE_NAME']
WORKSPACE_BUCKET = os.environ['WORKSPACE_BUCKET']
WORKSPACE_ATTRIBUTES = fapi.get_workspace(WORKSPACE_NAMESPACE, WORKSPACE_NAME).json().get('workspace',{}).get('attributes',{})

## Print the information to check we are in the proper release and billing 
## This will be different for you, the user, depending on the billing project your workspace is on
print('Billing and Workspace')
print(f'Workspace Name @ `WORKSPACE_NAME`: {WORKSPACE_NAME}')
print(f'Billing Project @ `BILLING_PROJECT_ID`: {BILLING_PROJECT_ID}')
print(f'Workspace Bucket, where you can upload and download data @ `WORKSPACE_BUCKET`: {WORKSPACE_BUCKET}')
print('')

## AMP-PD v4.0
# Explicitly define release v4.0 path 
AMP_RELEASE_GENO = 'gs://path/removed'

print('AMP-PD v4.0')
print(f'Path to AMP-PD v4.0 genetic data: {AMP_RELEASE_GENO}')

## Load R for Python


In [ ]:
%load_ext rpy2.ipython

## Make directories

In [ ]:
# Make a general directory called "/home/jupyter/multiTRS/"
GENO_DIR = f'/home/jupyter/multiTRS/geno'
!echo $GENO_DIR
!mkdir -p $GENO_DIR
!mkdir -p $GENO_DIR/raw
!mkdir -p $GENO_DIR/keep_files
!mkdir -p $GENO_DIR/software
!mkdir -p $GENO_DIR/software/GenoTools
!mkdir -p $GENO_DIR/PRS_scorefiles

# Create keep files for each cohort  

We need to create keep files for each cohort so we can QC these seperately.  

We can use the IDs from any of the score files, as all these individuals were used for TRS

##  Copy ancestry labels across

In [ ]:
!ls /home/jupyter/multiTRS/geno/

In [ ]:
# Copy over the case/control and demographics data etc to the working directory
shell_do(f'gsutil -u {BILLING_PROJECT_ID} -m cp -n -r {WORKSPACE_BUCKET}/AMPPD_v25_GenoTools_Predictions_PDBP_PPMI_HBS_EUR.txt /home/jupyter/multiTRS/geno/keep_files/')

shell_do(f'gsutil -u {BILLING_PROJECT_ID} -m cp -n -r {WORKSPACE_BUCKET}/AMPPD_v25_GenoTools_Predictions.txt /home/jupyter/multiTRS/geno/keep_files/')

In [ ]:
%%R

library(data.table)
library(dplyr)
library(stringr)

ancestry <- fread("/home/jupyter/multiTRS/geno/keep_files/AMPPD_v25_GenoTools_Predictions.txt")

cohorts <- c("PDBP","PPMI","HBS")

for (cohort in cohorts){

df <- fread(paste0("/home/jupyter/multiTRS/scores/", cohort, "_TWAS_SMR_FDR_scores.txt")) %>% select(participant_id)
    
df <- df %>% rename(`#FID` = participant_id)

df$IID <- df$`#FID`
  
print(paste0("The number of individuals used for TRS for ",cohort," is ",nrow(df)))
    
df_ancestry <- merge(df,ancestry, by ="IID")
    
print(table(df_ancestry$label))
    
df_EUR <- df_ancestry %>% filter(label == "EUR")

df_EUR <- df_EUR %>% select(`#FID`,IID)

print(paste0("The number of genetically EUR individuals for ",cohort," is ",nrow(df_EUR)))
print("========")
    
write.table(df_EUR, file = paste0("/home/jupyter/multiTRS/geno/keep_files/", cohort, "_EUR_keep.txt"), sep = "\t", quote = FALSE, row.names = FALSE, col.names = FALSE)

if (cohort == "PPMI"){
    
    df_AJ <- df_ancestry %>% filter(label == "AJ")

    df_AJ <- df_AJ %>% select(`#FID`,IID)

    print(paste0("The number of genetically AJ individuals for ",cohort," is ",nrow(df_AJ)))
    print("========")
    
write.table(df_AJ, file = paste0("/home/jupyter/multiTRS/geno/keep_files/", cohort, "_AJ_keep.txt"), sep = "\t", quote = FALSE, row.names = FALSE, col.names = FALSE)
   
}
}
  

## Make a pheno file and sex file for updating later

In [ ]:
%%R

library(data.table)
library(dplyr)
library(stringr)

df <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt")

df <- df %>% select(participant_id,case_control_other_at_baseline, sex)

df$IID <- df$participant_id

df <- df %>% rename("#FID" = participant_id, PHENO1 = case_control_other_at_baseline, SEX = sex)

print(head(df))

# Pheno file
df_pheno <- df %>% select(`#FID`, IID, PHENO1)

df_pheno$PHENO1 <- ifelse(df$PHENO1 == 0, 1,
                    ifelse(df$PHENO1 == 1, 2, df$PHENO1))

print(head(df_pheno))
                          
write.table(df_pheno, file = paste0("/home/jupyter/multiTRS/geno/keep_files/pheno_file.txt"), sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)
                          
# Sex file
df_sex <- df %>% select(`#FID`, IID, SEX)
                          
df_sex$SEX <- ifelse(df_sex$SEX == 1, 2,
                 ifelse(df_sex$SEX == 0, 1, 0))
                          
print(head(df_sex))
                          
write.table(df_sex, file = paste0("/home/jupyter/multiTRS/geno/keep_files/sex_file.txt"), sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)

# Download PLINK 1.9 and PLINK 2

## PLINK 1.9

In [ ]:
%%bash

if test -e /home/jupyter/multiTRS/geno/software/plink_linux_x86_64_20250819.zip; then
    echo "Plink 1.9 is already downloaded in /home/jupyter/multiTRS/geno/software/"
else
    echo -e "Downloading Plink 1.9 \n    -------"
    wget -N -P /home/jupyter/multiTRS/geno/software https://s3.amazonaws.com/plink1-assets/plink_linux_x86_64_20250819.zip
    unzip -o /home/jupyter/multiTRS/geno/software/plink_linux_x86_64_20250819.zip -d /home/jupyter/multiTRS/geno/software
    echo -e "\nPlink downloaded and unzipped in /home/jupyter/multiTRS/geno/software\n"
fi

## PLINK 2

In [ ]:
%%bash

if test -e /home/jupyter/multiTRS/geno/software/plink2_linux_x86_64_20260110.zip; then
    echo "Plink 2 is already downloaded in /home/jupyter/multiTRS/geno/software/"
else
    echo -e "Downloading Plink 2 \n    -------"
    wget -N -P /home/jupyter/multiTRS/geno/software https://s3.amazonaws.com/plink2-assets/plink2_linux_x86_64_20260110.zip
    unzip -o /home/jupyter/multiTRS/geno/software/plink2_linux_x86_64_20260110.zip -d /home/jupyter/multiTRS/geno/software
    echo -e "\nPlink downloaded and unzipped in /home/jupyter/multiTRS/geno/software\n"
fi


## Check they have downloaded

In [ ]:
!ls /home/jupyter/multiTRS/geno/software/

# Install GenoTools and obtain reference panel

## Install

In [ ]:
!pip install the-real-genotools

## Download reference panel

In [ ]:
!genotools-download --destination /home/jupyter/multiTRS/geno/GenoTools

There is a checksum error although the files themselves appear to be fine. You can unzip manually, and the error is flagged on Github

In [ ]:
!ls /home/jupyter/multiTRS/geno/GenoTools/ref_panel/

# Process AMP-PD genetic data

## Check genetic data exists, copy it over and check it was copied

In [ ]:
#Check the case/control data is in the main AMP-PD release 4 release path
shell_do(f'gsutil -u {BILLING_PROJECT_ID} ls {AMP_RELEASE_GENO}/chr*')

## Copy genetic data across and process

The chromosome files are very large as they contain everyone in AMP-PD and variants are derived from whole genome sequencing data. Downloading and keeping all of these files is unnecessary as it would take up a lot of storage.  

We only want to retain those individuals for whom we already have transcriptional data and have used in our TRS analyses. Also, we only want SNPs (no indels or multi-allelic variants), and do not need rare variants as generally not used for PRS.  

We can filter in a loop, downloading each chromosome, keeping only those in each cohort, restricting to SNPs with  2 alleles max and only ACGT, and filtering for MAF 0.01. We can also remove any duplicates based on chr_bp_allele_allele, with alleles ordered alphabetically, making matching with scorefiles easier later on (though there do not appear to be any when this pipeline was tested). This will significantly reduce the file size for later and are necessary filtering steps regardless.

There has already been some QC performed on the genetic data available in AMP-PD. For example, a sex-check has already been performed and individuals not matching genetic and phenotypic sex are excluded. 

For more details see: https://amp-pdrd.org/whole-genome-data

**NOTE: This section of code is efficient for storage but it takes a while to run**

In [ ]:
GENO_RAW = f'{GENO_DIR}/raw'
GENO_FILTERED = f'{GENO_DIR}/raw_filtered'
os.makedirs(GENO_RAW, exist_ok=True)
os.makedirs(GENO_FILTERED, exist_ok=True)

cohorts = {
    "PDBP_EUR": f"{GENO_DIR}/keep_files/PDBP_EUR_keep.txt",
    "PPMI_EUR": f"{GENO_DIR}/keep_files/PPMI_EUR_keep.txt",
    "PPMI_AJ": f"{GENO_DIR}/keep_files/PPMI_AJ_keep.txt",
    "HBS_EUR": f"{GENO_DIR}/keep_files/HBS_EUR_keep.txt"
}

chromosomes = list(map(str, range(1, 23)))

for chr_num in chromosomes:
    chr_prefix = f'chr{chr_num}'
    local_prefix = f'{GENO_RAW}/{chr_prefix}'

    # Download the chromosome file
    for ext in ['pgen', 'pvar', 'psam']:
        local_file = f'{local_prefix}.{ext}'
        if not os.path.exists(local_file):
            remote_file = f'{AMP_RELEASE_GENO}/{chr_prefix}.{ext}'
            print(f"Downloading {remote_file} -> {local_file}")
            subprocess.run(f'gsutil -u {BILLING_PROJECT_ID} cp {remote_file} {local_file}', shell=True, check=True)

    # Process the chromosome for each cohort
    for cohort_name, keep_file in cohorts.items():
        filtered_prefix = f'{GENO_FILTERED}/{chr_prefix}_{cohort_name}_MAF_0.01'

        plink_cmd = f"""
        /home/jupyter/multiTRS/geno/software/plink2 \
        --pfile {local_prefix} \
        --keep {keep_file} \
        --chr {chr_num} \
        --set-all-var-ids @_#_\$1_\$2 \
        --snps-only just-acgt \
        --max-alleles 2 \
        --maf 0.01 \
        --rm-dup exclude-all \
        --update-sex /home/jupyter/multiTRS/geno/keep_files/sex_file.txt \
        --pheno /home/jupyter/multiTRS/geno/keep_files/pheno_file.txt \
        --pheno-name PHENO1 \
        --threads 20 \
        --make-pgen \
        --out {filtered_prefix}
        """
        subprocess.run(plink_cmd, shell=True, check=True)
        print(f"Chromosome {chr_num} processed for {cohort_name}.")

    # Delete the raw chromosome files to save space
    for ext in ['pgen', 'pvar', 'psam']:
        raw_file = f'{local_prefix}.{ext}'
        if os.path.exists(raw_file):
            os.remove(raw_file)
            print(f"Deleted raw file: {raw_file}")

    print(f"Finished chromosome {chr_num}. Raw files removed, filtered files are in {GENO_FILTERED}\n")


## Perform some checks on the files

### Check the filtered files are all there

There should be 353 (.psam, .pgen, .pvar and a .log file for each chromsome and a logs folder)

In [ ]:
! mkdir -p $GENO_DIR/raw_filtered/logs
!ls $GENO_DIR/raw_filtered/ | wc -l

Also, the raw folder should be empty

In [ ]:
!ls $GENO_DIR/raw/

Move the logs

In [ ]:
!mv $GENO_DIR/raw_filtered/*.log $GENO_DIR/raw_filtered/logs

## Combine into one file per cohort

### Make a directory

In [ ]:
!mkdir -p /home/jupyter/multiTRS/geno/combined_filtered_maf

### Create lists for merging

In [ ]:
cohorts = ["PDBP_EUR", "PPMI_EUR", "PPMI_AJ", "HBS_EUR"]
chromosomes = list(map(str, range(1, 23)))  # full chromosomes 1-22

for cohort in cohorts:
    merge_list_file = f"{GENO_DIR}/raw_filtered/{cohort}_merge_list.txt"
    with open(merge_list_file, 'w') as f:
        for chr_num in chromosomes:
            prefix = f"{GENO_FILTERED}/chr{chr_num}_{cohort}_MAF_0.01"
            f.write(f"{prefix}\n")
    print(f"Merge list created for {cohort}: {merge_list_file}")

### Check files

In [ ]:
!head /home/jupyter/multiTRS/geno/raw_filtered/PDBP_EUR_merge_list.txt
!head /home/jupyter/multiTRS/geno/raw_filtered/PPMI_EUR_merge_list.txt
!head /home/jupyter/multiTRS/geno/raw_filtered/PPMI_AJ_merge_list.txt
!head /home/jupyter/multiTRS/geno/raw_filtered/HBS_EUR_merge_list.txt

### Merge plink files

In [ ]:
%%bash

GENO_DIR=/home/jupyter/multiTRS/geno
OUTPUT_DIR=${GENO_DIR}/combined_filtered_maf
mkdir -p $OUTPUT_DIR

cohorts=("PDBP_EUR" "PPMI_EUR" "PPMI_AJ" "HBS_EUR")

for cohort in "${cohorts[@]}"; do
    MERGE_LIST=${GENO_DIR}/raw_filtered/${cohort}_merge_list.txt
    OUT_PREFIX=${OUTPUT_DIR}/${cohort}_MAF_0.01_all_chr

    echo "Merging all chromosomes for $cohort..."
    /home/jupyter/multiTRS/geno/software/plink2 \
        --pmerge-list $MERGE_LIST \
        --out $OUT_PREFIX

    echo "Done: $OUT_PREFIX"
done

### We can remove the per chromsome files once we are done for space

In [ ]:
!rm /home/jupyter/multiTRS/geno/raw_filtered/*.pvar
!rm /home/jupyter/multiTRS/geno/raw_filtered/*.pgen
!rm /home/jupyter/multiTRS/geno/raw_filtered/*.psam
!ls /home/jupyter/multiTRS/geno/raw_filtered/

### Check the files exist

In [ ]:
!ls /home/jupyter/multiTRS/geno/combined_filtered_maf

# Run GenoTools on the genetic data

## Make directory for clean genetic data

In [ ]:
! mkdir /home/jupyter/multiTRS/geno/clean

## Run Genotools

In [ ]:
!genotools --help

## Perform QC using GenoTools

In [ ]:
%%bash

OUT_DIR=/home/jupyter/multiTRS/geno/clean
mkdir -p "$OUT_DIR"

cohorts=("PDBP_EUR" "PPMI_EUR" "PPMI_AJ" "HBS_EUR")

for cohort in "${cohorts[@]}"; do

OUT_PREFIX="${OUT_DIR}/${cohort}_MAF_0.01_mind_0.02_geno_0.01_hwe_0.000001_rel_0.044"


if [[ -f "$OUT_PREFIX.psam" ]]; then
    echo "Skipping ${cohort}: output already exists (${OUT_PREFIX}.psam)"
    continue
fi

echo "Processing ${cohort}..."

genotools \
  --pfile /home/jupyter/multiTRS/geno/combined_filtered_maf/${cohort}_MAF_0.01_all_chr \
  --related \
  --related_cutoff 0.044 \
  --prune_related \
  --callrate 0.02 \
  --case_control 1e-4 \
  --haplotype 1e-4 \
  --filter_controls \
  --geno 0.01 \
  --hwe 1e-6 \
  --out "$OUT_PREFIX"

echo "Finished ${cohort}"

done

### Check files exist

In [ ]:
!ls /home/jupyter/multiTRS/geno/clean/

### Check sample and variant removal numbers

In [ ]:
%%bash

echo "=== GenoTools QC summary for all cohorts) ==="

cohorts=("PDBP_EUR" "PPMI_EUR" "PPMI_AJ" "HBS_EUR")

for cohort in "${cohorts[@]}"; do

RAW_PREFIX=/home/jupyter/multiTRS/geno/combined_filtered_maf/${cohort}_MAF_0.01_all_chr
CLEAN_PREFIX=/home/jupyter/multiTRS/geno/clean/${cohort}_MAF_0.01_mind_0.02_geno_0.01_hwe_0.000001_rel_0.044

# Check files exist
if [[ ! -f "${RAW_PREFIX}.psam" || ! -f "${CLEAN_PREFIX}.psam" ]]; then
    echo "Skipping ${cohort} as missing files..."
    continue
fi

# Sample counts (subtract header)
raw_samples=$(($(wc -l < "${RAW_PREFIX}.psam") - 1))
clean_samples=$(($(wc -l < "${CLEAN_PREFIX}.psam") - 1))

# Variant counts (subtract header)
raw_variants=$(($(wc -l < "${RAW_PREFIX}.pvar") - 1))
clean_variants=$(($(wc -l < "${CLEAN_PREFIX}.pvar") - 1))

# Differences
removed_samples=$((raw_samples - clean_samples))
removed_variants=$((raw_variants - clean_variants))

# Percentages
sample_retention=$(awk "BEGIN {printf \"%.2f\", (${clean_samples}/${raw_samples})*100}")
variant_retention=$(awk "BEGIN {printf \"%.2f\", (${clean_variants}/${raw_variants})*100}")

# Output
echo ""
echo "=== ${cohort} ==="
printf "Samples  : Raw=%d | Clean=%d | Removed=%d | Retained=%s%%\n" \
    "$raw_samples" "$clean_samples" "$removed_samples" "$sample_retention"

printf "Variants : Raw=%d | Clean=%d | Removed=%d | Retained=%s%%\n" \
    "$raw_variants" "$clean_variants" "$removed_variants" "$variant_retention"

done

### Calculate PCs for final sample

In [ ]:
%%bash

PLINK2="/home/jupyter/multiTRS/geno/software/plink2"
OUT_DIR="/home/jupyter/multiTRS/geno/clean"
mkdir -p "$OUT_DIR"

# Write long-range LD exclusions for GRCh38 (same as GenoTools)
cat > ${OUT_DIR}/long_range_ld.txt << 'EOF'
1   47534328    51534328    r1
2   133742429   137242430   r2
2   182135273   189135274   r3
3   47458510    49962567    r4
3   83450849    86950850    r5
5   98664296    101164296   r6
5   129664307   132664308   r7
5   136164311   139164311   r8
6   24999772    35032223    r9
6   139678863   142178863   r10
8   7142478     13142491    r11
8   110987771   113987771   r12
11  87789108    90766832    r13
12  109062195   111562196   r14
20  33412194    35912078    r15
EOF

cohorts=("PDBP_EUR" "PPMI_EUR" "PPMI_AJ" "HBS_EUR")

for cohort in "${cohorts[@]}"; do

    PFILE="/home/jupyter/multiTRS/geno/clean/${cohort}_MAF_0.01_mind_0.02_geno_0.01_hwe_0.000001_rel_0.044"
    OUT_PREFIX="${OUT_DIR}/${cohort}_PCs"

    # Skip if already done
    if [[ -f "${OUT_PREFIX}.eigenvec" ]]; then
        echo "Skipping ${cohort}: output already exists (${OUT_PREFIX}.eigenvec)"
        continue
    fi

    echo "========================================="
    echo "Processing ${cohort}..."
    echo "========================================="

    # Step 1: Filter variants
    echo "[${cohort}] Step 1: Filtering variants..."
    $PLINK2 \
        --pfile ${PFILE} \
        --maf 0.05 \
        --geno 0.01 \
        --hwe 1e-6 \
        --autosome \
        --exclude range ${OUT_DIR}/long_range_ld.txt \
        --make-pgen psam-cols=fid,parents,sex,pheno1,phenos \
        --out ${OUT_PREFIX}_tmp

    # Step 2: LD pruning
    echo "[${cohort}] Step 2: LD pruning..."
    $PLINK2 \
        --pfile ${OUT_PREFIX}_tmp \
        --indep-pairwise 1000 100 0.2 \
        --autosome \
        --out ${OUT_PREFIX}_pruned

    # Step 3: PCA
    echo "[${cohort}] Step 3: Running PCA..."
    $PLINK2 \
        --pfile ${OUT_PREFIX}_tmp \
        --extract ${OUT_PREFIX}_pruned.prune.in \
        --pca 20 \
        --out ${OUT_PREFIX}

    # Cleanup tmp files
    echo "[${cohort}] Cleaning up tmp files..."
    rm -f ${OUT_PREFIX}_tmp.{pgen,psam,pvar}
    rm -f ${OUT_PREFIX}_pruned.{prune.in,prune.out}

    echo "Finished ${cohort}"

done

echo "========================================="
echo "All cohorts complete"
echo "========================================="

In [ ]:
#! rm /home/jupyter/multiTRS/geno/clean/*PCs*
! ls /home/jupyter/multiTRS/geno/clean/*PCs*

In [ ]:
!head /home/jupyter/multiTRS/geno/clean/HBS_EUR_PCs.eigenvec

### Plot scree

In [ ]:
%%R
library(data.table)
library(dplyr)
library(ggplot2)
library(patchwork)

cohorts <- c("PDBP_EUR", "PPMI_EUR", "PPMI_AJ", "HBS_EUR")

# Initialise empty list to collect plots
plot_list <- list()

for (i in cohorts) {

  # 1. Load eigenvalues
  eig <- fread(paste0("/home/jupyter/multiTRS/geno/clean/", i, "_PCs.eigenval"), header = FALSE)
  colnames(eig) <- c("eigenvalue")

  # Add PC index and variance explained
  eig_dt <- eig %>%
    mutate(
      PC = row_number(),
      pct_var = eigenvalue / sum(eigenvalue) * 100
    )

  # 2. Scree plot
  plot_list[[i]] <- ggplot(eig_dt, aes(x = PC, y = pct_var)) +
    geom_point(size = 2) +
    geom_line() +
    labs(
      title = paste0("Scree Plot (", i, ")"),
      x = "Principal Component",
      y = "% Variance Explained"
    ) +
    theme_minimal()

}

# 3. Combine all cohort plots
combined_plot <- wrap_plots(plot_list, ncol = 2)

# 4. Display
print(combined_plot)

# 5. Save
ggsave(
  filename = "/home/jupyter/multiTRS/geno/clean/PCA_scree_plots.png",
  plot     = combined_plot,
  width    = 12,
  height   = 10,
  dpi      = 300
)

# PRS

## Copy over PRS scorefiles

In [ ]:
#Check the score files are in the uploads folder
shell_do(f'gsutil -u {BILLING_PROJECT_ID} ls {WORKSPACE_BUCKET}/uploads/PRS_scorefiles/')

In [ ]:
# Copy over the case/control and demographics data etc to the working directory
shell_do(f'gsutil -u {BILLING_PROJECT_ID} -m cp -n -r {WORKSPACE_BUCKET}/uploads/PRS_scorefiles/*.snpRes /home/jupyter/multiTRS/geno/PRS_scorefiles')

In [ ]:
!ls /home/jupyter/multiTRS/geno/PRS_scorefiles

In [ ]:
!head /home/jupyter/multiTRS/geno/PRS_scorefiles/Vitamin_B12_Finngen_R12_EUR_2024_SBayesRC_hapmap3_SBayesRC_calc.snpRes

## Install GWASLab

In [ ]:
pip install gwaslab==4

## Use the GWASLab LiftOver function to convert scorefiles to GRCh38

In [ ]:
import gwaslab as gl
import glob
import os

# Find all scorefiles containing "SBayesRC"
search_dir = "/home/jupyter/multiTRS/geno/PRS_scorefiles"
scorefiles = glob.glob(os.path.join(search_dir, "**/*.snpRes"), recursive=True)

print(f"Found {len(scorefiles)} SBayesRC scorefiles: {scorefiles}")

for sumstats_path in scorefiles:
    print(f"\nProcessing: {sumstats_path}")

    # Build output path with .txt extension
    base = os.path.splitext(sumstats_path)[0]
    output_path = f"{base}_GRCh38.txt"

    try:
        mysumstats = gl.Sumstats(
            sumstats_path,
            fmt=None,
            snpid=None,
            rsid="Name",
            chrom="Chrom",
            pos="Position",
            ea="A1",
            nea="A2",
            beta="A1Effect",
            sep="\s+",
            build="19"
            )
        
        mysumstats.data["STATUS"] = 0

        mysumstats.liftover(
            from_build="19",
            to_build="38",
            remove=True
        )

        mysumstats.to_csv(output_path, sep="\t", index=False)
        print(f"  Saved → {output_path}")

    except Exception as e:
        print(f"  ERROR processing {sumstats_path}: {e}")

print("\nDone.")

In [ ]:
!ls /home/jupyter/multiTRS/geno/PRS_scorefiles/*GRCh38.txt

### Create a SNP ID from the update positions

In [ ]:
!head /home/jupyter/multiTRS/geno/PRS_scorefiles/AD_no_UKB_SBayesRC_hapmap3_SBayesRC_calc_GRCh38.txt

In [ ]:
%%R
library(data.table)
library(dplyr)
library(stringr)

setwd("/home/jupyter/multiTRS/geno/PRS_scorefiles/") 
score_files <- list.files(pattern = "GRCh38")       
# TEST: score_files <- score_files[1]

for (i in score_files){
  score <- fread(i)
  score$ID_GRCh38 <- paste0(
    score$CHR, "_", score$POS, "_",        
    pmin(score$EA, score$NEA), "_",
    pmax(score$EA, score$NEA))
    
    score <- score %>% relocate(ID_GRCh38, .before = rsID)
    
                          
write.table(score, file = paste0("/home/jupyter/multiTRS/geno/PRS_scorefiles/",i), sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)

}

## Calculate scores using PLINK2

In [ ]:
%%bash
PLINK2="/home/jupyter/multiTRS/geno/software/plink2"
GENO_QC_DIR="/home/jupyter/multiTRS/geno/clean"
SCORE_DIR="/home/jupyter/multiTRS/geno/PRS_scorefiles"
OUT_DIR="/home/jupyter/multiTRS/geno/PRS_scores_out"
COHORTS=("PDBP_EUR" "PPMI_EUR" "PPMI_AJ" "HBS_EUR")

mkdir -p ${OUT_DIR}

for COHORT in "${COHORTS[@]}"; do
    for score in ${SCORE_DIR}/*_GRCh38.txt; do
        basename=$(basename ${score} _GRCh38.txt)
        basename=${basename%_SBayesRC_hapmap3_SBayesRC_calc}
        OUT_FILE="${OUT_DIR}/${COHORT}_${basename}"

        if [ -f "${OUT_FILE}.sscore" ]; then
            echo "Skipping (already exists): $OUT_FILE"
            continue
        fi

        $PLINK2 \
            --pfile ${GENO_QC_DIR}/${COHORT}_MAF_0.01_mind_0.02_geno_0.01_hwe_0.000001_rel_0.044 \
            --score ${score} 1 5 8 header center cols=+scoresums,-scoreavgs \
            --out ${OUT_FILE}

        echo "Done: $OUT_FILE"
    done
done

In [ ]:
!mkdir -p /home/jupyter/multiTRS/geno/PRS_scores_out/logs
!mv /home/jupyter/multiTRS/geno/PRS_scores_out/*.log /home/jupyter/multiTRS/geno/PRS_scores_out/logs

##  Test associations between PRS and PD case/control status

### Make a data frame with covariates, PCs and scores

In [ ]:
!mkdir -p /home/jupyter/multiTRS/geno/PRS_scores_out/all_PRS_combined

In [ ]:
%%R

library(data.table)
library(dplyr)
library(stringr)

clinical <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt") %>%
            select(participant_id, case_control_other_at_baseline, age_at_baseline, sex) %>%
            rename(IID = participant_id)

cohorts <- c("PDBP_EUR", "PPMI_EUR", "PPMI_AJ", "HBS_EUR")

for (i in cohorts) {

    scores <- list.files(path = "/home/jupyter/multiTRS/geno/PRS_scores_out/",
                         pattern = i)

    PCs <- fread(paste0("/home/jupyter/multiTRS/geno/clean/", i, "_PCs.eigenvec"), header = TRUE) %>%
           select(IID, PC1, PC2, PC3, PC4, PC5)

    # Start with clinical + PCs as the base
    combined <- inner_join(clinical, PCs, by = "IID")

    # Add each score as a new column
    for (j in scores) {
        score <- fread(paste0("/home/jupyter/multiTRS/geno/PRS_scores_out/", j)) %>%
                 select(IID, SCORE1_SUM)

        score_name <- str_remove(j, paste0("^", i, "_"))
        score_name <- str_remove(score_name, "\\.sscore$")  # strip file extension

        score <- score %>% rename(!!score_name := SCORE1_SUM)

        combined <- inner_join(combined, score, by = "IID")
    }

    # Save one file per cohort
    out_path <- paste0("/home/jupyter/multiTRS/geno/PRS_scores_out/all_PRS_combined/", i, "_clinical_PCs_all_PRS.txt")
    fwrite(combined, out_path, sep = "\t")
    cat("Saved:", out_path, "\n")

}

In [ ]:
!head /home/jupyter/multiTRS/geno/PRS_scores_out/all_PRS_combined/PDBP_EUR_clinical_PCs_all_PRS.txt

### Run regression

In [ ]:
%%R
library(data.table)
library(dplyr)
library(stringr)
library(pROC)

full_results <- data.table()
cohorts <- c("PDBP_EUR", "PPMI_EUR", "PPMI_AJ", "HBS_EUR")

for (i in cohorts) {

    df <- fread(paste0("/home/jupyter/multiTRS/geno/PRS_scores_out/all_PRS_combined/", i, "_clinical_PCs_all_PRS.txt"))
    score_names <- colnames(df[, -c(1:9)])
    

    null <- glm(case_control_other_at_baseline ~ age_at_baseline + sex + PC1 + PC2 + PC3 + PC4 + PC5,
                data = df, family = "binomial")

    
    null_probs <- predict(null, type = "response")
    
    roc_obj_null <- pROC::roc(df$case_control_other_at_baseline, null_probs)

    
    null_auc <- pROC::auc(roc_obj_null)
    null_auc_ci <- pROC::ci.auc(roc_obj_null)
    null_auc_lower <- null_auc_ci[1]
    null_auc_upper <- null_auc_ci[3]
    
    null_coords <- coords(roc_obj_null, x = "best", best.method = "closest.topleft",
                          ret = c("threshold", "sensitivity", "specificity", "accuracy"))
    null_sensitivity <- as.numeric(null_coords["sensitivity"])
    null_specificity <- as.numeric(null_coords["specificity"])
    null_accuracy    <- as.numeric(null_coords["accuracy"])
    
    null_r2 <- fmsb::NagelkerkeR2(null)$R2

    for (j in score_names) {

        formula <- as.formula(paste0("case_control_other_at_baseline ~ scale(", j,
                                     ") + age_at_baseline + sex + PC1 + PC2 + PC3 + PC4 + PC5"))
        
        model <- glm(formula = formula, data = df, family = "binomial")
        
        model_probs <- predict(model, type = "response")
        
        roc_obj_model <- pROC::roc(df$case_control_other_at_baseline, model_probs) 
        
        model_auc <- pROC::auc(roc_obj_model)
        
        model_auc_ci <- pROC::ci.auc(roc_obj_model)
        
        model_auc_lower <- model_auc_ci[1]
        
        model_auc_upper <- model_auc_ci[3]
        
        model_coords <- coords(roc_obj_model, x = "best", best.method = "closest.topleft",
                               ret = c("threshold", "sensitivity", "specificity", "accuracy"))
        
        model_sensitivity <- as.numeric(model_coords["sensitivity"])
        model_specificity <- as.numeric(model_coords["specificity"])
        model_accuracy    <- as.numeric(model_coords["accuracy"])

        auc_diff <- model_auc - null_auc
        r2 <- fmsb::NagelkerkeR2(model)$R2 - null_r2
        coefs <- summary(model)$coefficients
        predictor_row <- coefs[2, ]
        delong_test <- roc.test(roc_obj_null, roc_obj_model, method = "delong")

        results <- data.table(
            cohort         = i,
            predictor      = j,
            AUC_null       = as.numeric(null_auc),
            AUC_lower_null = null_auc_lower,
            AUC_upper_null = null_auc_upper,
            AUC_full       = as.numeric(model_auc),
            AUC_lower_full = model_auc_lower,
            AUC_upper_full = model_auc_upper,
            AUC_diff       = as.numeric(auc_diff),
            sens_null      = null_sensitivity,
            sens_full      = model_sensitivity,
            sens_diff      = model_sensitivity - null_sensitivity,
            spec_null      = null_specificity,
            spec_full      = model_specificity,
            spec_diff      = model_specificity - null_specificity,
            acc_null       = null_accuracy,
            acc_full       = model_accuracy,
            acc_diff       = model_accuracy - null_accuracy,
            R2             = r2,
            estimate       = predictor_row["Estimate"],
            std_error      = predictor_row["Std. Error"],
            p_value        = predictor_row["Pr(>|z|)"],
            delong_z       = delong_test$statistic,
            delong_p       = delong_test$p.value
        )

        full_results <- rbind(full_results, results)
    }
}

# Post-loop: calculate ORs and sort — moved outside both loops
full_results$OR       <- exp(full_results$estimate)
full_results$OR_lower <- exp(full_results$estimate - 1.96 * full_results$std_error)
full_results$OR_upper <- exp(full_results$estimate + 1.96 * full_results$std_error)
full_results <- full_results %>% arrange(p_value)

print(full_results)
out_path <- "/home/jupyter/multiTRS/geno/PRS_scores_out/all_PRS_combined/all_cohorts_PRS_results.txt"
write.table(full_results,file = out_path,sep = "\t",row.names = FALSE,col.names = TRUE,quote = FALSE)

### Copy across per-cohort results

In [ ]:
!ls /home/jupyter/multiTRS/geno/PRS_scores_out/all_PRS_combined/
shell_do(f'gsutil -u {BILLING_PROJECT_ID} -m cp -r /home/jupyter/multiTRS/geno/PRS_scores_out/all_PRS_combined/* {WORKSPACE_BUCKET}')

## Meta-analyses

###  Fixed-effects

In [ ]:
%%R

library(data.table)
library(dplyr)
library(metafor)
library(stringr)


PRS_results <- fread("/home/jupyter/multiTRS/geno/PRS_scores_out/all_PRS_combined/all_cohorts_PRS_results.txt")

# Obtain unique predictors
predictors <- unique(PRS_results$predictor)

meta_results <- data.table(
  predictor = character(),
  meta_beta = numeric(),
  meta_se   = numeric(),
  z         = numeric(),
  p_value         = numeric(),
  beta_lower     = numeric(),
  beta_upper     = numeric(),
  Q         = numeric(),
  Q_p_value    = numeric(),
  I2        = numeric(),
  Direction_consistent = character(),
  Majority_nominal = character()
)


for (pred in predictors) {
  
  # Filter rows for this predictor
  meta_table <- PRS_results %>%
    filter(predictor == pred)
    
    #print(meta_table)
  Direction_consistent <- if (length(unique(sign(meta_table$estimate[meta_table$estimate != 0]))) == 1) {
  "PASS"
} else {
  "FAIL"
}
    
  Majority_nominal <- if (sum(meta_table$p_value <= 0.05) >= 3) {
  "PASS"
} else {
  "FAIL"
}

  # Fixed-effect IVW meta-analysis
  res <- rma(yi = estimate,     # make sure your column names match
             sei = std_error,
             data = meta_table,
             method = "FE")  
    
    

  
  # Extract heterogeneity statistics
  Q_val     <- res$QE
  Q_p       <- res$QEp
  I2_val    <- res$I2
  
  meta_results <- rbind(meta_results, data.table(
    predictor = pred,
    meta_beta = as.numeric(res$beta),
    meta_se   = res$se,
    z         = res$zval,
    p_value   = res$pval,
    beta_lower = res$ci.lb,
    beta_upper = res$ci.ub,
    Q         = Q_val,
    Q_p_value    = Q_p,
    I2        = I2_val,
    Direction_consistent = Direction_consistent,
    Majority_nominal = Majority_nominal
  ))
}

meta_results$OR <- exp(meta_results$meta_beta)
meta_results$OR_lower <- exp(meta_results$beta_lower)
meta_results$OR_upper <- exp(meta_results$beta_upper)

meta_results <- meta_results %>% arrange(p_value)

print(meta_results)

write.table(meta_results,paste0("/home/jupyter/multiTRS/geno/PRS_scores_out/all_PRS_combined/fixed_effects_meta_PRS_results.txt"), sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)

### Random-effects

In [ ]:
%%R

library(data.table)
library(dplyr)
library(metafor)
library(stringr)


PRS_results <- fread("/home/jupyter/multiTRS/geno/PRS_scores_out/all_PRS_combined/all_cohorts_PRS_results.txt")

# Obtain unique predictors
predictors <- unique(PRS_results$predictor)

meta_results <- data.table(
  predictor = character(),
  meta_beta = numeric(),
  meta_se   = numeric(),
  z         = numeric(),
  p_value         = numeric(),
  beta_lower     = numeric(),
  beta_upper     = numeric(),
  Q         = numeric(),
  Q_p_value    = numeric(),
  I2        = numeric(),
  Direction_consistent = character(),
  Majority_nominal = character()
)


for (pred in predictors) {
  
  # Filter rows for this predictor
  meta_table <- PRS_results %>%
    filter(predictor == pred)
    
    #print(meta_table)
  Direction_consistent <- if (length(unique(sign(meta_table$estimate[meta_table$estimate != 0]))) == 1) {
  "PASS"
} else {
  "FAIL"
}
    
  Majority_nominal <- if (sum(meta_table$p_value <= 0.05) >= 3) {
  "PASS"
} else {
  "FAIL"
}

  # Fixed-effect IVW meta-analysis
  res <- rma(yi = estimate,     # make sure your column names match
             sei = std_error,
             data = meta_table,
             method = "REML")  
    
    

  
  # Extract heterogeneity statistics
  Q_val     <- res$QE
  Q_p       <- res$QEp
  I2_val    <- res$I2
  
  meta_results <- rbind(meta_results, data.table(
    predictor = pred,
    meta_beta = as.numeric(res$beta),
    meta_se   = res$se,
    z         = res$zval,
    p_value   = res$pval,
    beta_lower = res$ci.lb,
    beta_upper = res$ci.ub,
    Q         = Q_val,
    Q_p_value    = Q_p,
    I2        = I2_val,
    Direction_consistent = Direction_consistent,
    Majority_nominal = Majority_nominal
  ))
}

meta_results$OR <- exp(meta_results$meta_beta)
meta_results$OR_lower <- exp(meta_results$beta_lower)
meta_results$OR_upper <- exp(meta_results$beta_upper)

meta_results <- meta_results %>% arrange(p_value)

print(meta_results)

write.table(meta_results,paste0("/home/jupyter/multiTRS/geno/PRS_scores_out/all_PRS_combined/random_effects_REML_meta_PRS_results.txt"), sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)

## Liability scale R2

### Run linear regression

In [ ]:
%%R
library(data.table)
library(dplyr)
library(stringr)
library(pROC)

full_results <- data.table()
cohorts <- c("PDBP_EUR", "PPMI_EUR", "PPMI_AJ", "HBS_EUR")

for (i in cohorts) {

    df <- fread(paste0("/home/jupyter/multiTRS/geno/PRS_scores_out/all_PRS_combined/", i, "_clinical_PCs_all_PRS.txt"))
    score_names <- colnames(df[, -c(1:9)])
    

    null <- lm(case_control_other_at_baseline ~ age_at_baseline + sex + PC1 + PC2 + PC3 + PC4 + PC5,
                data = df)

    
    
    null_r2 <- summary(null)$r.squared

    for (j in score_names) {

        formula <- as.formula(paste0("case_control_other_at_baseline ~ scale(", j,
                                     ") + age_at_baseline + sex + PC1 + PC2 + PC3 + PC4 + PC5"))
        
        model <- lm(formula = formula, data = df)
        

        r2 <- summary(model)$r.squared - null_r2

        results <- data.table(
            cohort         = i,
            predictor      = j,
            R2             = r2
        )

        full_results <- rbind(full_results, results)
    }
}

full_results <- full_results %>% arrange(desc(R2))

print(full_results)
out_path <- "/home/jupyter/multiTRS/geno/PRS_scores_out/all_PRS_combined/all_cohorts_PRS_linear_regression_R2.txt"
write.table(full_results,file = out_path,sep = "\t",row.names = FALSE,col.names = TRUE,quote = FALSE)

### Convert to liability scale

In [ ]:
%%R

library(data.table)
library(dplyr)
library(stringr)

PRS_r2 <- fread("/home/jupyter/multiTRS/geno/PRS_scores_out/all_PRS_combined/all_cohorts_PRS_linear_regression_R2.txt")

full_results <- PRS_r2 %>%
  mutate(
    sampprev = case_when(
      cohort == "PDBP_EUR" ~ 0.633,
      cohort == "PPMI_EUR" ~ 0.704,
      cohort == "PPMI_AJ"  ~ 0.554,
      cohort == "HBS_EUR"   ~ 0.543
    ),
    N = case_when(
      cohort == "PDBP_EUR" ~ 991,
      cohort == "PPMI_EUR" ~ 490,
      cohort == "PPMI_AJ"  ~ 370,
      cohort == "HBS_EUR"   ~ 557
    ),
    N_predictors = 1
  )

# Liability scale conversion
R2liab <- function(k, r2, p) {
  x = qnorm(1 - k)
  z = dnorm(x)
  i = z / k
  C = k * (1 - k) * k * (1 - k) / (z^2 * p * (1 - p))
  theta = i * ((p - k) / (1 - k)) * (i * ((p - k) / (1 - k)) - x)
  R2l = C * r2 / (1 + C * theta * r2)
  return(R2l)
}

full_results$R2liability_0.005 <- R2liab(
  k = 0.005,
  r2 = full_results$R2,
  p  = full_results$sampprev
)

print(full_results)

out_path <- "/home/jupyter/multiTRS/geno/PRS_scores_out/all_PRS_combined/all_cohorts_PRS_R2_liability.txt"
write.table(full_results,file = out_path,sep = "\t",row.names = FALSE,col.names = TRUE,quote = FALSE)

### Meta-analyse

In [ ]:
%%R

library(data.table)
library(dplyr)
library(metafor)
library(stringr)

all_liability_r2 <- fread("/home/jupyter/multiTRS/geno/PRS_scores_out/all_PRS_combined/all_cohorts_PRS_R2_liability.txt")

# Obtain unique predictors
predictors <- unique(all_liability_r2$predictor)

# Initialize results table
full_meta_results <- data.table(
  predictor = character(),
  meta_estimate = numeric(),
  meta_estimate_lower= numeric(),
  meta_estimate_upper= numeric(),
  z         = numeric(),
  p_value   = numeric()
)

for (pred in predictors) {
  
  # Filter rows for this predictor
  meta_table <- all_liability_r2 %>%
    filter(predictor == pred)

  # Skip if there are less than 2 studies
  if(nrow(meta_table) < 2){
      print(paste0("There are less than 2 studies for ", pred, ". Need to investigate..."))
      next 
  }
  
  # Compute effect sizes for this predictor
  es <- escalc(
    measure = "ZR2", 
    r2i = meta_table$R2liability_0.005, 
    ni  = meta_table$N, 
    mi  = meta_table$N_predictors
  )
  
  # Meta-analysis using fixed effects
  res <- rma(yi = yi, vi = vi, data = es, method = "FE")
    
    
    meta_results <- data.table(predictor = pred,
                               meta_estimate = tanh(as.numeric(res$beta))^2,
                               meta_estimate_lower = tanh(res$ci.lb)^2,
                               meta_estimate_upper = tanh(res$ci.ub)^2,
                               z = res$zval,
                               p_value = res$pval)
    
    
    full_meta_results <- rbind(full_meta_results,meta_results)

}

full_meta_results <- full_meta_results %>% arrange(desc(meta_estimate))

print(full_meta_results)

out_path <- "/home/jupyter/multiTRS/geno/PRS_scores_out/all_PRS_combined/meta_PRS_R2_liability.txt"
write.table(full_results,file = out_path,sep = "\t",row.names = FALSE,col.names = TRUE,quote = FALSE)

### Copy results across

In [ ]:
!ls /home/jupyter/multiTRS/geno/PRS_scores_out/all_PRS_combined/
shell_do(f'gsutil -u {BILLING_PROJECT_ID} -m cp -r /home/jupyter/multiTRS/geno/PRS_scores_out/all_PRS_combined/* {WORKSPACE_BUCKET}')

# TRS recalculation  

We need to re-calculate the TRS in these genetically homogenous sample so we can:

1. Perform a senstivity analysis for the significant TRS
2. Calculate multi-TRS models including polygenic scores

In [ ]:
!ls /home/jupyter/multiTRS/RNA/clean

## Subset the RNA-seq data by cohort

### PDBP and HBS

In [ ]:
%%R

library(data.table)
library(dplyr)
library(stringr)
library(sva)

set.seed(1)

setwd("/home/jupyter/multiTRS/RNA/clean/")

case_control_covar <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt")

ancestries <- c("PDBP_EUR","HBS_EUR")

for (i in ancestries) {

    prs_ids <- fread(paste0("/home/jupyter/multiTRS/geno/PRS_scores_out/all_PRS_combined/", i, "_clinical_PCs_all_PRS.txt")) %>% 
        select(IID)

    cohort <- str_remove(i, "_EUR")

    # Read gene counts
    gene_counts <- read.table(
        paste0("rnaseq_",cohort,"_baseline_cleaned.txt"),
        row.names = 1,
        header = TRUE,
        check.names = FALSE
    )

 
    common_ids <- Reduce(intersect, list(
        prs_ids$IID,
        case_control_covar$participant_id,
        colnames(gene_counts)
    ))

    if (length(common_ids) == 0) {
        stop(paste0("No overlapping samples for ", cohort))
    }

    # Subset gene counts (columns = samples)
    gene_counts <- gene_counts[, common_ids]

    # Build model vars
    model_vars <- case_control_covar %>%
    select(participant_id, case_control_other_at_baseline, sex, age_at_baseline) %>%
    filter(participant_id %in% common_ids) %>%
    distinct(participant_id, .keep_all = TRUE) %>%
    as.data.frame()

    # Set rownames and reorder
    rownames(model_vars) <- model_vars$participant_id
    model_vars <- model_vars[common_ids, ]
    model_vars$participant_id <- NULL

    # -------------------------------
    # Sanity checks
    # -------------------------------
    stopifnot(all(colnames(gene_counts) == rownames(model_vars)))
    stopifnot(!anyDuplicated(rownames(model_vars)))

    print("Model variables for SVA are:")
    print(head(model_vars))

    print(paste0("The mean age of ",i," is ", round(mean(model_vars$age_at_baseline),2)))
    print(paste0("The sd age of ",i," is ", round(sd(model_vars$age_at_baseline),2)))
    print(paste0("The percentage female of ", i, " is ",
                 round(100 * sum(model_vars$sex == 1)/nrow(model_vars), 2), "%"))

    print(paste0("The number of cases and controls for ",i," is:"))
    print(table(model_vars$case_control_other_at_baseline))

    # Models
    mod  <- model.matrix(~case_control_other_at_baseline + age_at_baseline + sex, data=model_vars)
    mod0 <- model.matrix(~1, data=model_vars)

    n.sv <- num.sv(as.matrix(gene_counts), mod, method="leek")

    print(paste0("The number of identified SVs in ",i," is ",n.sv))

    # -------------------------------
    # No SV case
    # -------------------------------
    if (n.sv == 0) {

        print("No SVs identified, creating data frame for TRS...")

        gene_counts_transposed <- as.data.frame(t(gene_counts))
        gene_counts_transposed$participant_id <- rownames(gene_counts_transposed)
        rownames(gene_counts_transposed) <- NULL

        gene_counts_transposed <- gene_counts_transposed %>%
            select(participant_id, everything())

        gene_counts_transposed[ , -1] <- as.data.frame(scale(gene_counts_transposed[ , -1]))

        write.table(
            gene_counts_transposed,
            paste0("/home/jupyter/multiTRS/RNA/clean/rnaseq_",i,"_sens_baseline_cleaned_transposed_scaled.txt"),
            sep = "\t",
            row.names = FALSE,
            col.names = TRUE,
            quote = FALSE)

        next
    }

    # -------------------------------
    # Run SVA
    # -------------------------------
    print("Running SVA...")

    svobj <- sva(as.matrix(gene_counts), mod, mod0, n.sv = n.sv)

    svs <- svobj$sv
    colnames(svs) <- paste0("SV", seq_len(ncol(svs)))

    svs_df <- as.data.frame(svs)
    svs_df$participant_id <- rownames(model_vars)
    svs_df <- svs_df %>% select(participant_id, everything())

    print(head(svs_df))

    print(paste0("Writing SVs for TRS for ",i,"..."))
    
    write.table(
            svs_df,
            paste0("/home/jupyter/multiTRS/RNA/clean/",i,"_sens_SVs.txt"),
            sep = "\t", 
            row.names = FALSE, 
            col.names = TRUE, 
            quote = FALSE) 

    # -------------------------------
    # TRS matrix
    # -------------------------------
    print("Creating data frame for TRS")

    gene_counts_transposed <- as.data.frame(t(gene_counts))
    gene_counts_transposed$participant_id <- rownames(gene_counts_transposed)
    rownames(gene_counts_transposed) <- NULL

    gene_counts_transposed <- gene_counts_transposed %>%
        select(participant_id, everything())

    gene_counts_transposed[ , -1] <- as.data.frame(scale(gene_counts_transposed[ , -1]))
    
        write.table(
            gene_counts_transposed,
            paste0("/home/jupyter/multiTRS/RNA/clean/rnaseq_",i,"_sens_baseline_cleaned_transposed_scaled.txt"),
            sep = "\t",
            row.names = FALSE,
            col.names = TRUE,
            quote = FALSE)

    print(paste0("Done processing ",i,"!"))
}

### Both PPMI

In [ ]:
%%R

library(data.table)
library(dplyr)
library(stringr)
library(sva)

set.seed(1)

setwd("/home/jupyter/multiTRS/RNA/clean/")

case_control_covar <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt")

ancestries <- c("PPMI_EUR","PPMI_AJ")

for (i in ancestries) {

    prs_ids <- fread(paste0("/home/jupyter/multiTRS/geno/PRS_scores_out/all_PRS_combined/", i, "_clinical_PCs_all_PRS.txt")) %>% 
        select(IID)
    
    cohort <- str_remove(i, "_EUR|_AJ")

    # Read gene counts
    gene_counts <- read.table(
        paste0("rnaseq_",cohort,"_baseline_cleaned.txt"),
        row.names = 1,
        header = TRUE,
        check.names = FALSE
    )

 
    common_ids <- Reduce(intersect, list(
        prs_ids$IID,
        case_control_covar$participant_id,
        colnames(gene_counts)
    ))

    if (length(common_ids) == 0) {
        stop(paste0("No overlapping samples for ", cohort))
    }

    # Subset gene counts (columns = samples)
    gene_counts <- gene_counts[, common_ids]

    # Build model vars
    model_vars <- case_control_covar %>%
    select(participant_id, case_control_other_at_baseline, sex, age_at_baseline) %>%
    filter(participant_id %in% common_ids) %>%
    distinct(participant_id, .keep_all = TRUE) %>%
    as.data.frame()

    # Set rownames and reorder
    rownames(model_vars) <- model_vars$participant_id
    model_vars <- model_vars[common_ids, ]
    model_vars$participant_id <- NULL

    # -------------------------------
    # 🔒 Sanity checks
    # -------------------------------
    stopifnot(all(colnames(gene_counts) == rownames(model_vars)))
    stopifnot(!anyDuplicated(rownames(model_vars)))

    print("Model variables for SVA are:")
    print(head(model_vars))

    print(paste0("The mean age of ",i," is ", round(mean(model_vars$age_at_baseline),2)))
    print(paste0("The sd age of ",i," is ", round(sd(model_vars$age_at_baseline),2)))
    print(paste0("The percentage female of ", i, " is ",
                 round(100 * sum(model_vars$sex == 1)/nrow(model_vars), 2), "%"))

    print(paste0("The number of cases and controls for ",i," is:"))
    print(table(model_vars$case_control_other_at_baseline))

    # Models
    mod  <- model.matrix(~case_control_other_at_baseline + age_at_baseline + sex, data=model_vars)
    mod0 <- model.matrix(~1, data=model_vars)

    n.sv <- num.sv(as.matrix(gene_counts), mod, method="leek")

    print(paste0("The number of identified SVs in ",i," is ",n.sv))

    # -------------------------------
    # No SV case
    # -------------------------------
    if (n.sv == 0) {

        print("No SVs identified, creating data frame for TRS...")

        gene_counts_transposed <- as.data.frame(t(gene_counts))
        gene_counts_transposed$participant_id <- rownames(gene_counts_transposed)
        rownames(gene_counts_transposed) <- NULL

        gene_counts_transposed <- gene_counts_transposed %>%
            select(participant_id, everything())

        gene_counts_transposed[ , -1] <- as.data.frame(scale(gene_counts_transposed[ , -1]))

        write.table(
            gene_counts_transposed,
            paste0("/home/jupyter/multiTRS/RNA/clean/rnaseq_",i,"_sens_baseline_cleaned_transposed_scaled.txt"),
            sep = "\t",
            row.names = FALSE,
            col.names = TRUE,
            quote = FALSE)

        next
    }

    # -------------------------------
    # Run SVA
    # -------------------------------
    print("Running SVA...")

    svobj <- sva(as.matrix(gene_counts), mod, mod0, n.sv = n.sv)

    svs <- svobj$sv
    colnames(svs) <- paste0("SV", seq_len(ncol(svs)))

    svs_df <- as.data.frame(svs)
    svs_df$participant_id <- rownames(model_vars)
    svs_df <- svs_df %>% select(participant_id, everything())

    print(head(svs_df))

    print(paste0("Writing SVs for TRS for ",i,"..."))
    
    write.table(
            svs_df,
            paste0("/home/jupyter/multiTRS/RNA/clean/",i,"_sens_SVs.txt"),
            sep = "\t", 
            row.names = FALSE, 
            col.names = TRUE, 
            quote = FALSE) 

    # -------------------------------
    # TRS matrix
    # -------------------------------
    print("Creating data frame for TRS")

    gene_counts_transposed <- as.data.frame(t(gene_counts))
    gene_counts_transposed$participant_id <- rownames(gene_counts_transposed)
    rownames(gene_counts_transposed) <- NULL

    gene_counts_transposed <- gene_counts_transposed %>%
        select(participant_id, everything())

    gene_counts_transposed[ , -1] <- as.data.frame(scale(gene_counts_transposed[ , -1]))
    
        write.table(
            gene_counts_transposed,
            paste0("/home/jupyter/multiTRS/RNA/clean/rnaseq_",i,"_sens_baseline_cleaned_transposed_scaled.txt"),
            sep = "\t",
            row.names = FALSE,
            col.names = TRUE,
            quote = FALSE)

    print(paste0("Done processing ",i,"!"))
}

## TRS re-calculation for all scores

## TRS re-calculation for post-process passes

In [ ]:
%%R

library(data.table)
library(dplyr)
library(stringr)


setwd("/home/jupyter/multiTRS/scorefiles/TWAS_SMR_FDR_scorefiles/")

weights_files <- list.files(pattern = "\\.txt$")

target_files <- c(
"Vitamin_B12_Finngen_R12_EUR_2024_HEIDI_FDR_SMR_post_process_clumped_scorefile.txt",
"Diastolic_blood_pressure_EUR_2024_COLOC_FDR_fusion_post_process_included_scorefile.txt",
"FEV_FVC_ratio_EUR_2023_HEIDI_FDR_SMR_post_process_clumped_scorefile.txt",
"Coronary_artery_plaque_burden_EUR_2025_HEIDI_FDR_SMR_post_process_clumped_scorefile.txt",
"Essential_tremor_EUR_2024_FDR_fusion_post_process_included_scorefile.txt",
"Gout_EUR_2024_FDR_SMR_post_process_clumped_scorefile.txt",
"Coronary_artery_plaque_burden_EUR_2025_FDR_SMR_post_process_clumped_scorefile.txt",
"Liver_cyrosis_EUR_2024_FDR_fusion_post_process_included_scorefile.txt",
"AD_no_UKB_COLOC_FDR_fusion_post_process_included_scorefile.txt",
"IBD_EUR_EA_2023_FDR_fusion_post_process_included_scorefile.txt",
"Peak_expiratory_flow_EUR_2023_COLOC_FDR_fusion_post_process_included_scorefile.txt",
"HDL_EUR_2021_COLOC_FDR_fusion_post_process_included_scorefile.txt",
"Insomnia_EUR_2019_FDR_SMR_single_SNP_post_process_clumped_scorefile.txt",
"Depression_PGC_EUR_2025_HEIDI_FDR_SMR_post_process_clumped_scorefile.txt",
"Varicose_veins_Finngen_R12_EUR_2024_HEIDI_FDR_SMR_post_process_clumped_scorefile.txt",
"Sleep_duration_EUR_2019_HEIDI_FDR_SMR_post_process_clumped_scorefile.txt",
"Sleep_diurnal_inactivity_EUR_2019_FDR_SMR_post_process_clumped_scorefile.txt",
"Depression_PGC_EUR_2025_FDR_SMR_post_process_clumped_scorefile.txt",
"Sleep_duration_EUR_2019_FDR_SMR_post_process_clumped_scorefile.txt",
"Depression_PGC_EUR_2025_FDR_SMR_single_SNP_post_process_clumped_scorefile.txt",
"Transient_global_amnesia_Finngen_R12_EUR_2024_HEIDI_FDR_SMR_post_process_clumped_scorefile.txt",
"LBD_EUR_2021_HEIDI_FDR_SMR_post_process_clumped_scorefile.txt",
"LBD_EUR_2021_HEIDI_FDR_SMR_single_SNP_post_process_clumped_scorefile.txt",
"AD_no_UKB_HEIDI_FDR_SMR_single_SNP_post_process_clumped_scorefile.txt",
"LBD_EUR_2021_FDR_SMR_post_process_clumped_scorefile.txt",
"Varicose_veins_Finngen_R12_EUR_2024_FDR_SMR_post_process_clumped_scorefile.txt",
"AD_no_UKB_FDR_SMR_post_process_clumped_scorefile.txt"
)

weights_files <- weights_files[weights_files %in% target_files]

#weights_files <- weights_files[1:4]

#print(weights_files)

# Define the RNA
rna_dir <- "/home/jupyter/multiTRS/RNA/clean/"

rna_files <- list.files(path = rna_dir, pattern = "_sens_baseline_cleaned_transposed_scaled.txt", full.names = TRUE)

#rna_files <- rna_files[1]

#print(rna_files)

for (i in rna_files){
    rna <- fread(i)
    print(rna[,1:4])
    cohort <- str_remove_all(i, "_sens_baseline_cleaned_transposed_scaled.txt")
    cohort <- str_remove_all(cohort, "rnaseq_")
    cohort <- str_remove_all(cohort, "/home/jupyter/multiTRS/RNA/clean/")
    
    colnames(rna)[-1] <- str_replace(colnames(rna)[-1], "\\..*", "")
    
    rna_scores_final <- data.frame(participant_id = rna$participant_id)
    
    skipped_weights <- data.table()
    
    for (j in weights_files){
        
        weights <- fread(j)
        
        pheno <- str_remove_all(j,"_scorefile.txt")
        
        # print(paste0("The phenotype being scored is ",pheno, " in ",cohort))
        
      if (nrow(weights) == 0) {
            skipped_weights <- rbind(
                skipped_weights, 
                data.table(weights_file = pheno, reason = "NA_sig_TWAS_SMR"))
            next
        }
        
        common_genes <- intersect(colnames(rna)[-1], weights$GENE)
        
        if(length(common_genes) == 0) {
        skipped_weights <- rbind(skipped_weights, data.table(weights_file = pheno, reason = "NA_common"))
        next
        }
        
        rna_genes_matched <- rna[, ..common_genes]
        
        weights_formatted <- setNames(weights$Z, weights$GENE)[common_genes]
        
        rna_genes_matched_scaled <- sweep(rna_genes_matched, 2, weights_formatted, `*`)
        
        summed_score <- rowSums(rna_genes_matched_scaled)

        rna_scores_final[[pheno]] <- summed_score
        
        
    }
    
    #print(head(rna_scores_final))
    print(skipped_weights)
    
    write.table(rna_scores_final,paste0("/home/jupyter/multiTRS/scores/",cohort,"_post_process_sens_TWAS_SMR_FDR_scores.txt"), sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)
   
}

In [ ]:
! ls /home/jupyter/multiTRS/scores/*_w_PGS_all_TWAS_SMR_FDR_scores.txt

## Re-test post-process passes associations

### PDBP

In [ ]:
%%R

library(data.table)
library(dplyr)
library(stringr)
library(pROC)

full_results <- data.table()

PD_pheno_covars <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt")

cohorts <- c("PDBP_EUR")

for (i in cohorts){

SVs <- fread(paste0("/home/jupyter/multiTRS/RNA/clean/",i,"_sens_SVs.txt"))

PD_pheno_covars_SVs <- inner_join(PD_pheno_covars,SVs, by = "participant_id")

scores <- fread(paste0("/home/jupyter/multiTRS/scores/",i,"_post_process_sens_TWAS_SMR_FDR_scores.txt"))

PD_pheno_covars_SVs_scores <- inner_join(PD_pheno_covars_SVs,scores, by = "participant_id")

print(nrow(PD_pheno_covars_SVs_scores))

score_names <- colnames(scores[,-1])

# score_names <- score_names[1]

null <- glm(case_control_other_at_baseline ~ age_at_baseline + sex + SV1 + SV2 + SV3, data = PD_pheno_covars_SVs_scores, family = "binomial")

null_probs <- predict(null, type = "response")

roc_obj_null <- roc(PD_pheno_covars_SVs_scores$case_control_other_at_baseline, null_probs)

null_auc <- auc(roc_obj_null)

null_auc_ci <- ci.auc(roc_obj_null)

null_auc_lower <- null_auc_ci[1]
null_auc_upper <- null_auc_ci[3]

null_coords <- coords(
  roc_obj_null,
  x = "best",
  best.method = "closest.topleft",
  ret = c("threshold", "sensitivity", "specificity", "accuracy")
)

null_sensitivity <- as.numeric(null_coords["sensitivity"])
null_specificity <- as.numeric(null_coords["specificity"])
null_accuracy <- as.numeric(null_coords["accuracy"])

for (j in score_names){

    
formula <- as.formula(paste0("case_control_other_at_baseline ~ scale(", j, ") + age_at_baseline + sex + SV1 + SV2 + SV3"))
  

model <- glm(formula = formula, data = PD_pheno_covars_SVs_scores, family = "binomial")
    
model_probs <- predict(model, type = "response")
    
roc_obj_model <- roc(PD_pheno_covars_SVs_scores$case_control_other_at_baseline, model_probs)

model_auc <- auc(roc_obj_model)
    
model_auc_ci <- ci.auc(roc_obj_model)

model_auc_lower <- model_auc_ci[1]
model_auc_upper <- model_auc_ci[3]

model_coords <- coords(
  roc_obj_model,
  x = "best",
  best.method = "closest.topleft",
  ret = c("threshold", "sensitivity", "specificity", "accuracy")
)

model_sensitivity <- as.numeric(model_coords["sensitivity"])
model_specificity <- as.numeric(model_coords["specificity"])
model_accuracy <- as.numeric(model_coords["accuracy"])


auc_diff <- model_auc - null_auc
    
r2 <- fmsb::NagelkerkeR2(model)$R2 - fmsb::NagelkerkeR2(null)$R2
    
coefs <- summary(model)$coefficients
    
predictor_row <- coefs[2, ]
    
delong_test <- roc.test(roc_obj_null, roc_obj_model, method = "delong")

    
results <- data.table(cohort = i,
                      predictor = j,
                      AUC_null = null_auc,
                      AUC_lower_null = null_auc_lower,
                      AUC_upper_null = null_auc_upper,
                      AUC_full = model_auc,
                      AUC_lower_full = model_auc_lower,
                      AUC_upper_full = model_auc_upper,
                      AUC_diff = auc_diff,
                      sens_null = null_sensitivity,
                      sens_full = model_sensitivity,
                      sens_diff = model_sensitivity - null_sensitivity,
                      spec_null = null_specificity,
                      spec_full = model_specificity,
                      spec_diff = model_specificity - null_specificity,
                      acc_null = null_accuracy,
                      acc_full = model_accuracy,
                      acc_diff = model_accuracy - null_accuracy,
                      R2 = r2,
                      estimate = predictor_row["Estimate"],
                      std_error = predictor_row["Std. Error"],
                      p_value = predictor_row["Pr(>|z|)"],
                      delong_z = delong_test$statistic,
                      delong_p = delong_test$p.value)

full_results <- rbind(full_results,results)
    
}
    
}

full_results <- full_results %>% arrange(p_value)

full_results$OR <- exp(full_results$estimate)
full_results$OR_lower <- exp(full_results$estimate - 1.96*full_results$std_error)
full_results$OR_upper <- exp(full_results$estimate + 1.96*full_results$std_error)

print(head(full_results))
    


write.table(full_results,paste0("/home/jupyter/multiTRS/results/",i,"_post_process_sens_FUSION_SMR_FDR_results.txt"), sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)

### PPMI (EUR)

In [ ]:
%%R

library(data.table)
library(dplyr)
library(stringr)
library(pROC)

full_results <- data.table()

PD_pheno_covars <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt")

cohorts <- c("PPMI_EUR")

for (i in cohorts){

SVs <- fread(paste0("/home/jupyter/multiTRS/RNA/clean/",i,"_sens_SVs.txt"))

PD_pheno_covars_SVs <- inner_join(PD_pheno_covars,SVs, by = "participant_id")

scores <- fread(paste0("/home/jupyter/multiTRS/scores/",i,"_post_process_sens_TWAS_SMR_FDR_scores.txt"))

PD_pheno_covars_SVs_scores <- inner_join(PD_pheno_covars_SVs,scores, by = "participant_id")

print(nrow(PD_pheno_covars_SVs_scores))

score_names <- colnames(scores[,-1])

# score_names <- score_names[1]

null <- glm(case_control_other_at_baseline ~ age_at_baseline + sex + SV1 + SV2 + SV3 + SV4 + SV5, data = PD_pheno_covars_SVs_scores, family = "binomial")

null_probs <- predict(null, type = "response")

roc_obj_null <- roc(PD_pheno_covars_SVs_scores$case_control_other_at_baseline, null_probs)

null_auc <- auc(roc_obj_null)

null_auc_ci <- ci.auc(roc_obj_null)

null_auc_lower <- null_auc_ci[1]
null_auc_upper <- null_auc_ci[3]

null_coords <- coords(
  roc_obj_null,
  x = "best",
  best.method = "closest.topleft",
  ret = c("threshold", "sensitivity", "specificity", "accuracy")
)

null_sensitivity <- as.numeric(null_coords["sensitivity"])
null_specificity <- as.numeric(null_coords["specificity"])
null_accuracy <- as.numeric(null_coords["accuracy"])

for (j in score_names){

    
formula <- as.formula(paste0("case_control_other_at_baseline ~ scale(", j, ") + age_at_baseline + sex + SV1 + SV2 + SV3 + SV4 + SV5"))
  

model <- glm(formula = formula, data = PD_pheno_covars_SVs_scores, family = "binomial")
    
model_probs <- predict(model, type = "response")
    
roc_obj_model <- roc(PD_pheno_covars_SVs_scores$case_control_other_at_baseline, model_probs)

model_auc <- auc(roc_obj_model)
    
model_auc_ci <- ci.auc(roc_obj_model)

model_auc_lower <- model_auc_ci[1]
model_auc_upper <- model_auc_ci[3]

model_coords <- coords(
  roc_obj_model,
  x = "best",
  best.method = "closest.topleft",
  ret = c("threshold", "sensitivity", "specificity", "accuracy")
)

model_sensitivity <- as.numeric(model_coords["sensitivity"])
model_specificity <- as.numeric(model_coords["specificity"])
model_accuracy <- as.numeric(model_coords["accuracy"])


auc_diff <- model_auc - null_auc
    
r2 <- fmsb::NagelkerkeR2(model)$R2 - fmsb::NagelkerkeR2(null)$R2
    
coefs <- summary(model)$coefficients
    
predictor_row <- coefs[2, ]
    
delong_test <- roc.test(roc_obj_null, roc_obj_model, method = "delong")

    
results <- data.table(cohort = i,
                      predictor = j,
                      AUC_null = null_auc,
                      AUC_lower_null = null_auc_lower,
                      AUC_upper_null = null_auc_upper,
                      AUC_full = model_auc,
                      AUC_lower_full = model_auc_lower,
                      AUC_upper_full = model_auc_upper,
                      AUC_diff = auc_diff,
                      sens_null = null_sensitivity,
                      sens_full = model_sensitivity,
                      sens_diff = model_sensitivity - null_sensitivity,
                      spec_null = null_specificity,
                      spec_full = model_specificity,
                      spec_diff = model_specificity - null_specificity,
                      acc_null = null_accuracy,
                      acc_full = model_accuracy,
                      acc_diff = model_accuracy - null_accuracy,
                      R2 = r2,
                      estimate = predictor_row["Estimate"],
                      std_error = predictor_row["Std. Error"],
                      p_value = predictor_row["Pr(>|z|)"],
                      delong_z = delong_test$statistic,
                      delong_p = delong_test$p.value)

full_results <- rbind(full_results,results)
    
}
    
}

full_results <- full_results %>% arrange(p_value)

full_results$OR <- exp(full_results$estimate)
full_results$OR_lower <- exp(full_results$estimate - 1.96*full_results$std_error)
full_results$OR_upper <- exp(full_results$estimate + 1.96*full_results$std_error)

print(head(full_results))
    


write.table(full_results,paste0("/home/jupyter/multiTRS/results/",i,"_post_process_sens_FUSION_SMR_FDR_results.txt"), sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)

### PPMI (AJ)

In [ ]:
%%R

library(data.table)
library(dplyr)
library(stringr)
library(pROC)

full_results <- data.table()

PD_pheno_covars <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt")

cohorts <- c("PPMI_AJ")

for (i in cohorts){

SVs <- fread(paste0("/home/jupyter/multiTRS/RNA/clean/",i,"_sens_SVs.txt"))

PD_pheno_covars_SVs <- inner_join(PD_pheno_covars,SVs, by = "participant_id")

scores <- fread(paste0("/home/jupyter/multiTRS/scores/",i,"_post_process_sens_TWAS_SMR_FDR_scores.txt"))

PD_pheno_covars_SVs_scores <- inner_join(PD_pheno_covars_SVs,scores, by = "participant_id")

print(nrow(PD_pheno_covars_SVs_scores))

score_names <- colnames(scores[,-1])

# score_names <- score_names[1]

null <- glm(case_control_other_at_baseline ~ age_at_baseline + sex + SV1 + SV2, data = PD_pheno_covars_SVs_scores, family = "binomial")

null_probs <- predict(null, type = "response")

roc_obj_null <- roc(PD_pheno_covars_SVs_scores$case_control_other_at_baseline, null_probs)

null_auc <- auc(roc_obj_null)

null_auc_ci <- ci.auc(roc_obj_null)

null_auc_lower <- null_auc_ci[1]
null_auc_upper <- null_auc_ci[3]

null_coords <- coords(
  roc_obj_null,
  x = "best",
  best.method = "closest.topleft",
  ret = c("threshold", "sensitivity", "specificity", "accuracy")
)

null_sensitivity <- as.numeric(null_coords["sensitivity"])
null_specificity <- as.numeric(null_coords["specificity"])
null_accuracy <- as.numeric(null_coords["accuracy"])

for (j in score_names){

    
formula <- as.formula(paste0("case_control_other_at_baseline ~ scale(", j, ") + age_at_baseline + sex + SV1 + SV2"))
  

model <- glm(formula = formula, data = PD_pheno_covars_SVs_scores, family = "binomial")
    
model_probs <- predict(model, type = "response")
    
roc_obj_model <- roc(PD_pheno_covars_SVs_scores$case_control_other_at_baseline, model_probs)

model_auc <- auc(roc_obj_model)
    
model_auc_ci <- ci.auc(roc_obj_model)

model_auc_lower <- model_auc_ci[1]
model_auc_upper <- model_auc_ci[3]

model_coords <- coords(
  roc_obj_model,
  x = "best",
  best.method = "closest.topleft",
  ret = c("threshold", "sensitivity", "specificity", "accuracy")
)

model_sensitivity <- as.numeric(model_coords["sensitivity"])
model_specificity <- as.numeric(model_coords["specificity"])
model_accuracy <- as.numeric(model_coords["accuracy"])


auc_diff <- model_auc - null_auc
    
r2 <- fmsb::NagelkerkeR2(model)$R2 - fmsb::NagelkerkeR2(null)$R2
    
coefs <- summary(model)$coefficients
    
predictor_row <- coefs[2, ]
    
delong_test <- roc.test(roc_obj_null, roc_obj_model, method = "delong")

    
results <- data.table(cohort = i,
                      predictor = j,
                      AUC_null = null_auc,
                      AUC_lower_null = null_auc_lower,
                      AUC_upper_null = null_auc_upper,
                      AUC_full = model_auc,
                      AUC_lower_full = model_auc_lower,
                      AUC_upper_full = model_auc_upper,
                      AUC_diff = auc_diff,
                      sens_null = null_sensitivity,
                      sens_full = model_sensitivity,
                      sens_diff = model_sensitivity - null_sensitivity,
                      spec_null = null_specificity,
                      spec_full = model_specificity,
                      spec_diff = model_specificity - null_specificity,
                      acc_null = null_accuracy,
                      acc_full = model_accuracy,
                      acc_diff = model_accuracy - null_accuracy,
                      R2 = r2,
                      estimate = predictor_row["Estimate"],
                      std_error = predictor_row["Std. Error"],
                      p_value = predictor_row["Pr(>|z|)"],
                      delong_z = delong_test$statistic,
                      delong_p = delong_test$p.value)

full_results <- rbind(full_results,results)
    
}
    
}

full_results <- full_results %>% arrange(p_value)

full_results$OR <- exp(full_results$estimate)
full_results$OR_lower <- exp(full_results$estimate - 1.96*full_results$std_error)
full_results$OR_upper <- exp(full_results$estimate + 1.96*full_results$std_error)

print(head(full_results))
    

write.table(full_results,paste0("/home/jupyter/multiTRS/results/",i,"_post_process_sens_FUSION_SMR_FDR_results.txt"), sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)

### HBS (EUR)

In [ ]:
%%R

library(data.table)
library(dplyr)
library(stringr)
library(pROC)

full_results <- data.table()

PD_pheno_covars <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt")

cohorts <- c("HBS_EUR")

for (i in cohorts){

PD_pheno_covars_SVs <- PD_pheno_covars

scores <- fread(paste0("/home/jupyter/multiTRS/scores/",i,"_post_process_sens_TWAS_SMR_FDR_scores.txt"))

PD_pheno_covars_SVs_scores <- inner_join(PD_pheno_covars_SVs,scores, by = "participant_id")

print(nrow(PD_pheno_covars_SVs_scores))

score_names <- colnames(scores[,-1])

# score_names <- score_names[1]

null <- glm(case_control_other_at_baseline ~ age_at_baseline + sex, data = PD_pheno_covars_SVs_scores, family = "binomial")

null_probs <- predict(null, type = "response")

roc_obj_null <- roc(PD_pheno_covars_SVs_scores$case_control_other_at_baseline, null_probs)

null_auc <- auc(roc_obj_null)

null_auc_ci <- ci.auc(roc_obj_null)

null_auc_lower <- null_auc_ci[1]
null_auc_upper <- null_auc_ci[3]

null_coords <- coords(
  roc_obj_null,
  x = "best",
  best.method = "closest.topleft",
  ret = c("threshold", "sensitivity", "specificity", "accuracy")
)

null_sensitivity <- as.numeric(null_coords["sensitivity"])
null_specificity <- as.numeric(null_coords["specificity"])
null_accuracy <- as.numeric(null_coords["accuracy"])

for (j in score_names){

    
formula <- as.formula(paste0("case_control_other_at_baseline ~ scale(", j, ") + age_at_baseline + sex"))
  

model <- glm(formula = formula, data = PD_pheno_covars_SVs_scores, family = "binomial")
    
model_probs <- predict(model, type = "response")
    
roc_obj_model <- roc(PD_pheno_covars_SVs_scores$case_control_other_at_baseline, model_probs)

model_auc <- auc(roc_obj_model)
    
model_auc_ci <- ci.auc(roc_obj_model)

model_auc_lower <- model_auc_ci[1]
model_auc_upper <- model_auc_ci[3]

model_coords <- coords(
  roc_obj_model,
  x = "best",
  best.method = "closest.topleft",
  ret = c("threshold", "sensitivity", "specificity", "accuracy")
)

model_sensitivity <- as.numeric(model_coords["sensitivity"])
model_specificity <- as.numeric(model_coords["specificity"])
model_accuracy <- as.numeric(model_coords["accuracy"])


auc_diff <- model_auc - null_auc
    
r2 <- fmsb::NagelkerkeR2(model)$R2 - fmsb::NagelkerkeR2(null)$R2
    
coefs <- summary(model)$coefficients
    
predictor_row <- coefs[2, ]
    
delong_test <- roc.test(roc_obj_null, roc_obj_model, method = "delong")

    
results <- data.table(cohort = i,
                      predictor = j,
                      AUC_null = null_auc,
                      AUC_lower_null = null_auc_lower,
                      AUC_upper_null = null_auc_upper,
                      AUC_full = model_auc,
                      AUC_lower_full = model_auc_lower,
                      AUC_upper_full = model_auc_upper,
                      AUC_diff = auc_diff,
                      sens_null = null_sensitivity,
                      sens_full = model_sensitivity,
                      sens_diff = model_sensitivity - null_sensitivity,
                      spec_null = null_specificity,
                      spec_full = model_specificity,
                      spec_diff = model_specificity - null_specificity,
                      acc_null = null_accuracy,
                      acc_full = model_accuracy,
                      acc_diff = model_accuracy - null_accuracy,
                      R2 = r2,
                      estimate = predictor_row["Estimate"],
                      std_error = predictor_row["Std. Error"],
                      p_value = predictor_row["Pr(>|z|)"],
                      delong_z = delong_test$statistic,
                      delong_p = delong_test$p.value)

full_results <- rbind(full_results,results)
    
}
    
}

full_results <- full_results %>% arrange(p_value)

full_results$OR <- exp(full_results$estimate)
full_results$OR_lower <- exp(full_results$estimate - 1.96*full_results$std_error)
full_results$OR_upper <- exp(full_results$estimate + 1.96*full_results$std_error)

print(head(full_results))
    

write.table(full_results,paste0("/home/jupyter/multiTRS/results/",i,"_post_process_sens_FUSION_SMR_FDR_results.txt"), sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)

## Combine per cohort output

In [ ]:
%%R

library(data.table)
library(dplyr)
library(stringr)

setwd("/home/jupyter/multiTRS/results/")

## First we need to bind the results from the three samples
results_files <- list.files(pattern = "_post_process_sens_FUSION_SMR_FDR_results.txt")

print(results_files)

full_results <- data.frame()

for (file in results_files){
    
    results_table <- fread(file)
    cohort <- str_remove_all(file,"_post_process_sens_FUSION_SMR_FDR_results.txt")
    results_table$cohort <- cohort
    results_table <- results_table %>% select(cohort, everything())
    
    full_results <- rbind(full_results,results_table)
      
}

write.table(full_results,paste0("/home/jupyter/multiTRS/results/ALL_FUSION_SMR_FDR_results_per_cohort_combined_post_process_sens_PRS_matched.txt"), sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)


## Meta-analyse

In [ ]:
%%R

library(data.table)
library(dplyr)
library(metafor)
library(stringr)

full_results <- fread("/home/jupyter/multiTRS/results/ALL_FUSION_SMR_FDR_results_per_cohort_combined_post_process_sens_PRS_matched.txt")

# Obtain unique predictors
predictors <- unique(full_results$predictor)

#predictors <- predictors[grepl("ALS", predictors)]

meta_results <- data.table(
  predictor = character(),
  meta_beta = numeric(),
  meta_se   = numeric(),
  z         = numeric(),
  p_value         = numeric(),
  beta_lower     = numeric(),
  beta_upper     = numeric(),
  Q         = numeric(),
  Q_p_value    = numeric(),
  I2        = numeric(),
  Direction_consistent = character(),
  Majority_sig = character()
)


for (pred in predictors) {
  
  # Filter rows for this predictor
  meta_table <- full_results %>%
    filter(predictor == pred)
    print(meta_table)
    
    Direction_consistent <- if (length(unique(sign(meta_table$estimate[meta_table$estimate != 0]))) == 1) {
  "PASS"
} else {
  "FAIL"
}
    
    
  Majority_sig <- if (sum(meta_table$p_value <= 0.001) >= 2) {
  "PASS"
} else {
  "FAIL"
}

    
    #print(meta_table)

  
  # Fixed-effect IVW meta-analysis
  res <- rma(yi = estimate,     # make sure your column names match
             sei = std_error,
             data = meta_table,
             method = "FE")  
    
    

  
  # Extract heterogeneity statistics
  Q_val     <- res$QE
  Q_p       <- res$QEp
  I2_val    <- res$I2
  
  meta_results <- rbind(meta_results, data.table(
    predictor = pred,
    meta_beta = as.numeric(res$beta),
    meta_se   = res$se,
    z         = res$zval,
    p_value   = res$pval,
    beta_lower = res$ci.lb,
    beta_upper = res$ci.ub,
    Q         = Q_val,
    Q_p_value    = Q_p,
    I2        = I2_val,
    Direction_consistent = Direction_consistent,
    Majority_sig = Majority_sig
  ))
}

  meta_results$OR <- exp(meta_results$meta_beta)
  meta_results$OR_lower <- exp(meta_results$beta_lower)
  meta_results$OR_upper <- exp(meta_results$beta_upper)

meta_results <- meta_results %>% arrange(predictor)

meta_results <- meta_results %>%
  mutate(weighting_method = case_when(
    grepl("_COLOC_FDR_fusion", predictor) ~ "FUSION-COLOC",
    grepl("_FDR_fusion", predictor) ~ "FUSION",
    grepl("HEIDI_FDR_SMR_single_SNP", predictor) ~ "SMR-HEIDI",
    grepl("single_SNP", predictor) ~ "SMR",
    grepl("_HEIDI_FDR_SMR", predictor) ~ "SMR-multi-HEIDI",
    grepl("_FDR_SMR", predictor) ~ "SMR-multi",
    TRUE ~ "Other"
  ))



meta_results <- meta_results %>% relocate(weighting_method, .before = meta_beta)


meta_results <- meta_results %>% arrange(p_value)

print(meta_results)

meta_results$predictor <- str_remove_all(meta_results$predictor,"_COLOC_FDR_fusion")
meta_results$predictor <- str_remove_all(meta_results$predictor,"_FDR_fusion")
meta_results$predictor <- str_remove_all(meta_results$predictor,"_HEIDI_FDR_SMR")
meta_results$predictor <- str_remove_all(meta_results$predictor,"_FDR_SMR")
meta_results$predictor <- str_remove_all(meta_results$predictor,"_single_SNP")

print(meta_results)  

write.table(meta_results,paste0("/home/jupyter/multiTRS/results/ALL_FUSION_SMR_FDR_results_meta_analysed_post_process_sens_PRS_matched.txt"), sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)

## Transfer across results

In [ ]:
!ls /home/jupyter/multiTRS/results/
shell_do(f'gsutil -u {BILLING_PROJECT_ID} -m cp -r /home/jupyter/multiTRS/results/ALL_FUSION_SMR_FDR_results_per_cohort_combined_post_process_sens_PRS_matched.txt {WORKSPACE_BUCKET}')
shell_do(f'gsutil -u {BILLING_PROJECT_ID} -m cp -r /home/jupyter/multiTRS/results/ALL_FUSION_SMR_FDR_results_meta_analysed_post_process_sens_PRS_matched.txt {WORKSPACE_BUCKET}')

# Multi-TRS+PD-PRS

## Residualise the PRS for the PCs

In [ ]:
!ls /home/jupyter/multiTRS/geno/PRS_scores_out/
!head /home/jupyter/multiTRS/geno/PRS_scores_out/HBS_EUR_Parkinsons_disease_GP2_BIOBANK_ONLY_EUR_2025.sscore

In [ ]:
%%R
# Load packages
library(data.table)
library(dplyr)
library(stringr)

# List of cohorts
cohorts <- c("PDBP_EUR", "PPMI_EUR", "PPMI_AJ", "HBS_EUR")

# Loop over cohorts
for (cohort in cohorts) {
  
  cat("\nProcessing cohort:", cohort, "\n")
  
  # ---------------------------
  # 1. Read scores and PC files
  # ---------------------------
  scores_path <- paste0("/home/jupyter/multiTRS/geno/PRS_scores_out/", cohort, "_Parkinsons_disease_GP2_BIOBANK_ONLY_EUR_2025.sscore")
  PC_path <- paste0("/home/jupyter/multiTRS/geno/clean/", cohort, "_PCs.eigenvec")
  
  if (!file.exists(scores_path)) stop(paste("Scores file not found:", scores_path))
  if (!file.exists(PC_path)) stop(paste("PC file not found:", PC_path))
  
  scores <- fread(scores_path, header = TRUE) %>% select(IID,SCORE1_SUM) %>% rename(participant_id = IID, PD_PGS = SCORE1_SUM) %>% as.data.frame()
  print(head(scores[,2]))
  
  PC <- fread(PC_path, header = TRUE) %>% select(IID, PC1, PC2, PC3, PC4, PC5) %>% rename(participant_id = IID) %>%  as.data.frame()
  
  # ---------------------------
  # 2. Join by ID column
  # ---------------------------
  combined_scores_PC <- inner_join(scores,PC, by = "participant_id")
  
  # ---------------------------
  # 3. Identify columns
  # ---------------------------
  id_col <- combined_scores_PC %>% select(participant_id)
  print(paste0("The ID column is ", names(id_col)))
  
  PC_cols <- combined_scores_PC %>% select(starts_with("PC"))
  print(paste0("The PC columns are ", names(PC_cols)))
  
  score_cols <- combined_scores_PC %>% 
    select(-all_of(names(id_col)), -all_of(names(PC_cols)))
  
  # ---------------------------
  # 4. Residualise each score for PCs
  # ---------------------------
  resid_scores <- lapply(score_cols, function(y) {
    #print(summary(lm(y ~ ., data = PC_cols)))
    resid(lm(y ~ ., data = PC_cols))
  })
  
  resid_scores <- as.data.frame(resid_scores)
  names(resid_scores) <- paste0(names(score_cols), "_resid")
  
  # ---------------------------
  # 5. Combine ID + residualised scores
  # ---------------------------
  df_resid <- cbind(id_col, resid_scores)
  print(head(df_resid[,2]))
  
  # ---------------------------
  # 6. Save file
  # ---------------------------
  outfile <- paste0("/home/jupyter/multiTRS/geno/PRS_scores_out/", cohort, "_PD_PGS_resid.txt")
  print(paste0("Saving residualised file: ", outfile))
  print(head(df_resid))
  write.table(df_resid, outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)
  
}

## Residualise the TRS for SVs

In [ ]:
%%R

# Load packages
library(data.table)
library(dplyr)
library(stringr)

# List of cohorts
cohorts <- c("PDBP_EUR", "PPMI_EUR", "PPMI_AJ")

# Loop over cohorts
for (cohort in cohorts) {
  
  cat("\nProcessing cohort:", cohort, "\n")
  
  # ---------------------------
  # 1. Read scores and SV files
  # ---------------------------
  scores_path <- paste0("/home/jupyter/multiTRS/scores/", cohort, "_w_PGS_all_TWAS_SMR_FDR_scores.txt")
  SV_path <- paste0("/home/jupyter/multiTRS/RNA/clean/", cohort, "_sens_SVs.txt")
  
  if (!file.exists(scores_path)) stop(paste("Scores file not found:", scores_path))
  if (!file.exists(SV_path)) stop(paste("SV file not found:", SV_path))
  
  scores <- fread(scores_path) %>% as.data.frame()
    
  print(head(scores[,2]))
    
  SV <- fread(SV_path) %>% as.data.frame()
  
  # ---------------------------
  # 2. Join by ID column
  # ---------------------------
  # Replace 'ID' with the actual common ID column name
  combined_scores_SV <- inner_join(SV, scores, by = "participant_id")
  
  # ---------------------------
  # 3. Identify columns
  # ---------------------------
  id_col <- combined_scores_SV %>% select(participant_id)
  print(paste0("The ID column is ",names(id_col)))
    
  SV_cols <- combined_scores_SV %>% select(starts_with("SV"))
  print(paste0("The SV columns are ",names(SV_cols)))
    
  score_cols <- combined_scores_SV %>% 
  select(-all_of(names(id_col)), -all_of(names(SV_cols)))
  #print(paste0("The score columns are ",names(score_cols)))
  
  # ---------------------------
  # 4. Residualise each score for SVs
  # ---------------------------
  resid_scores <- lapply(score_cols, function(y) {
    resid(lm(y ~ ., data = SV_cols))
    #print(summary(lm(y ~ ., data = SV_cols)))
  })
  
  resid_scores <- as.data.frame(resid_scores)
  names(resid_scores) <- paste0(names(score_cols), "_resid")
  
  # ---------------------------
  # 5. Combine ID + residualised scores
  # ---------------------------
  df_resid <- cbind(id_col, resid_scores)
  
  # Print first few rows for sanity check
  print(head(df_resid[,2]))
  
  # ---------------------------
  # 6. Save file
  # ---------------------------
  outfile <- paste0("/home/jupyter/multiTRS/scores/",cohort,"_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt")
  print(paste0("Saving residualised file: ",outfile))
  write.table(df_resid, outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)
  
}

In [ ]:
%%R

# Load packages
library(data.table)
library(dplyr)
library(stringr)

HBS_scores <- fread("/home/jupyter/multiTRS/scores/HBS_EUR_w_PGS_all_TWAS_SMR_FDR_scores.txt")

id_col <- HBS_scores %>% select(participant_id)
print(paste0("The ID column is ", names(id_col)))

score_cols <- setdiff(names(HBS_scores), "participant_id")

setnames(
  HBS_scores,
  old = score_cols,
  new = paste0(score_cols, "_resid")
)

write.table(HBS_scores, "/home/jupyter/multiTRS/scores/HBS_EUR_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt", sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)

## LASSO

### SMR-multi

#### Nested-CV

In [ ]:
%%R

library(data.table)
library(dplyr)
library(glmnet)
library(caret)
library(pROC)

set.seed(1)

# Create a data.table for all performance results
all_performance_results <- data.table()

print("This script calculates LASSO multi-TRS models with PRS using SMR-multi scores... Running LASSO models")

outfile <- "/home/jupyter/multiTRS/results/multi_TRS_w_PRS_SMR_multi_LASSO_nestedcv.txt"

# Read in the clinical data
clinical <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt") %>%
  select(participant_id, case_control_other_at_baseline, sex, age_at_baseline)

# List of cohorts to loop over
TRS_cohorts <- c("PDBP")

# Loop over cohorts
for (TRS_cohort in TRS_cohorts) {
    
  # Read PRS files
  PRS_path <- paste0("/home/jupyter/multiTRS/geno/PRS_scores_out/", TRS_cohort, "_EUR_PD_PGS_resid.txt")
  PRS <- fread(PRS_path)
  
  # Read TRS scores
  TRS_path <- paste0("/home/jupyter/multiTRS/scores/", TRS_cohort, "_EUR_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt")
  TRS <- fread(TRS_path) %>% select(participant_id, contains("SMR"), -contains("single_SNP"))
  
  combined_data <- clinical %>%
    inner_join(PRS, by = "participant_id")
    
  combined_data <- combined_data %>%
    inner_join(TRS, by = "participant_id")
  
  cat("Rows in combined data for", TRS_cohort, ":", nrow(combined_data), "\n")
  
  # Null model with age + sex
  null_model <- glm(case_control_other_at_baseline ~ sex + age_at_baseline + PD_PGS_resid,
                    data = combined_data,
                    family = binomial)
  probs_null <- predict(null_model, type = "response")
  roc_obj_null <- roc(combined_data$case_control_other_at_baseline, probs_null)
  auc_null <- auc(roc_obj_null)
  cat("AUC of null model:", auc_null, "\n")
  
  null_coords <- coords(roc_obj_null, x = "best", best.method = "closest.topleft",
                        ret = c("threshold", "sensitivity", "specificity", "accuracy"))
  null_sensitivity <- as.numeric(null_coords["sensitivity"])
  null_specificity <- as.numeric(null_coords["specificity"])
  null_accuracy <- as.numeric(null_coords["accuracy"])
  
  # Prepare predictor matrix X and outcome y
  X <- combined_data %>%
    select(-participant_id, -case_control_other_at_baseline) %>%
    as.matrix()
  y <- as.factor(combined_data$case_control_other_at_baseline)
  
  cat("Number of predictors:", ncol(X), "\n")
  
  # Define penalty factors: unpenalized for sex and age
  penalty <- ifelse(colnames(X) %in% c("sex", "age_at_baseline","PD_PGS_resid"), 0, 1)
  
  # Outer 10-fold CV
  outer_folds <- createFolds(y, k = 5, returnTrain = TRUE)
  
  # Initialize vectors for storing performance
  outer_auc <- c()
  lambda_vals <- c()
  outer_sensitivity <- c()
  outer_specificity <- c()
  outer_accuracy <- c()
  
  for (i in seq_along(outer_folds)) {
    cat("Outer fold:", i, "\n")
    
    train_idx <- outer_folds[[i]]
    test_idx <- setdiff(seq_along(y), train_idx)
    
    X_train <- X[train_idx, , drop = FALSE]
    y_train <- y[train_idx]
    X_test <- X[test_idx, , drop = FALSE]
    y_test <- y[test_idx]
    
    # Standardise variables (exclude sex)
    vars_to_scale <- setdiff(colnames(X), "sex")
    scaler <- preProcess(X_train[, vars_to_scale, drop = FALSE], method = c("center", "scale"))
    
    X_train_scaled <- X_train
    X_train_scaled[, vars_to_scale] <- predict(scaler, X_train[, vars_to_scale, drop = FALSE])
    
    X_test_scaled <- X_test
    X_test_scaled[, vars_to_scale] <- predict(scaler, X_test[, vars_to_scale, drop = FALSE])
    
    # Inner CV for LASSO (alpha = 1)
    cvfit <- cv.glmnet(
      x = X_train_scaled,
      y = y_train,
      family = "binomial",
      alpha = 1,
      nfolds = 5,
      type.measure = "auc",
      penalty.factor = penalty,
      standardize = FALSE
    )
    
    best_lambda <- cvfit$lambda.1se
    lambda_vals[i] <- best_lambda
    
    # Refit model on full training set with best lambda
    final_model <- glmnet(
      x = X_train_scaled,
      y = y_train,
      family = "binomial",
      alpha = 1,
      lambda = best_lambda,
      penalty.factor = penalty,
      standardize = FALSE
    )
    
    # Predict on outer test set
    probs <- predict(final_model, newx = X_test_scaled, type = "response")
    roc_obj_outer <- roc(y_test, as.numeric(probs))
    auc_val <- auc(roc_obj_outer)
    
    model_coords <- coords(roc_obj_outer, x = "best", best.method = "closest.topleft",
                           ret = c("sensitivity", "specificity", "accuracy"))
    
    outer_auc[i] <- auc_val
    outer_sensitivity[i] <- as.numeric(model_coords["sensitivity"])
    outer_specificity[i] <- as.numeric(model_coords["specificity"])
    outer_accuracy[i] <- as.numeric(model_coords["accuracy"])
  }
  
  # Median lambda across outer folds
  lambda_final <- median(lambda_vals)
    
  cat("AUC for each fold in ", TRS_cohort, ":", outer_auc, "\n")
  
  cat("Median lambda for cohort", TRS_cohort, ":", lambda_final, "\n")
  cat("Mean outer AUC:", mean(outer_auc), "SD:", sd(outer_auc), "\n")

  
  # Store performance results
  performance_results <- data.table(
    cohort = TRS_cohort,
    scores = "SMR-multi",
    model = "LASSO",
    covariate_treatment = "unpenalised",
    median_lambda = lambda_final,
    median_alpha = "Not applicable",
    AUC_null = auc_null,
    AUC_full_mean_cv = mean(outer_auc),
    AUC_full_sd_cv = sd(outer_auc),
    AUC_diff = mean(outer_auc) - auc_null,
    sens_null = null_sensitivity,
    sens_full_mean_cv = mean(outer_sensitivity),
    sens_full_sd_cv = sd(outer_sensitivity),
    sens_diff = mean(outer_sensitivity) - null_sensitivity,
    spec_null = null_specificity,
    spec_full_mean_cv = mean(outer_specificity),
    spec_full_sd_cv = sd(outer_specificity),
    spec_diff = mean(outer_specificity) - null_specificity,
    acc_null = null_accuracy,
    acc_full_mean_cv = mean(outer_accuracy),
    acc_full_sd_cv = sd(outer_accuracy),
    acc_diff = mean(outer_accuracy) - null_accuracy
  )
  
  all_performance_results <- rbind(all_performance_results, performance_results)
  
}


print(all_performance_results)

write.table(all_performance_results, outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)

#### Test in PPMI (EUR), PPMI (AJ) and HBS (EUR)

In [ ]:
%%R

library(data.table)
library(dplyr)
library(glmnet)
library(caret)
library(pROC)

set.seed(1)

# Read in the data
clinical <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt") %>% 
  select(participant_id, case_control_other_at_baseline, sex, age_at_baseline)

# Read in the PGS
PRS_PDBP_EUR <- fread("/home/jupyter/multiTRS/geno/PRS_scores_out/PDBP_EUR_PD_PGS_resid.txt")
PRS_PPMI_EUR <- fread("/home/jupyter/multiTRS/geno/PRS_scores_out/PPMI_EUR_PD_PGS_resid.txt")
PRS_PPMI_AJ  <- fread("/home/jupyter/multiTRS/geno/PRS_scores_out/PPMI_AJ_PD_PGS_resid.txt")
PRS_HBS_EUR  <- fread("/home/jupyter/multiTRS/geno/PRS_scores_out/HBS_EUR_PD_PGS_resid.txt")

# Read TRS scores - select columns based on PDBP first
TRS_PDBP_EUR <- fread("/home/jupyter/multiTRS/scores/PDBP_EUR_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt") %>% 
  select(participant_id, contains("SMR"), -contains("single_SNP"))

TRS_PPMI_EUR <- fread("/home/jupyter/multiTRS/scores/PPMI_EUR_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt") %>% 
  select(all_of(colnames(TRS_PDBP_EUR)))

TRS_PPMI_AJ  <- fread("/home/jupyter/multiTRS/scores/PPMI_AJ_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt") %>%  
  select(all_of(colnames(TRS_PDBP_EUR)))

TRS_HBS_EUR  <- fread("/home/jupyter/multiTRS/scores/HBS_EUR_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt") %>% 
  select(all_of(colnames(TRS_PDBP_EUR)))

# -----------------------------------------------
# Create combined datasets
# -----------------------------------------------

# PDBP (training)
combined_data <- clinical %>%
  inner_join(PRS_PDBP_EUR, by = "participant_id") %>%
  inner_join(TRS_PDBP_EUR, by = "participant_id")

# PPMI_EUR (test)
combined_data_PPMI_EUR <- clinical %>%
  inner_join(PRS_PPMI_EUR, by = "participant_id") %>%
  inner_join(TRS_PPMI_EUR, by = "participant_id")

# PPMI_AJ (test)
combined_data_PPMI_AJ <- clinical %>%
  inner_join(PRS_PPMI_AJ, by = "participant_id") %>%
  inner_join(TRS_PPMI_AJ, by = "participant_id")

# HBS_EUR (test)
combined_data_HBS_EUR <- clinical %>%
  inner_join(PRS_HBS_EUR, by = "participant_id") %>%
  inner_join(TRS_HBS_EUR, by = "participant_id")

cat("Testing SMR-multi model...\n")
cat("Rows in combined data - PDBP:", nrow(combined_data), "\n")
cat("Rows in combined data - PPMI_EUR:", nrow(combined_data_PPMI_EUR), "\n")
cat("Rows in combined data - PPMI_AJ:", nrow(combined_data_PPMI_AJ), "\n")
cat("Rows in combined data - HBS_EUR:", nrow(combined_data_HBS_EUR), "\n")

# -----------------------------------------------
# Null model (age + sex + PGS), fit on PDBP, applied to all
# -----------------------------------------------

null_model <- glm(case_control_other_at_baseline ~ scale(age_at_baseline) + sex + scale(PD_PGS_resid),
                  data = combined_data,
                  family = binomial)

# Null model predictions
null_probs_PDBP     <- predict(null_model, newdata = combined_data,          type = "response")
null_probs_PPMI_EUR <- predict(null_model, newdata = combined_data_PPMI_EUR, type = "response")
null_probs_PPMI_AJ  <- predict(null_model, newdata = combined_data_PPMI_AJ,  type = "response")
null_probs_HBS_EUR  <- predict(null_model, newdata = combined_data_HBS_EUR,  type = "response")

auc_null_PDBP     <- auc(roc(combined_data$case_control_other_at_baseline,          null_probs_PDBP))
auc_null_PPMI_EUR <- auc(roc(combined_data_PPMI_EUR$case_control_other_at_baseline, null_probs_PPMI_EUR))
auc_null_PPMI_AJ  <- auc(roc(combined_data_PPMI_AJ$case_control_other_at_baseline,  null_probs_PPMI_AJ))
auc_null_HBS_EUR  <- auc(roc(combined_data_HBS_EUR$case_control_other_at_baseline,  null_probs_HBS_EUR))

cat("Null model AUCs:\n")
cat("  PDBP:", auc_null_PDBP, "\n")
cat("  PPMI_EUR:", auc_null_PPMI_EUR, "\n")
cat("  PPMI_AJ:", auc_null_PPMI_AJ, "\n")
cat("  HBS_EUR:", auc_null_HBS_EUR, "\n")

# -----------------------------------------------
# Prepare predictor matrices
# -----------------------------------------------

X <- combined_data %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y <- as.factor(combined_data$case_control_other_at_baseline)

X_PPMI_EUR <- combined_data_PPMI_EUR %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y_PPMI_EUR <- as.factor(combined_data_PPMI_EUR$case_control_other_at_baseline)

X_PPMI_AJ <- combined_data_PPMI_AJ %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y_PPMI_AJ <- as.factor(combined_data_PPMI_AJ$case_control_other_at_baseline)

X_HBS_EUR <- combined_data_HBS_EUR %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y_HBS_EUR <- as.factor(combined_data_HBS_EUR$case_control_other_at_baseline)

cat("Number of predictors:", ncol(X), "\n")

# -----------------------------------------------
# Scale on PDBP, apply to all
# -----------------------------------------------

penalty <- ifelse(colnames(X) %in% c("PD_PGS_resid", "sex", "age_at_baseline"), 0, 1)
vars_to_scale <- setdiff(colnames(X), "sex")

scaler_full <- preProcess(X[, vars_to_scale], method = c("center", "scale"))

X_scaled          <- X
X_scaled[, vars_to_scale] <- predict(scaler_full, X[, vars_to_scale])

X_PPMI_EUR_scaled <- X_PPMI_EUR
X_PPMI_EUR_scaled[, vars_to_scale] <- predict(scaler_full, X_PPMI_EUR[, vars_to_scale])

X_PPMI_AJ_scaled  <- X_PPMI_AJ
X_PPMI_AJ_scaled[, vars_to_scale]  <- predict(scaler_full, X_PPMI_AJ[, vars_to_scale])

X_HBS_EUR_scaled  <- X_HBS_EUR
X_HBS_EUR_scaled[, vars_to_scale]  <- predict(scaler_full, X_HBS_EUR[, vars_to_scale])

# -----------------------------------------------
# Fit final LASSO model on PDBP
# -----------------------------------------------

final_model <- glmnet(
  x = X_scaled,
  y = y,
  family = "binomial",
  alpha = 1,
  lambda = 0.0535281,
  penalty.factor = penalty,
  standardize = FALSE
)

# Extract and print coefficients
coefs <- coef(final_model)
coef_df <- data.frame(
  feature = rownames(coefs),
  coefficient = as.numeric(coefs)
)

coef_df <- coef_df %>% arrange(desc(coefficient))

print(coef_df)

coef_outfile <- "/home/jupyter/multiTRS/results/multi_TRS_w_PRS_SMR_multi_LASSO_PDBP_full_model_coefficients.txt"
write.table(coef_df, coef_outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)
cat("Coefficients saved to:", coef_outfile, "\n")

# -----------------------------------------------
# Predictions and AUCs
# -----------------------------------------------

get_auc <- function(model, X_new, y_new) {
  probs <- predict(model, newx = X_new, type = "response")
  auc(roc(y_new, as.numeric(probs)))
}

auc_PDBP     <- get_auc(final_model, X_scaled,          y)
auc_PPMI_EUR <- get_auc(final_model, X_PPMI_EUR_scaled, y_PPMI_EUR)
auc_PPMI_AJ  <- get_auc(final_model, X_PPMI_AJ_scaled,  y_PPMI_AJ)
auc_HBS_EUR  <- get_auc(final_model, X_HBS_EUR_scaled,  y_HBS_EUR)

cat("Full model AUCs:\n")
cat("  PDBP:", auc_PDBP, "\n")
cat("  PPMI_EUR:", auc_PPMI_EUR, "\n")
cat("  PPMI_AJ:", auc_PPMI_AJ, "\n")
cat("  HBS_EUR:", auc_HBS_EUR, "\n")

# -----------------------------------------------
# ROC objects (created before CIs and plots)
# -----------------------------------------------

probs_PPMI_EUR <- predict(final_model, newx = X_PPMI_EUR_scaled, type = "response")
probs_PPMI_AJ  <- predict(final_model, newx = X_PPMI_AJ_scaled,  type = "response")
probs_HBS_EUR  <- predict(final_model, newx = X_HBS_EUR_scaled,  type = "response")

roc_null_PPMI_EUR <- roc(y_PPMI_EUR, null_probs_PPMI_EUR)
roc_full_PPMI_EUR <- roc(y_PPMI_EUR, as.numeric(probs_PPMI_EUR))

roc_null_PPMI_AJ  <- roc(y_PPMI_AJ,  null_probs_PPMI_AJ)
roc_full_PPMI_AJ  <- roc(y_PPMI_AJ,  as.numeric(probs_PPMI_AJ))

roc_null_HBS_EUR  <- roc(y_HBS_EUR,  null_probs_HBS_EUR)
roc_full_HBS_EUR  <- roc(y_HBS_EUR,  as.numeric(probs_HBS_EUR))

# -----------------------------------------------
# CIs for AUCs
# -----------------------------------------------

ci_null_PPMI_EUR <- ci.auc(roc_null_PPMI_EUR)
ci_null_PPMI_AJ  <- ci.auc(roc_null_PPMI_AJ)
ci_null_HBS_EUR  <- ci.auc(roc_null_HBS_EUR)

ci_full_PPMI_EUR <- ci.auc(roc_full_PPMI_EUR)
ci_full_PPMI_AJ  <- ci.auc(roc_full_PPMI_AJ)
ci_full_HBS_EUR  <- ci.auc(roc_full_HBS_EUR)

cat("Null model AUCs with 95% CI:\n")
cat("  PPMI_EUR:", round(auc_null_PPMI_EUR, 3), "(95% CI:", round(ci_null_PPMI_EUR[1], 3), "-", round(ci_null_PPMI_EUR[3], 3), ")\n")
cat("  PPMI_AJ:",  round(auc_null_PPMI_AJ, 3),  "(95% CI:", round(ci_null_PPMI_AJ[1], 3),  "-", round(ci_null_PPMI_AJ[3], 3),  ")\n")
cat("  HBS_EUR:",  round(auc_null_HBS_EUR, 3),   "(95% CI:", round(ci_null_HBS_EUR[1], 3),  "-", round(ci_null_HBS_EUR[3], 3),  ")\n")

cat("Full model AUCs with 95% CI:\n")
cat("  PPMI_EUR:", round(auc_PPMI_EUR, 3), "(95% CI:", round(ci_full_PPMI_EUR[1], 3), "-", round(ci_full_PPMI_EUR[3], 3), ")\n")
cat("  PPMI_AJ:",  round(auc_PPMI_AJ, 3),  "(95% CI:", round(ci_full_PPMI_AJ[1], 3),  "-", round(ci_full_PPMI_AJ[3], 3),  ")\n")
cat("  HBS_EUR:",  round(auc_HBS_EUR, 3),   "(95% CI:", round(ci_full_HBS_EUR[1], 3),  "-", round(ci_full_HBS_EUR[3], 3),  ")\n")

# -----------------------------------------------
# CI bands for plotting (ci.se required for ci.type="shape")
# -----------------------------------------------

ci_se_null_PPMI_EUR <- ci.se(roc_null_PPMI_EUR, specificities = seq(0, 1, 0.01))
ci_se_full_PPMI_EUR <- ci.se(roc_full_PPMI_EUR, specificities = seq(0, 1, 0.01))

ci_se_null_PPMI_AJ  <- ci.se(roc_null_PPMI_AJ,  specificities = seq(0, 1, 0.01))
ci_se_full_PPMI_AJ  <- ci.se(roc_full_PPMI_AJ,  specificities = seq(0, 1, 0.01))

ci_se_null_HBS_EUR  <- ci.se(roc_null_HBS_EUR,  specificities = seq(0, 1, 0.01))
ci_se_full_HBS_EUR  <- ci.se(roc_full_HBS_EUR,  specificities = seq(0, 1, 0.01))

# -----------------------------------------------
# ROC plots with CI shapes
# -----------------------------------------------

plot_roc <- function(roc_null, roc_full, ci_se_null, ci_se_full, ci_null, ci_full, title, full_col) {

  # Plot null model ROC
  plot.roc(roc_null, col = "black", lwd = 2, main = title,
           legacy.axes = TRUE, print.auc = FALSE)

  # Add CI shape for null model
  plot(ci_se_null, type = "shape",
       col = adjustcolor("black", alpha.f = 0.1), border = NA)

  # Add full model ROC
  plot.roc(roc_full, add = TRUE, col = full_col, lwd = 2)

  # Add CI shape for full model
  plot(ci_se_full, type = "shape",
       col = adjustcolor(full_col, alpha.f = 0.15), border = NA)

  # Redraw ROC lines on top of shading
  lines.roc(roc_null, col = "black", lwd = 2)
  lines.roc(roc_full, col = full_col, lwd = 2)

  # Legend with AUC + 95% CI
  legend("bottomright",
         legend = c(
           paste0("Age + Sex + PD-PGS (AUC = ", round(auc(roc_null), 3),
                  " [", round(ci_null[1], 3), "\u2013", round(ci_null[3], 3), "])"),
           paste0("Full Model (AUC = ", round(auc(roc_full), 3),
                  " [", round(ci_full[1], 3), "\u2013", round(ci_full[3], 3), "])")
         ),
         col = c("black", full_col), lwd = 2, bty = "n")
}

plot_roc(roc_null_PPMI_EUR, roc_full_PPMI_EUR, ci_se_null_PPMI_EUR, ci_se_full_PPMI_EUR, ci_null_PPMI_EUR, ci_full_PPMI_EUR, "PPMI EUR: Baseline vs Full Model", "goldenrod")
plot_roc(roc_null_PPMI_AJ,  roc_full_PPMI_AJ,  ci_se_null_PPMI_AJ,  ci_se_full_PPMI_AJ,  ci_null_PPMI_AJ,  ci_full_PPMI_AJ,  "PPMI AJ: Baseline vs Full Model",  "forestgreen")
plot_roc(roc_null_HBS_EUR,  roc_full_HBS_EUR,  ci_se_null_HBS_EUR,  ci_se_full_HBS_EUR,  ci_null_HBS_EUR,  ci_full_HBS_EUR,  "HBS EUR: Baseline vs Full Model",  "dodgerblue")

# -----------------------------------------------
# DeLong tests
# -----------------------------------------------

delong_PPMI_EUR <- roc.test(roc_null_PPMI_EUR, roc_full_PPMI_EUR, method = "delong")
delong_PPMI_AJ  <- roc.test(roc_null_PPMI_AJ,  roc_full_PPMI_AJ,  method = "delong")
delong_HBS_EUR  <- roc.test(roc_null_HBS_EUR,  roc_full_HBS_EUR,  method = "delong")

cat("DeLong test p-values:\n")
cat("  PPMI_EUR:", delong_PPMI_EUR$p.value, "\n")
cat("  PPMI_AJ:",  delong_PPMI_AJ$p.value,  "\n")
cat("  HBS_EUR:",  delong_HBS_EUR$p.value,  "\n")


# -----------------------------------------------
# Extract sensitivity, specificity, accuracy at
# optimal threshold (closest-to-top-left)
# -----------------------------------------------

coords_null_PPMI_EUR <- coords(roc_null_PPMI_EUR, x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]
coords_full_PPMI_EUR <- coords(roc_full_PPMI_EUR, x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]

coords_null_PPMI_AJ  <- coords(roc_null_PPMI_AJ,  x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]
coords_full_PPMI_AJ  <- coords(roc_full_PPMI_AJ,  x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]

coords_null_HBS_EUR  <- coords(roc_null_HBS_EUR,  x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]
coords_full_HBS_EUR  <- coords(roc_full_HBS_EUR,  x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]

# -----------------------------------------------
# Build results table
# -----------------------------------------------

results <- rbindlist(list(
  data.table(
    cohort         = "PPMI_EUR",
    AUC_null       = as.numeric(auc(roc_null_PPMI_EUR)),
    AUC_lower_null = as.numeric(ci_null_PPMI_EUR[1]),
    AUC_upper_null = as.numeric(ci_null_PPMI_EUR[3]),
    AUC_full       = as.numeric(auc(roc_full_PPMI_EUR)),
    AUC_lower_full = as.numeric(ci_full_PPMI_EUR[1]),
    AUC_upper_full = as.numeric(ci_full_PPMI_EUR[3]),
    AUC_diff       = as.numeric(auc(roc_full_PPMI_EUR)) - as.numeric(auc(roc_null_PPMI_EUR)),
    sens_null      = coords_null_PPMI_EUR$sensitivity,
    sens_full      = coords_full_PPMI_EUR$sensitivity,
    sens_diff      = coords_full_PPMI_EUR$sensitivity - coords_null_PPMI_EUR$sensitivity,
    spec_null      = coords_null_PPMI_EUR$specificity,
    spec_full      = coords_full_PPMI_EUR$specificity,
    spec_diff      = coords_full_PPMI_EUR$specificity - coords_null_PPMI_EUR$specificity,
    acc_null       = coords_null_PPMI_EUR$accuracy,
    acc_full       = coords_full_PPMI_EUR$accuracy,
    acc_diff       = coords_full_PPMI_EUR$accuracy - coords_null_PPMI_EUR$accuracy,
    delong_z       = delong_PPMI_EUR$statistic,
    delong_p       = delong_PPMI_EUR$p.value
  ),
  data.table(
    cohort         = "PPMI_AJ",
    AUC_null       = as.numeric(auc(roc_null_PPMI_AJ)),
    AUC_lower_null = as.numeric(ci_null_PPMI_AJ[1]),
    AUC_upper_null = as.numeric(ci_null_PPMI_AJ[3]),
    AUC_full       = as.numeric(auc(roc_full_PPMI_AJ)),
    AUC_lower_full = as.numeric(ci_full_PPMI_AJ[1]),
    AUC_upper_full = as.numeric(ci_full_PPMI_AJ[3]),
    AUC_diff       = as.numeric(auc(roc_full_PPMI_AJ)) - as.numeric(auc(roc_null_PPMI_AJ)),
    sens_null      = coords_null_PPMI_AJ$sensitivity,
    sens_full      = coords_full_PPMI_AJ$sensitivity,
    sens_diff      = coords_full_PPMI_AJ$sensitivity - coords_null_PPMI_AJ$sensitivity,
    spec_null      = coords_null_PPMI_AJ$specificity,
    spec_full      = coords_full_PPMI_AJ$specificity,
    spec_diff      = coords_full_PPMI_AJ$specificity - coords_null_PPMI_AJ$specificity,
    acc_null       = coords_null_PPMI_AJ$accuracy,
    acc_full       = coords_full_PPMI_AJ$accuracy,
    acc_diff       = coords_full_PPMI_AJ$accuracy - coords_null_PPMI_AJ$accuracy,
    delong_z       = delong_PPMI_AJ$statistic,
    delong_p       = delong_PPMI_AJ$p.value
  ),
  data.table(
    cohort         = "HBS_EUR",
    AUC_null       = as.numeric(auc(roc_null_HBS_EUR)),
    AUC_lower_null = as.numeric(ci_null_HBS_EUR[1]),
    AUC_upper_null = as.numeric(ci_null_HBS_EUR[3]),
    AUC_full       = as.numeric(auc(roc_full_HBS_EUR)),
    AUC_lower_full = as.numeric(ci_full_HBS_EUR[1]),
    AUC_upper_full = as.numeric(ci_full_HBS_EUR[3]),
    AUC_diff       = as.numeric(auc(roc_full_HBS_EUR)) - as.numeric(auc(roc_null_HBS_EUR)),
    sens_null      = coords_null_HBS_EUR$sensitivity,
    sens_full      = coords_full_HBS_EUR$sensitivity,
    sens_diff      = coords_full_HBS_EUR$sensitivity - coords_null_HBS_EUR$sensitivity,
    spec_null      = coords_null_HBS_EUR$specificity,
    spec_full      = coords_full_HBS_EUR$specificity,
    spec_diff      = coords_full_HBS_EUR$specificity - coords_null_HBS_EUR$specificity,
    acc_null       = coords_null_HBS_EUR$accuracy,
    acc_full       = coords_full_HBS_EUR$accuracy,
    acc_diff       = coords_full_HBS_EUR$accuracy - coords_null_HBS_EUR$accuracy,
    delong_z       = delong_HBS_EUR$statistic,
    delong_p       = delong_HBS_EUR$p.value
  )
))

print(results)

results_outfile <- "/home/jupyter/multiTRS/results/multi_TRS_w_PRS_SMR_multi_LASSO_external_validation_results_table.txt"
write.table(results, results_outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)
cat("Results table saved to:", results_outfile, "\n")

### SMR

#### Nested-CV

In [ ]:
%%R

library(data.table)
library(dplyr)
library(glmnet)
library(caret)
library(pROC)

set.seed(1)

# Create a data.table for all performance results
all_performance_results <- data.table()

print("This script calculates LASSO multi-TRS models using SMR single SNP scores... Running LASSO models")

outfile <- "/home/jupyter/multiTRS/results/multi_TRS_w_PRS_SMR_single_SNP_LASSO_nestedcv.txt"

# Read in the clinical data
clinical <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt") %>%
  select(participant_id, case_control_other_at_baseline, sex, age_at_baseline)

# List of cohorts to loop over
TRS_cohorts <- c("PDBP")

# Loop over cohorts
for (TRS_cohort in TRS_cohorts) {
    
  # Read PRS files
  PRS_path <- paste0("/home/jupyter/multiTRS/geno/PRS_scores_out/", TRS_cohort, "_EUR_PD_PGS_resid.txt")
  PRS <- fread(PRS_path)
  
  # Read TRS scores
  TRS_path <- paste0("/home/jupyter/multiTRS/scores/", TRS_cohort, "_EUR_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt")
  TRS <- fread(TRS_path) %>% select(participant_id, contains("single_SNP"),-contains("Hip_Fracture"))
  
  combined_data <- clinical %>%
    inner_join(PRS, by = "participant_id")
    
  combined_data <- combined_data %>%
    inner_join(TRS, by = "participant_id")
  
  cat("Rows in combined data for", TRS_cohort, ":", nrow(combined_data), "\n")
  
  # Null model with age + sex
  null_model <- glm(case_control_other_at_baseline ~ sex + age_at_baseline + PD_PGS_resid,
                    data = combined_data,
                    family = binomial)
  probs_null <- predict(null_model, type = "response")
  roc_obj_null <- roc(combined_data$case_control_other_at_baseline, probs_null)
  auc_null <- auc(roc_obj_null)
  cat("AUC of null model:", auc_null, "\n")
  
  null_coords <- coords(roc_obj_null, x = "best", best.method = "closest.topleft",
                        ret = c("threshold", "sensitivity", "specificity", "accuracy"))
  null_sensitivity <- as.numeric(null_coords["sensitivity"])
  null_specificity <- as.numeric(null_coords["specificity"])
  null_accuracy <- as.numeric(null_coords["accuracy"])
  
  # Prepare predictor matrix X and outcome y
  X <- combined_data %>%
    select(-participant_id, -case_control_other_at_baseline) %>%
    as.matrix()
  y <- as.factor(combined_data$case_control_other_at_baseline)
  
  cat("Number of predictors:", ncol(X), "\n")
  
  # Define penalty factors: unpenalized for sex and age
  penalty <- ifelse(colnames(X) %in% c("sex", "age_at_baseline","PD_PGS_resid"), 0, 1)
  
  # Outer 10-fold CV
  outer_folds <- createFolds(y, k = 5, returnTrain = TRUE)
  
  # Initialize vectors for storing performance
  outer_auc <- c()
  lambda_vals <- c()
  outer_sensitivity <- c()
  outer_specificity <- c()
  outer_accuracy <- c()
  
  for (i in seq_along(outer_folds)) {
    cat("Outer fold:", i, "\n")
    
    train_idx <- outer_folds[[i]]
    test_idx <- setdiff(seq_along(y), train_idx)
    
    X_train <- X[train_idx, , drop = FALSE]
    y_train <- y[train_idx]
    X_test <- X[test_idx, , drop = FALSE]
    y_test <- y[test_idx]
    
    # Standardise variables (exclude sex)
    vars_to_scale <- setdiff(colnames(X), "sex")
    scaler <- preProcess(X_train[, vars_to_scale, drop = FALSE], method = c("center", "scale"))
    
    X_train_scaled <- X_train
    X_train_scaled[, vars_to_scale] <- predict(scaler, X_train[, vars_to_scale, drop = FALSE])
    
    X_test_scaled <- X_test
    X_test_scaled[, vars_to_scale] <- predict(scaler, X_test[, vars_to_scale, drop = FALSE])
    
    # Inner CV for LASSO (alpha = 1)
    cvfit <- cv.glmnet(
      x = X_train_scaled,
      y = y_train,
      family = "binomial",
      alpha = 1,
      nfolds = 5,
      type.measure = "auc",
      penalty.factor = penalty,
      standardize = FALSE
    )
    
    best_lambda <- cvfit$lambda.1se
    lambda_vals[i] <- best_lambda
    
    # Refit model on full training set with best lambda
    final_model <- glmnet(
      x = X_train_scaled,
      y = y_train,
      family = "binomial",
      alpha = 1,
      lambda = best_lambda,
      penalty.factor = penalty,
      standardize = FALSE
    )
    
    # Predict on outer test set
    probs <- predict(final_model, newx = X_test_scaled, type = "response")
    roc_obj_outer <- roc(y_test, as.numeric(probs))
    auc_val <- auc(roc_obj_outer)
    
    model_coords <- coords(roc_obj_outer, x = "best", best.method = "closest.topleft",
                           ret = c("sensitivity", "specificity", "accuracy"))
    
    outer_auc[i] <- auc_val
    outer_sensitivity[i] <- as.numeric(model_coords["sensitivity"])
    outer_specificity[i] <- as.numeric(model_coords["specificity"])
    outer_accuracy[i] <- as.numeric(model_coords["accuracy"])
  }
  
  # Median lambda across outer folds
  lambda_final <- median(lambda_vals)
    
  cat("AUC for each fold in ", TRS_cohort, ":", outer_auc, "\n")
  
  cat("Median lambda for cohort", TRS_cohort, ":", lambda_final, "\n")
  cat("Mean outer AUC:", mean(outer_auc), "SD:", sd(outer_auc), "\n")

  
  # Store performance results
  performance_results <- data.table(
    cohort = TRS_cohort,
    scores = "SMR",
    model = "LASSO",
    covariate_treatment = "unpenalised",
    median_lambda = lambda_final,
    median_alpha = "Not applicable",
    AUC_null = auc_null,
    AUC_full_mean_cv = mean(outer_auc),
    AUC_full_sd_cv = sd(outer_auc),
    AUC_diff = mean(outer_auc) - auc_null,
    sens_null = null_sensitivity,
    sens_full_mean_cv = mean(outer_sensitivity),
    sens_full_sd_cv = sd(outer_sensitivity),
    sens_diff = mean(outer_sensitivity) - null_sensitivity,
    spec_null = null_specificity,
    spec_full_mean_cv = mean(outer_specificity),
    spec_full_sd_cv = sd(outer_specificity),
    spec_diff = mean(outer_specificity) - null_specificity,
    acc_null = null_accuracy,
    acc_full_mean_cv = mean(outer_accuracy),
    acc_full_sd_cv = sd(outer_accuracy),
    acc_diff = mean(outer_accuracy) - null_accuracy
  )
  
  all_performance_results <- rbind(all_performance_results, performance_results)
  
}


print(all_performance_results)

write.table(all_performance_results, outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)

#### Test in PPMI (EUR), PPMI (AJ) and HBS (EUR)

In [ ]:
%%R

library(data.table)
library(dplyr)
library(glmnet)
library(caret)
library(pROC)

set.seed(1)

# Read in the data
clinical <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt") %>% 
  select(participant_id, case_control_other_at_baseline, sex, age_at_baseline)

# Read in the PGS
PRS_PDBP_EUR <- fread("/home/jupyter/multiTRS/geno/PRS_scores_out/PDBP_EUR_PD_PGS_resid.txt")
PRS_PPMI_EUR <- fread("/home/jupyter/multiTRS/geno/PRS_scores_out/PPMI_EUR_PD_PGS_resid.txt")
PRS_PPMI_AJ  <- fread("/home/jupyter/multiTRS/geno/PRS_scores_out/PPMI_AJ_PD_PGS_resid.txt")
PRS_HBS_EUR  <- fread("/home/jupyter/multiTRS/geno/PRS_scores_out/HBS_EUR_PD_PGS_resid.txt")

# Read TRS scores - select columns based on PDBP first
TRS_PDBP_EUR <- fread("/home/jupyter/multiTRS/scores/PDBP_EUR_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt") %>% 
  select(participant_id, contains("single_SNP"),-contains("Hip_Fracture"))

TRS_PPMI_EUR <- fread("/home/jupyter/multiTRS/scores/PPMI_EUR_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt") %>% 
  select(all_of(colnames(TRS_PDBP_EUR)))

TRS_PPMI_AJ  <- fread("/home/jupyter/multiTRS/scores/PPMI_AJ_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt") %>%  
  select(all_of(colnames(TRS_PDBP_EUR)))

TRS_HBS_EUR  <- fread("/home/jupyter/multiTRS/scores/HBS_EUR_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt") %>% 
  select(all_of(colnames(TRS_PDBP_EUR)))

# -----------------------------------------------
# Create combined datasets
# -----------------------------------------------

# PDBP (training)
combined_data <- clinical %>%
  inner_join(PRS_PDBP_EUR, by = "participant_id") %>%
  inner_join(TRS_PDBP_EUR, by = "participant_id")

# PPMI_EUR (test)
combined_data_PPMI_EUR <- clinical %>%
  inner_join(PRS_PPMI_EUR, by = "participant_id") %>%
  inner_join(TRS_PPMI_EUR, by = "participant_id")

# PPMI_AJ (test)
combined_data_PPMI_AJ <- clinical %>%
  inner_join(PRS_PPMI_AJ, by = "participant_id") %>%
  inner_join(TRS_PPMI_AJ, by = "participant_id")

# HBS_EUR (test)
combined_data_HBS_EUR <- clinical %>%
  inner_join(PRS_HBS_EUR, by = "participant_id") %>%
  inner_join(TRS_HBS_EUR, by = "participant_id")

cat("Testing SMR-single-SNP model...\n")
cat("Rows in combined data - PDBP:", nrow(combined_data), "\n")
cat("Rows in combined data - PPMI_EUR:", nrow(combined_data_PPMI_EUR), "\n")
cat("Rows in combined data - PPMI_AJ:", nrow(combined_data_PPMI_AJ), "\n")
cat("Rows in combined data - HBS_EUR:", nrow(combined_data_HBS_EUR), "\n")

# -----------------------------------------------
# Null model (age + sex + PGS), fit on PDBP, applied to all
# -----------------------------------------------

null_model <- glm(case_control_other_at_baseline ~ scale(age_at_baseline) + sex + scale(PD_PGS_resid),
                  data = combined_data,
                  family = binomial)

# Null model predictions
null_probs_PDBP     <- predict(null_model, newdata = combined_data,          type = "response")
null_probs_PPMI_EUR <- predict(null_model, newdata = combined_data_PPMI_EUR, type = "response")
null_probs_PPMI_AJ  <- predict(null_model, newdata = combined_data_PPMI_AJ,  type = "response")
null_probs_HBS_EUR  <- predict(null_model, newdata = combined_data_HBS_EUR,  type = "response")

auc_null_PDBP     <- auc(roc(combined_data$case_control_other_at_baseline,          null_probs_PDBP))
auc_null_PPMI_EUR <- auc(roc(combined_data_PPMI_EUR$case_control_other_at_baseline, null_probs_PPMI_EUR))
auc_null_PPMI_AJ  <- auc(roc(combined_data_PPMI_AJ$case_control_other_at_baseline,  null_probs_PPMI_AJ))
auc_null_HBS_EUR  <- auc(roc(combined_data_HBS_EUR$case_control_other_at_baseline,  null_probs_HBS_EUR))

cat("Null model AUCs:\n")
cat("  PDBP:", auc_null_PDBP, "\n")
cat("  PPMI_EUR:", auc_null_PPMI_EUR, "\n")
cat("  PPMI_AJ:", auc_null_PPMI_AJ, "\n")
cat("  HBS_EUR:", auc_null_HBS_EUR, "\n")

# -----------------------------------------------
# Prepare predictor matrices
# -----------------------------------------------

X <- combined_data %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y <- as.factor(combined_data$case_control_other_at_baseline)

X_PPMI_EUR <- combined_data_PPMI_EUR %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y_PPMI_EUR <- as.factor(combined_data_PPMI_EUR$case_control_other_at_baseline)

X_PPMI_AJ <- combined_data_PPMI_AJ %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y_PPMI_AJ <- as.factor(combined_data_PPMI_AJ$case_control_other_at_baseline)

X_HBS_EUR <- combined_data_HBS_EUR %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y_HBS_EUR <- as.factor(combined_data_HBS_EUR$case_control_other_at_baseline)

cat("Number of predictors:", ncol(X), "\n")

# -----------------------------------------------
# Scale on PDBP, apply to all
# -----------------------------------------------

penalty <- ifelse(colnames(X) %in% c("PD_PGS_resid", "sex", "age_at_baseline"), 0, 1)
vars_to_scale <- setdiff(colnames(X), "sex")

scaler_full <- preProcess(X[, vars_to_scale], method = c("center", "scale"))

X_scaled          <- X
X_scaled[, vars_to_scale] <- predict(scaler_full, X[, vars_to_scale])

X_PPMI_EUR_scaled <- X_PPMI_EUR
X_PPMI_EUR_scaled[, vars_to_scale] <- predict(scaler_full, X_PPMI_EUR[, vars_to_scale])

X_PPMI_AJ_scaled  <- X_PPMI_AJ
X_PPMI_AJ_scaled[, vars_to_scale]  <- predict(scaler_full, X_PPMI_AJ[, vars_to_scale])

X_HBS_EUR_scaled  <- X_HBS_EUR
X_HBS_EUR_scaled[, vars_to_scale]  <- predict(scaler_full, X_HBS_EUR[, vars_to_scale])

# -----------------------------------------------
# Fit final LASSO model on PDBP
# -----------------------------------------------

final_model <- glmnet(
  x = X_scaled,
  y = y,
  family = "binomial",
  alpha = 1,
  lambda = 0.05440629,
  penalty.factor = penalty,
  standardize = FALSE
)

# Extract and print coefficients
coefs <- coef(final_model)
coef_df <- data.frame(
  feature = rownames(coefs),
  coefficient = as.numeric(coefs)
)

coef_df <- coef_df %>% arrange(desc(coefficient))

print(coef_df)

coef_outfile <- "/home/jupyter/multiTRS/results/multi_TRS_w_PRS_SMR_single_SNP_LASSO_PDBP_full_model_coefficients.txt"
write.table(coef_df, coef_outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)
cat("Coefficients saved to:", coef_outfile, "\n")

# -----------------------------------------------
# Predictions and AUCs
# -----------------------------------------------

get_auc <- function(model, X_new, y_new) {
  probs <- predict(model, newx = X_new, type = "response")
  auc(roc(y_new, as.numeric(probs)))
}

auc_PDBP     <- get_auc(final_model, X_scaled,          y)
auc_PPMI_EUR <- get_auc(final_model, X_PPMI_EUR_scaled, y_PPMI_EUR)
auc_PPMI_AJ  <- get_auc(final_model, X_PPMI_AJ_scaled,  y_PPMI_AJ)
auc_HBS_EUR  <- get_auc(final_model, X_HBS_EUR_scaled,  y_HBS_EUR)

cat("Full model AUCs:\n")
cat("  PDBP:", auc_PDBP, "\n")
cat("  PPMI_EUR:", auc_PPMI_EUR, "\n")
cat("  PPMI_AJ:", auc_PPMI_AJ, "\n")
cat("  HBS_EUR:", auc_HBS_EUR, "\n")

# -----------------------------------------------
# ROC objects (created before CIs and plots)
# -----------------------------------------------

probs_PPMI_EUR <- predict(final_model, newx = X_PPMI_EUR_scaled, type = "response")
probs_PPMI_AJ  <- predict(final_model, newx = X_PPMI_AJ_scaled,  type = "response")
probs_HBS_EUR  <- predict(final_model, newx = X_HBS_EUR_scaled,  type = "response")

roc_null_PPMI_EUR <- roc(y_PPMI_EUR, null_probs_PPMI_EUR)
roc_full_PPMI_EUR <- roc(y_PPMI_EUR, as.numeric(probs_PPMI_EUR))

roc_null_PPMI_AJ  <- roc(y_PPMI_AJ,  null_probs_PPMI_AJ)
roc_full_PPMI_AJ  <- roc(y_PPMI_AJ,  as.numeric(probs_PPMI_AJ))

roc_null_HBS_EUR  <- roc(y_HBS_EUR,  null_probs_HBS_EUR)
roc_full_HBS_EUR  <- roc(y_HBS_EUR,  as.numeric(probs_HBS_EUR))

# -----------------------------------------------
# CIs for AUCs
# -----------------------------------------------

ci_null_PPMI_EUR <- ci.auc(roc_null_PPMI_EUR)
ci_null_PPMI_AJ  <- ci.auc(roc_null_PPMI_AJ)
ci_null_HBS_EUR  <- ci.auc(roc_null_HBS_EUR)

ci_full_PPMI_EUR <- ci.auc(roc_full_PPMI_EUR)
ci_full_PPMI_AJ  <- ci.auc(roc_full_PPMI_AJ)
ci_full_HBS_EUR  <- ci.auc(roc_full_HBS_EUR)

cat("Null model AUCs with 95% CI:\n")
cat("  PPMI_EUR:", round(auc_null_PPMI_EUR, 3), "(95% CI:", round(ci_null_PPMI_EUR[1], 3), "-", round(ci_null_PPMI_EUR[3], 3), ")\n")
cat("  PPMI_AJ:",  round(auc_null_PPMI_AJ, 3),  "(95% CI:", round(ci_null_PPMI_AJ[1], 3),  "-", round(ci_null_PPMI_AJ[3], 3),  ")\n")
cat("  HBS_EUR:",  round(auc_null_HBS_EUR, 3),   "(95% CI:", round(ci_null_HBS_EUR[1], 3),  "-", round(ci_null_HBS_EUR[3], 3),  ")\n")

cat("Full model AUCs with 95% CI:\n")
cat("  PPMI_EUR:", round(auc_PPMI_EUR, 3), "(95% CI:", round(ci_full_PPMI_EUR[1], 3), "-", round(ci_full_PPMI_EUR[3], 3), ")\n")
cat("  PPMI_AJ:",  round(auc_PPMI_AJ, 3),  "(95% CI:", round(ci_full_PPMI_AJ[1], 3),  "-", round(ci_full_PPMI_AJ[3], 3),  ")\n")
cat("  HBS_EUR:",  round(auc_HBS_EUR, 3),   "(95% CI:", round(ci_full_HBS_EUR[1], 3),  "-", round(ci_full_HBS_EUR[3], 3),  ")\n")

# -----------------------------------------------
# CI bands for plotting (ci.se required for ci.type="shape")
# -----------------------------------------------

ci_se_null_PPMI_EUR <- ci.se(roc_null_PPMI_EUR, specificities = seq(0, 1, 0.01))
ci_se_full_PPMI_EUR <- ci.se(roc_full_PPMI_EUR, specificities = seq(0, 1, 0.01))

ci_se_null_PPMI_AJ  <- ci.se(roc_null_PPMI_AJ,  specificities = seq(0, 1, 0.01))
ci_se_full_PPMI_AJ  <- ci.se(roc_full_PPMI_AJ,  specificities = seq(0, 1, 0.01))

ci_se_null_HBS_EUR  <- ci.se(roc_null_HBS_EUR,  specificities = seq(0, 1, 0.01))
ci_se_full_HBS_EUR  <- ci.se(roc_full_HBS_EUR,  specificities = seq(0, 1, 0.01))

# -----------------------------------------------
# ROC plots with CI shapes
# -----------------------------------------------

plot_roc <- function(roc_null, roc_full, ci_se_null, ci_se_full, ci_null, ci_full, title, full_col) {

  # Plot null model ROC
  plot.roc(roc_null, col = "black", lwd = 2, main = title,
           legacy.axes = TRUE, print.auc = FALSE)

  # Add CI shape for null model
  plot(ci_se_null, type = "shape",
       col = adjustcolor("black", alpha.f = 0.1), border = NA)

  # Add full model ROC
  plot.roc(roc_full, add = TRUE, col = full_col, lwd = 2)

  # Add CI shape for full model
  plot(ci_se_full, type = "shape",
       col = adjustcolor(full_col, alpha.f = 0.15), border = NA)

  # Redraw ROC lines on top of shading
  lines.roc(roc_null, col = "black", lwd = 2)
  lines.roc(roc_full, col = full_col, lwd = 2)

  # Legend with AUC + 95% CI
  legend("bottomright",
         legend = c(
           paste0("Age + Sex + PD-PGS (AUC = ", round(auc(roc_null), 3),
                  " [", round(ci_null[1], 3), "\u2013", round(ci_null[3], 3), "])"),
           paste0("Full Model (AUC = ", round(auc(roc_full), 3),
                  " [", round(ci_full[1], 3), "\u2013", round(ci_full[3], 3), "])")
         ),
         col = c("black", full_col), lwd = 2, bty = "n")
}

plot_roc(roc_null_PPMI_EUR, roc_full_PPMI_EUR, ci_se_null_PPMI_EUR, ci_se_full_PPMI_EUR, ci_null_PPMI_EUR, ci_full_PPMI_EUR, "PPMI EUR: Baseline vs Full Model", "goldenrod")
plot_roc(roc_null_PPMI_AJ,  roc_full_PPMI_AJ,  ci_se_null_PPMI_AJ,  ci_se_full_PPMI_AJ,  ci_null_PPMI_AJ,  ci_full_PPMI_AJ,  "PPMI AJ: Baseline vs Full Model",  "forestgreen")
plot_roc(roc_null_HBS_EUR,  roc_full_HBS_EUR,  ci_se_null_HBS_EUR,  ci_se_full_HBS_EUR,  ci_null_HBS_EUR,  ci_full_HBS_EUR,  "HBS EUR: Baseline vs Full Model",  "dodgerblue")

# -----------------------------------------------
# DeLong tests
# -----------------------------------------------

delong_PPMI_EUR <- roc.test(roc_null_PPMI_EUR, roc_full_PPMI_EUR, method = "delong")
delong_PPMI_AJ  <- roc.test(roc_null_PPMI_AJ,  roc_full_PPMI_AJ,  method = "delong")
delong_HBS_EUR  <- roc.test(roc_null_HBS_EUR,  roc_full_HBS_EUR,  method = "delong")

cat("DeLong test p-values:\n")
cat("  PPMI_EUR:", delong_PPMI_EUR$p.value, "\n")
cat("  PPMI_AJ:",  delong_PPMI_AJ$p.value,  "\n")
cat("  HBS_EUR:",  delong_HBS_EUR$p.value,  "\n")

# -----------------------------------------------
# Extract sensitivity, specificity, accuracy at
# optimal threshold (closest-to-top-left)
# -----------------------------------------------

coords_null_PPMI_EUR <- coords(roc_null_PPMI_EUR, x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]
coords_full_PPMI_EUR <- coords(roc_full_PPMI_EUR, x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]

coords_null_PPMI_AJ  <- coords(roc_null_PPMI_AJ,  x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]
coords_full_PPMI_AJ  <- coords(roc_full_PPMI_AJ,  x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]

coords_null_HBS_EUR  <- coords(roc_null_HBS_EUR,  x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]
coords_full_HBS_EUR  <- coords(roc_full_HBS_EUR,  x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]

# -----------------------------------------------
# Build results table
# -----------------------------------------------

results <- rbindlist(list(
  data.table(
    cohort         = "PPMI_EUR",
    AUC_null       = as.numeric(auc(roc_null_PPMI_EUR)),
    AUC_lower_null = as.numeric(ci_null_PPMI_EUR[1]),
    AUC_upper_null = as.numeric(ci_null_PPMI_EUR[3]),
    AUC_full       = as.numeric(auc(roc_full_PPMI_EUR)),
    AUC_lower_full = as.numeric(ci_full_PPMI_EUR[1]),
    AUC_upper_full = as.numeric(ci_full_PPMI_EUR[3]),
    AUC_diff       = as.numeric(auc(roc_full_PPMI_EUR)) - as.numeric(auc(roc_null_PPMI_EUR)),
    sens_null      = coords_null_PPMI_EUR$sensitivity,
    sens_full      = coords_full_PPMI_EUR$sensitivity,
    sens_diff      = coords_full_PPMI_EUR$sensitivity - coords_null_PPMI_EUR$sensitivity,
    spec_null      = coords_null_PPMI_EUR$specificity,
    spec_full      = coords_full_PPMI_EUR$specificity,
    spec_diff      = coords_full_PPMI_EUR$specificity - coords_null_PPMI_EUR$specificity,
    acc_null       = coords_null_PPMI_EUR$accuracy,
    acc_full       = coords_full_PPMI_EUR$accuracy,
    acc_diff       = coords_full_PPMI_EUR$accuracy - coords_null_PPMI_EUR$accuracy,
    delong_z       = delong_PPMI_EUR$statistic,
    delong_p       = delong_PPMI_EUR$p.value
  ),
  data.table(
    cohort         = "PPMI_AJ",
    AUC_null       = as.numeric(auc(roc_null_PPMI_AJ)),
    AUC_lower_null = as.numeric(ci_null_PPMI_AJ[1]),
    AUC_upper_null = as.numeric(ci_null_PPMI_AJ[3]),
    AUC_full       = as.numeric(auc(roc_full_PPMI_AJ)),
    AUC_lower_full = as.numeric(ci_full_PPMI_AJ[1]),
    AUC_upper_full = as.numeric(ci_full_PPMI_AJ[3]),
    AUC_diff       = as.numeric(auc(roc_full_PPMI_AJ)) - as.numeric(auc(roc_null_PPMI_AJ)),
    sens_null      = coords_null_PPMI_AJ$sensitivity,
    sens_full      = coords_full_PPMI_AJ$sensitivity,
    sens_diff      = coords_full_PPMI_AJ$sensitivity - coords_null_PPMI_AJ$sensitivity,
    spec_null      = coords_null_PPMI_AJ$specificity,
    spec_full      = coords_full_PPMI_AJ$specificity,
    spec_diff      = coords_full_PPMI_AJ$specificity - coords_null_PPMI_AJ$specificity,
    acc_null       = coords_null_PPMI_AJ$accuracy,
    acc_full       = coords_full_PPMI_AJ$accuracy,
    acc_diff       = coords_full_PPMI_AJ$accuracy - coords_null_PPMI_AJ$accuracy,
    delong_z       = delong_PPMI_AJ$statistic,
    delong_p       = delong_PPMI_AJ$p.value
  ),
  data.table(
    cohort         = "HBS_EUR",
    AUC_null       = as.numeric(auc(roc_null_HBS_EUR)),
    AUC_lower_null = as.numeric(ci_null_HBS_EUR[1]),
    AUC_upper_null = as.numeric(ci_null_HBS_EUR[3]),
    AUC_full       = as.numeric(auc(roc_full_HBS_EUR)),
    AUC_lower_full = as.numeric(ci_full_HBS_EUR[1]),
    AUC_upper_full = as.numeric(ci_full_HBS_EUR[3]),
    AUC_diff       = as.numeric(auc(roc_full_HBS_EUR)) - as.numeric(auc(roc_null_HBS_EUR)),
    sens_null      = coords_null_HBS_EUR$sensitivity,
    sens_full      = coords_full_HBS_EUR$sensitivity,
    sens_diff      = coords_full_HBS_EUR$sensitivity - coords_null_HBS_EUR$sensitivity,
    spec_null      = coords_null_HBS_EUR$specificity,
    spec_full      = coords_full_HBS_EUR$specificity,
    spec_diff      = coords_full_HBS_EUR$specificity - coords_null_HBS_EUR$specificity,
    acc_null       = coords_null_HBS_EUR$accuracy,
    acc_full       = coords_full_HBS_EUR$accuracy,
    acc_diff       = coords_full_HBS_EUR$accuracy - coords_null_HBS_EUR$accuracy,
    delong_z       = delong_HBS_EUR$statistic,
    delong_p       = delong_HBS_EUR$p.value
  )
))

print(results)

results_outfile <- "/home/jupyter/multiTRS/results/multi_TRS_w_PRS_SMR_single_SNP_LASSO_external_validation_results_table.txt"
write.table(results, results_outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)
cat("Results table saved to:", results_outfile, "\n")

### FUSION (TWAS)

#### Nested-CV

In [ ]:
%%R

library(data.table)
library(dplyr)
library(glmnet)
library(caret)
library(pROC)

set.seed(1)

# Create a data.table for all performance results
all_performance_results <- data.table()

print("This script calculates LASSO multi-TRS models using SMR-multi scores... Running LASSO models")

outfile <- "/home/jupyter/multiTRS/results/multi_TRS_w_PRS_FUSION_SNP_LASSO_nestedcv.txt"

# Read in the clinical data
clinical <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt") %>%
  select(participant_id, case_control_other_at_baseline, sex, age_at_baseline)

# List of cohorts to loop over
TRS_cohorts <- c("PDBP")

# Loop over cohorts
for (TRS_cohort in TRS_cohorts) {
    
  # Read PRS files
  PRS_path <- paste0("/home/jupyter/multiTRS/geno/PRS_scores_out/", TRS_cohort, "_EUR_PD_PGS_resid.txt")
  PRS <- fread(PRS_path)
  
  # Read TRS scores
  TRS_path <- paste0("/home/jupyter/multiTRS/scores/", TRS_cohort, "_EUR_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt")
  TRS <- fread(TRS_path) %>% select(participant_id, contains("FUSION"))
  
  combined_data <- clinical %>%
    inner_join(PRS, by = "participant_id")
    
  combined_data <- combined_data %>%
    inner_join(TRS, by = "participant_id")
  
  cat("Rows in combined data for", TRS_cohort, ":", nrow(combined_data), "\n")
  
  # Null model with age + sex
  null_model <- glm(case_control_other_at_baseline ~ sex + age_at_baseline + PD_PGS_resid,
                    data = combined_data,
                    family = binomial)
  probs_null <- predict(null_model, type = "response")
  roc_obj_null <- roc(combined_data$case_control_other_at_baseline, probs_null)
  auc_null <- auc(roc_obj_null)
  cat("AUC of null model:", auc_null, "\n")
  
  null_coords <- coords(roc_obj_null, x = "best", best.method = "closest.topleft",
                        ret = c("threshold", "sensitivity", "specificity", "accuracy"))
  null_sensitivity <- as.numeric(null_coords["sensitivity"])
  null_specificity <- as.numeric(null_coords["specificity"])
  null_accuracy <- as.numeric(null_coords["accuracy"])
  
  # Prepare predictor matrix X and outcome y
  X <- combined_data %>%
    select(-participant_id, -case_control_other_at_baseline) %>%
    as.matrix()
  y <- as.factor(combined_data$case_control_other_at_baseline)
  
  cat("Number of predictors:", ncol(X), "\n")
  
  # Define penalty factors: unpenalized for sex and age
  penalty <- ifelse(colnames(X) %in% c("sex", "age_at_baseline","PD_PGS_resid"), 0, 1)
  
  # Outer 10-fold CV
  outer_folds <- createFolds(y, k = 5, returnTrain = TRUE)
  
  # Initialize vectors for storing performance
  outer_auc <- c()
  lambda_vals <- c()
  outer_sensitivity <- c()
  outer_specificity <- c()
  outer_accuracy <- c()
  
  for (i in seq_along(outer_folds)) {
    cat("Outer fold:", i, "\n")
    
    train_idx <- outer_folds[[i]]
    test_idx <- setdiff(seq_along(y), train_idx)
    
    X_train <- X[train_idx, , drop = FALSE]
    y_train <- y[train_idx]
    X_test <- X[test_idx, , drop = FALSE]
    y_test <- y[test_idx]
    
    # Standardise variables (exclude sex)
    vars_to_scale <- setdiff(colnames(X), "sex")
    scaler <- preProcess(X_train[, vars_to_scale, drop = FALSE], method = c("center", "scale"))
    
    X_train_scaled <- X_train
    X_train_scaled[, vars_to_scale] <- predict(scaler, X_train[, vars_to_scale, drop = FALSE])
    
    X_test_scaled <- X_test
    X_test_scaled[, vars_to_scale] <- predict(scaler, X_test[, vars_to_scale, drop = FALSE])
    
    # Inner CV for LASSO (alpha = 1)
    cvfit <- cv.glmnet(
      x = X_train_scaled,
      y = y_train,
      family = "binomial",
      alpha = 1,
      nfolds = 5,
      type.measure = "auc",
      penalty.factor = penalty,
      standardize = FALSE
    )
    
    best_lambda <- cvfit$lambda.1se
    lambda_vals[i] <- best_lambda
    
    # Refit model on full training set with best lambda
    final_model <- glmnet(
      x = X_train_scaled,
      y = y_train,
      family = "binomial",
      alpha = 1,
      lambda = best_lambda,
      penalty.factor = penalty,
      standardize = FALSE
    )
    
    # Predict on outer test set
    probs <- predict(final_model, newx = X_test_scaled, type = "response")
    roc_obj_outer <- roc(y_test, as.numeric(probs))
    auc_val <- auc(roc_obj_outer)
    
    model_coords <- coords(roc_obj_outer, x = "best", best.method = "closest.topleft",
                           ret = c("sensitivity", "specificity", "accuracy"))
    
    outer_auc[i] <- auc_val
    outer_sensitivity[i] <- as.numeric(model_coords["sensitivity"])
    outer_specificity[i] <- as.numeric(model_coords["specificity"])
    outer_accuracy[i] <- as.numeric(model_coords["accuracy"])
  }
  
  # Median lambda across outer folds
  lambda_final <- median(lambda_vals)
    
  cat("AUC for each fold in ", TRS_cohort, ":", outer_auc, "\n")
  
  cat("Median lambda for cohort", TRS_cohort, ":", lambda_final, "\n")
  cat("Mean outer AUC:", mean(outer_auc), "SD:", sd(outer_auc), "\n")

  
  # Store performance results
  performance_results <- data.table(
    cohort = TRS_cohort,
    scores = "FUSION",
    model = "LASSO",
    covariate_treatment = "unpenalised",
    median_lambda = lambda_final,
    median_alpha = "Not applicable",
    AUC_null = auc_null,
    AUC_full_mean_cv = mean(outer_auc),
    AUC_full_sd_cv = sd(outer_auc),
    AUC_diff = mean(outer_auc) - auc_null,
    sens_null = null_sensitivity,
    sens_full_mean_cv = mean(outer_sensitivity),
    sens_full_sd_cv = sd(outer_sensitivity),
    sens_diff = mean(outer_sensitivity) - null_sensitivity,
    spec_null = null_specificity,
    spec_full_mean_cv = mean(outer_specificity),
    spec_full_sd_cv = sd(outer_specificity),
    spec_diff = mean(outer_specificity) - null_specificity,
    acc_null = null_accuracy,
    acc_full_mean_cv = mean(outer_accuracy),
    acc_full_sd_cv = sd(outer_accuracy),
    acc_diff = mean(outer_accuracy) - null_accuracy
  )
  
  all_performance_results <- rbind(all_performance_results, performance_results)
  
}


print(all_performance_results)

write.table(all_performance_results, outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)

#### Test in PPMI (EUR), PPMI (AJ) and HBS (EUR)

In [ ]:
%%R

library(data.table)
library(dplyr)
library(glmnet)
library(caret)
library(pROC)

set.seed(1)

# Read in the data
clinical <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt") %>% 
  select(participant_id, case_control_other_at_baseline, sex, age_at_baseline)

# Read in the PGS
PRS_PDBP_EUR <- fread("/home/jupyter/multiTRS/geno/PRS_scores_out/PDBP_EUR_PD_PGS_resid.txt")
PRS_PPMI_EUR <- fread("/home/jupyter/multiTRS/geno/PRS_scores_out/PPMI_EUR_PD_PGS_resid.txt")
PRS_PPMI_AJ  <- fread("/home/jupyter/multiTRS/geno/PRS_scores_out/PPMI_AJ_PD_PGS_resid.txt")
PRS_HBS_EUR  <- fread("/home/jupyter/multiTRS/geno/PRS_scores_out/HBS_EUR_PD_PGS_resid.txt")

# Read TRS scores - select columns based on PDBP first
TRS_PDBP_EUR <- fread("/home/jupyter/multiTRS/scores/PDBP_EUR_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt") %>% 
  select(participant_id, contains("FUSION"))

TRS_PPMI_EUR <- fread("/home/jupyter/multiTRS/scores/PPMI_EUR_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt") %>% 
  select(all_of(colnames(TRS_PDBP_EUR)))  # Fixed: TRS_PDBP -> TRS_PDBP_EUR

TRS_PPMI_AJ  <- fread("/home/jupyter/multiTRS/scores/PPMI_AJ_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt") %>%  
  select(all_of(colnames(TRS_PDBP_EUR)))  # Fixed: TRS_PDBP -> TRS_PDBP_EUR

TRS_HBS_EUR  <- fread("/home/jupyter/multiTRS/scores/HBS_EUR_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt") %>% 
  select(all_of(colnames(TRS_PDBP_EUR)))  # Fixed: TRS_PDBP -> TRS_PDBP_EUR

# -----------------------------------------------
# Create combined datasets
# -----------------------------------------------

# PDBP (training)
combined_data <- clinical %>%
  inner_join(PRS_PDBP_EUR, by = "participant_id") %>%
  inner_join(TRS_PDBP_EUR, by = "participant_id")

# PPMI_EUR (test)
combined_data_PPMI_EUR <- clinical %>%
  inner_join(PRS_PPMI_EUR, by = "participant_id") %>%
  inner_join(TRS_PPMI_EUR, by = "participant_id")

# PPMI_AJ (test) - Fixed: was accidentally using combined_data_PPMI_EUR as base
combined_data_PPMI_AJ <- clinical %>%
  inner_join(PRS_PPMI_AJ, by = "participant_id") %>%
  inner_join(TRS_PPMI_AJ, by = "participant_id")

# HBS_EUR (test)
combined_data_HBS_EUR <- clinical %>%
  inner_join(PRS_HBS_EUR, by = "participant_id") %>%
  inner_join(TRS_HBS_EUR, by = "participant_id")

cat("Testing FUSION model...\n")
cat("Rows in combined data - PDBP:", nrow(combined_data), "\n")
cat("Rows in combined data - PPMI_EUR:", nrow(combined_data_PPMI_EUR), "\n")
cat("Rows in combined data - PPMI_AJ:", nrow(combined_data_PPMI_AJ), "\n")
cat("Rows in combined data - HBS_EUR:", nrow(combined_data_HBS_EUR), "\n")

# -----------------------------------------------
# Null model (age + sex), fit on PDBP, applied to all
# -----------------------------------------------

null_model <- glm(case_control_other_at_baseline ~ scale(age_at_baseline) + sex + scale(PD_PGS_resid),
                  data = combined_data,
                  family = binomial)

# Null model predictions
null_probs_PDBP     <- predict(null_model, newdata = combined_data,          type = "response")
null_probs_PPMI_EUR <- predict(null_model, newdata = combined_data_PPMI_EUR, type = "response")
null_probs_PPMI_AJ  <- predict(null_model, newdata = combined_data_PPMI_AJ,  type = "response")
null_probs_HBS_EUR  <- predict(null_model, newdata = combined_data_HBS_EUR,  type = "response")

auc_null_PDBP     <- auc(roc(combined_data$case_control_other_at_baseline,          null_probs_PDBP))
auc_null_PPMI_EUR <- auc(roc(combined_data_PPMI_EUR$case_control_other_at_baseline, null_probs_PPMI_EUR))
auc_null_PPMI_AJ  <- auc(roc(combined_data_PPMI_AJ$case_control_other_at_baseline,  null_probs_PPMI_AJ))
auc_null_HBS_EUR  <- auc(roc(combined_data_HBS_EUR$case_control_other_at_baseline,  null_probs_HBS_EUR))

cat("Null model AUCs:\n")
cat("  PDBP:", auc_null_PDBP, "\n")
cat("  PPMI_EUR:", auc_null_PPMI_EUR, "\n")
cat("  PPMI_AJ:", auc_null_PPMI_AJ, "\n")
cat("  HBS_EUR:", auc_null_HBS_EUR, "\n")

# -----------------------------------------------
# Prepare predictor matrices
# -----------------------------------------------

X <- combined_data %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y <- as.factor(combined_data$case_control_other_at_baseline)

X_PPMI_EUR <- combined_data_PPMI_EUR %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y_PPMI_EUR <- as.factor(combined_data_PPMI_EUR$case_control_other_at_baseline)

X_PPMI_AJ <- combined_data_PPMI_AJ %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y_PPMI_AJ <- as.factor(combined_data_PPMI_AJ$case_control_other_at_baseline)

X_HBS_EUR <- combined_data_HBS_EUR %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y_HBS_EUR <- as.factor(combined_data_HBS_EUR$case_control_other_at_baseline)

cat("Number of predictors:", ncol(X), "\n")

# -----------------------------------------------
# Scale on PDBP, apply to all
# -----------------------------------------------

penalty <- ifelse(colnames(X) %in% c("PD_PGS_resid", "sex", "age_at_baseline"), 0, 1)
vars_to_scale <- setdiff(colnames(X), "sex")

scaler_full <- preProcess(X[, vars_to_scale], method = c("center", "scale"))

X_scaled          <- X
X_scaled[, vars_to_scale] <- predict(scaler_full, X[, vars_to_scale])

X_PPMI_EUR_scaled <- X_PPMI_EUR
X_PPMI_EUR_scaled[, vars_to_scale] <- predict(scaler_full, X_PPMI_EUR[, vars_to_scale])

X_PPMI_AJ_scaled  <- X_PPMI_AJ
X_PPMI_AJ_scaled[, vars_to_scale]  <- predict(scaler_full, X_PPMI_AJ[, vars_to_scale])

X_HBS_EUR_scaled  <- X_HBS_EUR
X_HBS_EUR_scaled[, vars_to_scale]  <- predict(scaler_full, X_HBS_EUR[, vars_to_scale])

# -----------------------------------------------
# Fit final LASSO model on PDBP
# -----------------------------------------------

final_model <- glmnet(
  x = X_scaled,
  y = y,
  family = "binomial",
  alpha = 1,
  lambda = 0.05282453,
  penalty.factor = penalty,
  standardize = FALSE
)


# Extract and print coefficients
coefs <- coef(final_model)
coef_df <- data.frame(
  feature = rownames(coefs),
  coefficient = as.numeric(coefs)
)

coef_df <- coef_df %>% arrange(desc(coefficient))

# Print all coefficients (including zeros)
print(coef_df)

coef_outfile <- "/home/jupyter/multiTRS/results/multi_TRS_w_PRS_FUSION_LASSO_PDBP_full_model_coefficients.txt"
write.table(coef_df, coef_outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)
cat("Coefficients saved to:", coef_outfile, "\n")

# -----------------------------------------------
# Predictions and AUCs
# -----------------------------------------------

get_auc <- function(model, X_new, y_new) {
  probs <- predict(model, newx = X_new, type = "response")
  auc(roc(y_new, as.numeric(probs)))
}

auc_PDBP     <- get_auc(final_model, X_scaled,          y)
auc_PPMI_EUR <- get_auc(final_model, X_PPMI_EUR_scaled, y_PPMI_EUR)
auc_PPMI_AJ  <- get_auc(final_model, X_PPMI_AJ_scaled,  y_PPMI_AJ)
auc_HBS_EUR  <- get_auc(final_model, X_HBS_EUR_scaled,  y_HBS_EUR)

cat("Full model AUCs:\n")
cat("  PDBP:", auc_PDBP, "\n")
cat("  PPMI_EUR:", auc_PPMI_EUR, "\n")
cat("  PPMI_AJ:", auc_PPMI_AJ, "\n")
cat("  HBS_EUR:", auc_HBS_EUR, "\n")

# -----------------------------------------------
# CIs for model AUCs
# -----------------------------------------------

ci_null_PPMI_EUR <- ci.auc(roc(combined_data_PPMI_EUR$case_control_other_at_baseline, null_probs_PPMI_EUR))
ci_null_PPMI_AJ  <- ci.auc(roc(combined_data_PPMI_AJ$case_control_other_at_baseline,  null_probs_PPMI_AJ))
ci_null_HBS_EUR  <- ci.auc(roc(combined_data_HBS_EUR$case_control_other_at_baseline,  null_probs_HBS_EUR))

# CIs for full model AUCs
ci_full_PPMI_EUR <- ci.auc(roc_full_PPMI_EUR)
ci_full_PPMI_AJ  <- ci.auc(roc_full_PPMI_AJ)
ci_full_HBS_EUR  <- ci.auc(roc_full_HBS_EUR)

cat("Null model AUCs with 95% CI:\n")
cat("  PPMI_EUR:", round(auc_null_PPMI_EUR, 3), "(95% CI:", round(ci_null_PPMI_EUR[1], 3), "-", round(ci_null_PPMI_EUR[3], 3), ")\n")
cat("  PPMI_AJ:",  round(auc_null_PPMI_AJ, 3),  "(95% CI:", round(ci_null_PPMI_AJ[1], 3),  "-", round(ci_null_PPMI_AJ[3], 3),  ")\n")
cat("  HBS_EUR:",  round(auc_null_HBS_EUR, 3),   "(95% CI:", round(ci_null_HBS_EUR[1], 3),  "-", round(ci_null_HBS_EUR[3], 3),  ")\n")

cat("Full model AUCs with 95% CI:\n")
cat("  PPMI_EUR:", round(auc_PPMI_EUR, 3), "(95% CI:", round(ci_full_PPMI_EUR[1], 3), "-", round(ci_full_PPMI_EUR[3], 3), ")\n")
cat("  PPMI_AJ:",  round(auc_PPMI_AJ, 3),  "(95% CI:", round(ci_full_PPMI_AJ[1], 3),  "-", round(ci_full_PPMI_AJ[3], 3),  ")\n")
cat("  HBS_EUR:",  round(auc_HBS_EUR, 3),   "(95% CI:", round(ci_full_HBS_EUR[1], 3),  "-", round(ci_full_HBS_EUR[3], 3),  ")\n")
# -----------------------------------------------
# ROC plots
# -----------------------------------------------

probs_PPMI_EUR <- predict(final_model, newx = X_PPMI_EUR_scaled, type = "response")
probs_PPMI_AJ  <- predict(final_model, newx = X_PPMI_AJ_scaled,  type = "response")
probs_HBS_EUR  <- predict(final_model, newx = X_HBS_EUR_scaled,  type = "response")

roc_null_PPMI_EUR <- roc(y_PPMI_EUR, null_probs_PPMI_EUR)
roc_full_PPMI_EUR <- roc(y_PPMI_EUR, as.numeric(probs_PPMI_EUR))

roc_null_PPMI_AJ  <- roc(y_PPMI_AJ,  null_probs_PPMI_AJ)
roc_full_PPMI_AJ  <- roc(y_PPMI_AJ,  as.numeric(probs_PPMI_AJ))

roc_null_HBS_EUR  <- roc(y_HBS_EUR,  null_probs_HBS_EUR)
roc_full_HBS_EUR  <- roc(y_HBS_EUR,  as.numeric(probs_HBS_EUR))

# Plot helper function
plot_roc <- function(roc_null, roc_full, title, full_col) {
  plot.roc(roc_null, col = "black", lwd = 2, main = title,
           legacy.axes = TRUE, print.auc = FALSE)
  lines.roc(roc_full, col = full_col, lwd = 2)
  legend("bottomright",
         legend = c(
           paste0("Age + Sex + PD-PGS (AUC = ", round(auc(roc_null), 3), ")"),
           paste0("Full Model (AUC = ", round(auc(roc_full), 3), ")")
         ),
         col = c("black", full_col), lwd = 2, bty = "n")
}

plot_roc(roc_null_PPMI_EUR, roc_full_PPMI_EUR, "PPMI EUR: Baseline vs Full Model", "goldenrod")
plot_roc(roc_null_PPMI_AJ,  roc_full_PPMI_AJ,  "PPMI AJ: Baseline vs Full Model",  "forestgreen")
plot_roc(roc_null_HBS_EUR,  roc_full_HBS_EUR,  "HBS EUR: Baseline vs Full Model",  "dodgerblue")

# -----------------------------------------------
# DeLong tests
# -----------------------------------------------

delong_PPMI_EUR <- roc.test(roc_null_PPMI_EUR, roc_full_PPMI_EUR, method = "delong")
delong_PPMI_AJ  <- roc.test(roc_null_PPMI_AJ,  roc_full_PPMI_AJ,  method = "delong")
delong_HBS_EUR  <- roc.test(roc_null_HBS_EUR,  roc_full_HBS_EUR,  method = "delong")

cat("DeLong test p-values:\n")
cat("  PPMI_EUR:", delong_PPMI_EUR$p.value, "\n")
cat("  PPMI_AJ:",  delong_PPMI_AJ$p.value,  "\n")
cat("  HBS_EUR:",  delong_HBS_EUR$p.value,  "\n")

# -----------------------------------------------
# Extract sensitivity, specificity, accuracy at
# optimal threshold (closest-to-top-left)
# -----------------------------------------------

coords_null_PPMI_EUR <- coords(roc_null_PPMI_EUR, x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]
coords_full_PPMI_EUR <- coords(roc_full_PPMI_EUR, x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]

coords_null_PPMI_AJ  <- coords(roc_null_PPMI_AJ,  x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]
coords_full_PPMI_AJ  <- coords(roc_full_PPMI_AJ,  x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]

coords_null_HBS_EUR  <- coords(roc_null_HBS_EUR,  x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]
coords_full_HBS_EUR  <- coords(roc_full_HBS_EUR,  x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]

# -----------------------------------------------
# Build results table
# -----------------------------------------------

results <- rbindlist(list(
  data.table(
    cohort         = "PPMI_EUR",
    AUC_null       = as.numeric(auc(roc_null_PPMI_EUR)),
    AUC_lower_null = as.numeric(ci_null_PPMI_EUR[1]),
    AUC_upper_null = as.numeric(ci_null_PPMI_EUR[3]),
    AUC_full       = as.numeric(auc(roc_full_PPMI_EUR)),
    AUC_lower_full = as.numeric(ci_full_PPMI_EUR[1]),
    AUC_upper_full = as.numeric(ci_full_PPMI_EUR[3]),
    AUC_diff       = as.numeric(auc(roc_full_PPMI_EUR)) - as.numeric(auc(roc_null_PPMI_EUR)),
    sens_null      = coords_null_PPMI_EUR$sensitivity,
    sens_full      = coords_full_PPMI_EUR$sensitivity,
    sens_diff      = coords_full_PPMI_EUR$sensitivity - coords_null_PPMI_EUR$sensitivity,
    spec_null      = coords_null_PPMI_EUR$specificity,
    spec_full      = coords_full_PPMI_EUR$specificity,
    spec_diff      = coords_full_PPMI_EUR$specificity - coords_null_PPMI_EUR$specificity,
    acc_null       = coords_null_PPMI_EUR$accuracy,
    acc_full       = coords_full_PPMI_EUR$accuracy,
    acc_diff       = coords_full_PPMI_EUR$accuracy - coords_null_PPMI_EUR$accuracy,
    delong_z       = delong_PPMI_EUR$statistic,
    delong_p       = delong_PPMI_EUR$p.value
  ),
  data.table(
    cohort         = "PPMI_AJ",
    AUC_null       = as.numeric(auc(roc_null_PPMI_AJ)),
    AUC_lower_null = as.numeric(ci_null_PPMI_AJ[1]),
    AUC_upper_null = as.numeric(ci_null_PPMI_AJ[3]),
    AUC_full       = as.numeric(auc(roc_full_PPMI_AJ)),
    AUC_lower_full = as.numeric(ci_full_PPMI_AJ[1]),
    AUC_upper_full = as.numeric(ci_full_PPMI_AJ[3]),
    AUC_diff       = as.numeric(auc(roc_full_PPMI_AJ)) - as.numeric(auc(roc_null_PPMI_AJ)),
    sens_null      = coords_null_PPMI_AJ$sensitivity,
    sens_full      = coords_full_PPMI_AJ$sensitivity,
    sens_diff      = coords_full_PPMI_AJ$sensitivity - coords_null_PPMI_AJ$sensitivity,
    spec_null      = coords_null_PPMI_AJ$specificity,
    spec_full      = coords_full_PPMI_AJ$specificity,
    spec_diff      = coords_full_PPMI_AJ$specificity - coords_null_PPMI_AJ$specificity,
    acc_null       = coords_null_PPMI_AJ$accuracy,
    acc_full       = coords_full_PPMI_AJ$accuracy,
    acc_diff       = coords_full_PPMI_AJ$accuracy - coords_null_PPMI_AJ$accuracy,
    delong_z       = delong_PPMI_AJ$statistic,
    delong_p       = delong_PPMI_AJ$p.value
  ),
  data.table(
    cohort         = "HBS_EUR",
    AUC_null       = as.numeric(auc(roc_null_HBS_EUR)),
    AUC_lower_null = as.numeric(ci_null_HBS_EUR[1]),
    AUC_upper_null = as.numeric(ci_null_HBS_EUR[3]),
    AUC_full       = as.numeric(auc(roc_full_HBS_EUR)),
    AUC_lower_full = as.numeric(ci_full_HBS_EUR[1]),
    AUC_upper_full = as.numeric(ci_full_HBS_EUR[3]),
    AUC_diff       = as.numeric(auc(roc_full_HBS_EUR)) - as.numeric(auc(roc_null_HBS_EUR)),
    sens_null      = coords_null_HBS_EUR$sensitivity,
    sens_full      = coords_full_HBS_EUR$sensitivity,
    sens_diff      = coords_full_HBS_EUR$sensitivity - coords_null_HBS_EUR$sensitivity,
    spec_null      = coords_null_HBS_EUR$specificity,
    spec_full      = coords_full_HBS_EUR$specificity,
    spec_diff      = coords_full_HBS_EUR$specificity - coords_null_HBS_EUR$specificity,
    acc_null       = coords_null_HBS_EUR$accuracy,
    acc_full       = coords_full_HBS_EUR$accuracy,
    acc_diff       = coords_full_HBS_EUR$accuracy - coords_null_HBS_EUR$accuracy,
    delong_z       = delong_HBS_EUR$statistic,
    delong_p       = delong_HBS_EUR$p.value
  )
))

print(results)

results_outfile <- "/home/jupyter/multiTRS/results/multi_TRS_w_PRS_FUSION_LASSO_external_validation_results_table.txt"
write.table(results, results_outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)
cat("Results table saved to:", results_outfile, "\n")

### All scores

#### Nested-CV

In [ ]:
%%R

library(data.table)
library(dplyr)
library(glmnet)
library(caret)
library(pROC)

set.seed(1)

# Create a data.table for all performance results
all_performance_results <- data.table()

print("This script calculates LASSO multi-TRS models using SMR-multi scores... Running LASSO models")

outfile <- "/home/jupyter/multiTRS/results/multi_TRS_w_PRS_all_scores_LASSO_nestedcv.txt"

# Read in the clinical data
clinical <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt") %>%
  select(participant_id, case_control_other_at_baseline, sex, age_at_baseline)

# List of cohorts to loop over
TRS_cohorts <- c("PDBP")

# Loop over cohorts
for (TRS_cohort in TRS_cohorts) {
    
  # Read PRS files
  PRS_path <- paste0("/home/jupyter/multiTRS/geno/PRS_scores_out/", TRS_cohort, "_EUR_PD_PGS_resid.txt")
  PRS <- fread(PRS_path)
  
  # Read TRS scores
  TRS_path <- paste0("/home/jupyter/multiTRS/scores/", TRS_cohort, "_EUR_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt")
  TRS <- fread(TRS_path) %>% select(-contains("Hip_fracture_EUR_2022_FDR_SMR_single_SNP"))
  
  combined_data <- clinical %>%
    inner_join(PRS, by = "participant_id")
    
  combined_data <- combined_data %>%
    inner_join(TRS, by = "participant_id")
  
  cat("Rows in combined data for", TRS_cohort, ":", nrow(combined_data), "\n")
  
  # Null model with age + sex
  null_model <- glm(case_control_other_at_baseline ~ sex + age_at_baseline + PD_PGS_resid,
                    data = combined_data,
                    family = binomial)
  probs_null <- predict(null_model, type = "response")
  roc_obj_null <- roc(combined_data$case_control_other_at_baseline, probs_null)
  auc_null <- auc(roc_obj_null)
  cat("AUC of null model:", auc_null, "\n")
  
  null_coords <- coords(roc_obj_null, x = "best", best.method = "closest.topleft",
                        ret = c("threshold", "sensitivity", "specificity", "accuracy"))
  null_sensitivity <- as.numeric(null_coords["sensitivity"])
  null_specificity <- as.numeric(null_coords["specificity"])
  null_accuracy <- as.numeric(null_coords["accuracy"])
  
  # Prepare predictor matrix X and outcome y
  X <- combined_data %>%
    select(-participant_id, -case_control_other_at_baseline) %>%
    as.matrix()
  y <- as.factor(combined_data$case_control_other_at_baseline)
  
  cat("Number of predictors:", ncol(X), "\n")
  
  # Define penalty factors: unpenalized for sex and age
  penalty <- ifelse(colnames(X) %in% c("sex", "age_at_baseline","PD_PGS_resid"), 0, 1)
  
  # Outer 10-fold CV
  outer_folds <- createFolds(y, k = 5, returnTrain = TRUE)
  
  # Initialize vectors for storing performance
  outer_auc <- c()
  lambda_vals <- c()
  outer_sensitivity <- c()
  outer_specificity <- c()
  outer_accuracy <- c()
  
  for (i in seq_along(outer_folds)) {
    cat("Outer fold:", i, "\n")
    
    train_idx <- outer_folds[[i]]
    test_idx <- setdiff(seq_along(y), train_idx)
    
    X_train <- X[train_idx, , drop = FALSE]
    y_train <- y[train_idx]
    X_test <- X[test_idx, , drop = FALSE]
    y_test <- y[test_idx]
    
    # Standardise variables (exclude sex)
    vars_to_scale <- setdiff(colnames(X), "sex")
    scaler <- preProcess(X_train[, vars_to_scale, drop = FALSE], method = c("center", "scale"))
    
    X_train_scaled <- X_train
    X_train_scaled[, vars_to_scale] <- predict(scaler, X_train[, vars_to_scale, drop = FALSE])
    
    X_test_scaled <- X_test
    X_test_scaled[, vars_to_scale] <- predict(scaler, X_test[, vars_to_scale, drop = FALSE])
    
    # Inner CV for LASSO (alpha = 1)
    cvfit <- cv.glmnet(
      x = X_train_scaled,
      y = y_train,
      family = "binomial",
      alpha = 1,
      nfolds = 5,
      type.measure = "auc",
      penalty.factor = penalty,
      standardize = FALSE
    )
    
    best_lambda <- cvfit$lambda.1se
    lambda_vals[i] <- best_lambda
    
    # Refit model on full training set with best lambda
    final_model <- glmnet(
      x = X_train_scaled,
      y = y_train,
      family = "binomial",
      alpha = 1,
      lambda = best_lambda,
      penalty.factor = penalty,
      standardize = FALSE
    )
    
    # Predict on outer test set
    probs <- predict(final_model, newx = X_test_scaled, type = "response")
    roc_obj_outer <- roc(y_test, as.numeric(probs))
    auc_val <- auc(roc_obj_outer)
    
    model_coords <- coords(roc_obj_outer, x = "best", best.method = "closest.topleft",
                           ret = c("sensitivity", "specificity", "accuracy"))
    
    outer_auc[i] <- auc_val
    outer_sensitivity[i] <- as.numeric(model_coords["sensitivity"])
    outer_specificity[i] <- as.numeric(model_coords["specificity"])
    outer_accuracy[i] <- as.numeric(model_coords["accuracy"])
  }
  
  # Median lambda across outer folds
  lambda_final <- median(lambda_vals)
    
  cat("AUC for each fold in ", TRS_cohort, ":", outer_auc, "\n")
  
  cat("Median lambda for cohort", TRS_cohort, ":", lambda_final, "\n")
  cat("Mean outer AUC:", mean(outer_auc), "SD:", sd(outer_auc), "\n")

  
  # Store performance results
  performance_results <- data.table(
    cohort = TRS_cohort,
    scores = "all_scores",
    model = "LASSO",
    covariate_treatment = "unpenalised",
    median_lambda = lambda_final,
    median_alpha = "Not applicable",
    AUC_null = auc_null,
    AUC_full_mean_cv = mean(outer_auc),
    AUC_full_sd_cv = sd(outer_auc),
    AUC_diff = mean(outer_auc) - auc_null,
    sens_null = null_sensitivity,
    sens_full_mean_cv = mean(outer_sensitivity),
    sens_full_sd_cv = sd(outer_sensitivity),
    sens_diff = mean(outer_sensitivity) - null_sensitivity,
    spec_null = null_specificity,
    spec_full_mean_cv = mean(outer_specificity),
    spec_full_sd_cv = sd(outer_specificity),
    spec_diff = mean(outer_specificity) - null_specificity,
    acc_null = null_accuracy,
    acc_full_mean_cv = mean(outer_accuracy),
    acc_full_sd_cv = sd(outer_accuracy),
    acc_diff = mean(outer_accuracy) - null_accuracy
  )
  
  all_performance_results <- rbind(all_performance_results, performance_results)
  
}


print(all_performance_results)

#write.table(all_performance_results, outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)

#### Test in PPMI (EUR), PPMI (AJ) and HBS (EUR)

In [ ]:
%%R

library(data.table)
library(dplyr)
library(glmnet)
library(caret)
library(pROC)

set.seed(1)

# Read in the data
clinical <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt") %>% 
  select(participant_id, case_control_other_at_baseline, sex, age_at_baseline)

# Read in the PGS
PRS_PDBP_EUR <- fread("/home/jupyter/multiTRS/geno/PRS_scores_out/PDBP_EUR_PD_PGS_resid.txt")
PRS_PPMI_EUR <- fread("/home/jupyter/multiTRS/geno/PRS_scores_out/PPMI_EUR_PD_PGS_resid.txt")
PRS_PPMI_AJ  <- fread("/home/jupyter/multiTRS/geno/PRS_scores_out/PPMI_AJ_PD_PGS_resid.txt")
PRS_HBS_EUR  <- fread("/home/jupyter/multiTRS/geno/PRS_scores_out/HBS_EUR_PD_PGS_resid.txt")

# Read TRS scores - select columns based on PDBP first
TRS_PDBP_EUR <- fread("/home/jupyter/multiTRS/scores/PDBP_EUR_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt") %>% 
  select(-contains("Hip_fracture_EUR_2022_FDR_SMR_single_SNP"))

TRS_PPMI_EUR <- fread("/home/jupyter/multiTRS/scores/PPMI_EUR_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt") %>% 
  select(all_of(colnames(TRS_PDBP_EUR)))  # Fixed: TRS_PDBP -> TRS_PDBP_EUR

TRS_PPMI_AJ  <- fread("/home/jupyter/multiTRS/scores/PPMI_AJ_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt") %>%  
  select(all_of(colnames(TRS_PDBP_EUR)))  # Fixed: TRS_PDBP -> TRS_PDBP_EUR

TRS_HBS_EUR  <- fread("/home/jupyter/multiTRS/scores/HBS_EUR_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt") %>% 
  select(all_of(colnames(TRS_PDBP_EUR)))  # Fixed: TRS_PDBP -> TRS_PDBP_EUR

# -----------------------------------------------
# Create combined datasets
# -----------------------------------------------

# PDBP (training)
combined_data <- clinical %>%
  inner_join(PRS_PDBP_EUR, by = "participant_id") %>%
  inner_join(TRS_PDBP_EUR, by = "participant_id")

# PPMI_EUR (test)
combined_data_PPMI_EUR <- clinical %>%
  inner_join(PRS_PPMI_EUR, by = "participant_id") %>%
  inner_join(TRS_PPMI_EUR, by = "participant_id")

# PPMI_AJ (test) - Fixed: was accidentally using combined_data_PPMI_EUR as base
combined_data_PPMI_AJ <- clinical %>%
  inner_join(PRS_PPMI_AJ, by = "participant_id") %>%
  inner_join(TRS_PPMI_AJ, by = "participant_id")

# HBS_EUR (test)
combined_data_HBS_EUR <- clinical %>%
  inner_join(PRS_HBS_EUR, by = "participant_id") %>%
  inner_join(TRS_HBS_EUR, by = "participant_id")

cat("Testing all scores model...\n")
cat("Rows in combined data - PDBP:", nrow(combined_data), "\n")
cat("Rows in combined data - PPMI_EUR:", nrow(combined_data_PPMI_EUR), "\n")
cat("Rows in combined data - PPMI_AJ:", nrow(combined_data_PPMI_AJ), "\n")
cat("Rows in combined data - HBS_EUR:", nrow(combined_data_HBS_EUR), "\n")

# -----------------------------------------------
# Null model (age + sex), fit on PDBP, applied to all
# -----------------------------------------------

null_model <- glm(case_control_other_at_baseline ~ scale(age_at_baseline) + sex + scale(PD_PGS_resid),
                  data = combined_data,
                  family = binomial)

# Null model predictions
null_probs_PDBP     <- predict(null_model, newdata = combined_data,          type = "response")
null_probs_PPMI_EUR <- predict(null_model, newdata = combined_data_PPMI_EUR, type = "response")
null_probs_PPMI_AJ  <- predict(null_model, newdata = combined_data_PPMI_AJ,  type = "response")
null_probs_HBS_EUR  <- predict(null_model, newdata = combined_data_HBS_EUR,  type = "response")

auc_null_PDBP     <- auc(roc(combined_data$case_control_other_at_baseline,          null_probs_PDBP))
auc_null_PPMI_EUR <- auc(roc(combined_data_PPMI_EUR$case_control_other_at_baseline, null_probs_PPMI_EUR))
auc_null_PPMI_AJ  <- auc(roc(combined_data_PPMI_AJ$case_control_other_at_baseline,  null_probs_PPMI_AJ))
auc_null_HBS_EUR  <- auc(roc(combined_data_HBS_EUR$case_control_other_at_baseline,  null_probs_HBS_EUR))

cat("Null model AUCs:\n")
cat("  PDBP:", auc_null_PDBP, "\n")
cat("  PPMI_EUR:", auc_null_PPMI_EUR, "\n")
cat("  PPMI_AJ:", auc_null_PPMI_AJ, "\n")
cat("  HBS_EUR:", auc_null_HBS_EUR, "\n")

# -----------------------------------------------
# Prepare predictor matrices
# -----------------------------------------------

X <- combined_data %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y <- as.factor(combined_data$case_control_other_at_baseline)

X_PPMI_EUR <- combined_data_PPMI_EUR %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y_PPMI_EUR <- as.factor(combined_data_PPMI_EUR$case_control_other_at_baseline)

X_PPMI_AJ <- combined_data_PPMI_AJ %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y_PPMI_AJ <- as.factor(combined_data_PPMI_AJ$case_control_other_at_baseline)

X_HBS_EUR <- combined_data_HBS_EUR %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y_HBS_EUR <- as.factor(combined_data_HBS_EUR$case_control_other_at_baseline)

cat("Number of predictors:", ncol(X), "\n")

# -----------------------------------------------
# Scale on PDBP, apply to all
# -----------------------------------------------

penalty <- ifelse(colnames(X) %in% c("PD_PGS_resid", "sex", "age_at_baseline"), 0, 1)
vars_to_scale <- setdiff(colnames(X), "sex")

scaler_full <- preProcess(X[, vars_to_scale], method = c("center", "scale"))

X_scaled          <- X
X_scaled[, vars_to_scale] <- predict(scaler_full, X[, vars_to_scale])

X_PPMI_EUR_scaled <- X_PPMI_EUR
X_PPMI_EUR_scaled[, vars_to_scale] <- predict(scaler_full, X_PPMI_EUR[, vars_to_scale])

X_PPMI_AJ_scaled  <- X_PPMI_AJ
X_PPMI_AJ_scaled[, vars_to_scale]  <- predict(scaler_full, X_PPMI_AJ[, vars_to_scale])

X_HBS_EUR_scaled  <- X_HBS_EUR
X_HBS_EUR_scaled[, vars_to_scale]  <- predict(scaler_full, X_HBS_EUR[, vars_to_scale])

# -----------------------------------------------
# Fit final LASSO model on PDBP
# -----------------------------------------------

final_model <- glmnet(
  x = X_scaled,
  y = y,
  family = "binomial",
  alpha = 1,
  lambda = 0.05838129,
  penalty.factor = penalty,
  standardize = FALSE
)

# Extract and print coefficients
coefs <- coef(final_model)
coef_df <- data.frame(
  feature = rownames(coefs),
  coefficient = as.numeric(coefs)
)

coef_df <- coef_df %>% arrange(desc(coefficient))

# Print all coefficients (including zeros)
print(coef_df)

coef_outfile <- "/home/jupyter/multiTRS/results/multi_TRS_w_PRS_all_scores_LASSO_PDBP_full_model_coefficients.txt"
write.table(coef_df, coef_outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)
cat("Coefficients saved to:", coef_outfile, "\n")

# -----------------------------------------------
# Predictions and AUCs
# -----------------------------------------------

get_auc <- function(model, X_new, y_new) {
  probs <- predict(model, newx = X_new, type = "response")
  auc(roc(y_new, as.numeric(probs)))
}

auc_PDBP     <- get_auc(final_model, X_scaled,          y)
auc_PPMI_EUR <- get_auc(final_model, X_PPMI_EUR_scaled, y_PPMI_EUR)
auc_PPMI_AJ  <- get_auc(final_model, X_PPMI_AJ_scaled,  y_PPMI_AJ)
auc_HBS_EUR  <- get_auc(final_model, X_HBS_EUR_scaled,  y_HBS_EUR)

cat("Full model AUCs:\n")
cat("  PDBP:", auc_PDBP, "\n")
cat("  PPMI_EUR:", auc_PPMI_EUR, "\n")
cat("  PPMI_AJ:", auc_PPMI_AJ, "\n")
cat("  HBS_EUR:", auc_HBS_EUR, "\n")

# -----------------------------------------------
# CIs for model AUCs
# -----------------------------------------------

ci_null_PPMI_EUR <- ci.auc(roc(combined_data_PPMI_EUR$case_control_other_at_baseline, null_probs_PPMI_EUR))
ci_null_PPMI_AJ  <- ci.auc(roc(combined_data_PPMI_AJ$case_control_other_at_baseline,  null_probs_PPMI_AJ))
ci_null_HBS_EUR  <- ci.auc(roc(combined_data_HBS_EUR$case_control_other_at_baseline,  null_probs_HBS_EUR))

# CIs for full model AUCs
ci_full_PPMI_EUR <- ci.auc(roc_full_PPMI_EUR)
ci_full_PPMI_AJ  <- ci.auc(roc_full_PPMI_AJ)
ci_full_HBS_EUR  <- ci.auc(roc_full_HBS_EUR)

cat("Null model AUCs with 95% CI:\n")
cat("  PPMI_EUR:", round(auc_null_PPMI_EUR, 3), "(95% CI:", round(ci_null_PPMI_EUR[1], 3), "-", round(ci_null_PPMI_EUR[3], 3), ")\n")
cat("  PPMI_AJ:",  round(auc_null_PPMI_AJ, 3),  "(95% CI:", round(ci_null_PPMI_AJ[1], 3),  "-", round(ci_null_PPMI_AJ[3], 3),  ")\n")
cat("  HBS_EUR:",  round(auc_null_HBS_EUR, 3),   "(95% CI:", round(ci_null_HBS_EUR[1], 3),  "-", round(ci_null_HBS_EUR[3], 3),  ")\n")

cat("Full model AUCs with 95% CI:\n")
cat("  PPMI_EUR:", round(auc_PPMI_EUR, 3), "(95% CI:", round(ci_full_PPMI_EUR[1], 3), "-", round(ci_full_PPMI_EUR[3], 3), ")\n")
cat("  PPMI_AJ:",  round(auc_PPMI_AJ, 3),  "(95% CI:", round(ci_full_PPMI_AJ[1], 3),  "-", round(ci_full_PPMI_AJ[3], 3),  ")\n")
cat("  HBS_EUR:",  round(auc_HBS_EUR, 3),   "(95% CI:", round(ci_full_HBS_EUR[1], 3),  "-", round(ci_full_HBS_EUR[3], 3),  ")\n")
# -----------------------------------------------
# ROC plots
# -----------------------------------------------

probs_PPMI_EUR <- predict(final_model, newx = X_PPMI_EUR_scaled, type = "response")
probs_PPMI_AJ  <- predict(final_model, newx = X_PPMI_AJ_scaled,  type = "response")
probs_HBS_EUR  <- predict(final_model, newx = X_HBS_EUR_scaled,  type = "response")

roc_null_PPMI_EUR <- roc(y_PPMI_EUR, null_probs_PPMI_EUR)
roc_full_PPMI_EUR <- roc(y_PPMI_EUR, as.numeric(probs_PPMI_EUR))

roc_null_PPMI_AJ  <- roc(y_PPMI_AJ,  null_probs_PPMI_AJ)
roc_full_PPMI_AJ  <- roc(y_PPMI_AJ,  as.numeric(probs_PPMI_AJ))

roc_null_HBS_EUR  <- roc(y_HBS_EUR,  null_probs_HBS_EUR)
roc_full_HBS_EUR  <- roc(y_HBS_EUR,  as.numeric(probs_HBS_EUR))

# Plot helper function
plot_roc <- function(roc_null, roc_full, title, full_col) {
  plot.roc(roc_null, col = "black", lwd = 2, main = title,
           legacy.axes = TRUE, print.auc = FALSE)
  lines.roc(roc_full, col = full_col, lwd = 2)
  legend("bottomright",
         legend = c(
           paste0("Age + Sex + PD-PGS (AUC = ", round(auc(roc_null), 3), ")"),
           paste0("Full Model (AUC = ", round(auc(roc_full), 3), ")")
         ),
         col = c("black", full_col), lwd = 2, bty = "n")
}

plot_roc(roc_null_PPMI_EUR, roc_full_PPMI_EUR, "PPMI EUR: Baseline vs Full Model", "goldenrod")
plot_roc(roc_null_PPMI_AJ,  roc_full_PPMI_AJ,  "PPMI AJ: Baseline vs Full Model",  "forestgreen")
plot_roc(roc_null_HBS_EUR,  roc_full_HBS_EUR,  "HBS EUR: Baseline vs Full Model",  "dodgerblue")

# -----------------------------------------------
# DeLong tests
# -----------------------------------------------

delong_PPMI_EUR <- roc.test(roc_null_PPMI_EUR, roc_full_PPMI_EUR, method = "delong")
delong_PPMI_AJ  <- roc.test(roc_null_PPMI_AJ,  roc_full_PPMI_AJ,  method = "delong")
delong_HBS_EUR  <- roc.test(roc_null_HBS_EUR,  roc_full_HBS_EUR,  method = "delong")

cat("DeLong test p-values:\n")
cat("  PPMI_EUR:", delong_PPMI_EUR$p.value, "\n")
cat("  PPMI_AJ:",  delong_PPMI_AJ$p.value,  "\n")
cat("  HBS_EUR:",  delong_HBS_EUR$p.value,  "\n")


# -----------------------------------------------
# Extract sensitivity, specificity, accuracy at
# optimal threshold (closest-to-top-left)
# -----------------------------------------------

coords_null_PPMI_EUR <- coords(roc_null_PPMI_EUR, x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]
coords_full_PPMI_EUR <- coords(roc_full_PPMI_EUR, x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]

coords_null_PPMI_AJ  <- coords(roc_null_PPMI_AJ,  x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]
coords_full_PPMI_AJ  <- coords(roc_full_PPMI_AJ,  x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]

coords_null_HBS_EUR  <- coords(roc_null_HBS_EUR,  x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]
coords_full_HBS_EUR  <- coords(roc_full_HBS_EUR,  x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]

# -----------------------------------------------
# Build results table
# -----------------------------------------------

results <- rbindlist(list(
  data.table(
    cohort         = "PPMI_EUR",
    AUC_null       = as.numeric(auc(roc_null_PPMI_EUR)),
    AUC_lower_null = as.numeric(ci_null_PPMI_EUR[1]),
    AUC_upper_null = as.numeric(ci_null_PPMI_EUR[3]),
    AUC_full       = as.numeric(auc(roc_full_PPMI_EUR)),
    AUC_lower_full = as.numeric(ci_full_PPMI_EUR[1]),
    AUC_upper_full = as.numeric(ci_full_PPMI_EUR[3]),
    AUC_diff       = as.numeric(auc(roc_full_PPMI_EUR)) - as.numeric(auc(roc_null_PPMI_EUR)),
    sens_null      = coords_null_PPMI_EUR$sensitivity,
    sens_full      = coords_full_PPMI_EUR$sensitivity,
    sens_diff      = coords_full_PPMI_EUR$sensitivity - coords_null_PPMI_EUR$sensitivity,
    spec_null      = coords_null_PPMI_EUR$specificity,
    spec_full      = coords_full_PPMI_EUR$specificity,
    spec_diff      = coords_full_PPMI_EUR$specificity - coords_null_PPMI_EUR$specificity,
    acc_null       = coords_null_PPMI_EUR$accuracy,
    acc_full       = coords_full_PPMI_EUR$accuracy,
    acc_diff       = coords_full_PPMI_EUR$accuracy - coords_null_PPMI_EUR$accuracy,
    delong_z       = delong_PPMI_EUR$statistic,
    delong_p       = delong_PPMI_EUR$p.value
  ),
  data.table(
    cohort         = "PPMI_AJ",
    AUC_null       = as.numeric(auc(roc_null_PPMI_AJ)),
    AUC_lower_null = as.numeric(ci_null_PPMI_AJ[1]),
    AUC_upper_null = as.numeric(ci_null_PPMI_AJ[3]),
    AUC_full       = as.numeric(auc(roc_full_PPMI_AJ)),
    AUC_lower_full = as.numeric(ci_full_PPMI_AJ[1]),
    AUC_upper_full = as.numeric(ci_full_PPMI_AJ[3]),
    AUC_diff       = as.numeric(auc(roc_full_PPMI_AJ)) - as.numeric(auc(roc_null_PPMI_AJ)),
    sens_null      = coords_null_PPMI_AJ$sensitivity,
    sens_full      = coords_full_PPMI_AJ$sensitivity,
    sens_diff      = coords_full_PPMI_AJ$sensitivity - coords_null_PPMI_AJ$sensitivity,
    spec_null      = coords_null_PPMI_AJ$specificity,
    spec_full      = coords_full_PPMI_AJ$specificity,
    spec_diff      = coords_full_PPMI_AJ$specificity - coords_null_PPMI_AJ$specificity,
    acc_null       = coords_null_PPMI_AJ$accuracy,
    acc_full       = coords_full_PPMI_AJ$accuracy,
    acc_diff       = coords_full_PPMI_AJ$accuracy - coords_null_PPMI_AJ$accuracy,
    delong_z       = delong_PPMI_AJ$statistic,
    delong_p       = delong_PPMI_AJ$p.value
  ),
  data.table(
    cohort         = "HBS_EUR",
    AUC_null       = as.numeric(auc(roc_null_HBS_EUR)),
    AUC_lower_null = as.numeric(ci_null_HBS_EUR[1]),
    AUC_upper_null = as.numeric(ci_null_HBS_EUR[3]),
    AUC_full       = as.numeric(auc(roc_full_HBS_EUR)),
    AUC_lower_full = as.numeric(ci_full_HBS_EUR[1]),
    AUC_upper_full = as.numeric(ci_full_HBS_EUR[3]),
    AUC_diff       = as.numeric(auc(roc_full_HBS_EUR)) - as.numeric(auc(roc_null_HBS_EUR)),
    sens_null      = coords_null_HBS_EUR$sensitivity,
    sens_full      = coords_full_HBS_EUR$sensitivity,
    sens_diff      = coords_full_HBS_EUR$sensitivity - coords_null_HBS_EUR$sensitivity,
    spec_null      = coords_null_HBS_EUR$specificity,
    spec_full      = coords_full_HBS_EUR$specificity,
    spec_diff      = coords_full_HBS_EUR$specificity - coords_null_HBS_EUR$specificity,
    acc_null       = coords_null_HBS_EUR$accuracy,
    acc_full       = coords_full_HBS_EUR$accuracy,
    acc_diff       = coords_full_HBS_EUR$accuracy - coords_null_HBS_EUR$accuracy,
    delong_z       = delong_HBS_EUR$statistic,
    delong_p       = delong_HBS_EUR$p.value
  )
))

print(results)

results_outfile <- "/home/jupyter/multiTRS/results/multi_TRS_w_PRS_all_scores_LASSO_external_validation_results_table.txt"
write.table(results, results_outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)
cat("Results table saved to:", results_outfile, "\n")

## ENET

### SMR-multi

#### Nested-CV

In [ ]:
%%R

library(data.table)
library(dplyr)
library(glmnet)
library(glmnetUtils)
library(caret)
library(pROC)

set.seed(1)

# Create a data.table for all performance results
all_performance_results <- data.table()

print("This script calculates ENET multi-TRS models using SMR-multi scores... Running ENET models")

outfile <- "/home/jupyter/multiTRS/results/multi_TRS_w_PRS_SMR_multi_ENET_nestedcv.txt"

# Read in the clinical data
clinical <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt") %>%
  select(participant_id, case_control_other_at_baseline, sex, age_at_baseline)

# List of cohorts to loop over
TRS_cohorts <- c("PDBP")

# Loop over cohorts
for (TRS_cohort in TRS_cohorts) {

  # Read TRS scores
  TRS_path <- paste0("/home/jupyter/multiTRS/scores/", TRS_cohort, "_TWAS_SMR_FDR_scores_resid.txt")
  TRS <- fread(TRS_path) %>% select(participant_id, contains("SMR"), -contains("single_SNP"))

  # Read PGS scores
  PGS_path <- paste0("/home/jupyter/multiTRS/geno/PRS_scores_out/", TRS_cohort, "_EUR_PD_PGS_resid.txt")
  PGS <- fread(PGS_path)

  combined_data <- clinical %>%
    inner_join(PGS, by = "participant_id") %>%
    inner_join(TRS, by = "participant_id")

  cat("Rows in combined data for", TRS_cohort, ":", nrow(combined_data), "\n")

  # Null model with age + sex
  null_model <- glm(case_control_other_at_baseline ~ sex + scale(age_at_baseline) + scale(PD_PGS_resid),
                    data = combined_data,
                    family = binomial)
  probs_null <- predict(null_model, type = "response")
  roc_obj_null <- roc(combined_data$case_control_other_at_baseline, probs_null)
  auc_null <- auc(roc_obj_null)
  cat("AUC of null model:", auc_null, "\n")

  null_coords <- coords(roc_obj_null, x = "best", best.method = "closest.topleft",
                        ret = c("sensitivity", "specificity", "accuracy"))
  null_sensitivity <- as.numeric(null_coords["sensitivity"])
  null_specificity <- as.numeric(null_coords["specificity"])
  null_accuracy <- as.numeric(null_coords["accuracy"])

  # Prepare predictor matrix X and outcome y
  X <- combined_data %>%
    select(-participant_id, -case_control_other_at_baseline) %>%
    as.matrix()
  y <- as.factor(combined_data$case_control_other_at_baseline)

  cat("Number of predictors:", ncol(X), "\n")

  # Define penalty factors: unpenalized for sex, age and PD_PGS_resid
  penalty <- ifelse(colnames(X) %in% c("sex", "age_at_baseline", "PD_PGS_resid"), 0, 1)

# Create five outer folds
outer_folds <- createFolds(y, k = 5, returnTrain = TRUE)

# Store auc of outer folders
outer_auc <- c()
lambda_vals <- c()
alpha_vals  <- c()
outer_sensitivity <- c()
outer_specificity <- c()
outer_accuracy <- c()

for (i in 1:length(outer_folds)) {
  cat("Outer fold:", i, "\n")
  
  # Train/test split for this outer fold
  train_idx <- outer_folds[[i]]
  test_idx  <- setdiff(seq_along(y), train_idx)
  
  X_train <- X[train_idx, ]
  y_train <- y[train_idx]
  X_test  <- X[test_idx, ]
  y_test  <- y[test_idx]
  
  # Standardize inside training only
    vars_to_scale <- setdiff(colnames(X), "sex")
    scaler <- preProcess(X_train[, vars_to_scale, drop = FALSE], method = c("center", "scale"))
    
    X_train_scaled <- X_train
    X_train_scaled[, vars_to_scale] <- predict(scaler, X_train[, vars_to_scale, drop = FALSE])
    
    X_test_scaled <- X_test
    X_test_scaled[, vars_to_scale] <- predict(scaler, X_test[, vars_to_scale, drop = FALSE])
    
 # Specify alpha list 
  alphalist <- seq(0,1,by=0.1)
  
  # Inner CV with cv.glmnet (5-fold)
  cvfit <- glmnetUtils::cva.glmnet(
    x = X_train_scaled,
    y = y_train,
    family = "binomial",
    alpha = alphalist,
    nfolds = 5,
    type.measure = "auc",
    penalty.factor = penalty,
    standardize = FALSE
  )
  

    
  # Select best alpha and lambda
  lambda_1se <- sapply(cvfit$modlist, `[[`, "lambda.1se")
  error <- sapply(cvfit$modlist, function(mod) {
    idx <- which(mod$lambda == mod$lambda.1se)
    mod$cvm[idx]
  })
  best <- which.max(error)
  best_alpha  <- cvfit$alpha[best]
  best_lambda <- lambda_1se[best]
                  

  # Best lambda chosen inside
  lambda_vals[i] <- best_lambda
  alpha_vals[i]  <- best_alpha
    
  
  # Refit on full training set with best lambda
  final_model <- glmnet(
    x = X_train_scaled,
    y = y_train,
    family = "binomial",
    alpha = best_alpha,
    lambda = best_lambda,
    penalty.factor = penalty,
    standardize = FALSE
  )
  
  # Predict probabilities on outer test set
  probs <- predict(final_model, newx = X_test_scaled, type = "response")
  
# Compute ROC curve
  roc_obj_outer <- roc(y_test, as.numeric(probs))

# Compute AUC
  auc_val <- auc(roc_obj_outer)

# Get sens, spec, accuracy using topleft
model_coords <- coords(
  roc_obj_outer,
  x = "best",
  best.method = "closest.topleft",
  ret = c("threshold", "sensitivity", "specificity", "accuracy")
)

model_sensitivity <- as.numeric(model_coords["sensitivity"])
model_specificity <- as.numeric(model_coords["specificity"])
model_accuracy <- as.numeric(model_coords["accuracy"])
    
  outer_auc[i] <- auc_val
  outer_sensitivity[i] <- model_sensitivity
  outer_specificity[i] <- model_specificity
  outer_accuracy[i] <- model_accuracy
}

alpha_vals <- as.numeric(alpha_vals)
lambda_vals <- as.numeric(lambda_vals)

cat("Alpha per outer fold:\n")
print(alpha_vals)

cat("Lambda per outer fold:\n")
print(lambda_vals)
                  
lambda_final <- median(lambda_vals)
alpha_final <- median(alpha_vals)


cat("AUC per outer fold:\n")
print(outer_auc)
cat("Mean AUC across outer folds:", mean(outer_auc), "\n")
cat("SD of the AUC across outer folds:", sd(outer_auc), "\n")

cat("Median alpha:\n")
print(alpha_final)
cat("Median lambda:\n")
print(lambda_final)


                  
performance_results <- data.table(cohort = TRS_cohort,
                                  scores = "SMR-multi",
                                  model = "ENET",
                                  covariate_treatment = "unpenalised",
                                  median_lambda = lambda_final,
                                  median_alpha = alpha_final,
                                  AUC_null = auc_null,
                                  AUC_full_mean_cv =  mean(outer_auc),
                                  AUC_full_sd_cv = sd(outer_auc),
                                  AUC_diff = mean(outer_auc) - auc_null,
                                  sens_null = null_sensitivity,
                                  sens_full_mean_cv = mean(outer_sensitivity),
                                  sens_full_sd_cv = sd(outer_sensitivity),
                                  sens_diff = mean(outer_sensitivity) - null_sensitivity,
                                  spec_null = null_specificity,
                                  spec_full_mean_cv = mean(outer_specificity),
                                  spec_full_sd_cv = sd(outer_specificity),
                                  spec_diff = mean(outer_specificity) - null_specificity,
                                  acc_null = null_accuracy,
                                  acc_full_mean_cv = mean(outer_accuracy),
                                  acc_full_sd_cv = sd(outer_accuracy),
                                  acc_diff = mean(outer_accuracy) - null_accuracy)
    

all_performance_results <- rbind(all_performance_results,performance_results)


}
                  
print(all_performance_results)
                  

write.table(all_performance_results, outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)

#### Test in PPMI (EUR), PPMI (AJ) and HBS (EUR)

In [ ]:
%%R

library(data.table)
library(dplyr)
library(glmnet)
library(caret)
library(pROC)

set.seed(1)

# Read in the data
clinical <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt") %>% 
  select(participant_id, case_control_other_at_baseline, sex, age_at_baseline)

# Read in the PGS
PRS_PDBP_EUR <- fread("/home/jupyter/multiTRS/geno/PRS_scores_out/PDBP_EUR_PD_PGS_resid.txt")
PRS_PPMI_EUR <- fread("/home/jupyter/multiTRS/geno/PRS_scores_out/PPMI_EUR_PD_PGS_resid.txt")
PRS_PPMI_AJ  <- fread("/home/jupyter/multiTRS/geno/PRS_scores_out/PPMI_AJ_PD_PGS_resid.txt")
PRS_HBS_EUR  <- fread("/home/jupyter/multiTRS/geno/PRS_scores_out/HBS_EUR_PD_PGS_resid.txt")

# Read TRS scores - select columns based on PDBP first
TRS_PDBP_EUR <- fread("/home/jupyter/multiTRS/scores/PDBP_EUR_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt") %>% 
  select(participant_id, contains("SMR"), -contains("single_SNP"))

TRS_PPMI_EUR <- fread("/home/jupyter/multiTRS/scores/PPMI_EUR_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt") %>% 
  select(all_of(colnames(TRS_PDBP_EUR)))

TRS_PPMI_AJ  <- fread("/home/jupyter/multiTRS/scores/PPMI_AJ_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt") %>%  
  select(all_of(colnames(TRS_PDBP_EUR)))

TRS_HBS_EUR  <- fread("/home/jupyter/multiTRS/scores/HBS_EUR_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt") %>% 
  select(all_of(colnames(TRS_PDBP_EUR)))

# -----------------------------------------------
# Create combined datasets
# -----------------------------------------------

# PDBP (training)
combined_data <- clinical %>%
  inner_join(PRS_PDBP_EUR, by = "participant_id") %>%
  inner_join(TRS_PDBP_EUR, by = "participant_id")

# PPMI_EUR (test)
combined_data_PPMI_EUR <- clinical %>%
  inner_join(PRS_PPMI_EUR, by = "participant_id") %>%
  inner_join(TRS_PPMI_EUR, by = "participant_id")

# PPMI_AJ (test)
combined_data_PPMI_AJ <- clinical %>%
  inner_join(PRS_PPMI_AJ, by = "participant_id") %>%
  inner_join(TRS_PPMI_AJ, by = "participant_id")

# HBS_EUR (test)
combined_data_HBS_EUR <- clinical %>%
  inner_join(PRS_HBS_EUR, by = "participant_id") %>%
  inner_join(TRS_HBS_EUR, by = "participant_id")

cat("Testing SMR-multi model...\n")
cat("Rows in combined data - PDBP:", nrow(combined_data), "\n")
cat("Rows in combined data - PPMI_EUR:", nrow(combined_data_PPMI_EUR), "\n")
cat("Rows in combined data - PPMI_AJ:", nrow(combined_data_PPMI_AJ), "\n")
cat("Rows in combined data - HBS_EUR:", nrow(combined_data_HBS_EUR), "\n")

# -----------------------------------------------
# Null model (age + sex + PGS), fit on PDBP, applied to all
# -----------------------------------------------

null_model <- glm(case_control_other_at_baseline ~ scale(age_at_baseline) + sex + scale(PD_PGS_resid),
                  data = combined_data,
                  family = binomial)

# Null model predictions
null_probs_PDBP     <- predict(null_model, newdata = combined_data,          type = "response")
null_probs_PPMI_EUR <- predict(null_model, newdata = combined_data_PPMI_EUR, type = "response")
null_probs_PPMI_AJ  <- predict(null_model, newdata = combined_data_PPMI_AJ,  type = "response")
null_probs_HBS_EUR  <- predict(null_model, newdata = combined_data_HBS_EUR,  type = "response")

auc_null_PDBP     <- auc(roc(combined_data$case_control_other_at_baseline,          null_probs_PDBP))
auc_null_PPMI_EUR <- auc(roc(combined_data_PPMI_EUR$case_control_other_at_baseline, null_probs_PPMI_EUR))
auc_null_PPMI_AJ  <- auc(roc(combined_data_PPMI_AJ$case_control_other_at_baseline,  null_probs_PPMI_AJ))
auc_null_HBS_EUR  <- auc(roc(combined_data_HBS_EUR$case_control_other_at_baseline,  null_probs_HBS_EUR))

cat("Null model AUCs:\n")
cat("  PDBP:", auc_null_PDBP, "\n")
cat("  PPMI_EUR:", auc_null_PPMI_EUR, "\n")
cat("  PPMI_AJ:", auc_null_PPMI_AJ, "\n")
cat("  HBS_EUR:", auc_null_HBS_EUR, "\n")

# -----------------------------------------------
# Prepare predictor matrices
# -----------------------------------------------

X <- combined_data %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y <- as.factor(combined_data$case_control_other_at_baseline)

X_PPMI_EUR <- combined_data_PPMI_EUR %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y_PPMI_EUR <- as.factor(combined_data_PPMI_EUR$case_control_other_at_baseline)

X_PPMI_AJ <- combined_data_PPMI_AJ %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y_PPMI_AJ <- as.factor(combined_data_PPMI_AJ$case_control_other_at_baseline)

X_HBS_EUR <- combined_data_HBS_EUR %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y_HBS_EUR <- as.factor(combined_data_HBS_EUR$case_control_other_at_baseline)

cat("Number of predictors:", ncol(X), "\n")

# -----------------------------------------------
# Scale on PDBP, apply to all
# -----------------------------------------------

penalty <- ifelse(colnames(X) %in% c("PD_PGS_resid", "sex", "age_at_baseline"), 0, 1)
vars_to_scale <- setdiff(colnames(X), "sex")

scaler_full <- preProcess(X[, vars_to_scale], method = c("center", "scale"))

X_scaled          <- X
X_scaled[, vars_to_scale] <- predict(scaler_full, X[, vars_to_scale])

X_PPMI_EUR_scaled <- X_PPMI_EUR
X_PPMI_EUR_scaled[, vars_to_scale] <- predict(scaler_full, X_PPMI_EUR[, vars_to_scale])

X_PPMI_AJ_scaled  <- X_PPMI_AJ
X_PPMI_AJ_scaled[, vars_to_scale]  <- predict(scaler_full, X_PPMI_AJ[, vars_to_scale])

X_HBS_EUR_scaled  <- X_HBS_EUR
X_HBS_EUR_scaled[, vars_to_scale]  <- predict(scaler_full, X_HBS_EUR[, vars_to_scale])

# -----------------------------------------------
# Fit final LASSO model on PDBP
# -----------------------------------------------

final_model <- glmnet(
  x = X_scaled,
  y = y,
  family = "binomial",
  alpha = 0.9,
  lambda = 0.04783882,
  penalty.factor = penalty,
  standardize = FALSE
)

# Extract and print coefficients
coefs <- coef(final_model)
coef_df <- data.frame(
  feature = rownames(coefs),
  coefficient = as.numeric(coefs)
)

coef_df <- coef_df %>% arrange(desc(coefficient))

print(coef_df)

coef_outfile <- "/home/jupyter/multiTRS/results/multi_TRS_w_PRS_SMR_multi_ENET_PDBP_full_model_coefficients.txt"
write.table(coef_df, coef_outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)
cat("Coefficients saved to:", coef_outfile, "\n")

# -----------------------------------------------
# Predictions and AUCs
# -----------------------------------------------

get_auc <- function(model, X_new, y_new) {
  probs <- predict(model, newx = X_new, type = "response")
  auc(roc(y_new, as.numeric(probs)))
}

auc_PDBP     <- get_auc(final_model, X_scaled,          y)
auc_PPMI_EUR <- get_auc(final_model, X_PPMI_EUR_scaled, y_PPMI_EUR)
auc_PPMI_AJ  <- get_auc(final_model, X_PPMI_AJ_scaled,  y_PPMI_AJ)
auc_HBS_EUR  <- get_auc(final_model, X_HBS_EUR_scaled,  y_HBS_EUR)

cat("Full model AUCs:\n")
cat("  PDBP:", auc_PDBP, "\n")
cat("  PPMI_EUR:", auc_PPMI_EUR, "\n")
cat("  PPMI_AJ:", auc_PPMI_AJ, "\n")
cat("  HBS_EUR:", auc_HBS_EUR, "\n")

# -----------------------------------------------
# ROC objects (created before CIs and plots)
# -----------------------------------------------

probs_PPMI_EUR <- predict(final_model, newx = X_PPMI_EUR_scaled, type = "response")
probs_PPMI_AJ  <- predict(final_model, newx = X_PPMI_AJ_scaled,  type = "response")
probs_HBS_EUR  <- predict(final_model, newx = X_HBS_EUR_scaled,  type = "response")

roc_null_PPMI_EUR <- roc(y_PPMI_EUR, null_probs_PPMI_EUR)
roc_full_PPMI_EUR <- roc(y_PPMI_EUR, as.numeric(probs_PPMI_EUR))

roc_null_PPMI_AJ  <- roc(y_PPMI_AJ,  null_probs_PPMI_AJ)
roc_full_PPMI_AJ  <- roc(y_PPMI_AJ,  as.numeric(probs_PPMI_AJ))

roc_null_HBS_EUR  <- roc(y_HBS_EUR,  null_probs_HBS_EUR)
roc_full_HBS_EUR  <- roc(y_HBS_EUR,  as.numeric(probs_HBS_EUR))

# -----------------------------------------------
# CIs for AUCs
# -----------------------------------------------

ci_null_PPMI_EUR <- ci.auc(roc_null_PPMI_EUR)
ci_null_PPMI_AJ  <- ci.auc(roc_null_PPMI_AJ)
ci_null_HBS_EUR  <- ci.auc(roc_null_HBS_EUR)

ci_full_PPMI_EUR <- ci.auc(roc_full_PPMI_EUR)
ci_full_PPMI_AJ  <- ci.auc(roc_full_PPMI_AJ)
ci_full_HBS_EUR  <- ci.auc(roc_full_HBS_EUR)

cat("Null model AUCs with 95% CI:\n")
cat("  PPMI_EUR:", round(auc_null_PPMI_EUR, 3), "(95% CI:", round(ci_null_PPMI_EUR[1], 3), "-", round(ci_null_PPMI_EUR[3], 3), ")\n")
cat("  PPMI_AJ:",  round(auc_null_PPMI_AJ, 3),  "(95% CI:", round(ci_null_PPMI_AJ[1], 3),  "-", round(ci_null_PPMI_AJ[3], 3),  ")\n")
cat("  HBS_EUR:",  round(auc_null_HBS_EUR, 3),   "(95% CI:", round(ci_null_HBS_EUR[1], 3),  "-", round(ci_null_HBS_EUR[3], 3),  ")\n")

cat("Full model AUCs with 95% CI:\n")
cat("  PPMI_EUR:", round(auc_PPMI_EUR, 3), "(95% CI:", round(ci_full_PPMI_EUR[1], 3), "-", round(ci_full_PPMI_EUR[3], 3), ")\n")
cat("  PPMI_AJ:",  round(auc_PPMI_AJ, 3),  "(95% CI:", round(ci_full_PPMI_AJ[1], 3),  "-", round(ci_full_PPMI_AJ[3], 3),  ")\n")
cat("  HBS_EUR:",  round(auc_HBS_EUR, 3),   "(95% CI:", round(ci_full_HBS_EUR[1], 3),  "-", round(ci_full_HBS_EUR[3], 3),  ")\n")

# -----------------------------------------------
# CI bands for plotting (ci.se required for ci.type="shape")
# -----------------------------------------------

ci_se_null_PPMI_EUR <- ci.se(roc_null_PPMI_EUR, specificities = seq(0, 1, 0.01))
ci_se_full_PPMI_EUR <- ci.se(roc_full_PPMI_EUR, specificities = seq(0, 1, 0.01))

ci_se_null_PPMI_AJ  <- ci.se(roc_null_PPMI_AJ,  specificities = seq(0, 1, 0.01))
ci_se_full_PPMI_AJ  <- ci.se(roc_full_PPMI_AJ,  specificities = seq(0, 1, 0.01))

ci_se_null_HBS_EUR  <- ci.se(roc_null_HBS_EUR,  specificities = seq(0, 1, 0.01))
ci_se_full_HBS_EUR  <- ci.se(roc_full_HBS_EUR,  specificities = seq(0, 1, 0.01))

# -----------------------------------------------
# ROC plots with CI shapes
# -----------------------------------------------

plot_roc <- function(roc_null, roc_full, ci_se_null, ci_se_full, ci_null, ci_full, title, full_col) {

  # Plot null model ROC
  plot.roc(roc_null, col = "black", lwd = 2, main = title,
           legacy.axes = TRUE, print.auc = FALSE)

  # Add CI shape for null model
  plot(ci_se_null, type = "shape",
       col = adjustcolor("black", alpha.f = 0.1), border = NA)

  # Add full model ROC
  plot.roc(roc_full, add = TRUE, col = full_col, lwd = 2)

  # Add CI shape for full model
  plot(ci_se_full, type = "shape",
       col = adjustcolor(full_col, alpha.f = 0.15), border = NA)

  # Redraw ROC lines on top of shading
  lines.roc(roc_null, col = "black", lwd = 2)
  lines.roc(roc_full, col = full_col, lwd = 2)

  # Legend with AUC + 95% CI
  legend("bottomright",
         legend = c(
           paste0("Age + Sex + PD-PGS (AUC = ", round(auc(roc_null), 3),
                  " [", round(ci_null[1], 3), "\u2013", round(ci_null[3], 3), "])"),
           paste0("Full Model (AUC = ", round(auc(roc_full), 3),
                  " [", round(ci_full[1], 3), "\u2013", round(ci_full[3], 3), "])")
         ),
         col = c("black", full_col), lwd = 2, bty = "n")
}

plot_roc(roc_null_PPMI_EUR, roc_full_PPMI_EUR, ci_se_null_PPMI_EUR, ci_se_full_PPMI_EUR, ci_null_PPMI_EUR, ci_full_PPMI_EUR, "PPMI EUR: Baseline vs Full Model", "goldenrod")
plot_roc(roc_null_PPMI_AJ,  roc_full_PPMI_AJ,  ci_se_null_PPMI_AJ,  ci_se_full_PPMI_AJ,  ci_null_PPMI_AJ,  ci_full_PPMI_AJ,  "PPMI AJ: Baseline vs Full Model",  "forestgreen")
plot_roc(roc_null_HBS_EUR,  roc_full_HBS_EUR,  ci_se_null_HBS_EUR,  ci_se_full_HBS_EUR,  ci_null_HBS_EUR,  ci_full_HBS_EUR,  "HBS EUR: Baseline vs Full Model",  "dodgerblue")

# -----------------------------------------------
# DeLong tests
# -----------------------------------------------

delong_PPMI_EUR <- roc.test(roc_null_PPMI_EUR, roc_full_PPMI_EUR, method = "delong")
delong_PPMI_AJ  <- roc.test(roc_null_PPMI_AJ,  roc_full_PPMI_AJ,  method = "delong")
delong_HBS_EUR  <- roc.test(roc_null_HBS_EUR,  roc_full_HBS_EUR,  method = "delong")

cat("DeLong test p-values:\n")
cat("  PPMI_EUR:", delong_PPMI_EUR$p.value, "\n")
cat("  PPMI_AJ:",  delong_PPMI_AJ$p.value,  "\n")
cat("  HBS_EUR:",  delong_HBS_EUR$p.value,  "\n")

# -----------------------------------------------
# Extract sensitivity, specificity, accuracy at
# optimal threshold (closest-to-top-left)
# -----------------------------------------------

coords_null_PPMI_EUR <- coords(roc_null_PPMI_EUR, x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]
coords_full_PPMI_EUR <- coords(roc_full_PPMI_EUR, x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]

coords_null_PPMI_AJ  <- coords(roc_null_PPMI_AJ,  x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]
coords_full_PPMI_AJ  <- coords(roc_full_PPMI_AJ,  x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]

coords_null_HBS_EUR  <- coords(roc_null_HBS_EUR,  x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]
coords_full_HBS_EUR  <- coords(roc_full_HBS_EUR,  x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]

# -----------------------------------------------
# Build results table
# -----------------------------------------------

results <- rbindlist(list(
  data.table(
    cohort         = "PPMI_EUR",
    AUC_null       = as.numeric(auc(roc_null_PPMI_EUR)),
    AUC_lower_null = as.numeric(ci_null_PPMI_EUR[1]),
    AUC_upper_null = as.numeric(ci_null_PPMI_EUR[3]),
    AUC_full       = as.numeric(auc(roc_full_PPMI_EUR)),
    AUC_lower_full = as.numeric(ci_full_PPMI_EUR[1]),
    AUC_upper_full = as.numeric(ci_full_PPMI_EUR[3]),
    AUC_diff       = as.numeric(auc(roc_full_PPMI_EUR)) - as.numeric(auc(roc_null_PPMI_EUR)),
    sens_null      = coords_null_PPMI_EUR$sensitivity,
    sens_full      = coords_full_PPMI_EUR$sensitivity,
    sens_diff      = coords_full_PPMI_EUR$sensitivity - coords_null_PPMI_EUR$sensitivity,
    spec_null      = coords_null_PPMI_EUR$specificity,
    spec_full      = coords_full_PPMI_EUR$specificity,
    spec_diff      = coords_full_PPMI_EUR$specificity - coords_null_PPMI_EUR$specificity,
    acc_null       = coords_null_PPMI_EUR$accuracy,
    acc_full       = coords_full_PPMI_EUR$accuracy,
    acc_diff       = coords_full_PPMI_EUR$accuracy - coords_null_PPMI_EUR$accuracy,
    delong_z       = delong_PPMI_EUR$statistic,
    delong_p       = delong_PPMI_EUR$p.value
  ),
  data.table(
    cohort         = "PPMI_AJ",
    AUC_null       = as.numeric(auc(roc_null_PPMI_AJ)),
    AUC_lower_null = as.numeric(ci_null_PPMI_AJ[1]),
    AUC_upper_null = as.numeric(ci_null_PPMI_AJ[3]),
    AUC_full       = as.numeric(auc(roc_full_PPMI_AJ)),
    AUC_lower_full = as.numeric(ci_full_PPMI_AJ[1]),
    AUC_upper_full = as.numeric(ci_full_PPMI_AJ[3]),
    AUC_diff       = as.numeric(auc(roc_full_PPMI_AJ)) - as.numeric(auc(roc_null_PPMI_AJ)),
    sens_null      = coords_null_PPMI_AJ$sensitivity,
    sens_full      = coords_full_PPMI_AJ$sensitivity,
    sens_diff      = coords_full_PPMI_AJ$sensitivity - coords_null_PPMI_AJ$sensitivity,
    spec_null      = coords_null_PPMI_AJ$specificity,
    spec_full      = coords_full_PPMI_AJ$specificity,
    spec_diff      = coords_full_PPMI_AJ$specificity - coords_null_PPMI_AJ$specificity,
    acc_null       = coords_null_PPMI_AJ$accuracy,
    acc_full       = coords_full_PPMI_AJ$accuracy,
    acc_diff       = coords_full_PPMI_AJ$accuracy - coords_null_PPMI_AJ$accuracy,
    delong_z       = delong_PPMI_AJ$statistic,
    delong_p       = delong_PPMI_AJ$p.value
  ),
  data.table(
    cohort         = "HBS_EUR",
    AUC_null       = as.numeric(auc(roc_null_HBS_EUR)),
    AUC_lower_null = as.numeric(ci_null_HBS_EUR[1]),
    AUC_upper_null = as.numeric(ci_null_HBS_EUR[3]),
    AUC_full       = as.numeric(auc(roc_full_HBS_EUR)),
    AUC_lower_full = as.numeric(ci_full_HBS_EUR[1]),
    AUC_upper_full = as.numeric(ci_full_HBS_EUR[3]),
    AUC_diff       = as.numeric(auc(roc_full_HBS_EUR)) - as.numeric(auc(roc_null_HBS_EUR)),
    sens_null      = coords_null_HBS_EUR$sensitivity,
    sens_full      = coords_full_HBS_EUR$sensitivity,
    sens_diff      = coords_full_HBS_EUR$sensitivity - coords_null_HBS_EUR$sensitivity,
    spec_null      = coords_null_HBS_EUR$specificity,
    spec_full      = coords_full_HBS_EUR$specificity,
    spec_diff      = coords_full_HBS_EUR$specificity - coords_null_HBS_EUR$specificity,
    acc_null       = coords_null_HBS_EUR$accuracy,
    acc_full       = coords_full_HBS_EUR$accuracy,
    acc_diff       = coords_full_HBS_EUR$accuracy - coords_null_HBS_EUR$accuracy,
    delong_z       = delong_HBS_EUR$statistic,
    delong_p       = delong_HBS_EUR$p.value
  )
))

print(results)

results_outfile <- "/home/jupyter/multiTRS/results/multi_TRS_w_PRS_SMR_multi_ENET_external_validation_results_table.txt"
write.table(results, results_outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)
cat("Results table saved to:", results_outfile, "\n")

### SMR

#### Nested-CV

In [ ]:
%%R

library(data.table)
library(dplyr)
library(glmnet)
library(glmnetUtils)
library(caret)
library(pROC)

set.seed(1)

# Create a data.table for all performance results
all_performance_results <- data.table()

print("This script calculates ENET multi-TRS models using SMR-single-SNP scores... Running ENET models")

outfile <- "/home/jupyter/multiTRS/results/multi_TRS_w_PRS_SMR_single_SNP_ENET_nestedcv.txt"

# Read in the clinical data
clinical <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt") %>%
  select(participant_id, case_control_other_at_baseline, sex, age_at_baseline)

# List of cohorts to loop over
TRS_cohorts <- c("PDBP")

# Loop over cohorts
for (TRS_cohort in TRS_cohorts) {

  # Read TRS scores
  TRS_path <- paste0("/home/jupyter/multiTRS/scores/", TRS_cohort, "_TWAS_SMR_FDR_scores_resid.txt")
  TRS <- fread(TRS_path) %>% select(participant_id, contains("single_SNP"),-contains("Hip_Fracture"))

  # Read PGS scores
  PGS_path <- paste0("/home/jupyter/multiTRS/geno/PRS_scores_out/", TRS_cohort, "_EUR_PD_PGS_resid.txt")
  PGS <- fread(PGS_path)

  combined_data <- clinical %>%
    inner_join(PGS, by = "participant_id") %>%
    inner_join(TRS, by = "participant_id")

  cat("Rows in combined data for", TRS_cohort, ":", nrow(combined_data), "\n")

  # Null model with age + sex
  null_model <- glm(case_control_other_at_baseline ~ sex + scale(age_at_baseline) + scale(PD_PGS_resid),
                    data = combined_data,
                    family = binomial)
  probs_null <- predict(null_model, type = "response")
  roc_obj_null <- roc(combined_data$case_control_other_at_baseline, probs_null)
  auc_null <- auc(roc_obj_null)
  cat("AUC of null model:", auc_null, "\n")

  null_coords <- coords(roc_obj_null, x = "best", best.method = "closest.topleft",
                        ret = c("sensitivity", "specificity", "accuracy"))
  null_sensitivity <- as.numeric(null_coords["sensitivity"])
  null_specificity <- as.numeric(null_coords["specificity"])
  null_accuracy <- as.numeric(null_coords["accuracy"])

  # Prepare predictor matrix X and outcome y
  X <- combined_data %>%
    select(-participant_id, -case_control_other_at_baseline) %>%
    as.matrix()
  y <- as.factor(combined_data$case_control_other_at_baseline)

  cat("Number of predictors:", ncol(X), "\n")

  # Define penalty factors: unpenalized for sex, age and PD_PGS_resid
  penalty <- ifelse(colnames(X) %in% c("sex", "age_at_baseline", "PD_PGS_resid"), 0, 1)

# Create five outer folds
outer_folds <- createFolds(y, k = 5, returnTrain = TRUE)

# Store auc of outer folders
outer_auc <- c()
lambda_vals <- c()
alpha_vals  <- c()
outer_sensitivity <- c()
outer_specificity <- c()
outer_accuracy <- c()

for (i in 1:length(outer_folds)) {
  cat("Outer fold:", i, "\n")
  
  # Train/test split for this outer fold
  train_idx <- outer_folds[[i]]
  test_idx  <- setdiff(seq_along(y), train_idx)
  
  X_train <- X[train_idx, ]
  y_train <- y[train_idx]
  X_test  <- X[test_idx, ]
  y_test  <- y[test_idx]
  
  # Standardize inside training only
    vars_to_scale <- setdiff(colnames(X), "sex")
    scaler <- preProcess(X_train[, vars_to_scale, drop = FALSE], method = c("center", "scale"))
    
    X_train_scaled <- X_train
    X_train_scaled[, vars_to_scale] <- predict(scaler, X_train[, vars_to_scale, drop = FALSE])
    
    X_test_scaled <- X_test
    X_test_scaled[, vars_to_scale] <- predict(scaler, X_test[, vars_to_scale, drop = FALSE])
    
 # Specify alpha list 
  alphalist <- seq(0,1,by=0.1)
  
  # Inner CV with cv.glmnet (5-fold)
  cvfit <- glmnetUtils::cva.glmnet(
    x = X_train_scaled,
    y = y_train,
    family = "binomial",
    alpha = alphalist,
    nfolds = 5,
    type.measure = "auc",
    penalty.factor = penalty,
    standardize = FALSE
  )
  

    
  # Select best alpha and lambda
  lambda_1se <- sapply(cvfit$modlist, `[[`, "lambda.1se")
  error <- sapply(cvfit$modlist, function(mod) {
    idx <- which(mod$lambda == mod$lambda.1se)
    mod$cvm[idx]
  })
  best <- which.max(error)
  best_alpha  <- cvfit$alpha[best]
  best_lambda <- lambda_1se[best]
                  

  # Best lambda chosen inside
  lambda_vals[i] <- best_lambda
  alpha_vals[i]  <- best_alpha
    
  
  # Refit on full training set with best lambda
  final_model <- glmnet(
    x = X_train_scaled,
    y = y_train,
    family = "binomial",
    alpha = best_alpha,
    lambda = best_lambda,
    penalty.factor = penalty,
    standardize = FALSE
  )
  
  # Predict probabilities on outer test set
  probs <- predict(final_model, newx = X_test_scaled, type = "response")
  
# Compute ROC curve
  roc_obj_outer <- roc(y_test, as.numeric(probs))

# Compute AUC
  auc_val <- auc(roc_obj_outer)

# Get sens, spec, accuracy using topleft
model_coords <- coords(
  roc_obj_outer,
  x = "best",
  best.method = "closest.topleft",
  ret = c("threshold", "sensitivity", "specificity", "accuracy")
)

model_sensitivity <- as.numeric(model_coords["sensitivity"])
model_specificity <- as.numeric(model_coords["specificity"])
model_accuracy <- as.numeric(model_coords["accuracy"])
    
  outer_auc[i] <- auc_val
  outer_sensitivity[i] <- model_sensitivity
  outer_specificity[i] <- model_specificity
  outer_accuracy[i] <- model_accuracy
}

alpha_vals <- as.numeric(alpha_vals)
lambda_vals <- as.numeric(lambda_vals)

cat("Alpha per outer fold:\n")
print(alpha_vals)

cat("Lambda per outer fold:\n")
print(lambda_vals)
                  
lambda_final <- median(lambda_vals)
alpha_final <- median(alpha_vals)


cat("AUC per outer fold:\n")
print(outer_auc)
cat("Mean AUC across outer folds:", mean(outer_auc), "\n")
cat("SD of the AUC across outer folds:", sd(outer_auc), "\n")

cat("Median alpha:\n")
print(alpha_final)
cat("Median lambda:\n")
print(lambda_final)


                  
performance_results <- data.table(cohort = TRS_cohort,
                                  scores = "SMR-multi",
                                  model = "ENET",
                                  covariate_treatment = "unpenalised",
                                  median_lambda = lambda_final,
                                  median_alpha = alpha_final,
                                  AUC_null = auc_null,
                                  AUC_full_mean_cv =  mean(outer_auc),
                                  AUC_full_sd_cv = sd(outer_auc),
                                  AUC_diff = mean(outer_auc) - auc_null,
                                  sens_null = null_sensitivity,
                                  sens_full_mean_cv = mean(outer_sensitivity),
                                  sens_full_sd_cv = sd(outer_sensitivity),
                                  sens_diff = mean(outer_sensitivity) - null_sensitivity,
                                  spec_null = null_specificity,
                                  spec_full_mean_cv = mean(outer_specificity),
                                  spec_full_sd_cv = sd(outer_specificity),
                                  spec_diff = mean(outer_specificity) - null_specificity,
                                  acc_null = null_accuracy,
                                  acc_full_mean_cv = mean(outer_accuracy),
                                  acc_full_sd_cv = sd(outer_accuracy),
                                  acc_diff = mean(outer_accuracy) - null_accuracy)
    

all_performance_results <- rbind(all_performance_results,performance_results)


}
                  
print(all_performance_results)
                  

write.table(all_performance_results, outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)

#### Test in PPMI (EUR), PPMI (AJ) and HBS (EUR)

In [ ]:
%%R

library(data.table)
library(dplyr)
library(glmnet)
library(caret)
library(pROC)

set.seed(1)

# Read in the data
clinical <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt") %>% 
  select(participant_id, case_control_other_at_baseline, sex, age_at_baseline)

# Read in the PGS
PRS_PDBP_EUR <- fread("/home/jupyter/multiTRS/geno/PRS_scores_out/PDBP_EUR_PD_PGS_resid.txt")
PRS_PPMI_EUR <- fread("/home/jupyter/multiTRS/geno/PRS_scores_out/PPMI_EUR_PD_PGS_resid.txt")
PRS_PPMI_AJ  <- fread("/home/jupyter/multiTRS/geno/PRS_scores_out/PPMI_AJ_PD_PGS_resid.txt")
PRS_HBS_EUR  <- fread("/home/jupyter/multiTRS/geno/PRS_scores_out/HBS_EUR_PD_PGS_resid.txt")

# Read TRS scores - select columns based on PDBP first
TRS_PDBP_EUR <- fread("/home/jupyter/multiTRS/scores/PDBP_EUR_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt") %>% 
  select(participant_id, contains("single_SNP"),-contains("Hip_Fracture"))

TRS_PPMI_EUR <- fread("/home/jupyter/multiTRS/scores/PPMI_EUR_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt") %>% 
  select(all_of(colnames(TRS_PDBP_EUR)))

TRS_PPMI_AJ  <- fread("/home/jupyter/multiTRS/scores/PPMI_AJ_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt") %>%  
  select(all_of(colnames(TRS_PDBP_EUR)))

TRS_HBS_EUR  <- fread("/home/jupyter/multiTRS/scores/HBS_EUR_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt") %>% 
  select(all_of(colnames(TRS_PDBP_EUR)))

# -----------------------------------------------
# Create combined datasets
# -----------------------------------------------

# PDBP (training)
combined_data <- clinical %>%
  inner_join(PRS_PDBP_EUR, by = "participant_id") %>%
  inner_join(TRS_PDBP_EUR, by = "participant_id")

# PPMI_EUR (test)
combined_data_PPMI_EUR <- clinical %>%
  inner_join(PRS_PPMI_EUR, by = "participant_id") %>%
  inner_join(TRS_PPMI_EUR, by = "participant_id")

# PPMI_AJ (test)
combined_data_PPMI_AJ <- clinical %>%
  inner_join(PRS_PPMI_AJ, by = "participant_id") %>%
  inner_join(TRS_PPMI_AJ, by = "participant_id")

# HBS_EUR (test)
combined_data_HBS_EUR <- clinical %>%
  inner_join(PRS_HBS_EUR, by = "participant_id") %>%
  inner_join(TRS_HBS_EUR, by = "participant_id")

cat("Testing SMR-single-SNP model...\n")
cat("Rows in combined data - PDBP:", nrow(combined_data), "\n")
cat("Rows in combined data - PPMI_EUR:", nrow(combined_data_PPMI_EUR), "\n")
cat("Rows in combined data - PPMI_AJ:", nrow(combined_data_PPMI_AJ), "\n")
cat("Rows in combined data - HBS_EUR:", nrow(combined_data_HBS_EUR), "\n")

# -----------------------------------------------
# Null model (age + sex + PGS), fit on PDBP, applied to all
# -----------------------------------------------

null_model <- glm(case_control_other_at_baseline ~ scale(age_at_baseline) + sex + scale(PD_PGS_resid),
                  data = combined_data,
                  family = binomial)

# Null model predictions
null_probs_PDBP     <- predict(null_model, newdata = combined_data,          type = "response")
null_probs_PPMI_EUR <- predict(null_model, newdata = combined_data_PPMI_EUR, type = "response")
null_probs_PPMI_AJ  <- predict(null_model, newdata = combined_data_PPMI_AJ,  type = "response")
null_probs_HBS_EUR  <- predict(null_model, newdata = combined_data_HBS_EUR,  type = "response")

auc_null_PDBP     <- auc(roc(combined_data$case_control_other_at_baseline,          null_probs_PDBP))
auc_null_PPMI_EUR <- auc(roc(combined_data_PPMI_EUR$case_control_other_at_baseline, null_probs_PPMI_EUR))
auc_null_PPMI_AJ  <- auc(roc(combined_data_PPMI_AJ$case_control_other_at_baseline,  null_probs_PPMI_AJ))
auc_null_HBS_EUR  <- auc(roc(combined_data_HBS_EUR$case_control_other_at_baseline,  null_probs_HBS_EUR))

cat("Null model AUCs:\n")
cat("  PDBP:", auc_null_PDBP, "\n")
cat("  PPMI_EUR:", auc_null_PPMI_EUR, "\n")
cat("  PPMI_AJ:", auc_null_PPMI_AJ, "\n")
cat("  HBS_EUR:", auc_null_HBS_EUR, "\n")

# -----------------------------------------------
# Prepare predictor matrices
# -----------------------------------------------

X <- combined_data %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y <- as.factor(combined_data$case_control_other_at_baseline)

X_PPMI_EUR <- combined_data_PPMI_EUR %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y_PPMI_EUR <- as.factor(combined_data_PPMI_EUR$case_control_other_at_baseline)

X_PPMI_AJ <- combined_data_PPMI_AJ %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y_PPMI_AJ <- as.factor(combined_data_PPMI_AJ$case_control_other_at_baseline)

X_HBS_EUR <- combined_data_HBS_EUR %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y_HBS_EUR <- as.factor(combined_data_HBS_EUR$case_control_other_at_baseline)

cat("Number of predictors:", ncol(X), "\n")

# -----------------------------------------------
# Scale on PDBP, apply to all
# -----------------------------------------------

penalty <- ifelse(colnames(X) %in% c("PD_PGS_resid", "sex", "age_at_baseline"), 0, 1)
vars_to_scale <- setdiff(colnames(X), "sex")

scaler_full <- preProcess(X[, vars_to_scale], method = c("center", "scale"))

X_scaled          <- X
X_scaled[, vars_to_scale] <- predict(scaler_full, X[, vars_to_scale])

X_PPMI_EUR_scaled <- X_PPMI_EUR
X_PPMI_EUR_scaled[, vars_to_scale] <- predict(scaler_full, X_PPMI_EUR[, vars_to_scale])

X_PPMI_AJ_scaled  <- X_PPMI_AJ
X_PPMI_AJ_scaled[, vars_to_scale]  <- predict(scaler_full, X_PPMI_AJ[, vars_to_scale])

X_HBS_EUR_scaled  <- X_HBS_EUR
X_HBS_EUR_scaled[, vars_to_scale]  <- predict(scaler_full, X_HBS_EUR[, vars_to_scale])

# -----------------------------------------------
# Fit final LASSO model on PDBP
# -----------------------------------------------

final_model <- glmnet(
  x = X_scaled,
  y = y,
  family = "binomial",
  alpha = 0,
  lambda = 4.114934,
  penalty.factor = penalty,
  standardize = FALSE
)

# Extract and print coefficients
coefs <- coef(final_model)
coef_df <- data.frame(
  feature = rownames(coefs),
  coefficient = as.numeric(coefs)
)

coef_df <- coef_df %>% arrange(desc(coefficient))

print(coef_df)

coef_outfile <- "/home/jupyter/multiTRS/results/multi_TRS_w_PRS_SMR_single_SNP_ENET_PDBP_full_model_coefficients.txt"
write.table(coef_df, coef_outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)
cat("Coefficients saved to:", coef_outfile, "\n")

# -----------------------------------------------
# Predictions and AUCs
# -----------------------------------------------

get_auc <- function(model, X_new, y_new) {
  probs <- predict(model, newx = X_new, type = "response")
  auc(roc(y_new, as.numeric(probs)))
}

auc_PDBP     <- get_auc(final_model, X_scaled,          y)
auc_PPMI_EUR <- get_auc(final_model, X_PPMI_EUR_scaled, y_PPMI_EUR)
auc_PPMI_AJ  <- get_auc(final_model, X_PPMI_AJ_scaled,  y_PPMI_AJ)
auc_HBS_EUR  <- get_auc(final_model, X_HBS_EUR_scaled,  y_HBS_EUR)

cat("Full model AUCs:\n")
cat("  PDBP:", auc_PDBP, "\n")
cat("  PPMI_EUR:", auc_PPMI_EUR, "\n")
cat("  PPMI_AJ:", auc_PPMI_AJ, "\n")
cat("  HBS_EUR:", auc_HBS_EUR, "\n")

# -----------------------------------------------
# ROC objects (created before CIs and plots)
# -----------------------------------------------

probs_PPMI_EUR <- predict(final_model, newx = X_PPMI_EUR_scaled, type = "response")
probs_PPMI_AJ  <- predict(final_model, newx = X_PPMI_AJ_scaled,  type = "response")
probs_HBS_EUR  <- predict(final_model, newx = X_HBS_EUR_scaled,  type = "response")

roc_null_PPMI_EUR <- roc(y_PPMI_EUR, null_probs_PPMI_EUR)
roc_full_PPMI_EUR <- roc(y_PPMI_EUR, as.numeric(probs_PPMI_EUR))

roc_null_PPMI_AJ  <- roc(y_PPMI_AJ,  null_probs_PPMI_AJ)
roc_full_PPMI_AJ  <- roc(y_PPMI_AJ,  as.numeric(probs_PPMI_AJ))

roc_null_HBS_EUR  <- roc(y_HBS_EUR,  null_probs_HBS_EUR)
roc_full_HBS_EUR  <- roc(y_HBS_EUR,  as.numeric(probs_HBS_EUR))

# -----------------------------------------------
# CIs for AUCs
# -----------------------------------------------

ci_null_PPMI_EUR <- ci.auc(roc_null_PPMI_EUR)
ci_null_PPMI_AJ  <- ci.auc(roc_null_PPMI_AJ)
ci_null_HBS_EUR  <- ci.auc(roc_null_HBS_EUR)

ci_full_PPMI_EUR <- ci.auc(roc_full_PPMI_EUR)
ci_full_PPMI_AJ  <- ci.auc(roc_full_PPMI_AJ)
ci_full_HBS_EUR  <- ci.auc(roc_full_HBS_EUR)

cat("Null model AUCs with 95% CI:\n")
cat("  PPMI_EUR:", round(auc_null_PPMI_EUR, 3), "(95% CI:", round(ci_null_PPMI_EUR[1], 3), "-", round(ci_null_PPMI_EUR[3], 3), ")\n")
cat("  PPMI_AJ:",  round(auc_null_PPMI_AJ, 3),  "(95% CI:", round(ci_null_PPMI_AJ[1], 3),  "-", round(ci_null_PPMI_AJ[3], 3),  ")\n")
cat("  HBS_EUR:",  round(auc_null_HBS_EUR, 3),   "(95% CI:", round(ci_null_HBS_EUR[1], 3),  "-", round(ci_null_HBS_EUR[3], 3),  ")\n")

cat("Full model AUCs with 95% CI:\n")
cat("  PPMI_EUR:", round(auc_PPMI_EUR, 3), "(95% CI:", round(ci_full_PPMI_EUR[1], 3), "-", round(ci_full_PPMI_EUR[3], 3), ")\n")
cat("  PPMI_AJ:",  round(auc_PPMI_AJ, 3),  "(95% CI:", round(ci_full_PPMI_AJ[1], 3),  "-", round(ci_full_PPMI_AJ[3], 3),  ")\n")
cat("  HBS_EUR:",  round(auc_HBS_EUR, 3),   "(95% CI:", round(ci_full_HBS_EUR[1], 3),  "-", round(ci_full_HBS_EUR[3], 3),  ")\n")

# -----------------------------------------------
# CI bands for plotting (ci.se required for ci.type="shape")
# -----------------------------------------------

ci_se_null_PPMI_EUR <- ci.se(roc_null_PPMI_EUR, specificities = seq(0, 1, 0.01))
ci_se_full_PPMI_EUR <- ci.se(roc_full_PPMI_EUR, specificities = seq(0, 1, 0.01))

ci_se_null_PPMI_AJ  <- ci.se(roc_null_PPMI_AJ,  specificities = seq(0, 1, 0.01))
ci_se_full_PPMI_AJ  <- ci.se(roc_full_PPMI_AJ,  specificities = seq(0, 1, 0.01))

ci_se_null_HBS_EUR  <- ci.se(roc_null_HBS_EUR,  specificities = seq(0, 1, 0.01))
ci_se_full_HBS_EUR  <- ci.se(roc_full_HBS_EUR,  specificities = seq(0, 1, 0.01))

# -----------------------------------------------
# ROC plots with CI shapes
# -----------------------------------------------

plot_roc <- function(roc_null, roc_full, ci_se_null, ci_se_full, ci_null, ci_full, title, full_col) {

  # Plot null model ROC
  plot.roc(roc_null, col = "black", lwd = 2, main = title,
           legacy.axes = TRUE, print.auc = FALSE)

  # Add CI shape for null model
  plot(ci_se_null, type = "shape",
       col = adjustcolor("black", alpha.f = 0.1), border = NA)

  # Add full model ROC
  plot.roc(roc_full, add = TRUE, col = full_col, lwd = 2)

  # Add CI shape for full model
  plot(ci_se_full, type = "shape",
       col = adjustcolor(full_col, alpha.f = 0.15), border = NA)

  # Redraw ROC lines on top of shading
  lines.roc(roc_null, col = "black", lwd = 2)
  lines.roc(roc_full, col = full_col, lwd = 2)

  # Legend with AUC + 95% CI
  legend("bottomright",
         legend = c(
           paste0("Age + Sex + PD-PGS (AUC = ", round(auc(roc_null), 3),
                  " [", round(ci_null[1], 3), "\u2013", round(ci_null[3], 3), "])"),
           paste0("Full Model (AUC = ", round(auc(roc_full), 3),
                  " [", round(ci_full[1], 3), "\u2013", round(ci_full[3], 3), "])")
         ),
         col = c("black", full_col), lwd = 2, bty = "n")
}

plot_roc(roc_null_PPMI_EUR, roc_full_PPMI_EUR, ci_se_null_PPMI_EUR, ci_se_full_PPMI_EUR, ci_null_PPMI_EUR, ci_full_PPMI_EUR, "PPMI EUR: Baseline vs Full Model", "goldenrod")
plot_roc(roc_null_PPMI_AJ,  roc_full_PPMI_AJ,  ci_se_null_PPMI_AJ,  ci_se_full_PPMI_AJ,  ci_null_PPMI_AJ,  ci_full_PPMI_AJ,  "PPMI AJ: Baseline vs Full Model",  "forestgreen")
plot_roc(roc_null_HBS_EUR,  roc_full_HBS_EUR,  ci_se_null_HBS_EUR,  ci_se_full_HBS_EUR,  ci_null_HBS_EUR,  ci_full_HBS_EUR,  "HBS EUR: Baseline vs Full Model",  "dodgerblue")

# -----------------------------------------------
# DeLong tests
# -----------------------------------------------

delong_PPMI_EUR <- roc.test(roc_null_PPMI_EUR, roc_full_PPMI_EUR, method = "delong")
delong_PPMI_AJ  <- roc.test(roc_null_PPMI_AJ,  roc_full_PPMI_AJ,  method = "delong")
delong_HBS_EUR  <- roc.test(roc_null_HBS_EUR,  roc_full_HBS_EUR,  method = "delong")

cat("DeLong test p-values:\n")
cat("  PPMI_EUR:", delong_PPMI_EUR$p.value, "\n")
cat("  PPMI_AJ:",  delong_PPMI_AJ$p.value,  "\n")
cat("  HBS_EUR:",  delong_HBS_EUR$p.value,  "\n")

# -----------------------------------------------
# Extract sensitivity, specificity, accuracy at
# optimal threshold (closest-to-top-left)
# -----------------------------------------------

coords_null_PPMI_EUR <- coords(roc_null_PPMI_EUR, x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]
coords_full_PPMI_EUR <- coords(roc_full_PPMI_EUR, x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]

coords_null_PPMI_AJ  <- coords(roc_null_PPMI_AJ,  x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]
coords_full_PPMI_AJ  <- coords(roc_full_PPMI_AJ,  x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]

coords_null_HBS_EUR  <- coords(roc_null_HBS_EUR,  x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]
coords_full_HBS_EUR  <- coords(roc_full_HBS_EUR,  x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]

# -----------------------------------------------
# Build results table
# -----------------------------------------------

results <- rbindlist(list(
  data.table(
    cohort         = "PPMI_EUR",
    AUC_null       = as.numeric(auc(roc_null_PPMI_EUR)),
    AUC_lower_null = as.numeric(ci_null_PPMI_EUR[1]),
    AUC_upper_null = as.numeric(ci_null_PPMI_EUR[3]),
    AUC_full       = as.numeric(auc(roc_full_PPMI_EUR)),
    AUC_lower_full = as.numeric(ci_full_PPMI_EUR[1]),
    AUC_upper_full = as.numeric(ci_full_PPMI_EUR[3]),
    AUC_diff       = as.numeric(auc(roc_full_PPMI_EUR)) - as.numeric(auc(roc_null_PPMI_EUR)),
    sens_null      = coords_null_PPMI_EUR$sensitivity,
    sens_full      = coords_full_PPMI_EUR$sensitivity,
    sens_diff      = coords_full_PPMI_EUR$sensitivity - coords_null_PPMI_EUR$sensitivity,
    spec_null      = coords_null_PPMI_EUR$specificity,
    spec_full      = coords_full_PPMI_EUR$specificity,
    spec_diff      = coords_full_PPMI_EUR$specificity - coords_null_PPMI_EUR$specificity,
    acc_null       = coords_null_PPMI_EUR$accuracy,
    acc_full       = coords_full_PPMI_EUR$accuracy,
    acc_diff       = coords_full_PPMI_EUR$accuracy - coords_null_PPMI_EUR$accuracy,
    delong_z       = delong_PPMI_EUR$statistic,
    delong_p       = delong_PPMI_EUR$p.value
  ),
  data.table(
    cohort         = "PPMI_AJ",
    AUC_null       = as.numeric(auc(roc_null_PPMI_AJ)),
    AUC_lower_null = as.numeric(ci_null_PPMI_AJ[1]),
    AUC_upper_null = as.numeric(ci_null_PPMI_AJ[3]),
    AUC_full       = as.numeric(auc(roc_full_PPMI_AJ)),
    AUC_lower_full = as.numeric(ci_full_PPMI_AJ[1]),
    AUC_upper_full = as.numeric(ci_full_PPMI_AJ[3]),
    AUC_diff       = as.numeric(auc(roc_full_PPMI_AJ)) - as.numeric(auc(roc_null_PPMI_AJ)),
    sens_null      = coords_null_PPMI_AJ$sensitivity,
    sens_full      = coords_full_PPMI_AJ$sensitivity,
    sens_diff      = coords_full_PPMI_AJ$sensitivity - coords_null_PPMI_AJ$sensitivity,
    spec_null      = coords_null_PPMI_AJ$specificity,
    spec_full      = coords_full_PPMI_AJ$specificity,
    spec_diff      = coords_full_PPMI_AJ$specificity - coords_null_PPMI_AJ$specificity,
    acc_null       = coords_null_PPMI_AJ$accuracy,
    acc_full       = coords_full_PPMI_AJ$accuracy,
    acc_diff       = coords_full_PPMI_AJ$accuracy - coords_null_PPMI_AJ$accuracy,
    delong_z       = delong_PPMI_AJ$statistic,
    delong_p       = delong_PPMI_AJ$p.value
  ),
  data.table(
    cohort         = "HBS_EUR",
    AUC_null       = as.numeric(auc(roc_null_HBS_EUR)),
    AUC_lower_null = as.numeric(ci_null_HBS_EUR[1]),
    AUC_upper_null = as.numeric(ci_null_HBS_EUR[3]),
    AUC_full       = as.numeric(auc(roc_full_HBS_EUR)),
    AUC_lower_full = as.numeric(ci_full_HBS_EUR[1]),
    AUC_upper_full = as.numeric(ci_full_HBS_EUR[3]),
    AUC_diff       = as.numeric(auc(roc_full_HBS_EUR)) - as.numeric(auc(roc_null_HBS_EUR)),
    sens_null      = coords_null_HBS_EUR$sensitivity,
    sens_full      = coords_full_HBS_EUR$sensitivity,
    sens_diff      = coords_full_HBS_EUR$sensitivity - coords_null_HBS_EUR$sensitivity,
    spec_null      = coords_null_HBS_EUR$specificity,
    spec_full      = coords_full_HBS_EUR$specificity,
    spec_diff      = coords_full_HBS_EUR$specificity - coords_null_HBS_EUR$specificity,
    acc_null       = coords_null_HBS_EUR$accuracy,
    acc_full       = coords_full_HBS_EUR$accuracy,
    acc_diff       = coords_full_HBS_EUR$accuracy - coords_null_HBS_EUR$accuracy,
    delong_z       = delong_HBS_EUR$statistic,
    delong_p       = delong_HBS_EUR$p.value
  )
))

print(results)

results_outfile <- "/home/jupyter/multiTRS/results/multi_TRS_w_PRS_SMR_single_SNP_ENET_external_validation_results_table.txt"
write.table(results, results_outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)
cat("Results table saved to:", results_outfile, "\n")

### FUSION (TWAS)

#### Nested-CV

In [ ]:
%%R

library(data.table)
library(dplyr)
library(glmnet)
library(glmnetUtils)
library(caret)
library(pROC)

set.seed(1)

# Create a data.table for all performance results
all_performance_results <- data.table()

print("This script calculates ENET multi-TRS models using FUSION scores... Running ENET models")

outfile <- "/home/jupyter/multiTRS/results/multi_TRS_w_PRS_FUSION_ENET_nestedcv.txt"

# Read in the clinical data
clinical <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt") %>%
  select(participant_id, case_control_other_at_baseline, sex, age_at_baseline)

# List of cohorts to loop over
TRS_cohorts <- c("PDBP")

# Loop over cohorts
for (TRS_cohort in TRS_cohorts) {

  # Read TRS scores
  TRS_path <- paste0("/home/jupyter/multiTRS/scores/", TRS_cohort, "_TWAS_SMR_FDR_scores_resid.txt")
  TRS <- fread(TRS_path) %>% select(participant_id, contains("FUSION"))

  # Read PGS scores
  PGS_path <- paste0("/home/jupyter/multiTRS/geno/PRS_scores_out/", TRS_cohort, "_EUR_PD_PGS_resid.txt")
  PGS <- fread(PGS_path)

  combined_data <- clinical %>%
    inner_join(PGS, by = "participant_id") %>%
    inner_join(TRS, by = "participant_id")

  cat("Rows in combined data for", TRS_cohort, ":", nrow(combined_data), "\n")

  # Null model with age + sex
  null_model <- glm(case_control_other_at_baseline ~ sex + scale(age_at_baseline) + scale(PD_PGS_resid),
                    data = combined_data,
                    family = binomial)
  probs_null <- predict(null_model, type = "response")
  roc_obj_null <- roc(combined_data$case_control_other_at_baseline, probs_null)
  auc_null <- auc(roc_obj_null)
  cat("AUC of null model:", auc_null, "\n")

  null_coords <- coords(roc_obj_null, x = "best", best.method = "closest.topleft",
                        ret = c("sensitivity", "specificity", "accuracy"))
  null_sensitivity <- as.numeric(null_coords["sensitivity"])
  null_specificity <- as.numeric(null_coords["specificity"])
  null_accuracy <- as.numeric(null_coords["accuracy"])

  # Prepare predictor matrix X and outcome y
  X <- combined_data %>%
    select(-participant_id, -case_control_other_at_baseline) %>%
    as.matrix()
  y <- as.factor(combined_data$case_control_other_at_baseline)

  cat("Number of predictors:", ncol(X), "\n")

  # Define penalty factors: unpenalized for sex, age and PD_PGS_resid
  penalty <- ifelse(colnames(X) %in% c("sex", "age_at_baseline", "PD_PGS_resid"), 0, 1)

# Create five outer folds
outer_folds <- createFolds(y, k = 5, returnTrain = TRUE)

# Store auc of outer folders
outer_auc <- c()
lambda_vals <- c()
alpha_vals  <- c()
outer_sensitivity <- c()
outer_specificity <- c()
outer_accuracy <- c()

for (i in 1:length(outer_folds)) {
  cat("Outer fold:", i, "\n")
  
  # Train/test split for this outer fold
  train_idx <- outer_folds[[i]]
  test_idx  <- setdiff(seq_along(y), train_idx)
  
  X_train <- X[train_idx, ]
  y_train <- y[train_idx]
  X_test  <- X[test_idx, ]
  y_test  <- y[test_idx]
  
  # Standardize inside training only
    vars_to_scale <- setdiff(colnames(X), "sex")
    scaler <- preProcess(X_train[, vars_to_scale, drop = FALSE], method = c("center", "scale"))
    
    X_train_scaled <- X_train
    X_train_scaled[, vars_to_scale] <- predict(scaler, X_train[, vars_to_scale, drop = FALSE])
    
    X_test_scaled <- X_test
    X_test_scaled[, vars_to_scale] <- predict(scaler, X_test[, vars_to_scale, drop = FALSE])
    
 # Specify alpha list 
  alphalist <- seq(0,1,by=0.1)
  
  # Inner CV with cv.glmnet (5-fold)
  cvfit <- glmnetUtils::cva.glmnet(
    x = X_train_scaled,
    y = y_train,
    family = "binomial",
    alpha = alphalist,
    nfolds = 5,
    type.measure = "auc",
    penalty.factor = penalty,
    standardize = FALSE
  )
  

    
  # Select best alpha and lambda
  lambda_1se <- sapply(cvfit$modlist, `[[`, "lambda.1se")
  error <- sapply(cvfit$modlist, function(mod) {
    idx <- which(mod$lambda == mod$lambda.1se)
    mod$cvm[idx]
  })
  best <- which.max(error)
  best_alpha  <- cvfit$alpha[best]
  best_lambda <- lambda_1se[best]
                  

  # Best lambda chosen inside
  lambda_vals[i] <- best_lambda
  alpha_vals[i]  <- best_alpha
    
  
  # Refit on full training set with best lambda
  final_model <- glmnet(
    x = X_train_scaled,
    y = y_train,
    family = "binomial",
    alpha = best_alpha,
    lambda = best_lambda,
    penalty.factor = penalty,
    standardize = FALSE
  )
  
  # Predict probabilities on outer test set
  probs <- predict(final_model, newx = X_test_scaled, type = "response")
  
# Compute ROC curve
  roc_obj_outer <- roc(y_test, as.numeric(probs))

# Compute AUC
  auc_val <- auc(roc_obj_outer)

# Get sens, spec, accuracy using topleft
model_coords <- coords(
  roc_obj_outer,
  x = "best",
  best.method = "closest.topleft",
  ret = c("threshold", "sensitivity", "specificity", "accuracy")
)

model_sensitivity <- as.numeric(model_coords["sensitivity"])
model_specificity <- as.numeric(model_coords["specificity"])
model_accuracy <- as.numeric(model_coords["accuracy"])
    
  outer_auc[i] <- auc_val
  outer_sensitivity[i] <- model_sensitivity
  outer_specificity[i] <- model_specificity
  outer_accuracy[i] <- model_accuracy
}

alpha_vals <- as.numeric(alpha_vals)
lambda_vals <- as.numeric(lambda_vals)

cat("Alpha per outer fold:\n")
print(alpha_vals)

cat("Lambda per outer fold:\n")
print(lambda_vals)
                  
lambda_final <- median(lambda_vals)
alpha_final <- median(alpha_vals)


cat("AUC per outer fold:\n")
print(outer_auc)
cat("Mean AUC across outer folds:", mean(outer_auc), "\n")
cat("SD of the AUC across outer folds:", sd(outer_auc), "\n")

cat("Median alpha:\n")
print(alpha_final)
cat("Median lambda:\n")
print(lambda_final)


                  
performance_results <- data.table(cohort = TRS_cohort,
                                  scores = "FUSION",
                                  model = "ENET",
                                  covariate_treatment = "unpenalised",
                                  median_lambda = lambda_final,
                                  median_alpha = alpha_final,
                                  AUC_null = auc_null,
                                  AUC_full_mean_cv =  mean(outer_auc),
                                  AUC_full_sd_cv = sd(outer_auc),
                                  AUC_diff = mean(outer_auc) - auc_null,
                                  sens_null = null_sensitivity,
                                  sens_full_mean_cv = mean(outer_sensitivity),
                                  sens_full_sd_cv = sd(outer_sensitivity),
                                  sens_diff = mean(outer_sensitivity) - null_sensitivity,
                                  spec_null = null_specificity,
                                  spec_full_mean_cv = mean(outer_specificity),
                                  spec_full_sd_cv = sd(outer_specificity),
                                  spec_diff = mean(outer_specificity) - null_specificity,
                                  acc_null = null_accuracy,
                                  acc_full_mean_cv = mean(outer_accuracy),
                                  acc_full_sd_cv = sd(outer_accuracy),
                                  acc_diff = mean(outer_accuracy) - null_accuracy)
    

all_performance_results <- rbind(all_performance_results,performance_results)


}
                  
print(all_performance_results)
                  

write.table(all_performance_results, outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)

#### Test in PPMI (EUR), PPMI (AJ) and HBS (EUR)

In [ ]:
%%R

library(data.table)
library(dplyr)
library(glmnet)
library(caret)
library(pROC)

set.seed(1)

# Read in the data
clinical <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt") %>% 
  select(participant_id, case_control_other_at_baseline, sex, age_at_baseline)

# Read in the PGS
PRS_PDBP_EUR <- fread("/home/jupyter/multiTRS/geno/PRS_scores_out/PDBP_EUR_PD_PGS_resid.txt")
PRS_PPMI_EUR <- fread("/home/jupyter/multiTRS/geno/PRS_scores_out/PPMI_EUR_PD_PGS_resid.txt")
PRS_PPMI_AJ  <- fread("/home/jupyter/multiTRS/geno/PRS_scores_out/PPMI_AJ_PD_PGS_resid.txt")
PRS_HBS_EUR  <- fread("/home/jupyter/multiTRS/geno/PRS_scores_out/HBS_EUR_PD_PGS_resid.txt")

# Read TRS scores - select columns based on PDBP first
TRS_PDBP_EUR <- fread("/home/jupyter/multiTRS/scores/PDBP_EUR_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt") %>% 
   select(participant_id, contains("FUSION"))

TRS_PPMI_EUR <- fread("/home/jupyter/multiTRS/scores/PPMI_EUR_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt") %>% 
  select(all_of(colnames(TRS_PDBP_EUR)))

TRS_PPMI_AJ  <- fread("/home/jupyter/multiTRS/scores/PPMI_AJ_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt") %>%  
  select(all_of(colnames(TRS_PDBP_EUR)))

TRS_HBS_EUR  <- fread("/home/jupyter/multiTRS/scores/HBS_EUR_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt") %>% 
  select(all_of(colnames(TRS_PDBP_EUR)))

# -----------------------------------------------
# Create combined datasets
# -----------------------------------------------

# PDBP (training)
combined_data <- clinical %>%
  inner_join(PRS_PDBP_EUR, by = "participant_id") %>%
  inner_join(TRS_PDBP_EUR, by = "participant_id")

# PPMI_EUR (test)
combined_data_PPMI_EUR <- clinical %>%
  inner_join(PRS_PPMI_EUR, by = "participant_id") %>%
  inner_join(TRS_PPMI_EUR, by = "participant_id")

# PPMI_AJ (test)
combined_data_PPMI_AJ <- clinical %>%
  inner_join(PRS_PPMI_AJ, by = "participant_id") %>%
  inner_join(TRS_PPMI_AJ, by = "participant_id")

# HBS_EUR (test)
combined_data_HBS_EUR <- clinical %>%
  inner_join(PRS_HBS_EUR, by = "participant_id") %>%
  inner_join(TRS_HBS_EUR, by = "participant_id")

cat("Testing FUSION model...\n")
cat("Rows in combined data - PDBP:", nrow(combined_data), "\n")
cat("Rows in combined data - PPMI_EUR:", nrow(combined_data_PPMI_EUR), "\n")
cat("Rows in combined data - PPMI_AJ:", nrow(combined_data_PPMI_AJ), "\n")
cat("Rows in combined data - HBS_EUR:", nrow(combined_data_HBS_EUR), "\n")

# -----------------------------------------------
# Null model (age + sex + PGS), fit on PDBP, applied to all
# -----------------------------------------------

null_model <- glm(case_control_other_at_baseline ~ scale(age_at_baseline) + sex + scale(PD_PGS_resid),
                  data = combined_data,
                  family = binomial)

# Null model predictions
null_probs_PDBP     <- predict(null_model, newdata = combined_data,          type = "response")
null_probs_PPMI_EUR <- predict(null_model, newdata = combined_data_PPMI_EUR, type = "response")
null_probs_PPMI_AJ  <- predict(null_model, newdata = combined_data_PPMI_AJ,  type = "response")
null_probs_HBS_EUR  <- predict(null_model, newdata = combined_data_HBS_EUR,  type = "response")

auc_null_PDBP     <- auc(roc(combined_data$case_control_other_at_baseline,          null_probs_PDBP))
auc_null_PPMI_EUR <- auc(roc(combined_data_PPMI_EUR$case_control_other_at_baseline, null_probs_PPMI_EUR))
auc_null_PPMI_AJ  <- auc(roc(combined_data_PPMI_AJ$case_control_other_at_baseline,  null_probs_PPMI_AJ))
auc_null_HBS_EUR  <- auc(roc(combined_data_HBS_EUR$case_control_other_at_baseline,  null_probs_HBS_EUR))

cat("Null model AUCs:\n")
cat("  PDBP:", auc_null_PDBP, "\n")
cat("  PPMI_EUR:", auc_null_PPMI_EUR, "\n")
cat("  PPMI_AJ:", auc_null_PPMI_AJ, "\n")
cat("  HBS_EUR:", auc_null_HBS_EUR, "\n")

# -----------------------------------------------
# Prepare predictor matrices
# -----------------------------------------------

X <- combined_data %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y <- as.factor(combined_data$case_control_other_at_baseline)

X_PPMI_EUR <- combined_data_PPMI_EUR %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y_PPMI_EUR <- as.factor(combined_data_PPMI_EUR$case_control_other_at_baseline)

X_PPMI_AJ <- combined_data_PPMI_AJ %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y_PPMI_AJ <- as.factor(combined_data_PPMI_AJ$case_control_other_at_baseline)

X_HBS_EUR <- combined_data_HBS_EUR %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y_HBS_EUR <- as.factor(combined_data_HBS_EUR$case_control_other_at_baseline)

cat("Number of predictors:", ncol(X), "\n")

# -----------------------------------------------
# Scale on PDBP, apply to all
# -----------------------------------------------

penalty <- ifelse(colnames(X) %in% c("PD_PGS_resid", "sex", "age_at_baseline"), 0, 1)
vars_to_scale <- setdiff(colnames(X), "sex")

scaler_full <- preProcess(X[, vars_to_scale], method = c("center", "scale"))

X_scaled          <- X
X_scaled[, vars_to_scale] <- predict(scaler_full, X[, vars_to_scale])

X_PPMI_EUR_scaled <- X_PPMI_EUR
X_PPMI_EUR_scaled[, vars_to_scale] <- predict(scaler_full, X_PPMI_EUR[, vars_to_scale])

X_PPMI_AJ_scaled  <- X_PPMI_AJ
X_PPMI_AJ_scaled[, vars_to_scale]  <- predict(scaler_full, X_PPMI_AJ[, vars_to_scale])

X_HBS_EUR_scaled  <- X_HBS_EUR
X_HBS_EUR_scaled[, vars_to_scale]  <- predict(scaler_full, X_HBS_EUR[, vars_to_scale])

# -----------------------------------------------
# Fit final LASSO model on PDBP
# -----------------------------------------------

final_model <- glmnet(
  x = X_scaled,
  y = y,
  family = "binomial",
  alpha = 0.1,
  lambda = 0.3764203,
  penalty.factor = penalty,
  standardize = FALSE
)

# Extract and print coefficients
coefs <- coef(final_model)
coef_df <- data.frame(
  feature = rownames(coefs),
  coefficient = as.numeric(coefs)
)

coef_df <- coef_df %>% arrange(desc(coefficient))

print(coef_df)

coef_outfile <- "/home/jupyter/multiTRS/results/multi_TRS_w_PRS_FUSION_ENET_PDBP_full_model_coefficients.txt"
write.table(coef_df, coef_outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)
cat("Coefficients saved to:", coef_outfile, "\n")

# -----------------------------------------------
# Predictions and AUCs
# -----------------------------------------------

get_auc <- function(model, X_new, y_new) {
  probs <- predict(model, newx = X_new, type = "response")
  auc(roc(y_new, as.numeric(probs)))
}

auc_PDBP     <- get_auc(final_model, X_scaled,          y)
auc_PPMI_EUR <- get_auc(final_model, X_PPMI_EUR_scaled, y_PPMI_EUR)
auc_PPMI_AJ  <- get_auc(final_model, X_PPMI_AJ_scaled,  y_PPMI_AJ)
auc_HBS_EUR  <- get_auc(final_model, X_HBS_EUR_scaled,  y_HBS_EUR)

cat("Full model AUCs:\n")
cat("  PDBP:", auc_PDBP, "\n")
cat("  PPMI_EUR:", auc_PPMI_EUR, "\n")
cat("  PPMI_AJ:", auc_PPMI_AJ, "\n")
cat("  HBS_EUR:", auc_HBS_EUR, "\n")

# -----------------------------------------------
# ROC objects (created before CIs and plots)
# -----------------------------------------------

probs_PPMI_EUR <- predict(final_model, newx = X_PPMI_EUR_scaled, type = "response")
probs_PPMI_AJ  <- predict(final_model, newx = X_PPMI_AJ_scaled,  type = "response")
probs_HBS_EUR  <- predict(final_model, newx = X_HBS_EUR_scaled,  type = "response")

roc_null_PPMI_EUR <- roc(y_PPMI_EUR, null_probs_PPMI_EUR)
roc_full_PPMI_EUR <- roc(y_PPMI_EUR, as.numeric(probs_PPMI_EUR))

roc_null_PPMI_AJ  <- roc(y_PPMI_AJ,  null_probs_PPMI_AJ)
roc_full_PPMI_AJ  <- roc(y_PPMI_AJ,  as.numeric(probs_PPMI_AJ))

roc_null_HBS_EUR  <- roc(y_HBS_EUR,  null_probs_HBS_EUR)
roc_full_HBS_EUR  <- roc(y_HBS_EUR,  as.numeric(probs_HBS_EUR))

# -----------------------------------------------
# CIs for AUCs
# -----------------------------------------------

ci_null_PPMI_EUR <- ci.auc(roc_null_PPMI_EUR)
ci_null_PPMI_AJ  <- ci.auc(roc_null_PPMI_AJ)
ci_null_HBS_EUR  <- ci.auc(roc_null_HBS_EUR)

ci_full_PPMI_EUR <- ci.auc(roc_full_PPMI_EUR)
ci_full_PPMI_AJ  <- ci.auc(roc_full_PPMI_AJ)
ci_full_HBS_EUR  <- ci.auc(roc_full_HBS_EUR)

cat("Null model AUCs with 95% CI:\n")
cat("  PPMI_EUR:", round(auc_null_PPMI_EUR, 3), "(95% CI:", round(ci_null_PPMI_EUR[1], 3), "-", round(ci_null_PPMI_EUR[3], 3), ")\n")
cat("  PPMI_AJ:",  round(auc_null_PPMI_AJ, 3),  "(95% CI:", round(ci_null_PPMI_AJ[1], 3),  "-", round(ci_null_PPMI_AJ[3], 3),  ")\n")
cat("  HBS_EUR:",  round(auc_null_HBS_EUR, 3),   "(95% CI:", round(ci_null_HBS_EUR[1], 3),  "-", round(ci_null_HBS_EUR[3], 3),  ")\n")

cat("Full model AUCs with 95% CI:\n")
cat("  PPMI_EUR:", round(auc_PPMI_EUR, 3), "(95% CI:", round(ci_full_PPMI_EUR[1], 3), "-", round(ci_full_PPMI_EUR[3], 3), ")\n")
cat("  PPMI_AJ:",  round(auc_PPMI_AJ, 3),  "(95% CI:", round(ci_full_PPMI_AJ[1], 3),  "-", round(ci_full_PPMI_AJ[3], 3),  ")\n")
cat("  HBS_EUR:",  round(auc_HBS_EUR, 3),   "(95% CI:", round(ci_full_HBS_EUR[1], 3),  "-", round(ci_full_HBS_EUR[3], 3),  ")\n")

# -----------------------------------------------
# CI bands for plotting (ci.se required for ci.type="shape")
# -----------------------------------------------

ci_se_null_PPMI_EUR <- ci.se(roc_null_PPMI_EUR, specificities = seq(0, 1, 0.01))
ci_se_full_PPMI_EUR <- ci.se(roc_full_PPMI_EUR, specificities = seq(0, 1, 0.01))

ci_se_null_PPMI_AJ  <- ci.se(roc_null_PPMI_AJ,  specificities = seq(0, 1, 0.01))
ci_se_full_PPMI_AJ  <- ci.se(roc_full_PPMI_AJ,  specificities = seq(0, 1, 0.01))

ci_se_null_HBS_EUR  <- ci.se(roc_null_HBS_EUR,  specificities = seq(0, 1, 0.01))
ci_se_full_HBS_EUR  <- ci.se(roc_full_HBS_EUR,  specificities = seq(0, 1, 0.01))

# -----------------------------------------------
# ROC plots with CI shapes
# -----------------------------------------------

plot_roc <- function(roc_null, roc_full, ci_se_null, ci_se_full, ci_null, ci_full, title, full_col) {

  # Plot null model ROC
  plot.roc(roc_null, col = "black", lwd = 2, main = title,
           legacy.axes = TRUE, print.auc = FALSE)

  # Add CI shape for null model
  plot(ci_se_null, type = "shape",
       col = adjustcolor("black", alpha.f = 0.1), border = NA)

  # Add full model ROC
  plot.roc(roc_full, add = TRUE, col = full_col, lwd = 2)

  # Add CI shape for full model
  plot(ci_se_full, type = "shape",
       col = adjustcolor(full_col, alpha.f = 0.15), border = NA)

  # Redraw ROC lines on top of shading
  lines.roc(roc_null, col = "black", lwd = 2)
  lines.roc(roc_full, col = full_col, lwd = 2)

  # Legend with AUC + 95% CI
  legend("bottomright",
         legend = c(
           paste0("Age + Sex + PD-PGS (AUC = ", round(auc(roc_null), 3),
                  " [", round(ci_null[1], 3), "\u2013", round(ci_null[3], 3), "])"),
           paste0("Full Model (AUC = ", round(auc(roc_full), 3),
                  " [", round(ci_full[1], 3), "\u2013", round(ci_full[3], 3), "])")
         ),
         col = c("black", full_col), lwd = 2, bty = "n")
}

plot_roc(roc_null_PPMI_EUR, roc_full_PPMI_EUR, ci_se_null_PPMI_EUR, ci_se_full_PPMI_EUR, ci_null_PPMI_EUR, ci_full_PPMI_EUR, "PPMI EUR: Baseline vs Full Model", "goldenrod")
plot_roc(roc_null_PPMI_AJ,  roc_full_PPMI_AJ,  ci_se_null_PPMI_AJ,  ci_se_full_PPMI_AJ,  ci_null_PPMI_AJ,  ci_full_PPMI_AJ,  "PPMI AJ: Baseline vs Full Model",  "forestgreen")
plot_roc(roc_null_HBS_EUR,  roc_full_HBS_EUR,  ci_se_null_HBS_EUR,  ci_se_full_HBS_EUR,  ci_null_HBS_EUR,  ci_full_HBS_EUR,  "HBS EUR: Baseline vs Full Model",  "dodgerblue")

# -----------------------------------------------
# DeLong tests
# -----------------------------------------------

delong_PPMI_EUR <- roc.test(roc_null_PPMI_EUR, roc_full_PPMI_EUR, method = "delong")
delong_PPMI_AJ  <- roc.test(roc_null_PPMI_AJ,  roc_full_PPMI_AJ,  method = "delong")
delong_HBS_EUR  <- roc.test(roc_null_HBS_EUR,  roc_full_HBS_EUR,  method = "delong")

cat("DeLong test p-values:\n")
cat("  PPMI_EUR:", delong_PPMI_EUR$p.value, "\n")
cat("  PPMI_AJ:",  delong_PPMI_AJ$p.value,  "\n")
cat("  HBS_EUR:",  delong_HBS_EUR$p.value,  "\n")

# -----------------------------------------------
# Extract sensitivity, specificity, accuracy at
# optimal threshold (closest-to-top-left)
# -----------------------------------------------

coords_null_PPMI_EUR <- coords(roc_null_PPMI_EUR, x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]
coords_full_PPMI_EUR <- coords(roc_full_PPMI_EUR, x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]

coords_null_PPMI_AJ  <- coords(roc_null_PPMI_AJ,  x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]
coords_full_PPMI_AJ  <- coords(roc_full_PPMI_AJ,  x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]

coords_null_HBS_EUR  <- coords(roc_null_HBS_EUR,  x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]
coords_full_HBS_EUR  <- coords(roc_full_HBS_EUR,  x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]

# -----------------------------------------------
# Build results table
# -----------------------------------------------

results <- rbindlist(list(
  data.table(
    cohort         = "PPMI_EUR",
    AUC_null       = as.numeric(auc(roc_null_PPMI_EUR)),
    AUC_lower_null = as.numeric(ci_null_PPMI_EUR[1]),
    AUC_upper_null = as.numeric(ci_null_PPMI_EUR[3]),
    AUC_full       = as.numeric(auc(roc_full_PPMI_EUR)),
    AUC_lower_full = as.numeric(ci_full_PPMI_EUR[1]),
    AUC_upper_full = as.numeric(ci_full_PPMI_EUR[3]),
    AUC_diff       = as.numeric(auc(roc_full_PPMI_EUR)) - as.numeric(auc(roc_null_PPMI_EUR)),
    sens_null      = coords_null_PPMI_EUR$sensitivity,
    sens_full      = coords_full_PPMI_EUR$sensitivity,
    sens_diff      = coords_full_PPMI_EUR$sensitivity - coords_null_PPMI_EUR$sensitivity,
    spec_null      = coords_null_PPMI_EUR$specificity,
    spec_full      = coords_full_PPMI_EUR$specificity,
    spec_diff      = coords_full_PPMI_EUR$specificity - coords_null_PPMI_EUR$specificity,
    acc_null       = coords_null_PPMI_EUR$accuracy,
    acc_full       = coords_full_PPMI_EUR$accuracy,
    acc_diff       = coords_full_PPMI_EUR$accuracy - coords_null_PPMI_EUR$accuracy,
    delong_z       = delong_PPMI_EUR$statistic,
    delong_p       = delong_PPMI_EUR$p.value
  ),
  data.table(
    cohort         = "PPMI_AJ",
    AUC_null       = as.numeric(auc(roc_null_PPMI_AJ)),
    AUC_lower_null = as.numeric(ci_null_PPMI_AJ[1]),
    AUC_upper_null = as.numeric(ci_null_PPMI_AJ[3]),
    AUC_full       = as.numeric(auc(roc_full_PPMI_AJ)),
    AUC_lower_full = as.numeric(ci_full_PPMI_AJ[1]),
    AUC_upper_full = as.numeric(ci_full_PPMI_AJ[3]),
    AUC_diff       = as.numeric(auc(roc_full_PPMI_AJ)) - as.numeric(auc(roc_null_PPMI_AJ)),
    sens_null      = coords_null_PPMI_AJ$sensitivity,
    sens_full      = coords_full_PPMI_AJ$sensitivity,
    sens_diff      = coords_full_PPMI_AJ$sensitivity - coords_null_PPMI_AJ$sensitivity,
    spec_null      = coords_null_PPMI_AJ$specificity,
    spec_full      = coords_full_PPMI_AJ$specificity,
    spec_diff      = coords_full_PPMI_AJ$specificity - coords_null_PPMI_AJ$specificity,
    acc_null       = coords_null_PPMI_AJ$accuracy,
    acc_full       = coords_full_PPMI_AJ$accuracy,
    acc_diff       = coords_full_PPMI_AJ$accuracy - coords_null_PPMI_AJ$accuracy,
    delong_z       = delong_PPMI_AJ$statistic,
    delong_p       = delong_PPMI_AJ$p.value
  ),
  data.table(
    cohort         = "HBS_EUR",
    AUC_null       = as.numeric(auc(roc_null_HBS_EUR)),
    AUC_lower_null = as.numeric(ci_null_HBS_EUR[1]),
    AUC_upper_null = as.numeric(ci_null_HBS_EUR[3]),
    AUC_full       = as.numeric(auc(roc_full_HBS_EUR)),
    AUC_lower_full = as.numeric(ci_full_HBS_EUR[1]),
    AUC_upper_full = as.numeric(ci_full_HBS_EUR[3]),
    AUC_diff       = as.numeric(auc(roc_full_HBS_EUR)) - as.numeric(auc(roc_null_HBS_EUR)),
    sens_null      = coords_null_HBS_EUR$sensitivity,
    sens_full      = coords_full_HBS_EUR$sensitivity,
    sens_diff      = coords_full_HBS_EUR$sensitivity - coords_null_HBS_EUR$sensitivity,
    spec_null      = coords_null_HBS_EUR$specificity,
    spec_full      = coords_full_HBS_EUR$specificity,
    spec_diff      = coords_full_HBS_EUR$specificity - coords_null_HBS_EUR$specificity,
    acc_null       = coords_null_HBS_EUR$accuracy,
    acc_full       = coords_full_HBS_EUR$accuracy,
    acc_diff       = coords_full_HBS_EUR$accuracy - coords_null_HBS_EUR$accuracy,
    delong_z       = delong_HBS_EUR$statistic,
    delong_p       = delong_HBS_EUR$p.value
  )
))

print(results)

results_outfile <- "/home/jupyter/multiTRS/results/multi_TRS_w_PRS_FUSION_ENET_external_validation_results_table.txt"
write.table(results, results_outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)
cat("Results table saved to:", results_outfile, "\n")

### All scores

#### Nested-CV

In [ ]:
%%R

library(data.table)
library(dplyr)
library(glmnet)
library(glmnetUtils)
library(caret)
library(pROC)

set.seed(1)

# Create a data.table for all performance results
all_performance_results <- data.table()

print("This script calculates ENET multi-TRS models using all scores... Running ENET models")

outfile <- "/home/jupyter/multiTRS/results/multi_TRS_w_PRS_all_scores_ENET_nestedcv.txt"

# Read in the clinical data
clinical <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt") %>%
  select(participant_id, case_control_other_at_baseline, sex, age_at_baseline)

# List of cohorts to loop over
TRS_cohorts <- c("PDBP")

# Loop over cohorts
for (TRS_cohort in TRS_cohorts) {

  # Read TRS scores
  TRS_path <- paste0("/home/jupyter/multiTRS/scores/", TRS_cohort, "_TWAS_SMR_FDR_scores_resid.txt")
  TRS <- fread(TRS_path) %>% select(-contains("Hip_fracture_EUR_2022_FDR_SMR_single_SNP"))

  # Read PGS scores
  PGS_path <- paste0("/home/jupyter/multiTRS/geno/PRS_scores_out/", TRS_cohort, "_EUR_PD_PGS_resid.txt")
  PGS <- fread(PGS_path)

  combined_data <- clinical %>%
    inner_join(PGS, by = "participant_id") %>%
    inner_join(TRS, by = "participant_id")

  cat("Rows in combined data for", TRS_cohort, ":", nrow(combined_data), "\n")

  # Null model with age + sex
  null_model <- glm(case_control_other_at_baseline ~ sex + scale(age_at_baseline) + scale(PD_PGS_resid),
                    data = combined_data,
                    family = binomial)
  probs_null <- predict(null_model, type = "response")
  roc_obj_null <- roc(combined_data$case_control_other_at_baseline, probs_null)
  auc_null <- auc(roc_obj_null)
  cat("AUC of null model:", auc_null, "\n")

  null_coords <- coords(roc_obj_null, x = "best", best.method = "closest.topleft",
                        ret = c("sensitivity", "specificity", "accuracy"))
  null_sensitivity <- as.numeric(null_coords["sensitivity"])
  null_specificity <- as.numeric(null_coords["specificity"])
  null_accuracy <- as.numeric(null_coords["accuracy"])

  # Prepare predictor matrix X and outcome y
  X <- combined_data %>%
    select(-participant_id, -case_control_other_at_baseline) %>%
    as.matrix()
  y <- as.factor(combined_data$case_control_other_at_baseline)

  cat("Number of predictors:", ncol(X), "\n")

  # Define penalty factors: unpenalized for sex, age and PD_PGS_resid
  penalty <- ifelse(colnames(X) %in% c("sex", "age_at_baseline", "PD_PGS_resid"), 0, 1)

# Create five outer folds
outer_folds <- createFolds(y, k = 5, returnTrain = TRUE)

# Store auc of outer folders
outer_auc <- c()
lambda_vals <- c()
alpha_vals  <- c()
outer_sensitivity <- c()
outer_specificity <- c()
outer_accuracy <- c()

for (i in 1:length(outer_folds)) {
  cat("Outer fold:", i, "\n")
  
  # Train/test split for this outer fold
  train_idx <- outer_folds[[i]]
  test_idx  <- setdiff(seq_along(y), train_idx)
  
  X_train <- X[train_idx, ]
  y_train <- y[train_idx]
  X_test  <- X[test_idx, ]
  y_test  <- y[test_idx]
  
  # Standardize inside training only
    vars_to_scale <- setdiff(colnames(X), "sex")
    scaler <- preProcess(X_train[, vars_to_scale, drop = FALSE], method = c("center", "scale"))
    
    X_train_scaled <- X_train
    X_train_scaled[, vars_to_scale] <- predict(scaler, X_train[, vars_to_scale, drop = FALSE])
    
    X_test_scaled <- X_test
    X_test_scaled[, vars_to_scale] <- predict(scaler, X_test[, vars_to_scale, drop = FALSE])
    
 # Specify alpha list 
  alphalist <- seq(0,1,by=0.1)
  
  # Inner CV with cv.glmnet (5-fold)
  cvfit <- glmnetUtils::cva.glmnet(
    x = X_train_scaled,
    y = y_train,
    family = "binomial",
    alpha = alphalist,
    nfolds = 5,
    type.measure = "auc",
    penalty.factor = penalty,
    standardize = FALSE
  )
  

    
  # Select best alpha and lambda
  lambda_1se <- sapply(cvfit$modlist, `[[`, "lambda.1se")
  error <- sapply(cvfit$modlist, function(mod) {
    idx <- which(mod$lambda == mod$lambda.1se)
    mod$cvm[idx]
  })
  best <- which.max(error)
  best_alpha  <- cvfit$alpha[best]
  best_lambda <- lambda_1se[best]
                  

  # Best lambda chosen inside
  lambda_vals[i] <- best_lambda
  alpha_vals[i]  <- best_alpha
    
  
  # Refit on full training set with best lambda
  final_model <- glmnet(
    x = X_train_scaled,
    y = y_train,
    family = "binomial",
    alpha = best_alpha,
    lambda = best_lambda,
    penalty.factor = penalty,
    standardize = FALSE
  )
  
  # Predict probabilities on outer test set
  probs <- predict(final_model, newx = X_test_scaled, type = "response")
  
# Compute ROC curve
  roc_obj_outer <- roc(y_test, as.numeric(probs))

# Compute AUC
  auc_val <- auc(roc_obj_outer)

# Get sens, spec, accuracy using topleft
model_coords <- coords(
  roc_obj_outer,
  x = "best",
  best.method = "closest.topleft",
  ret = c("threshold", "sensitivity", "specificity", "accuracy")
)

model_sensitivity <- as.numeric(model_coords["sensitivity"])
model_specificity <- as.numeric(model_coords["specificity"])
model_accuracy <- as.numeric(model_coords["accuracy"])
    
  outer_auc[i] <- auc_val
  outer_sensitivity[i] <- model_sensitivity
  outer_specificity[i] <- model_specificity
  outer_accuracy[i] <- model_accuracy
}

alpha_vals <- as.numeric(alpha_vals)
lambda_vals <- as.numeric(lambda_vals)

cat("Alpha per outer fold:\n")
print(alpha_vals)

cat("Lambda per outer fold:\n")
print(lambda_vals)
                  
lambda_final <- median(lambda_vals)
alpha_final <- median(alpha_vals)


cat("AUC per outer fold:\n")
print(outer_auc)
cat("Mean AUC across outer folds:", mean(outer_auc), "\n")
cat("SD of the AUC across outer folds:", sd(outer_auc), "\n")

cat("Median alpha:\n")
print(alpha_final)
cat("Median lambda:\n")
print(lambda_final)


                  
performance_results <- data.table(cohort = TRS_cohort,
                                  scores = "all_scores",
                                  model = "ENET",
                                  covariate_treatment = "unpenalised",
                                  median_lambda = lambda_final,
                                  median_alpha = alpha_final,
                                  AUC_null = auc_null,
                                  AUC_full_mean_cv =  mean(outer_auc),
                                  AUC_full_sd_cv = sd(outer_auc),
                                  AUC_diff = mean(outer_auc) - auc_null,
                                  sens_null = null_sensitivity,
                                  sens_full_mean_cv = mean(outer_sensitivity),
                                  sens_full_sd_cv = sd(outer_sensitivity),
                                  sens_diff = mean(outer_sensitivity) - null_sensitivity,
                                  spec_null = null_specificity,
                                  spec_full_mean_cv = mean(outer_specificity),
                                  spec_full_sd_cv = sd(outer_specificity),
                                  spec_diff = mean(outer_specificity) - null_specificity,
                                  acc_null = null_accuracy,
                                  acc_full_mean_cv = mean(outer_accuracy),
                                  acc_full_sd_cv = sd(outer_accuracy),
                                  acc_diff = mean(outer_accuracy) - null_accuracy)
    

all_performance_results <- rbind(all_performance_results,performance_results)


}
                  
print(all_performance_results)
                  

write.table(all_performance_results, outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)

#### Test in PPMI (EUR), PPMI (AJ) and HBS (EUR)

In [ ]:
%%R

library(data.table)
library(dplyr)
library(glmnet)
library(caret)
library(pROC)

set.seed(1)

# Read in the data
clinical <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt") %>% 
  select(participant_id, case_control_other_at_baseline, sex, age_at_baseline)

# Read in the PGS
PRS_PDBP_EUR <- fread("/home/jupyter/multiTRS/geno/PRS_scores_out/PDBP_EUR_PD_PGS_resid.txt")
PRS_PPMI_EUR <- fread("/home/jupyter/multiTRS/geno/PRS_scores_out/PPMI_EUR_PD_PGS_resid.txt")
PRS_PPMI_AJ  <- fread("/home/jupyter/multiTRS/geno/PRS_scores_out/PPMI_AJ_PD_PGS_resid.txt")
PRS_HBS_EUR  <- fread("/home/jupyter/multiTRS/geno/PRS_scores_out/HBS_EUR_PD_PGS_resid.txt")

# Read TRS scores - select columns based on PDBP first
TRS_PDBP_EUR <- fread("/home/jupyter/multiTRS/scores/PDBP_EUR_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt") %>% 
   select(-contains("Hip_fracture_EUR_2022_FDR_SMR_single_SNP"))

TRS_PPMI_EUR <- fread("/home/jupyter/multiTRS/scores/PPMI_EUR_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt") %>% 
  select(all_of(colnames(TRS_PDBP_EUR)))

TRS_PPMI_AJ  <- fread("/home/jupyter/multiTRS/scores/PPMI_AJ_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt") %>%  
  select(all_of(colnames(TRS_PDBP_EUR)))

TRS_HBS_EUR  <- fread("/home/jupyter/multiTRS/scores/HBS_EUR_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt") %>% 
  select(all_of(colnames(TRS_PDBP_EUR)))

# -----------------------------------------------
# Create combined datasets
# -----------------------------------------------

# PDBP (training)
combined_data <- clinical %>%
  inner_join(PRS_PDBP_EUR, by = "participant_id") %>%
  inner_join(TRS_PDBP_EUR, by = "participant_id")

# PPMI_EUR (test)
combined_data_PPMI_EUR <- clinical %>%
  inner_join(PRS_PPMI_EUR, by = "participant_id") %>%
  inner_join(TRS_PPMI_EUR, by = "participant_id")

# PPMI_AJ (test)
combined_data_PPMI_AJ <- clinical %>%
  inner_join(PRS_PPMI_AJ, by = "participant_id") %>%
  inner_join(TRS_PPMI_AJ, by = "participant_id")

# HBS_EUR (test)
combined_data_HBS_EUR <- clinical %>%
  inner_join(PRS_HBS_EUR, by = "participant_id") %>%
  inner_join(TRS_HBS_EUR, by = "participant_id")

cat("Testing all scores model...\n")
cat("Rows in combined data - PDBP:", nrow(combined_data), "\n")
cat("Rows in combined data - PPMI_EUR:", nrow(combined_data_PPMI_EUR), "\n")
cat("Rows in combined data - PPMI_AJ:", nrow(combined_data_PPMI_AJ), "\n")
cat("Rows in combined data - HBS_EUR:", nrow(combined_data_HBS_EUR), "\n")

# -----------------------------------------------
# Null model (age + sex + PGS), fit on PDBP, applied to all
# -----------------------------------------------

null_model <- glm(case_control_other_at_baseline ~ scale(age_at_baseline) + sex + scale(PD_PGS_resid),
                  data = combined_data,
                  family = binomial)

# Null model predictions
null_probs_PDBP     <- predict(null_model, newdata = combined_data,          type = "response")
null_probs_PPMI_EUR <- predict(null_model, newdata = combined_data_PPMI_EUR, type = "response")
null_probs_PPMI_AJ  <- predict(null_model, newdata = combined_data_PPMI_AJ,  type = "response")
null_probs_HBS_EUR  <- predict(null_model, newdata = combined_data_HBS_EUR,  type = "response")

auc_null_PDBP     <- auc(roc(combined_data$case_control_other_at_baseline,          null_probs_PDBP))
auc_null_PPMI_EUR <- auc(roc(combined_data_PPMI_EUR$case_control_other_at_baseline, null_probs_PPMI_EUR))
auc_null_PPMI_AJ  <- auc(roc(combined_data_PPMI_AJ$case_control_other_at_baseline,  null_probs_PPMI_AJ))
auc_null_HBS_EUR  <- auc(roc(combined_data_HBS_EUR$case_control_other_at_baseline,  null_probs_HBS_EUR))

cat("Null model AUCs:\n")
cat("  PDBP:", auc_null_PDBP, "\n")
cat("  PPMI_EUR:", auc_null_PPMI_EUR, "\n")
cat("  PPMI_AJ:", auc_null_PPMI_AJ, "\n")
cat("  HBS_EUR:", auc_null_HBS_EUR, "\n")

# -----------------------------------------------
# Prepare predictor matrices
# -----------------------------------------------

X <- combined_data %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y <- as.factor(combined_data$case_control_other_at_baseline)

X_PPMI_EUR <- combined_data_PPMI_EUR %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y_PPMI_EUR <- as.factor(combined_data_PPMI_EUR$case_control_other_at_baseline)

X_PPMI_AJ <- combined_data_PPMI_AJ %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y_PPMI_AJ <- as.factor(combined_data_PPMI_AJ$case_control_other_at_baseline)

X_HBS_EUR <- combined_data_HBS_EUR %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y_HBS_EUR <- as.factor(combined_data_HBS_EUR$case_control_other_at_baseline)

cat("Number of predictors:", ncol(X), "\n")

# -----------------------------------------------
# Scale on PDBP, apply to all
# -----------------------------------------------

penalty <- ifelse(colnames(X) %in% c("PD_PGS_resid", "sex", "age_at_baseline"), 0, 1)
vars_to_scale <- setdiff(colnames(X), "sex")

scaler_full <- preProcess(X[, vars_to_scale], method = c("center", "scale"))

X_scaled          <- X
X_scaled[, vars_to_scale] <- predict(scaler_full, X[, vars_to_scale])

X_PPMI_EUR_scaled <- X_PPMI_EUR
X_PPMI_EUR_scaled[, vars_to_scale] <- predict(scaler_full, X_PPMI_EUR[, vars_to_scale])

X_PPMI_AJ_scaled  <- X_PPMI_AJ
X_PPMI_AJ_scaled[, vars_to_scale]  <- predict(scaler_full, X_PPMI_AJ[, vars_to_scale])

X_HBS_EUR_scaled  <- X_HBS_EUR
X_HBS_EUR_scaled[, vars_to_scale]  <- predict(scaler_full, X_HBS_EUR[, vars_to_scale])

# -----------------------------------------------
# Fit final LASSO model on PDBP
# -----------------------------------------------

final_model <- glmnet(
  x = X_scaled,
  y = y,
  family = "binomial",
  alpha = 0.1,
  lambda = 0.4143562,
  penalty.factor = penalty,
  standardize = FALSE
)

# Extract and print coefficients
coefs <- coef(final_model)
coef_df <- data.frame(
  feature = rownames(coefs),
  coefficient = as.numeric(coefs)
)

coef_df <- coef_df %>% arrange(desc(coefficient))

print(coef_df)

coef_outfile <- "/home/jupyter/multiTRS/results/multi_TRS_w_PRS_all_scores_ENET_PDBP_full_model_coefficients.txt"
write.table(coef_df, coef_outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)
cat("Coefficients saved to:", coef_outfile, "\n")

# -----------------------------------------------
# Predictions and AUCs
# -----------------------------------------------

get_auc <- function(model, X_new, y_new) {
  probs <- predict(model, newx = X_new, type = "response")
  auc(roc(y_new, as.numeric(probs)))
}

auc_PDBP     <- get_auc(final_model, X_scaled,          y)
auc_PPMI_EUR <- get_auc(final_model, X_PPMI_EUR_scaled, y_PPMI_EUR)
auc_PPMI_AJ  <- get_auc(final_model, X_PPMI_AJ_scaled,  y_PPMI_AJ)
auc_HBS_EUR  <- get_auc(final_model, X_HBS_EUR_scaled,  y_HBS_EUR)

cat("Full model AUCs:\n")
cat("  PDBP:", auc_PDBP, "\n")
cat("  PPMI_EUR:", auc_PPMI_EUR, "\n")
cat("  PPMI_AJ:", auc_PPMI_AJ, "\n")
cat("  HBS_EUR:", auc_HBS_EUR, "\n")

# -----------------------------------------------
# ROC objects (created before CIs and plots)
# -----------------------------------------------

probs_PPMI_EUR <- predict(final_model, newx = X_PPMI_EUR_scaled, type = "response")
probs_PPMI_AJ  <- predict(final_model, newx = X_PPMI_AJ_scaled,  type = "response")
probs_HBS_EUR  <- predict(final_model, newx = X_HBS_EUR_scaled,  type = "response")

roc_null_PPMI_EUR <- roc(y_PPMI_EUR, null_probs_PPMI_EUR)
roc_full_PPMI_EUR <- roc(y_PPMI_EUR, as.numeric(probs_PPMI_EUR))

roc_null_PPMI_AJ  <- roc(y_PPMI_AJ,  null_probs_PPMI_AJ)
roc_full_PPMI_AJ  <- roc(y_PPMI_AJ,  as.numeric(probs_PPMI_AJ))

roc_null_HBS_EUR  <- roc(y_HBS_EUR,  null_probs_HBS_EUR)
roc_full_HBS_EUR  <- roc(y_HBS_EUR,  as.numeric(probs_HBS_EUR))

# -----------------------------------------------
# CIs for AUCs
# -----------------------------------------------

ci_null_PPMI_EUR <- ci.auc(roc_null_PPMI_EUR)
ci_null_PPMI_AJ  <- ci.auc(roc_null_PPMI_AJ)
ci_null_HBS_EUR  <- ci.auc(roc_null_HBS_EUR)

ci_full_PPMI_EUR <- ci.auc(roc_full_PPMI_EUR)
ci_full_PPMI_AJ  <- ci.auc(roc_full_PPMI_AJ)
ci_full_HBS_EUR  <- ci.auc(roc_full_HBS_EUR)

cat("Null model AUCs with 95% CI:\n")
cat("  PPMI_EUR:", round(auc_null_PPMI_EUR, 3), "(95% CI:", round(ci_null_PPMI_EUR[1], 3), "-", round(ci_null_PPMI_EUR[3], 3), ")\n")
cat("  PPMI_AJ:",  round(auc_null_PPMI_AJ, 3),  "(95% CI:", round(ci_null_PPMI_AJ[1], 3),  "-", round(ci_null_PPMI_AJ[3], 3),  ")\n")
cat("  HBS_EUR:",  round(auc_null_HBS_EUR, 3),   "(95% CI:", round(ci_null_HBS_EUR[1], 3),  "-", round(ci_null_HBS_EUR[3], 3),  ")\n")

cat("Full model AUCs with 95% CI:\n")
cat("  PPMI_EUR:", round(auc_PPMI_EUR, 3), "(95% CI:", round(ci_full_PPMI_EUR[1], 3), "-", round(ci_full_PPMI_EUR[3], 3), ")\n")
cat("  PPMI_AJ:",  round(auc_PPMI_AJ, 3),  "(95% CI:", round(ci_full_PPMI_AJ[1], 3),  "-", round(ci_full_PPMI_AJ[3], 3),  ")\n")
cat("  HBS_EUR:",  round(auc_HBS_EUR, 3),   "(95% CI:", round(ci_full_HBS_EUR[1], 3),  "-", round(ci_full_HBS_EUR[3], 3),  ")\n")

# -----------------------------------------------
# CI bands for plotting (ci.se required for ci.type="shape")
# -----------------------------------------------

ci_se_null_PPMI_EUR <- ci.se(roc_null_PPMI_EUR, specificities = seq(0, 1, 0.01))
ci_se_full_PPMI_EUR <- ci.se(roc_full_PPMI_EUR, specificities = seq(0, 1, 0.01))

ci_se_null_PPMI_AJ  <- ci.se(roc_null_PPMI_AJ,  specificities = seq(0, 1, 0.01))
ci_se_full_PPMI_AJ  <- ci.se(roc_full_PPMI_AJ,  specificities = seq(0, 1, 0.01))

ci_se_null_HBS_EUR  <- ci.se(roc_null_HBS_EUR,  specificities = seq(0, 1, 0.01))
ci_se_full_HBS_EUR  <- ci.se(roc_full_HBS_EUR,  specificities = seq(0, 1, 0.01))

# -----------------------------------------------
# ROC plots with CI shapes
# -----------------------------------------------

plot_roc <- function(roc_null, roc_full, ci_se_null, ci_se_full, ci_null, ci_full, title, full_col) {

  # Plot null model ROC
  plot.roc(roc_null, col = "black", lwd = 2, main = title,
           legacy.axes = TRUE, print.auc = FALSE)

  # Add CI shape for null model
  plot(ci_se_null, type = "shape",
       col = adjustcolor("black", alpha.f = 0.1), border = NA)

  # Add full model ROC
  plot.roc(roc_full, add = TRUE, col = full_col, lwd = 2)

  # Add CI shape for full model
  plot(ci_se_full, type = "shape",
       col = adjustcolor(full_col, alpha.f = 0.15), border = NA)

  # Redraw ROC lines on top of shading
  lines.roc(roc_null, col = "black", lwd = 2)
  lines.roc(roc_full, col = full_col, lwd = 2)

  # Legend with AUC + 95% CI
  legend("bottomright",
         legend = c(
           paste0("Age + Sex + PD-PGS (AUC = ", round(auc(roc_null), 3),
                  " [", round(ci_null[1], 3), "\u2013", round(ci_null[3], 3), "])"),
           paste0("Full Model (AUC = ", round(auc(roc_full), 3),
                  " [", round(ci_full[1], 3), "\u2013", round(ci_full[3], 3), "])")
         ),
         col = c("black", full_col), lwd = 2, bty = "n")
}

plot_roc(roc_null_PPMI_EUR, roc_full_PPMI_EUR, ci_se_null_PPMI_EUR, ci_se_full_PPMI_EUR, ci_null_PPMI_EUR, ci_full_PPMI_EUR, "PPMI EUR: Baseline vs Full Model", "goldenrod")
plot_roc(roc_null_PPMI_AJ,  roc_full_PPMI_AJ,  ci_se_null_PPMI_AJ,  ci_se_full_PPMI_AJ,  ci_null_PPMI_AJ,  ci_full_PPMI_AJ,  "PPMI AJ: Baseline vs Full Model",  "forestgreen")
plot_roc(roc_null_HBS_EUR,  roc_full_HBS_EUR,  ci_se_null_HBS_EUR,  ci_se_full_HBS_EUR,  ci_null_HBS_EUR,  ci_full_HBS_EUR,  "HBS EUR: Baseline vs Full Model",  "dodgerblue")

# -----------------------------------------------
# DeLong tests
# -----------------------------------------------

delong_PPMI_EUR <- roc.test(roc_null_PPMI_EUR, roc_full_PPMI_EUR, method = "delong")
delong_PPMI_AJ  <- roc.test(roc_null_PPMI_AJ,  roc_full_PPMI_AJ,  method = "delong")
delong_HBS_EUR  <- roc.test(roc_null_HBS_EUR,  roc_full_HBS_EUR,  method = "delong")

cat("DeLong test p-values:\n")
cat("  PPMI_EUR:", delong_PPMI_EUR$p.value, "\n")
cat("  PPMI_AJ:",  delong_PPMI_AJ$p.value,  "\n")
cat("  HBS_EUR:",  delong_HBS_EUR$p.value,  "\n")

# -----------------------------------------------
# Extract sensitivity, specificity, accuracy at
# optimal threshold (closest-to-top-left)
# -----------------------------------------------

coords_null_PPMI_EUR <- coords(roc_null_PPMI_EUR, x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]
coords_full_PPMI_EUR <- coords(roc_full_PPMI_EUR, x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]

coords_null_PPMI_AJ  <- coords(roc_null_PPMI_AJ,  x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]
coords_full_PPMI_AJ  <- coords(roc_full_PPMI_AJ,  x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]

coords_null_HBS_EUR  <- coords(roc_null_HBS_EUR,  x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]
coords_full_HBS_EUR  <- coords(roc_full_HBS_EUR,  x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]

# -----------------------------------------------
# Build results table
# -----------------------------------------------

results <- rbindlist(list(
  data.table(
    cohort         = "PPMI_EUR",
    AUC_null       = as.numeric(auc(roc_null_PPMI_EUR)),
    AUC_lower_null = as.numeric(ci_null_PPMI_EUR[1]),
    AUC_upper_null = as.numeric(ci_null_PPMI_EUR[3]),
    AUC_full       = as.numeric(auc(roc_full_PPMI_EUR)),
    AUC_lower_full = as.numeric(ci_full_PPMI_EUR[1]),
    AUC_upper_full = as.numeric(ci_full_PPMI_EUR[3]),
    AUC_diff       = as.numeric(auc(roc_full_PPMI_EUR)) - as.numeric(auc(roc_null_PPMI_EUR)),
    sens_null      = coords_null_PPMI_EUR$sensitivity,
    sens_full      = coords_full_PPMI_EUR$sensitivity,
    sens_diff      = coords_full_PPMI_EUR$sensitivity - coords_null_PPMI_EUR$sensitivity,
    spec_null      = coords_null_PPMI_EUR$specificity,
    spec_full      = coords_full_PPMI_EUR$specificity,
    spec_diff      = coords_full_PPMI_EUR$specificity - coords_null_PPMI_EUR$specificity,
    acc_null       = coords_null_PPMI_EUR$accuracy,
    acc_full       = coords_full_PPMI_EUR$accuracy,
    acc_diff       = coords_full_PPMI_EUR$accuracy - coords_null_PPMI_EUR$accuracy,
    delong_z       = delong_PPMI_EUR$statistic,
    delong_p       = delong_PPMI_EUR$p.value
  ),
  data.table(
    cohort         = "PPMI_AJ",
    AUC_null       = as.numeric(auc(roc_null_PPMI_AJ)),
    AUC_lower_null = as.numeric(ci_null_PPMI_AJ[1]),
    AUC_upper_null = as.numeric(ci_null_PPMI_AJ[3]),
    AUC_full       = as.numeric(auc(roc_full_PPMI_AJ)),
    AUC_lower_full = as.numeric(ci_full_PPMI_AJ[1]),
    AUC_upper_full = as.numeric(ci_full_PPMI_AJ[3]),
    AUC_diff       = as.numeric(auc(roc_full_PPMI_AJ)) - as.numeric(auc(roc_null_PPMI_AJ)),
    sens_null      = coords_null_PPMI_AJ$sensitivity,
    sens_full      = coords_full_PPMI_AJ$sensitivity,
    sens_diff      = coords_full_PPMI_AJ$sensitivity - coords_null_PPMI_AJ$sensitivity,
    spec_null      = coords_null_PPMI_AJ$specificity,
    spec_full      = coords_full_PPMI_AJ$specificity,
    spec_diff      = coords_full_PPMI_AJ$specificity - coords_null_PPMI_AJ$specificity,
    acc_null       = coords_null_PPMI_AJ$accuracy,
    acc_full       = coords_full_PPMI_AJ$accuracy,
    acc_diff       = coords_full_PPMI_AJ$accuracy - coords_null_PPMI_AJ$accuracy,
    delong_z       = delong_PPMI_AJ$statistic,
    delong_p       = delong_PPMI_AJ$p.value
  ),
  data.table(
    cohort         = "HBS_EUR",
    AUC_null       = as.numeric(auc(roc_null_HBS_EUR)),
    AUC_lower_null = as.numeric(ci_null_HBS_EUR[1]),
    AUC_upper_null = as.numeric(ci_null_HBS_EUR[3]),
    AUC_full       = as.numeric(auc(roc_full_HBS_EUR)),
    AUC_lower_full = as.numeric(ci_full_HBS_EUR[1]),
    AUC_upper_full = as.numeric(ci_full_HBS_EUR[3]),
    AUC_diff       = as.numeric(auc(roc_full_HBS_EUR)) - as.numeric(auc(roc_null_HBS_EUR)),
    sens_null      = coords_null_HBS_EUR$sensitivity,
    sens_full      = coords_full_HBS_EUR$sensitivity,
    sens_diff      = coords_full_HBS_EUR$sensitivity - coords_null_HBS_EUR$sensitivity,
    spec_null      = coords_null_HBS_EUR$specificity,
    spec_full      = coords_full_HBS_EUR$specificity,
    spec_diff      = coords_full_HBS_EUR$specificity - coords_null_HBS_EUR$specificity,
    acc_null       = coords_null_HBS_EUR$accuracy,
    acc_full       = coords_full_HBS_EUR$accuracy,
    acc_diff       = coords_full_HBS_EUR$accuracy - coords_null_HBS_EUR$accuracy,
    delong_z       = delong_HBS_EUR$statistic,
    delong_p       = delong_HBS_EUR$p.value
  )
))

print(results)

results_outfile <- "/home/jupyter/multiTRS/results/multi_TRS_w_PRS_all_scores_ENET_external_validation_results_table.txt"
write.table(results, results_outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)
cat("Results table saved to:", results_outfile, "\n")

## XGBoost

### Import packages

In [ ]:
from sklearn.model_selection import KFold, RandomizedSearchCV, train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve, make_scorer
from xgboost import XGBClassifier
from scipy.stats import uniform, randint, mode
from sklearn.base import BaseEstimator, ClassifierMixin
from scipy.stats import mode
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from scipy import stats
from MLstatkit import Delong_test
import shap
from sklearn.metrics import accuracy_score

### SMR-multi

#### Nested-CV

In [ ]:
# =========================
# Closest.topleft function (pROC equivalent)
# =========================
def closest_topleft_threshold(y_true, y_prob):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    distances = np.sqrt((fpr)**2 + (1 - tpr)**2)
    idx = np.argmin(distances)
    return thresholds[idx], tpr[idx], 1 - fpr[idx]

# =========================
# Load and join data
# =========================
clinical_path = "/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt"
TRS_path = "/home/jupyter/multiTRS/scores/PDBP_EUR_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt"
PGS_path = "/home/jupyter/multiTRS/geno/PRS_scores_out/PDBP_EUR_PD_PGS_resid.txt"

clinical = pd.read_csv(clinical_path, sep="\t")[[
    "participant_id", "case_control_other_at_baseline", "sex", "age_at_baseline"
]]

TRS = pd.read_csv(TRS_path, sep="\t")
TRS_cols = ["participant_id"] + [c for c in TRS.columns if "SMR" in c and "single_SNP" not in c]
TRS = TRS[TRS_cols]

PGS = pd.read_csv(PGS_path, sep="\t")[["participant_id", "PD_PGS_resid"]]

combined_data = pd.merge(clinical, PGS, on="participant_id", how="inner")
combined_data = pd.merge(combined_data, TRS, on="participant_id", how="inner")

print(f"Rows in combined PDBP data: {len(combined_data)}")

X = combined_data.drop(columns=["participant_id", "case_control_other_at_baseline"])
y = combined_data["case_control_other_at_baseline"]

print(f"Number of predictors: {X.shape[1]}")

# =========================
# Hyperparameter grid
# =========================
param_dist = {
    'n_estimators': [100, 200, 300, 500, 700, 1000],
    'max_depth': [3, 5, 7, 9],
    'min_child_weight': [5, 10, 20, 30, 40],
    'colsample_bytree': [0.4, 0.6, 0.8, 1],
    'subsample': [0.6, 0.8],
    'learning_rate': [0.001, 0.0015, 0.01, 0.015, 0.1],
    'gamma': [0, 0.1, 0.3, 0.5, 0.8, 1.0]
}

# =========================
# Nested CV
# =========================
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)

outer_auc = []
outer_sensitivity = []
outer_specificity = []
outer_accuracy = []

best_params_list = []

for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), 1):
    print(f"\n--- Outer fold {fold} ---")

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = XGBClassifier(eval_metric='auc', random_state=1)

    random_search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_dist,
        n_iter=100,
        scoring="roc_auc",
        cv=5,
        n_jobs=-1,
        random_state=1
    )

    random_search.fit(X_train, y_train)

    best_params_list.append(random_search.best_params_)

    # Predictions
    y_prob = random_search.best_estimator_.predict_proba(X_test)[:, 1]

    # AUC
    auc = roc_auc_score(y_test, y_prob)
    outer_auc.append(auc)

    # closest.topleft threshold
    thresh, sens, spec = closest_topleft_threshold(y_test, y_prob)

    y_pred = (y_prob >= thresh).astype(int)
    acc = (y_pred == y_test).mean()

    outer_sensitivity.append(sens)
    outer_specificity.append(spec)
    outer_accuracy.append(acc)

    print(f"AUC: {auc:.4f}, Sens: {sens:.3f}, Spec: {spec:.3f}, Acc: {acc:.3f}")

print("\nNested CV results:")
print(f"AUC mean: {np.mean(outer_auc):.4f} ± {np.std(outer_auc):.4f}")

# =========================
# Aggregate hyperparameters
# =========================
best_params_df = pd.DataFrame(best_params_list)
best_params_df.to_csv("/home/jupyter/multiTRS/results/multi_TRS_w_PRS_SMR_multi_XGBoost_nestedcv_hyperparameters_all_outer.txt",
                       sep="\t", index=False)
print("\nHyperparameters saved to: /home/jupyter/multiTRS/results/multi_TRS_w_PRS_SMR_multi_XGBoost_nestedcv_hyperparameters_all_outer.txt")
print(best_params_df)

final_params = {
    'n_estimators': int(mode(best_params_df['n_estimators'], keepdims=True).mode[0]),
    'max_depth': int(mode(best_params_df['max_depth'], keepdims=True).mode[0]),
    'min_child_weight': int(mode(best_params_df['min_child_weight'], keepdims=True).mode[0]),

    'subsample': best_params_df['subsample'].median(),
    'colsample_bytree': best_params_df['colsample_bytree'].median(),
    'learning_rate': best_params_df['learning_rate'].median(),
    'gamma': best_params_df['gamma'].median()
}

# convert dict → single-row DataFrame (FIXED)
final_params_df = pd.DataFrame([final_params])

# save DataFrame (FIXED: was incorrectly using dict)
final_params_df.to_csv(
    "/home/jupyter/multiTRS/results/multi_TRS_w_PRS_SMR_multi_XGBoost_nestedcv_hyperparameters_best.txt",
    sep="\t",
    index=False
)

print("\nHyperparameters saved to:")
print("/home/jupyter/multiTRS/results/multi_TRS_w_PRS_SMR_multi_XGBoost_nestedcv_hyperparameters_best.txt")

print("\nFinal aggregated hyperparameters:")
print(final_params_df)
# Snap continuous params to grid
#for param in ['subsample', 'colsample_bytree', 'learning_rate', 'gamma']:
 #   final_params[param] = min(param_dist[param], key=lambda x: abs(x - final_params[param]))


print("\nFinal aggregated hyperparameters:")
print(final_params)

# =========================
# Null model (baseline)
# =========================
X_baseline = combined_data[["sex", "age_at_baseline", "PD_PGS_resid"]].copy()

scaler = StandardScaler()
X_baseline_scaled = X_baseline.copy()
X_baseline_scaled[["age_at_baseline", "PD_PGS_resid"]] = scaler.fit_transform(
    X_baseline[["age_at_baseline", "PD_PGS_resid"]]
)

baseline_model = LogisticRegression(
    random_state=1,
    max_iter=2000,
    penalty=None,
    solver='newton-cholesky'
)

baseline_model.fit(X_baseline_scaled, y)

y_prob_null = baseline_model.predict_proba(X_baseline_scaled)[:, 1]

auc_null = roc_auc_score(y, y_prob_null)

# closest.topleft for null
null_thresh, null_sens, null_spec = closest_topleft_threshold(y, y_prob_null)

y_pred_null = (y_prob_null >= null_thresh).astype(int)
null_acc = (y_pred_null == y).mean()

print("\nNull model performance:")
print(f"AUC: {auc_null:.4f}, Sens: {null_sens:.3f}, Spec: {null_spec:.3f}, Acc: {null_acc:.3f}")

# =========================
# Final performance table
# =========================
performance_results = pd.DataFrame([{
    "cohort": "PDBP",
    "scores": "all_scores",
    "model": "XGBoost",

    "aggregated_params": str(final_params),

    "AUC_null": auc_null,
    "AUC_full_mean_cv": np.mean(outer_auc),
    "AUC_full_sd_cv": np.std(outer_auc),
    "AUC_diff": np.mean(outer_auc) - auc_null,

    "sens_null": null_sens,
    "sens_full_mean_cv": np.mean(outer_sensitivity),
    "sens_full_sd_cv": np.std(outer_sensitivity),
    "sens_diff": np.mean(outer_sensitivity) - null_sens,

    "spec_null": null_spec,
    "spec_full_mean_cv": np.mean(outer_specificity),
    "spec_full_sd_cv": np.std(outer_specificity),
    "spec_diff": np.mean(outer_specificity) - null_spec,

    "acc_null": null_acc,
    "acc_full_mean_cv": np.mean(outer_accuracy),
    "acc_full_sd_cv": np.std(outer_accuracy),
    "acc_diff": np.mean(outer_accuracy) - null_acc
}])

print("\n Performance summary:")
print(performance_results)

# Optional save
performance_results.to_csv("/home/jupyter/multiTRS/results/multi_TRS_w_PRS_SMR_multi_XGBoost_nestedcv.txt", sep="\t", index=False)

#### Test in PPMI (EUR), PPMI (AJ) and HBS

In [ ]:
# =========================
# Load and join data from PDBP
# =========================
clinical_path = "/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt"
TRS_path = "/home/jupyter/multiTRS/scores/PDBP_EUR_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt"
PGS_path = "/home/jupyter/multiTRS/geno/PRS_scores_out/PDBP_EUR_PD_PGS_resid.txt"

clinical = pd.read_csv(clinical_path, sep="\t")[["participant_id", "case_control_other_at_baseline", "sex", "age_at_baseline"]]

TRS = pd.read_csv(TRS_path, sep="\t")
TRS_cols = ["participant_id"] + [c for c in TRS.columns if "SMR" in c and "single_SNP" not in c]
TRS = TRS[TRS_cols]

PGS = pd.read_csv(PGS_path, sep="\t")[["participant_id", "PD_PGS_resid"]]

combined_data = pd.merge(clinical, PGS, on="participant_id", how="inner")
combined_data = pd.merge(combined_data, TRS, on="participant_id", how="inner")

print(f"Rows in combined PDBP data: {len(combined_data)}")

X = combined_data.drop(columns=["participant_id", "case_control_other_at_baseline"])
y = combined_data["case_control_other_at_baseline"]

print(f"Number of predictors: {X.shape[1]}")

# =========================
# Load saved hyperparameters and aggregate
# =========================
best_params_df = pd.read_csv("/home/jupyter/multiTRS/results/multi_TRS_w_PRS_SMR_multi_XGBoost_nestedcv_hyperparameters_all_outer.txt", sep="\t")

performance_df = pd.read_csv("/home/jupyter/multiTRS/results/multi_TRS_w_PRS_SMR_multi_XGBoost_nestedcv.txt", sep="\t")
print(f"\nLoaded mean nested CV AUC: {performance_df['AUC_full_mean_cv'].values[0]:.4f} SD: {performance_df['AUC_full_sd_cv'].values[0]:.4f}")

final_params = {
    'n_estimators':    int(mode(best_params_df['n_estimators'],    keepdims=True).mode[0]),
    'max_depth':       int(mode(best_params_df['max_depth'],       keepdims=True).mode[0]),
    'min_child_weight':int(mode(best_params_df['min_child_weight'],keepdims=True).mode[0]),
    'subsample':       best_params_df['subsample'].median(),
    'colsample_bytree':best_params_df['colsample_bytree'].median(),
    'learning_rate':   best_params_df['learning_rate'].median(),
    'gamma':           best_params_df['gamma'].median()
}

print("\nAggregated final hyperparameters:", final_params)

# =========================
# Fit final XGBoost model on full PDBP
# =========================
final_model = XGBClassifier(
    eval_metric='auc',
    random_state=1,
    **final_params
)

final_model.fit(X, y)

print("\nFinal XGBoost model fitted on full PDBP dataset.")

y_pred_prob = final_model.predict_proba(X)[:, 1]
final_auc = roc_auc_score(y, y_pred_prob)
print(f"Final XGBoost model AUC on full PDBP dataset: {final_auc:.4f}")

# =========================
# Extract and save SHAP values
# =========================
explainer = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X)
base_value = explainer.expected_value

print("\nSHAP values extracted from final XGBoost model.")
print(f"SHAP values shape: {shap_values.shape}")
print(f"Base value (expected model output): {base_value:.6f}")

shap_df = pd.DataFrame(shap_values, columns=X.columns)
shap_df.to_csv("/home/jupyter/multiTRS/results/multi_TRS_w_PRS_SMR_multi_XGBoost_PDBP_SHAP_values.txt",
               sep="\t", index=False)
print("SHAP values saved to: /home/jupyter/multiTRS/results/multi_TRS_w_PRS_SMR_multi_XGBoost_PDBP_SHAP_values.txt")

print("\nGenerating SHAP bar plot...")
plt.figure()
shap.summary_plot(shap_values, X, plot_type="bar", show=False)
plt.tight_layout()
plt.savefig("/home/jupyter/multiTRS/results/multi_TRS_w_PRS_SMR_multi_PDBP_SHAP_bar_plot.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()
print("SHAP bar plot saved to: /home/jupyter/multiTRS/results/multi_TRS_w_PRS_SMR_multi_XGBoost_PDBP_SHAP_bar_plot.png")

print("\nGenerating SHAP beeswarm plot...")
plt.figure()
shap.summary_plot(shap_values, X, show=False)
plt.tight_layout()
plt.savefig("/home/jupyter/multiTRS/results/multi_TRS_w_PRS_SMR_multi_XGBoost_PDBP_SHAP_beeswarm_plot.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()
print("Beeswarm plot saved to: /home/jupyter/multiTRS/results/multi_TRS_w_PRS_SMR_multi_XGBoost_PDBP_SHAP_beeswarm_plot.png")

# =========================
# Fit baseline logistic regression model on PDBP (sex + age + PGS)
# =========================
X_baseline = combined_data[["sex", "age_at_baseline", "PD_PGS_resid"]].copy()

scaler = StandardScaler()
cols_to_scale_baseline = ["age_at_baseline", "PD_PGS_resid"]
X_baseline_scaled = X_baseline.copy()
X_baseline_scaled[cols_to_scale_baseline] = scaler.fit_transform(X_baseline[cols_to_scale_baseline])

age_mean = scaler.mean_[0]
age_sd   = scaler.scale_[0]
pgs_mean = scaler.mean_[1]
pgs_sd   = scaler.scale_[1]

baseline_model = LogisticRegression(
    random_state=1,
    max_iter=2000,
    penalty=None,
    solver='newton-cholesky',
    fit_intercept=True
)
baseline_model.fit(X_baseline_scaled, y)

print("\nBaseline logistic regression model fitted on full PDBP dataset.")
print(f"Age scaling parameters - Mean: {age_mean:.4f}, SD: {age_sd:.4f}")
print(f"PGS scaling parameters - Mean: {pgs_mean:.4f}, SD: {pgs_sd:.4f}")
print(f"Baseline model intercept: {baseline_model.intercept_[0]:.6f}")
print(f"Baseline model coefficients (sex, age, PGS): {baseline_model.coef_[0]}")

y_baseline_pred_prob = baseline_model.predict_proba(X_baseline_scaled)[:, 1]
baseline_auc = roc_auc_score(y, y_baseline_pred_prob)
print(f"Baseline (sex + age + PGS) model AUC on full PDBP dataset: {baseline_auc:.4f}")

# =========================
# Closest.topleft function (pROC equivalent)
# =========================
def closest_topleft_threshold(y_true, y_prob):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    distances = np.sqrt((fpr)**2 + (1 - tpr)**2)
    idx = np.argmin(distances)
    return thresholds[idx], tpr[idx], 1 - fpr[idx]

def get_accuracy_at_threshold(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    return accuracy_score(y_true, y_pred)

# =========================
# External validation in PPMI_EUR, PPMI_AJ and HBS_EUR
# =========================
external_datasets = {
    "PPMI_EUR": {
        "TRS_path": "/home/jupyter/multiTRS/scores/PPMI_EUR_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt",
        "PGS_path": "/home/jupyter/multiTRS/geno/PRS_scores_out/PPMI_EUR_PD_PGS_resid.txt"
    },
    "PPMI_AJ": {
        "TRS_path": "/home/jupyter/multiTRS/scores/PPMI_AJ_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt",
        "PGS_path": "/home/jupyter/multiTRS/geno/PRS_scores_out/PPMI_AJ_PD_PGS_resid.txt"
    },
    "HBS_EUR": {
        "TRS_path": "/home/jupyter/multiTRS/scores/HBS_EUR_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt",
        "PGS_path": "/home/jupyter/multiTRS/geno/PRS_scores_out/HBS_EUR_PD_PGS_resid.txt"
    }
}

print("\n" + "="*70)
print("EXTERNAL VALIDATION RESULTS WITH DELONG'S TEST")
print("="*70)

full_results_summary = []

for dataset_name, paths in external_datasets.items():
    print(f"\n--- {dataset_name} Dataset ---")

    # Load TRS
    ext_TRS = pd.read_csv(paths["TRS_path"], sep="\t")
    TRS_cols = ["participant_id"] + [c for c in ext_TRS.columns if "SMR" in c and "single_SNP" not in c]
    ext_TRS = ext_TRS[TRS_cols]

    # Load PGS
    ext_PGS = pd.read_csv(paths["PGS_path"], sep="\t")[["participant_id", "PD_PGS_resid"]]

    # Merge
    ext_combined = pd.merge(clinical, ext_PGS, on="participant_id", how="inner")
    ext_combined = pd.merge(ext_combined, ext_TRS, on="participant_id", how="inner")
    print(f"Rows in combined data for {dataset_name}: {len(ext_combined)}")

    # Predictions - full model
    X_ext = ext_combined[X.columns]
    y_ext = ext_combined["case_control_other_at_baseline"]
    y_ext_pred_prob = final_model.predict_proba(X_ext)[:, 1]

    # Predictions - baseline model (scaled with PDBP parameters)
    X_ext_baseline = ext_combined[["sex", "age_at_baseline", "PD_PGS_resid"]].copy()
    X_ext_baseline_scaled = X_ext_baseline.copy()
    X_ext_baseline_scaled["age_at_baseline"] = (X_ext_baseline["age_at_baseline"] - age_mean) / age_sd
    X_ext_baseline_scaled["PD_PGS_resid"]    = (X_ext_baseline["PD_PGS_resid"]    - pgs_mean) / pgs_sd
    y_ext_baseline_pred_prob = baseline_model.predict_proba(X_ext_baseline_scaled)[:, 1]

    # DeLong test
    z_stat, p_value, ci_full, ci_null, auc_full, auc_null, info = Delong_test(
        y_ext, y_ext_pred_prob, y_ext_baseline_pred_prob,
        return_ci=True, return_auc=True, verbose=0
    )

    print(f"XGBoost model AUC:                   {auc_full:.4f} (95% CI: {ci_full[0]:.4f}-{ci_full[1]:.4f})")
    print(f"Baseline model AUC:                  {auc_null:.4f} (95% CI: {ci_null[0]:.4f}-{ci_null[1]:.4f})")
    print(f"AUC Difference (XGBoost - Baseline): {auc_full - auc_null:.4f}")
    print(f"Variance of AUC difference:          {info['var_diff']:.6f}")
    print(f"DeLong's test Z-statistic:           {z_stat:.4f}")
    print(f"DeLong's test p-value (2-tailed):    {p_value:.4e}")
    print(f"Result: {'Significantly different (p < 0.05)' if p_value < 0.05 else 'Not significantly different (p >= 0.05)'}")

    # Optimal threshold metrics
    thresh_full, sens_full, spec_full = closest_topleft_threshold(y_ext, y_ext_pred_prob)
    thresh_null, sens_null, spec_null = closest_topleft_threshold(y_ext, y_ext_baseline_pred_prob)

    acc_full = get_accuracy_at_threshold(y_ext, y_ext_pred_prob,          thresh_full)
    acc_null = get_accuracy_at_threshold(y_ext, y_ext_baseline_pred_prob, thresh_null)

    full_results_summary.append({
        'cohort':         dataset_name,
        'AUC_null':       auc_null,
        'AUC_lower_null': ci_null[0],
        'AUC_upper_null': ci_null[1],
        'AUC_full':       auc_full,
        'AUC_lower_full': ci_full[0],
        'AUC_upper_full': ci_full[1],
        'AUC_diff':       auc_full - auc_null,
        'sens_null':      sens_null,
        'sens_full':      sens_full,
        'sens_diff':      sens_full - sens_null,
        'spec_null':      spec_null,
        'spec_full':      spec_full,
        'spec_diff':      spec_full - spec_null,
        'acc_null':       acc_null,
        'acc_full':       acc_full,
        'acc_diff':       acc_full - acc_null,
        'delong_z':       z_stat,
        'delong_p':       p_value
    })

# =========================
# Save full results table
# =========================
print("\n" + "="*70)
print("FULL RESULTS TABLE")
print("="*70)
full_results_df = pd.DataFrame(full_results_summary)
print(full_results_df.to_string(index=False))

full_results_outfile = "/home/jupyter/multiTRS/results/multi_TRS_w_PRS_SMR_multi_XGBoost_external_validation_full_results.txt"
full_results_df.to_csv(full_results_outfile, sep="\t", index=False)
print(f"\nFull results saved to: {full_results_outfile}")

### SMR

#### Nested-CV

In [ ]:
# =========================
# Closest.topleft function (pROC equivalent)
# =========================
def closest_topleft_threshold(y_true, y_prob):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    distances = np.sqrt((fpr)**2 + (1 - tpr)**2)
    idx = np.argmin(distances)
    return thresholds[idx], tpr[idx], 1 - fpr[idx]

# =========================
# Load and join data
# =========================
clinical_path = "/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt"
TRS_path = "/home/jupyter/multiTRS/scores/PDBP_EUR_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt"
PGS_path = "/home/jupyter/multiTRS/geno/PRS_scores_out/PDBP_EUR_PD_PGS_resid.txt"

clinical = pd.read_csv(clinical_path, sep="\t")[[
    "participant_id", "case_control_other_at_baseline", "sex", "age_at_baseline"
]]

TRS = pd.read_csv(TRS_path, sep="\t")
TRS_cols = ["participant_id"] + [c for c in TRS.columns if "single_SNP" in c and "Hip_Fracture" not in c]
TRS = TRS[TRS_cols]

PGS = pd.read_csv(PGS_path, sep="\t")[["participant_id", "PD_PGS_resid"]]

combined_data = pd.merge(clinical, PGS, on="participant_id", how="inner")
combined_data = pd.merge(combined_data, TRS, on="participant_id", how="inner")

print(f"Rows in combined PDBP data: {len(combined_data)}")

X = combined_data.drop(columns=["participant_id", "case_control_other_at_baseline"])
y = combined_data["case_control_other_at_baseline"]

print(f"Number of predictors: {X.shape[1]}")

# =========================
# Hyperparameter grid
# =========================
param_dist = {
    'n_estimators': [100, 200, 300, 500, 700, 1000],
    'max_depth': [3, 5, 7, 9],
    'min_child_weight': [5, 10, 20, 30, 40],
    'colsample_bytree': [0.4, 0.6, 0.8, 1],
    'subsample': [0.6, 0.8],
    'learning_rate': [0.001, 0.0015, 0.01, 0.015, 0.1],
    'gamma': [0, 0.1, 0.3, 0.5, 0.8, 1.0]
}

# =========================
# Nested CV
# =========================
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)

outer_auc = []
outer_sensitivity = []
outer_specificity = []
outer_accuracy = []

best_params_list = []

for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), 1):
    print(f"\n--- Outer fold {fold} ---")

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = XGBClassifier(eval_metric='auc', random_state=1)

    random_search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_dist,
        n_iter=100,
        scoring="roc_auc",
        cv=5,
        n_jobs=-1,
        random_state=1
    )

    random_search.fit(X_train, y_train)

    best_params_list.append(random_search.best_params_)

    # Predictions
    y_prob = random_search.best_estimator_.predict_proba(X_test)[:, 1]

    # AUC
    auc = roc_auc_score(y_test, y_prob)
    outer_auc.append(auc)

    # closest.topleft threshold
    thresh, sens, spec = closest_topleft_threshold(y_test, y_prob)

    y_pred = (y_prob >= thresh).astype(int)
    acc = (y_pred == y_test).mean()

    outer_sensitivity.append(sens)
    outer_specificity.append(spec)
    outer_accuracy.append(acc)

    print(f"AUC: {auc:.4f}, Sens: {sens:.3f}, Spec: {spec:.3f}, Acc: {acc:.3f}")

print("\nNested CV results:")
print(f"AUC mean: {np.mean(outer_auc):.4f} ± {np.std(outer_auc):.4f}")

# =========================
# Aggregate hyperparameters
# =========================
best_params_df = pd.DataFrame(best_params_list)
best_params_df.to_csv("/home/jupyter/multiTRS/results/multi_TRS_w_PRS_SMR_single_SNP_XGBoost_nestedcv_hyperparameters_all_outer.txt",
                       sep="\t", index=False)
print("\nHyperparameters saved to: /home/jupyter/multiTRS/results/multi_TRS_w_PRS_SMR_single_SNP_XGBoost_nestedcv_hyperparameters_all_outer.txt")
print(best_params_df)

final_params = {
    'n_estimators': int(mode(best_params_df['n_estimators'], keepdims=True).mode[0]),
    'max_depth': int(mode(best_params_df['max_depth'], keepdims=True).mode[0]),
    'min_child_weight': int(mode(best_params_df['min_child_weight'], keepdims=True).mode[0]),

    'subsample': best_params_df['subsample'].median(),
    'colsample_bytree': best_params_df['colsample_bytree'].median(),
    'learning_rate': best_params_df['learning_rate'].median(),
    'gamma': best_params_df['gamma'].median()
}


final_params_df = pd.DataFrame([final_params])

final_params_df.to_csv(
    "/home/jupyter/multiTRS/results/multi_TRS_w_PRS_SMR_single_SNP_XGBoost_nestedcv_hyperparameters_best.txt",
    sep="\t",
    index=False
)
print("\nHyperparameters saved to: /home/jupyter/multiTRS/results/multi_TRS_w_PRS_SMR_single_SNP_XGBoost_nestedcv_hyperparameters_best.txt")
print(final_params_df)

# Snap continuous params to grid
#for param in ['subsample', 'colsample_bytree', 'learning_rate', 'gamma']:
 #   final_params[param] = min(param_dist[param], key=lambda x: abs(x - final_params[param]))


print("\nFinal aggregated hyperparameters:")
print(final_params)

# =========================
# Null model (baseline)
# =========================
X_baseline = combined_data[["sex", "age_at_baseline", "PD_PGS_resid"]].copy()

scaler = StandardScaler()
X_baseline_scaled = X_baseline.copy()
X_baseline_scaled[["age_at_baseline", "PD_PGS_resid"]] = scaler.fit_transform(
    X_baseline[["age_at_baseline", "PD_PGS_resid"]]
)

baseline_model = LogisticRegression(
    random_state=1,
    max_iter=2000,
    penalty=None,
    solver='newton-cholesky'
)

baseline_model.fit(X_baseline_scaled, y)

y_prob_null = baseline_model.predict_proba(X_baseline_scaled)[:, 1]

auc_null = roc_auc_score(y, y_prob_null)

# closest.topleft for null
null_thresh, null_sens, null_spec = closest_topleft_threshold(y, y_prob_null)

y_pred_null = (y_prob_null >= null_thresh).astype(int)
null_acc = (y_pred_null == y).mean()

print("\nNull model performance:")
print(f"AUC: {auc_null:.4f}, Sens: {null_sens:.3f}, Spec: {null_spec:.3f}, Acc: {null_acc:.3f}")

# =========================
# Final performance table
# =========================
performance_results = pd.DataFrame([{
    "cohort": "PDBP",
    "scores": "all_scores",
    "model": "XGBoost",

    "aggregated_params": str(final_params),

    "AUC_null": auc_null,
    "AUC_full_mean_cv": np.mean(outer_auc),
    "AUC_full_sd_cv": np.std(outer_auc),
    "AUC_diff": np.mean(outer_auc) - auc_null,

    "sens_null": null_sens,
    "sens_full_mean_cv": np.mean(outer_sensitivity),
    "sens_full_sd_cv": np.std(outer_sensitivity),
    "sens_diff": np.mean(outer_sensitivity) - null_sens,

    "spec_null": null_spec,
    "spec_full_mean_cv": np.mean(outer_specificity),
    "spec_full_sd_cv": np.std(outer_specificity),
    "spec_diff": np.mean(outer_specificity) - null_spec,

    "acc_null": null_acc,
    "acc_full_mean_cv": np.mean(outer_accuracy),
    "acc_full_sd_cv": np.std(outer_accuracy),
    "acc_diff": np.mean(outer_accuracy) - null_acc
}])

print("\n Performance summary:")
print(performance_results)

# Optional save
performance_results.to_csv("/home/jupyter/multiTRS/results/multi_TRS_w_PRS_SMR_single_SNP_XGBoost_nestedcv.txt", sep="\t", index=False)

#### Test in PPMI (EUR), PPMI (AJ) and HBS

In [ ]:
# =========================
# Load and join data from PDBP
# =========================
clinical_path = "/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt"
TRS_path = "/home/jupyter/multiTRS/scores/PDBP_EUR_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt"
PGS_path = "/home/jupyter/multiTRS/geno/PRS_scores_out/PDBP_EUR_PD_PGS_resid.txt"

clinical = pd.read_csv(clinical_path, sep="\t")[["participant_id", "case_control_other_at_baseline", "sex", "age_at_baseline"]]

TRS = pd.read_csv(TRS_path, sep="\t")
TRS_cols = ["participant_id"] + [c for c in TRS.columns if "single_SNP" in c and "Hip_Fracture" not in c]
TRS = TRS[TRS_cols]

PGS = pd.read_csv(PGS_path, sep="\t")[["participant_id", "PD_PGS_resid"]]

combined_data = pd.merge(clinical, PGS, on="participant_id", how="inner")
combined_data = pd.merge(combined_data, TRS, on="participant_id", how="inner")

print(f"Rows in combined PDBP data: {len(combined_data)}")

X = combined_data.drop(columns=["participant_id", "case_control_other_at_baseline"])
y = combined_data["case_control_other_at_baseline"]

print(f"Number of predictors: {X.shape[1]}")

# =========================
# Load saved hyperparameters and aggregate
# =========================
best_params_df = pd.read_csv("/home/jupyter/multiTRS/results/multi_TRS_w_PRS_SMR_single_SNP_XGBoost_nestedcv_hyperparameters_all_outer.txt", sep="\t")

performance_df = pd.read_csv("/home/jupyter/multiTRS/results/multi_TRS_w_PRS_SMR_single_SNP_XGBoost_nestedcv.txt", sep="\t")
print(f"\nLoaded mean nested CV AUC: {performance_df['AUC_full_mean_cv'].values[0]:.4f} SD: {performance_df['AUC_full_sd_cv'].values[0]:.4f}")

final_params = {
    'n_estimators':    int(mode(best_params_df['n_estimators'],    keepdims=True).mode[0]),
    'max_depth':       int(mode(best_params_df['max_depth'],       keepdims=True).mode[0]),
    'min_child_weight':int(mode(best_params_df['min_child_weight'],keepdims=True).mode[0]),
    'subsample':       best_params_df['subsample'].median(),
    'colsample_bytree':best_params_df['colsample_bytree'].median(),
    'learning_rate':   best_params_df['learning_rate'].median(),
    'gamma':           best_params_df['gamma'].median()
}

print("\nAggregated final hyperparameters:", final_params)

# =========================
# Fit final XGBoost model on full PDBP
# =========================
final_model = XGBClassifier(
    eval_metric='auc',
    random_state=1,
    **final_params
)

final_model.fit(X, y)

print("\nFinal XGBoost model fitted on full PDBP dataset.")

y_pred_prob = final_model.predict_proba(X)[:, 1]
final_auc = roc_auc_score(y, y_pred_prob)
print(f"Final XGBoost model AUC on full PDBP dataset: {final_auc:.4f}")

# =========================
# Extract and save SHAP values
# =========================
explainer = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X)
base_value = explainer.expected_value

print("\nSHAP values extracted from final XGBoost model.")
print(f"SHAP values shape: {shap_values.shape}")
print(f"Base value (expected model output): {base_value:.6f}")

shap_df = pd.DataFrame(shap_values, columns=X.columns)
shap_df.to_csv("/home/jupyter/multiTRS/results/multi_TRS_w_PRS_SMR_single_SNP_XGBoost_PDBP_SHAP_values.txt",
               sep="\t", index=False)
print("SHAP values saved to: /home/jupyter/multiTRS/results/multi_TRS_w_PRS_SMR_single_SNP_XGBoost_PDBP_SHAP_values.txt")

print("\nGenerating SHAP bar plot...")
plt.figure()
shap.summary_plot(shap_values, X, plot_type="bar", show=False)
plt.tight_layout()
plt.savefig("/home/jupyter/multiTRS/results/multi_TRS_w_PRS_SMR_single_SNP_XGBoost_PDBP_SHAP_bar_plot.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()
print("SHAP bar plot saved to: /home/jupyter/multiTRS/results/multi_TRS_w_PRS_SMR_single_SNP_XGBoost_PDBP_SHAP_bar_plot.png")

print("\nGenerating SHAP beeswarm plot...")
plt.figure()
shap.summary_plot(shap_values, X, show=False)
plt.tight_layout()
plt.savefig("/home/jupyter/multiTRS/results/multi_TRS_w_PRS_SMR_single_SNP_XGBoost_PDBP_SHAP_beeswarm_plot.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()
print("Beeswarm plot saved to: /home/jupyter/multiTRS/results/multi_TRS_w_PRS_SMR_single_SNP_XGBoost_PDBP_SHAP_beeswarm_plot.png")

# =========================
# Fit baseline logistic regression model on PDBP (sex + age + PGS)
# =========================
X_baseline = combined_data[["sex", "age_at_baseline", "PD_PGS_resid"]].copy()

scaler = StandardScaler()
cols_to_scale_baseline = ["age_at_baseline", "PD_PGS_resid"]
X_baseline_scaled = X_baseline.copy()
X_baseline_scaled[cols_to_scale_baseline] = scaler.fit_transform(X_baseline[cols_to_scale_baseline])

age_mean = scaler.mean_[0]
age_sd   = scaler.scale_[0]
pgs_mean = scaler.mean_[1]
pgs_sd   = scaler.scale_[1]

baseline_model = LogisticRegression(
    random_state=1,
    max_iter=2000,
    penalty=None,
    solver='newton-cholesky',
    fit_intercept=True
)
baseline_model.fit(X_baseline_scaled, y)

print("\nBaseline logistic regression model fitted on full PDBP dataset.")
print(f"Age scaling parameters - Mean: {age_mean:.4f}, SD: {age_sd:.4f}")
print(f"PGS scaling parameters - Mean: {pgs_mean:.4f}, SD: {pgs_sd:.4f}")
print(f"Baseline model intercept: {baseline_model.intercept_[0]:.6f}")
print(f"Baseline model coefficients (sex, age, PGS): {baseline_model.coef_[0]}")

y_baseline_pred_prob = baseline_model.predict_proba(X_baseline_scaled)[:, 1]
baseline_auc = roc_auc_score(y, y_baseline_pred_prob)
print(f"Baseline (sex + age + PGS) model AUC on full PDBP dataset: {baseline_auc:.4f}")

# =========================
# Closest.topleft function (pROC equivalent)
# =========================
def closest_topleft_threshold(y_true, y_prob):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    distances = np.sqrt((fpr)**2 + (1 - tpr)**2)
    idx = np.argmin(distances)
    return thresholds[idx], tpr[idx], 1 - fpr[idx]

def get_accuracy_at_threshold(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    return accuracy_score(y_true, y_pred)

# =========================
# External validation in PPMI_EUR, PPMI_AJ and HBS_EUR
# =========================
external_datasets = {
    "PPMI_EUR": {
        "TRS_path": "/home/jupyter/multiTRS/scores/PPMI_EUR_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt",
        "PGS_path": "/home/jupyter/multiTRS/geno/PRS_scores_out/PPMI_EUR_PD_PGS_resid.txt"
    },
    "PPMI_AJ": {
        "TRS_path": "/home/jupyter/multiTRS/scores/PPMI_AJ_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt",
        "PGS_path": "/home/jupyter/multiTRS/geno/PRS_scores_out/PPMI_AJ_PD_PGS_resid.txt"
    },
    "HBS_EUR": {
        "TRS_path": "/home/jupyter/multiTRS/scores/HBS_EUR_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt",
        "PGS_path": "/home/jupyter/multiTRS/geno/PRS_scores_out/HBS_EUR_PD_PGS_resid.txt"
    }
}

print("\n" + "="*70)
print("EXTERNAL VALIDATION RESULTS WITH DELONG'S TEST")
print("="*70)

full_results_summary = []

for dataset_name, paths in external_datasets.items():
    print(f"\n--- {dataset_name} Dataset ---")

    # Load TRS
    ext_TRS = pd.read_csv(paths["TRS_path"], sep="\t")
    TRS_cols = ["participant_id"] + [c for c in ext_TRS.columns if "single_SNP" in c and "Hip_Fracture" not in c]
    ext_TRS = ext_TRS[TRS_cols]

    # Load PGS
    ext_PGS = pd.read_csv(paths["PGS_path"], sep="\t")[["participant_id", "PD_PGS_resid"]]

    # Merge
    ext_combined = pd.merge(clinical, ext_PGS, on="participant_id", how="inner")
    ext_combined = pd.merge(ext_combined, ext_TRS, on="participant_id", how="inner")
    print(f"Rows in combined data for {dataset_name}: {len(ext_combined)}")

    # Predictions - full model
    X_ext = ext_combined[X.columns]
    y_ext = ext_combined["case_control_other_at_baseline"]
    y_ext_pred_prob = final_model.predict_proba(X_ext)[:, 1]

    # Predictions - baseline model (scaled with PDBP parameters)
    X_ext_baseline = ext_combined[["sex", "age_at_baseline", "PD_PGS_resid"]].copy()
    X_ext_baseline_scaled = X_ext_baseline.copy()
    X_ext_baseline_scaled["age_at_baseline"] = (X_ext_baseline["age_at_baseline"] - age_mean) / age_sd
    X_ext_baseline_scaled["PD_PGS_resid"]    = (X_ext_baseline["PD_PGS_resid"]    - pgs_mean) / pgs_sd
    y_ext_baseline_pred_prob = baseline_model.predict_proba(X_ext_baseline_scaled)[:, 1]

    # DeLong test
    z_stat, p_value, ci_full, ci_null, auc_full, auc_null, info = Delong_test(
        y_ext, y_ext_pred_prob, y_ext_baseline_pred_prob,
        return_ci=True, return_auc=True, verbose=0
    )

    print(f"XGBoost model AUC:                   {auc_full:.4f} (95% CI: {ci_full[0]:.4f}-{ci_full[1]:.4f})")
    print(f"Baseline model AUC:                  {auc_null:.4f} (95% CI: {ci_null[0]:.4f}-{ci_null[1]:.4f})")
    print(f"AUC Difference (XGBoost - Baseline): {auc_full - auc_null:.4f}")
    print(f"Variance of AUC difference:          {info['var_diff']:.6f}")
    print(f"DeLong's test Z-statistic:           {z_stat:.4f}")
    print(f"DeLong's test p-value (2-tailed):    {p_value:.4e}")
    print(f"Result: {'Significantly different (p < 0.05)' if p_value < 0.05 else 'Not significantly different (p >= 0.05)'}")

    # Optimal threshold metrics
    thresh_full, sens_full, spec_full = closest_topleft_threshold(y_ext, y_ext_pred_prob)
    thresh_null, sens_null, spec_null = closest_topleft_threshold(y_ext, y_ext_baseline_pred_prob)

    acc_full = get_accuracy_at_threshold(y_ext, y_ext_pred_prob,          thresh_full)
    acc_null = get_accuracy_at_threshold(y_ext, y_ext_baseline_pred_prob, thresh_null)

    full_results_summary.append({
        'cohort':         dataset_name,
        'AUC_null':       auc_null,
        'AUC_lower_null': ci_null[0],
        'AUC_upper_null': ci_null[1],
        'AUC_full':       auc_full,
        'AUC_lower_full': ci_full[0],
        'AUC_upper_full': ci_full[1],
        'AUC_diff':       auc_full - auc_null,
        'sens_null':      sens_null,
        'sens_full':      sens_full,
        'sens_diff':      sens_full - sens_null,
        'spec_null':      spec_null,
        'spec_full':      spec_full,
        'spec_diff':      spec_full - spec_null,
        'acc_null':       acc_null,
        'acc_full':       acc_full,
        'acc_diff':       acc_full - acc_null,
        'delong_z':       z_stat,
        'delong_p':       p_value
    })

# =========================
# Save full results table
# =========================
print("\n" + "="*70)
print("FULL RESULTS TABLE")
print("="*70)
full_results_df = pd.DataFrame(full_results_summary)
print(full_results_df.to_string(index=False))

full_results_outfile = "/home/jupyter/multiTRS/results/multi_TRS_w_PRS_SMR_single_SNP_XGBoost_external_validation_full_results.txt"
full_results_df.to_csv(full_results_outfile, sep="\t", index=False)
print(f"\nFull results saved to: {full_results_outfile}")

### FUSION (TWAS)

#### Nested-CV

In [ ]:
# =========================
# Closest.topleft function (pROC equivalent)
# =========================
def closest_topleft_threshold(y_true, y_prob):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    distances = np.sqrt((fpr)**2 + (1 - tpr)**2)
    idx = np.argmin(distances)
    return thresholds[idx], tpr[idx], 1 - fpr[idx]

# =========================
# Load and join data
# =========================
clinical_path = "/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt"
TRS_path = "/home/jupyter/multiTRS/scores/PDBP_EUR_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt"
PGS_path = "/home/jupyter/multiTRS/geno/PRS_scores_out/PDBP_EUR_PD_PGS_resid.txt"

clinical = pd.read_csv(clinical_path, sep="\t")[[
    "participant_id", "case_control_other_at_baseline", "sex", "age_at_baseline"
]]

TRS = pd.read_csv(TRS_path, sep="\t")
TRS_cols = ["participant_id"] + [c for c in TRS.columns if "fusion" in c]
TRS = TRS[TRS_cols]

PGS = pd.read_csv(PGS_path, sep="\t")[["participant_id", "PD_PGS_resid"]]

combined_data = pd.merge(clinical, PGS, on="participant_id", how="inner")
combined_data = pd.merge(combined_data, TRS, on="participant_id", how="inner")

print(f"Rows in combined PDBP data: {len(combined_data)}")

X = combined_data.drop(columns=["participant_id", "case_control_other_at_baseline"])
y = combined_data["case_control_other_at_baseline"]

print(f"Number of predictors: {X.shape[1]}")

# =========================
# Hyperparameter grid
# =========================
param_dist = {
    'n_estimators': [100, 200, 300, 500, 700, 1000],
    'max_depth': [3, 5, 7, 9],
    'min_child_weight': [5, 10, 20, 30, 40],
    'colsample_bytree': [0.4, 0.6, 0.8, 1],
    'subsample': [0.6, 0.8],
    'learning_rate': [0.001, 0.0015, 0.01, 0.015, 0.1],
    'gamma': [0, 0.1, 0.3, 0.5, 0.8, 1.0]
}

# =========================
# Nested CV
# =========================
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)

outer_auc = []
outer_sensitivity = []
outer_specificity = []
outer_accuracy = []

best_params_list = []

for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), 1):
    print(f"\n--- Outer fold {fold} ---")

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = XGBClassifier(eval_metric='auc', random_state=1)

    random_search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_dist,
        n_iter=100,
        scoring="roc_auc",
        cv=5,
        n_jobs=-1,
        random_state=1
    )

    random_search.fit(X_train, y_train)

    best_params_list.append(random_search.best_params_)

    # Predictions
    y_prob = random_search.best_estimator_.predict_proba(X_test)[:, 1]

    # AUC
    auc = roc_auc_score(y_test, y_prob)
    outer_auc.append(auc)

    # closest.topleft threshold
    thresh, sens, spec = closest_topleft_threshold(y_test, y_prob)

    y_pred = (y_prob >= thresh).astype(int)
    acc = (y_pred == y_test).mean()

    outer_sensitivity.append(sens)
    outer_specificity.append(spec)
    outer_accuracy.append(acc)

    print(f"AUC: {auc:.4f}, Sens: {sens:.3f}, Spec: {spec:.3f}, Acc: {acc:.3f}")

print("\nNested CV results:")
print(f"AUC mean: {np.mean(outer_auc):.4f} ± {np.std(outer_auc):.4f}")

# =========================
# Aggregate hyperparameters
# =========================
best_params_df = pd.DataFrame(best_params_list)
best_params_df.to_csv("/home/jupyter/multiTRS/results/multi_TRS_w_PRS_FUSION_XGBoost_nestedcv_hyperparameters_all_outer.txt",
                       sep="\t", index=False)
print("\nHyperparameters saved to: /home/jupyter/multiTRS/results/multi_TRS_w_PRS_FUSION_XGBoost_nestedcv_hyperparameters_all_outer.txt")
print(best_params_df)

final_params = {
    'n_estimators': int(mode(best_params_df['n_estimators'], keepdims=True).mode[0]),
    'max_depth': int(mode(best_params_df['max_depth'], keepdims=True).mode[0]),
    'min_child_weight': int(mode(best_params_df['min_child_weight'], keepdims=True).mode[0]),

    'subsample': best_params_df['subsample'].median(),
    'colsample_bytree': best_params_df['colsample_bytree'].median(),
    'learning_rate': best_params_df['learning_rate'].median(),
    'gamma': best_params_df['gamma'].median()
}


final_params_df = pd.DataFrame([final_params])

final_params_df.to_csv(
    "/home/jupyter/multiTRS/results/multi_TRS_w_PRS_FUSION_XGBoost_nestedcv_hyperparameters_best.txt",
    sep="\t",
    index=False
)
print("\nHyperparameters saved to: /home/jupyter/multiTRS/results/multi_TRS_w_PRS_FUSION_XGBoost_nestedcv_hyperparameters_best.txt")
print(final_params_df)

# Snap continuous params to grid
#for param in ['subsample', 'colsample_bytree', 'learning_rate', 'gamma']:
 #   final_params[param] = min(param_dist[param], key=lambda x: abs(x - final_params[param]))


print("\nFinal aggregated hyperparameters:")
print(final_params)

# =========================
# Null model (baseline)
# =========================
X_baseline = combined_data[["sex", "age_at_baseline", "PD_PGS_resid"]].copy()

scaler = StandardScaler()
X_baseline_scaled = X_baseline.copy()
X_baseline_scaled[["age_at_baseline", "PD_PGS_resid"]] = scaler.fit_transform(
    X_baseline[["age_at_baseline", "PD_PGS_resid"]]
)

baseline_model = LogisticRegression(
    random_state=1,
    max_iter=2000,
    penalty=None,
    solver='newton-cholesky'
)

baseline_model.fit(X_baseline_scaled, y)

y_prob_null = baseline_model.predict_proba(X_baseline_scaled)[:, 1]

auc_null = roc_auc_score(y, y_prob_null)

# closest.topleft for null
null_thresh, null_sens, null_spec = closest_topleft_threshold(y, y_prob_null)

y_pred_null = (y_prob_null >= null_thresh).astype(int)
null_acc = (y_pred_null == y).mean()

print("\nNull model performance:")
print(f"AUC: {auc_null:.4f}, Sens: {null_sens:.3f}, Spec: {null_spec:.3f}, Acc: {null_acc:.3f}")

# =========================
# Final performance table
# =========================
performance_results = pd.DataFrame([{
    "cohort": "PDBP",
    "scores": "all_scores",
    "model": "XGBoost",

    "aggregated_params": str(final_params),

    "AUC_null": auc_null,
    "AUC_full_mean_cv": np.mean(outer_auc),
    "AUC_full_sd_cv": np.std(outer_auc),
    "AUC_diff": np.mean(outer_auc) - auc_null,

    "sens_null": null_sens,
    "sens_full_mean_cv": np.mean(outer_sensitivity),
    "sens_full_sd_cv": np.std(outer_sensitivity),
    "sens_diff": np.mean(outer_sensitivity) - null_sens,

    "spec_null": null_spec,
    "spec_full_mean_cv": np.mean(outer_specificity),
    "spec_full_sd_cv": np.std(outer_specificity),
    "spec_diff": np.mean(outer_specificity) - null_spec,

    "acc_null": null_acc,
    "acc_full_mean_cv": np.mean(outer_accuracy),
    "acc_full_sd_cv": np.std(outer_accuracy),
    "acc_diff": np.mean(outer_accuracy) - null_acc
}])

print("\n Performance summary:")
print(performance_results)

# Optional save
performance_results.to_csv("/home/jupyter/multiTRS/results/multi_TRS_w_PRS_FUSION_XGBoost_nestedcv.txt", sep="\t", index=False)

#### Test in PPMI (EUR), PPMI (AJ) and HBS

In [ ]:
# =========================
# Load and join data from PDBP
# =========================
clinical_path = "/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt"
TRS_path = "/home/jupyter/multiTRS/scores/PDBP_EUR_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt"
PGS_path = "/home/jupyter/multiTRS/geno/PRS_scores_out/PDBP_EUR_PD_PGS_resid.txt"

clinical = pd.read_csv(clinical_path, sep="\t")[["participant_id", "case_control_other_at_baseline", "sex", "age_at_baseline"]]

TRS = pd.read_csv(TRS_path, sep="\t")
TRS_cols = ["participant_id"] + [c for c in TRS.columns if "fusion" in c]
TRS = TRS[TRS_cols]

PGS = pd.read_csv(PGS_path, sep="\t")[["participant_id", "PD_PGS_resid"]]

combined_data = pd.merge(clinical, PGS, on="participant_id", how="inner")
combined_data = pd.merge(combined_data, TRS, on="participant_id", how="inner")

print(f"Rows in combined PDBP data: {len(combined_data)}")

X = combined_data.drop(columns=["participant_id", "case_control_other_at_baseline"])
y = combined_data["case_control_other_at_baseline"]

print(f"Number of predictors: {X.shape[1]}")

# =========================
# Load saved hyperparameters and aggregate
# =========================
best_params_df = pd.read_csv("/home/jupyter/multiTRS/results/multi_TRS_w_PRS_FUSION_XGBoost_nestedcv_hyperparameters_all_outer.txt", sep="\t")

performance_df = pd.read_csv("/home/jupyter/multiTRS/results/multi_TRS_w_PRS_FUSION_XGBoost_nestedcv.txt", sep="\t")
print(f"\nLoaded mean nested CV AUC: {performance_df['AUC_full_mean_cv'].values[0]:.4f} SD: {performance_df['AUC_full_sd_cv'].values[0]:.4f}")

final_params = {
    'n_estimators':    int(mode(best_params_df['n_estimators'],    keepdims=True).mode[0]),
    'max_depth':       int(mode(best_params_df['max_depth'],       keepdims=True).mode[0]),
    'min_child_weight':int(mode(best_params_df['min_child_weight'],keepdims=True).mode[0]),
    'subsample':       best_params_df['subsample'].median(),
    'colsample_bytree':best_params_df['colsample_bytree'].median(),
    'learning_rate':   best_params_df['learning_rate'].median(),
    'gamma':           best_params_df['gamma'].median()
}

print("\nAggregated final hyperparameters:", final_params)

# =========================
# Fit final XGBoost model on full PDBP
# =========================
final_model = XGBClassifier(
    eval_metric='auc',
    random_state=1,
    **final_params
)

final_model.fit(X, y)

print("\nFinal XGBoost model fitted on full PDBP dataset.")

y_pred_prob = final_model.predict_proba(X)[:, 1]
final_auc = roc_auc_score(y, y_pred_prob)
print(f"Final XGBoost model AUC on full PDBP dataset: {final_auc:.4f}")

# =========================
# Extract and save SHAP values
# =========================
explainer = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X)
base_value = explainer.expected_value

print("\nSHAP values extracted from final XGBoost model.")
print(f"SHAP values shape: {shap_values.shape}")
print(f"Base value (expected model output): {base_value:.6f}")

shap_df = pd.DataFrame(shap_values, columns=X.columns)
shap_df.to_csv("/home/jupyter/multiTRS/results/multi_TRS_w_PRS_FUSION_XGBoost_PDBP_SHAP_values.txt",
               sep="\t", index=False)
print("SHAP values saved to: /home/jupyter/multiTRS/results/multi_TRS_w_PRS_FUSION_XGBoost_PDBP_SHAP_values.txt")

print("\nGenerating SHAP bar plot...")
plt.figure()
shap.summary_plot(shap_values, X, plot_type="bar", show=False)
plt.tight_layout()
plt.savefig("/home/jupyter/multiTRS/results/multi_TRS_w_PRS_FUSION_XGBoost_PDBP_SHAP_bar_plot.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()
print("SHAP bar plot saved to: /home/jupyter/multiTRS/results/multi_TRS_w_PRS_FUSION_XGBoost_PDBP_SHAP_bar_plot.png")

print("\nGenerating SHAP beeswarm plot...")
plt.figure()
shap.summary_plot(shap_values, X, show=False)
plt.tight_layout()
plt.savefig("/home/jupyter/multiTRS/results/multi_TRS_w_PRS_FUSION_XGBoost_PDBP_SHAP_beeswarm_plot.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()
print("Beeswarm plot saved to: /home/jupyter/multiTRS/results/multi_TRS_w_PRS_FUSION_XGBoost_PDBP_SHAP_beeswarm_plot.png")

# =========================
# Fit baseline logistic regression model on PDBP (sex + age + PGS)
# =========================
X_baseline = combined_data[["sex", "age_at_baseline", "PD_PGS_resid"]].copy()

scaler = StandardScaler()
cols_to_scale_baseline = ["age_at_baseline", "PD_PGS_resid"]
X_baseline_scaled = X_baseline.copy()
X_baseline_scaled[cols_to_scale_baseline] = scaler.fit_transform(X_baseline[cols_to_scale_baseline])

age_mean = scaler.mean_[0]
age_sd   = scaler.scale_[0]
pgs_mean = scaler.mean_[1]
pgs_sd   = scaler.scale_[1]

baseline_model = LogisticRegression(
    random_state=1,
    max_iter=2000,
    penalty=None,
    solver='newton-cholesky',
    fit_intercept=True
)
baseline_model.fit(X_baseline_scaled, y)

print("\nBaseline logistic regression model fitted on full PDBP dataset.")
print(f"Age scaling parameters - Mean: {age_mean:.4f}, SD: {age_sd:.4f}")
print(f"PGS scaling parameters - Mean: {pgs_mean:.4f}, SD: {pgs_sd:.4f}")
print(f"Baseline model intercept: {baseline_model.intercept_[0]:.6f}")
print(f"Baseline model coefficients (sex, age, PGS): {baseline_model.coef_[0]}")

y_baseline_pred_prob = baseline_model.predict_proba(X_baseline_scaled)[:, 1]
baseline_auc = roc_auc_score(y, y_baseline_pred_prob)
print(f"Baseline (sex + age + PGS) model AUC on full PDBP dataset: {baseline_auc:.4f}")

# =========================
# Closest.topleft function (pROC equivalent)
# =========================
def closest_topleft_threshold(y_true, y_prob):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    distances = np.sqrt((fpr)**2 + (1 - tpr)**2)
    idx = np.argmin(distances)
    return thresholds[idx], tpr[idx], 1 - fpr[idx]

def get_accuracy_at_threshold(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    return accuracy_score(y_true, y_pred)

# =========================
# External validation in PPMI_EUR, PPMI_AJ and HBS_EUR
# =========================
external_datasets = {
    "PPMI_EUR": {
        "TRS_path": "/home/jupyter/multiTRS/scores/PPMI_EUR_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt",
        "PGS_path": "/home/jupyter/multiTRS/geno/PRS_scores_out/PPMI_EUR_PD_PGS_resid.txt"
    },
    "PPMI_AJ": {
        "TRS_path": "/home/jupyter/multiTRS/scores/PPMI_AJ_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt",
        "PGS_path": "/home/jupyter/multiTRS/geno/PRS_scores_out/PPMI_AJ_PD_PGS_resid.txt"
    },
    "HBS_EUR": {
        "TRS_path": "/home/jupyter/multiTRS/scores/HBS_EUR_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt",
        "PGS_path": "/home/jupyter/multiTRS/geno/PRS_scores_out/HBS_EUR_PD_PGS_resid.txt"
    }
}

print("\n" + "="*70)
print("EXTERNAL VALIDATION RESULTS WITH DELONG'S TEST")
print("="*70)

full_results_summary = []

for dataset_name, paths in external_datasets.items():
    print(f"\n--- {dataset_name} Dataset ---")

    # Load TRS
    ext_TRS = pd.read_csv(paths["TRS_path"], sep="\t")
    TRS_cols = ["participant_id"] + [c for c in ext_TRS.columns if "fusion" in c]
    ext_TRS = ext_TRS[TRS_cols]

    # Load PGS
    ext_PGS = pd.read_csv(paths["PGS_path"], sep="\t")[["participant_id", "PD_PGS_resid"]]

    # Merge
    ext_combined = pd.merge(clinical, ext_PGS, on="participant_id", how="inner")
    ext_combined = pd.merge(ext_combined, ext_TRS, on="participant_id", how="inner")
    print(f"Rows in combined data for {dataset_name}: {len(ext_combined)}")

    # Predictions - full model
    X_ext = ext_combined[X.columns]
    y_ext = ext_combined["case_control_other_at_baseline"]
    y_ext_pred_prob = final_model.predict_proba(X_ext)[:, 1]

    # Predictions - baseline model (scaled with PDBP parameters)
    X_ext_baseline = ext_combined[["sex", "age_at_baseline", "PD_PGS_resid"]].copy()
    X_ext_baseline_scaled = X_ext_baseline.copy()
    X_ext_baseline_scaled["age_at_baseline"] = (X_ext_baseline["age_at_baseline"] - age_mean) / age_sd
    X_ext_baseline_scaled["PD_PGS_resid"]    = (X_ext_baseline["PD_PGS_resid"]    - pgs_mean) / pgs_sd
    y_ext_baseline_pred_prob = baseline_model.predict_proba(X_ext_baseline_scaled)[:, 1]

    # DeLong test
    z_stat, p_value, ci_full, ci_null, auc_full, auc_null, info = Delong_test(
        y_ext, y_ext_pred_prob, y_ext_baseline_pred_prob,
        return_ci=True, return_auc=True, verbose=0
    )

    print(f"XGBoost model AUC:                   {auc_full:.4f} (95% CI: {ci_full[0]:.4f}-{ci_full[1]:.4f})")
    print(f"Baseline model AUC:                  {auc_null:.4f} (95% CI: {ci_null[0]:.4f}-{ci_null[1]:.4f})")
    print(f"AUC Difference (XGBoost - Baseline): {auc_full - auc_null:.4f}")
    print(f"Variance of AUC difference:          {info['var_diff']:.6f}")
    print(f"DeLong's test Z-statistic:           {z_stat:.4f}")
    print(f"DeLong's test p-value (2-tailed):    {p_value:.4e}")
    print(f"Result: {'Significantly different (p < 0.05)' if p_value < 0.05 else 'Not significantly different (p >= 0.05)'}")

    # Optimal threshold metrics
    thresh_full, sens_full, spec_full = closest_topleft_threshold(y_ext, y_ext_pred_prob)
    thresh_null, sens_null, spec_null = closest_topleft_threshold(y_ext, y_ext_baseline_pred_prob)

    acc_full = get_accuracy_at_threshold(y_ext, y_ext_pred_prob,          thresh_full)
    acc_null = get_accuracy_at_threshold(y_ext, y_ext_baseline_pred_prob, thresh_null)

    full_results_summary.append({
        'cohort':         dataset_name,
        'AUC_null':       auc_null,
        'AUC_lower_null': ci_null[0],
        'AUC_upper_null': ci_null[1],
        'AUC_full':       auc_full,
        'AUC_lower_full': ci_full[0],
        'AUC_upper_full': ci_full[1],
        'AUC_diff':       auc_full - auc_null,
        'sens_null':      sens_null,
        'sens_full':      sens_full,
        'sens_diff':      sens_full - sens_null,
        'spec_null':      spec_null,
        'spec_full':      spec_full,
        'spec_diff':      spec_full - spec_null,
        'acc_null':       acc_null,
        'acc_full':       acc_full,
        'acc_diff':       acc_full - acc_null,
        'delong_z':       z_stat,
        'delong_p':       p_value
    })

# =========================
# Save full results table
# =========================
print("\n" + "="*70)
print("FULL RESULTS TABLE")
print("="*70)
full_results_df = pd.DataFrame(full_results_summary)
print(full_results_df.to_string(index=False))

full_results_outfile = "/home/jupyter/multiTRS/results/multi_TRS_w_PRS_FUSION_XGBoost_external_validation_full_results.txt"
full_results_df.to_csv(full_results_outfile, sep="\t", index=False)
print(f"\nFull results saved to: {full_results_outfile}")

### All scores

#### Nested-CV

In [ ]:
# =========================
# Closest.topleft function (pROC equivalent)
# =========================
def closest_topleft_threshold(y_true, y_prob):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    distances = np.sqrt((fpr)**2 + (1 - tpr)**2)
    idx = np.argmin(distances)
    return thresholds[idx], tpr[idx], 1 - fpr[idx]

# =========================
# Load and join data
# =========================
clinical_path = "/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt"
TRS_path = "/home/jupyter/multiTRS/scores/PDBP_EUR_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt"
PGS_path = "/home/jupyter/multiTRS/geno/PRS_scores_out/PDBP_EUR_PD_PGS_resid.txt"

clinical = pd.read_csv(clinical_path, sep="\t")[[
    "participant_id", "case_control_other_at_baseline", "sex", "age_at_baseline"
]]

TRS = pd.read_csv(TRS_path, sep="\t")
TRS_cols = [c for c in TRS.columns if "Hip_fracture_EUR_2022_FDR_SMR_single_SNP" not in c]
TRS = TRS[TRS_cols]

PGS = pd.read_csv(PGS_path, sep="\t")[["participant_id", "PD_PGS_resid"]]

combined_data = pd.merge(clinical, PGS, on="participant_id", how="inner")
combined_data = pd.merge(combined_data, TRS, on="participant_id", how="inner")

print(f"Rows in combined PDBP data: {len(combined_data)}")

X = combined_data.drop(columns=["participant_id", "case_control_other_at_baseline"])
y = combined_data["case_control_other_at_baseline"]

print(f"Number of predictors: {X.shape[1]}")

# =========================
# Hyperparameter grid
# =========================
param_dist = {
    'n_estimators': [100, 200, 300, 500, 700, 1000],
    'max_depth': [3, 5, 7, 9],
    'min_child_weight': [5, 10, 20, 30, 40],
    'colsample_bytree': [0.4, 0.6, 0.8, 1],
    'subsample': [0.6, 0.8],
    'learning_rate': [0.001, 0.0015, 0.01, 0.015, 0.1],
    'gamma': [0, 0.1, 0.3, 0.5, 0.8, 1.0]
}

# =========================
# Nested CV
# =========================
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)

outer_auc = []
outer_sensitivity = []
outer_specificity = []
outer_accuracy = []

best_params_list = []

for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), 1):
    print(f"\n--- Outer fold {fold} ---")

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = XGBClassifier(eval_metric='auc', random_state=1)

    random_search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_dist,
        n_iter=100,
        scoring="roc_auc",
        cv=5,
        n_jobs=-1,
        random_state=1
    )

    random_search.fit(X_train, y_train)

    best_params_list.append(random_search.best_params_)

    # Predictions
    y_prob = random_search.best_estimator_.predict_proba(X_test)[:, 1]

    # AUC
    auc = roc_auc_score(y_test, y_prob)
    outer_auc.append(auc)

    # closest.topleft threshold
    thresh, sens, spec = closest_topleft_threshold(y_test, y_prob)

    y_pred = (y_prob >= thresh).astype(int)
    acc = (y_pred == y_test).mean()

    outer_sensitivity.append(sens)
    outer_specificity.append(spec)
    outer_accuracy.append(acc)

    print(f"AUC: {auc:.4f}, Sens: {sens:.3f}, Spec: {spec:.3f}, Acc: {acc:.3f}")

print("\nNested CV results:")
print(f"AUC mean: {np.mean(outer_auc):.4f} ± {np.std(outer_auc):.4f}")

# =========================
# Aggregate hyperparameters
# =========================
best_params_df = pd.DataFrame(best_params_list)
best_params_df.to_csv("/home/jupyter/multiTRS/results/multi_TRS_w_PRS_all_scores_XGBoost_nestedcv_hyperparameters_all_outer.txt",
                       sep="\t", index=False)
print("\nHyperparameters saved to: /home/jupyter/multiTRS/results/multi_TRS_w_PRS_all_scores_XGBoost_nestedcv_hyperparameters_all_outer.txt")
print(best_params_df)

final_params = {
    'n_estimators': int(mode(best_params_df['n_estimators'], keepdims=True).mode[0]),
    'max_depth': int(mode(best_params_df['max_depth'], keepdims=True).mode[0]),
    'min_child_weight': int(mode(best_params_df['min_child_weight'], keepdims=True).mode[0]),

    'subsample': best_params_df['subsample'].median(),
    'colsample_bytree': best_params_df['colsample_bytree'].median(),
    'learning_rate': best_params_df['learning_rate'].median(),
    'gamma': best_params_df['gamma'].median()
}


final_params_df = pd.DataFrame([final_params])

final_params_df.to_csv(
    "/home/jupyter/multiTRS/results/multi_TRS_w_PRS_all_scores_XGBoost_nestedcv_hyperparameters_best.txt",
    sep="\t",
    index=False
)
print("\nHyperparameters saved to: /home/jupyter/multiTRS/results/multi_TRS_w_PRS_all_scores_XGBoost_nestedcv_hyperparameters_best.txt")
print(final_params_df)

# Snap continuous params to grid
#for param in ['subsample', 'colsample_bytree', 'learning_rate', 'gamma']:
 #   final_params[param] = min(param_dist[param], key=lambda x: abs(x - final_params[param]))


print("\nFinal aggregated hyperparameters:")
print(final_params)

# =========================
# Null model (baseline)
# =========================
X_baseline = combined_data[["sex", "age_at_baseline", "PD_PGS_resid"]].copy()

scaler = StandardScaler()
X_baseline_scaled = X_baseline.copy()
X_baseline_scaled[["age_at_baseline", "PD_PGS_resid"]] = scaler.fit_transform(
    X_baseline[["age_at_baseline", "PD_PGS_resid"]]
)

baseline_model = LogisticRegression(
    random_state=1,
    max_iter=2000,
    penalty=None,
    solver='newton-cholesky'
)

baseline_model.fit(X_baseline_scaled, y)

y_prob_null = baseline_model.predict_proba(X_baseline_scaled)[:, 1]

auc_null = roc_auc_score(y, y_prob_null)

# closest.topleft for null
null_thresh, null_sens, null_spec = closest_topleft_threshold(y, y_prob_null)

y_pred_null = (y_prob_null >= null_thresh).astype(int)
null_acc = (y_pred_null == y).mean()

print("\nNull model performance:")
print(f"AUC: {auc_null:.4f}, Sens: {null_sens:.3f}, Spec: {null_spec:.3f}, Acc: {null_acc:.3f}")

# =========================
# Final performance table
# =========================
performance_results = pd.DataFrame([{
    "cohort": "PDBP",
    "scores": "all_scores",
    "model": "XGBoost",

    "aggregated_params": str(final_params),

    "AUC_null": auc_null,
    "AUC_full_mean_cv": np.mean(outer_auc),
    "AUC_full_sd_cv": np.std(outer_auc),
    "AUC_diff": np.mean(outer_auc) - auc_null,

    "sens_null": null_sens,
    "sens_full_mean_cv": np.mean(outer_sensitivity),
    "sens_full_sd_cv": np.std(outer_sensitivity),
    "sens_diff": np.mean(outer_sensitivity) - null_sens,

    "spec_null": null_spec,
    "spec_full_mean_cv": np.mean(outer_specificity),
    "spec_full_sd_cv": np.std(outer_specificity),
    "spec_diff": np.mean(outer_specificity) - null_spec,

    "acc_null": null_acc,
    "acc_full_mean_cv": np.mean(outer_accuracy),
    "acc_full_sd_cv": np.std(outer_accuracy),
    "acc_diff": np.mean(outer_accuracy) - null_acc
}])

print("\n Performance summary:")
print(performance_results)

# Optional save
performance_results.to_csv("/home/jupyter/multiTRS/results/multi_TRS_w_PRS_all_scores_XGBoost_nestedcv.txt", sep="\t", index=False)

#### Test in PPMI (EUR), PPMI (AJ) and HBS

In [ ]:
from sklearn.metrics import accuracy_score

# =========================
# Load and join data from PDBP
# =========================
clinical_path = "/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt"
TRS_path = "/home/jupyter/multiTRS/scores/PDBP_EUR_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt"
PGS_path = "/home/jupyter/multiTRS/geno/PRS_scores_out/PDBP_EUR_PD_PGS_resid.txt"

clinical = pd.read_csv(clinical_path, sep="\t")[["participant_id", "case_control_other_at_baseline", "sex", "age_at_baseline"]]

TRS = pd.read_csv(TRS_path, sep="\t")
TRS_cols = [c for c in TRS.columns if "Hip_fracture_EUR_2022_FDR_SMR_single_SNP" not in c]
TRS = TRS[TRS_cols]

PGS = pd.read_csv(PGS_path, sep="\t")[["participant_id", "PD_PGS_resid"]]

combined_data = pd.merge(clinical, PGS, on="participant_id", how="inner")
combined_data = pd.merge(combined_data, TRS, on="participant_id", how="inner")

print(f"Rows in combined PDBP data: {len(combined_data)}")

X = combined_data.drop(columns=["participant_id", "case_control_other_at_baseline"])
y = combined_data["case_control_other_at_baseline"]

print(f"Number of predictors: {X.shape[1]}")

# =========================
# Load saved hyperparameters and aggregate
# =========================
best_params_df = pd.read_csv("/home/jupyter/multiTRS/results/multi_TRS_w_PRS_all_scores_XGBoost_nestedcv_hyperparameters_all_outer.txt", sep="\t")

performance_df = pd.read_csv("/home/jupyter/multiTRS/results/multi_TRS_w_PRS_all_scores_XGBoost_nestedcv.txt", sep="\t")
print(f"\nLoaded mean nested CV AUC: {performance_df['AUC_full_mean_cv'].values[0]:.4f} SD: {performance_df['AUC_full_sd_cv'].values[0]:.4f}")

final_params = {
    'n_estimators':    int(mode(best_params_df['n_estimators'],    keepdims=True).mode[0]),
    'max_depth':       int(mode(best_params_df['max_depth'],       keepdims=True).mode[0]),
    'min_child_weight':int(mode(best_params_df['min_child_weight'],keepdims=True).mode[0]),
    'subsample':       best_params_df['subsample'].median(),
    'colsample_bytree':best_params_df['colsample_bytree'].median(),
    'learning_rate':   best_params_df['learning_rate'].median(),
    'gamma':           best_params_df['gamma'].median()
}

print("\nAggregated final hyperparameters:", final_params)

# =========================
# Fit final XGBoost model on full PDBP
# =========================
final_model = XGBClassifier(
    eval_metric='auc',
    random_state=1,
    **final_params
)

final_model.fit(X, y)

print("\nFinal XGBoost model fitted on full PDBP dataset.")

y_pred_prob = final_model.predict_proba(X)[:, 1]
final_auc = roc_auc_score(y, y_pred_prob)
print(f"Final XGBoost model AUC on full PDBP dataset: {final_auc:.4f}")

# =========================
# Extract and save SHAP values
# =========================
explainer = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X)
base_value = explainer.expected_value

print("\nSHAP values extracted from final XGBoost model.")
print(f"SHAP values shape: {shap_values.shape}")
print(f"Base value (expected model output): {base_value:.6f}")

shap_df = pd.DataFrame(shap_values, columns=X.columns)
shap_df.to_csv("/home/jupyter/multiTRS/results/multi_TRS_w_PRS_all_scores_XGBoost_PDBP_SHAP_values.txt",
               sep="\t", index=False)
print("SHAP values saved to: /home/jupyter/multiTRS/results/multi_TRS_w_PRS_all_scores_XGBoost_PDBP_SHAP_values.txt")

print("\nGenerating SHAP bar plot...")
plt.figure()
shap.summary_plot(shap_values, X, plot_type="bar", show=False)
plt.tight_layout()
plt.savefig("/home/jupyter/multiTRS/results/multi_TRS_w_PRS_all_scores_XGBoost_PDBP_SHAP_bar_plot.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()
print("SHAP bar plot saved to: /home/jupyter/multiTRS/results/multi_TRS_w_PRS_all_scores_XGBoost_PDBP_SHAP_bar_plot.png")

print("\nGenerating SHAP beeswarm plot...")
plt.figure()
shap.summary_plot(shap_values, X, show=False)
plt.tight_layout()
plt.savefig("/home/jupyter/multiTRS/results/multi_TRS_w_PRS_all_scores_XGBoost_PDBP_SHAP_beeswarm_plot.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()
print("Beeswarm plot saved to: /home/jupyter/multiTRS/results/multi_TRS_w_PRS_all_scores_XGBoost_PDBP_SHAP_beeswarm_plot.png")

# =========================
# Fit baseline logistic regression model on PDBP (sex + age + PGS)
# =========================
X_baseline = combined_data[["sex", "age_at_baseline", "PD_PGS_resid"]].copy()

scaler = StandardScaler()
cols_to_scale_baseline = ["age_at_baseline", "PD_PGS_resid"]
X_baseline_scaled = X_baseline.copy()
X_baseline_scaled[cols_to_scale_baseline] = scaler.fit_transform(X_baseline[cols_to_scale_baseline])

age_mean = scaler.mean_[0]
age_sd   = scaler.scale_[0]
pgs_mean = scaler.mean_[1]
pgs_sd   = scaler.scale_[1]

baseline_model = LogisticRegression(
    random_state=1,
    max_iter=2000,
    penalty=None,
    solver='newton-cholesky',
    fit_intercept=True
)
baseline_model.fit(X_baseline_scaled, y)

print("\nBaseline logistic regression model fitted on full PDBP dataset.")
print(f"Age scaling parameters - Mean: {age_mean:.4f}, SD: {age_sd:.4f}")
print(f"PGS scaling parameters - Mean: {pgs_mean:.4f}, SD: {pgs_sd:.4f}")
print(f"Baseline model intercept: {baseline_model.intercept_[0]:.6f}")
print(f"Baseline model coefficients (sex, age, PGS): {baseline_model.coef_[0]}")

y_baseline_pred_prob = baseline_model.predict_proba(X_baseline_scaled)[:, 1]
baseline_auc = roc_auc_score(y, y_baseline_pred_prob)
print(f"Baseline (sex + age + PGS) model AUC on full PDBP dataset: {baseline_auc:.4f}")

# =========================
# Closest.topleft function (pROC equivalent)
# =========================
def closest_topleft_threshold(y_true, y_prob):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    distances = np.sqrt((fpr)**2 + (1 - tpr)**2)
    idx = np.argmin(distances)
    return thresholds[idx], tpr[idx], 1 - fpr[idx]

def get_accuracy_at_threshold(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    return accuracy_score(y_true, y_pred)

# =========================
# External validation in PPMI_EUR, PPMI_AJ and HBS_EUR
# =========================
external_datasets = {
    "PPMI_EUR": {
        "TRS_path": "/home/jupyter/multiTRS/scores/PPMI_EUR_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt",
        "PGS_path": "/home/jupyter/multiTRS/geno/PRS_scores_out/PPMI_EUR_PD_PGS_resid.txt"
    },
    "PPMI_AJ": {
        "TRS_path": "/home/jupyter/multiTRS/scores/PPMI_AJ_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt",
        "PGS_path": "/home/jupyter/multiTRS/geno/PRS_scores_out/PPMI_AJ_PD_PGS_resid.txt"
    },
    "HBS_EUR": {
        "TRS_path": "/home/jupyter/multiTRS/scores/HBS_EUR_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt",
        "PGS_path": "/home/jupyter/multiTRS/geno/PRS_scores_out/HBS_EUR_PD_PGS_resid.txt"
    }
}

print("\n" + "="*70)
print("EXTERNAL VALIDATION RESULTS WITH DELONG'S TEST")
print("="*70)

full_results_summary = []

for dataset_name, paths in external_datasets.items():
    print(f"\n--- {dataset_name} Dataset ---")

    # Load TRS
    ext_TRS = pd.read_csv(paths["TRS_path"], sep="\t")
    TRS_cols = [c for c in ext_TRS.columns if "Hip_fracture_EUR_2022_FDR_SMR_single_SNP" not in c]
    ext_TRS = ext_TRS[TRS_cols]

    # Load PGS
    ext_PGS = pd.read_csv(paths["PGS_path"], sep="\t")[["participant_id", "PD_PGS_resid"]]

    # Merge
    ext_combined = pd.merge(clinical, ext_PGS, on="participant_id", how="inner")
    ext_combined = pd.merge(ext_combined, ext_TRS, on="participant_id", how="inner")
    print(f"Rows in combined data for {dataset_name}: {len(ext_combined)}")

    # Predictions - full model
    X_ext = ext_combined[X.columns]
    y_ext = ext_combined["case_control_other_at_baseline"]
    y_ext_pred_prob = final_model.predict_proba(X_ext)[:, 1]

    # Predictions - baseline model (scaled with PDBP parameters)
    X_ext_baseline = ext_combined[["sex", "age_at_baseline", "PD_PGS_resid"]].copy()
    X_ext_baseline_scaled = X_ext_baseline.copy()
    X_ext_baseline_scaled["age_at_baseline"] = (X_ext_baseline["age_at_baseline"] - age_mean) / age_sd
    X_ext_baseline_scaled["PD_PGS_resid"]    = (X_ext_baseline["PD_PGS_resid"]    - pgs_mean) / pgs_sd
    y_ext_baseline_pred_prob = baseline_model.predict_proba(X_ext_baseline_scaled)[:, 1]

    # DeLong test
    z_stat, p_value, ci_full, ci_null, auc_full, auc_null, info = Delong_test(
        y_ext, y_ext_pred_prob, y_ext_baseline_pred_prob,
        return_ci=True, return_auc=True, verbose=0
    )

    print(f"XGBoost model AUC:                   {auc_full:.4f} (95% CI: {ci_full[0]:.4f}-{ci_full[1]:.4f})")
    print(f"Baseline model AUC:                  {auc_null:.4f} (95% CI: {ci_null[0]:.4f}-{ci_null[1]:.4f})")
    print(f"AUC Difference (XGBoost - Baseline): {auc_full - auc_null:.4f}")
    print(f"Variance of AUC difference:          {info['var_diff']:.6f}")
    print(f"DeLong's test Z-statistic:           {z_stat:.4f}")
    print(f"DeLong's test p-value (2-tailed):    {p_value:.4e}")
    print(f"Result: {'Significantly different (p < 0.05)' if p_value < 0.05 else 'Not significantly different (p >= 0.05)'}")

    # Optimal threshold metrics
    thresh_full, sens_full, spec_full = closest_topleft_threshold(y_ext, y_ext_pred_prob)
    thresh_null, sens_null, spec_null = closest_topleft_threshold(y_ext, y_ext_baseline_pred_prob)

    acc_full = get_accuracy_at_threshold(y_ext, y_ext_pred_prob,          thresh_full)
    acc_null = get_accuracy_at_threshold(y_ext, y_ext_baseline_pred_prob, thresh_null)

    full_results_summary.append({
        'cohort':         dataset_name,
        'AUC_null':       auc_null,
        'AUC_lower_null': ci_null[0],
        'AUC_upper_null': ci_null[1],
        'AUC_full':       auc_full,
        'AUC_lower_full': ci_full[0],
        'AUC_upper_full': ci_full[1],
        'AUC_diff':       auc_full - auc_null,
        'sens_null':      sens_null,
        'sens_full':      sens_full,
        'sens_diff':      sens_full - sens_null,
        'spec_null':      spec_null,
        'spec_full':      spec_full,
        'spec_diff':      spec_full - spec_null,
        'acc_null':       acc_null,
        'acc_full':       acc_full,
        'acc_diff':       acc_full - acc_null,
        'delong_z':       z_stat,
        'delong_p':       p_value
    })

# =========================
# Save full results table
# =========================
print("\n" + "="*70)
print("FULL RESULTS TABLE")
print("="*70)
full_results_df = pd.DataFrame(full_results_summary)
print(full_results_df.to_string(index=False))

full_results_outfile = "/home/jupyter/multiTRS/results/multi_TRS_w_PRS_all_scores_XGBoost_external_validation_full_results.txt"
full_results_df.to_csv(full_results_outfile, sep="\t", index=False)
print(f"\nFull results saved to: {full_results_outfile}")

### Transfer over results

In [ ]:
!ls /home/jupyter/multiTRS/results/
shell_do(f'gsutil -u {BILLING_PROJECT_ID} -m cp -r /home/jupyter/multiTRS/results/multi_TRS_* {WORKSPACE_BUCKET}')

# Fit a sparse model using PTS with absolute weight > 0.01

## ENET

### Nested-CV

In [ ]:
%%R

library(data.table)
library(dplyr)
library(glmnet)
library(glmnetUtils)
library(caret)
library(pROC)

set.seed(1)

# Create a data.table for all performance results
all_performance_results <- data.table()

print("This script calculates ENET multi-TRS models using SMR-multi scores... Running ENET models")
print("This is a sparse model based on previous ENET above...")

outfile <- "/home/jupyter/multiTRS/results/multi_TRS_w_PRS_SMR_multi_ENET_sparse_nestedcv.txt"

coef <- fread("/home/jupyter/multiTRS/results/multi_TRS_w_PRS_SMR_multi_ENET_PDBP_full_model_coefficients.txt")

coef <- coef %>% filter(abs(coefficient) > 0.01)

# Read in the clinical data
clinical <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt") %>%
  select(participant_id, case_control_other_at_baseline, sex, age_at_baseline)

# List of cohorts to loop over
TRS_cohorts <- c("PDBP")

# Loop over cohorts
for (TRS_cohort in TRS_cohorts) {

  # Read TRS scores
  TRS_path <- paste0("/home/jupyter/multiTRS/scores/", TRS_cohort, "_TWAS_SMR_FDR_scores_resid.txt")
  TRS <- fread(TRS_path) %>% select(participant_id, any_of(coef$feature))

  # Read PGS scores
  PGS_path <- paste0("/home/jupyter/multiTRS/geno/PRS_scores_out/", TRS_cohort, "_EUR_PD_PGS_resid.txt")
  PGS <- fread(PGS_path)

  combined_data <- clinical %>%
    inner_join(PGS, by = "participant_id") %>%
    inner_join(TRS, by = "participant_id")

  cat("Rows in combined data for", TRS_cohort, ":", nrow(combined_data), "\n")

  # Null model with age + sex
  null_model <- glm(case_control_other_at_baseline ~ sex + scale(age_at_baseline) + scale(PD_PGS_resid),
                    data = combined_data,
                    family = binomial)
  probs_null <- predict(null_model, type = "response")
  roc_obj_null <- roc(combined_data$case_control_other_at_baseline, probs_null)
  auc_null <- auc(roc_obj_null)
  cat("AUC of null model:", auc_null, "\n")

  null_coords <- coords(roc_obj_null, x = "best", best.method = "closest.topleft",
                        ret = c("sensitivity", "specificity", "accuracy"))
  null_sensitivity <- as.numeric(null_coords["sensitivity"])
  null_specificity <- as.numeric(null_coords["specificity"])
  null_accuracy <- as.numeric(null_coords["accuracy"])

  # Prepare predictor matrix X and outcome y
  X <- combined_data %>%
    select(-participant_id, -case_control_other_at_baseline) %>%
    as.matrix()
  y <- as.factor(combined_data$case_control_other_at_baseline)

  cat("Number of predictors:", ncol(X), "\n")

  # Define penalty factors: unpenalized for sex, age and PD_PGS_resid
  penalty <- ifelse(colnames(X) %in% c("sex", "age_at_baseline", "PD_PGS_resid"), 0, 1)

# Create five outer folds
outer_folds <- createFolds(y, k = 5, returnTrain = TRUE)

# Store auc of outer folders
outer_auc <- c()
lambda_vals <- c()
alpha_vals  <- c()
outer_sensitivity <- c()
outer_specificity <- c()
outer_accuracy <- c()

for (i in 1:length(outer_folds)) {
  cat("Outer fold:", i, "\n")
  
  # Train/test split for this outer fold
  train_idx <- outer_folds[[i]]
  test_idx  <- setdiff(seq_along(y), train_idx)
  
  X_train <- X[train_idx, ]
  y_train <- y[train_idx]
  X_test  <- X[test_idx, ]
  y_test  <- y[test_idx]
  
  # Standardize inside training only
    vars_to_scale <- setdiff(colnames(X), "sex")
    scaler <- preProcess(X_train[, vars_to_scale, drop = FALSE], method = c("center", "scale"))
    
    X_train_scaled <- X_train
    X_train_scaled[, vars_to_scale] <- predict(scaler, X_train[, vars_to_scale, drop = FALSE])
    
    X_test_scaled <- X_test
    X_test_scaled[, vars_to_scale] <- predict(scaler, X_test[, vars_to_scale, drop = FALSE])
    
 # Specify alpha list 
  alphalist <- seq(0,1,by=0.1)
  
  # Inner CV with cv.glmnet (5-fold)
  cvfit <- glmnetUtils::cva.glmnet(
    x = X_train_scaled,
    y = y_train,
    family = "binomial",
    alpha = alphalist,
    nfolds = 5,
    type.measure = "auc",
    penalty.factor = penalty,
    standardize = FALSE
  )
  

    
  # Select best alpha and lambda
  lambda_1se <- sapply(cvfit$modlist, `[[`, "lambda.1se")
  error <- sapply(cvfit$modlist, function(mod) {
    idx <- which(mod$lambda == mod$lambda.1se)
    mod$cvm[idx]
  })
  best <- which.max(error)
  best_alpha  <- cvfit$alpha[best]
  best_lambda <- lambda_1se[best]
                  

  # Best lambda chosen inside
  lambda_vals[i] <- best_lambda
  alpha_vals[i]  <- best_alpha
    
  
  # Refit on full training set with best lambda
  final_model <- glmnet(
    x = X_train_scaled,
    y = y_train,
    family = "binomial",
    alpha = best_alpha,
    lambda = best_lambda,
    penalty.factor = penalty,
    standardize = FALSE
  )
  
  # Predict probabilities on outer test set
  probs <- predict(final_model, newx = X_test_scaled, type = "response")
  
# Compute ROC curve
  roc_obj_outer <- roc(y_test, as.numeric(probs))

# Compute AUC
  auc_val <- auc(roc_obj_outer)

# Get sens, spec, accuracy using topleft
model_coords <- coords(
  roc_obj_outer,
  x = "best",
  best.method = "closest.topleft",
  ret = c("threshold", "sensitivity", "specificity", "accuracy")
)

model_sensitivity <- as.numeric(model_coords["sensitivity"])
model_specificity <- as.numeric(model_coords["specificity"])
model_accuracy <- as.numeric(model_coords["accuracy"])
    
  outer_auc[i] <- auc_val
  outer_sensitivity[i] <- model_sensitivity
  outer_specificity[i] <- model_specificity
  outer_accuracy[i] <- model_accuracy
}

alpha_vals <- as.numeric(alpha_vals)
lambda_vals <- as.numeric(lambda_vals)

cat("Alpha per outer fold:\n")
print(alpha_vals)

cat("Lambda per outer fold:\n")
print(lambda_vals)
                  
lambda_final <- median(lambda_vals)
alpha_final <- median(alpha_vals)


cat("AUC per outer fold:\n")
print(outer_auc)
cat("Mean AUC across outer folds:", mean(outer_auc), "\n")
cat("SD of the AUC across outer folds:", sd(outer_auc), "\n")

cat("Median alpha:\n")
print(alpha_final)
cat("Median lambda:\n")
print(lambda_final)


                  
performance_results <- data.table(cohort = TRS_cohort,
                                  scores = "SMR-multi",
                                  model = "ENET (sparse)",
                                  covariate_treatment = "unpenalised",
                                  median_lambda = lambda_final,
                                  median_alpha = alpha_final,
                                  AUC_null = auc_null,
                                  AUC_full_mean_cv =  mean(outer_auc),
                                  AUC_full_sd_cv = sd(outer_auc),
                                  AUC_diff = mean(outer_auc) - auc_null,
                                  sens_null = null_sensitivity,
                                  sens_full_mean_cv = mean(outer_sensitivity),
                                  sens_full_sd_cv = sd(outer_sensitivity),
                                  sens_diff = mean(outer_sensitivity) - null_sensitivity,
                                  spec_null = null_specificity,
                                  spec_full_mean_cv = mean(outer_specificity),
                                  spec_full_sd_cv = sd(outer_specificity),
                                  spec_diff = mean(outer_specificity) - null_specificity,
                                  acc_null = null_accuracy,
                                  acc_full_mean_cv = mean(outer_accuracy),
                                  acc_full_sd_cv = sd(outer_accuracy),
                                  acc_diff = mean(outer_accuracy) - null_accuracy)
    

all_performance_results <- rbind(all_performance_results,performance_results)


}
                  
print(all_performance_results)
                  

write.table(all_performance_results, outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)

### Test in PPMI (EUR), PPMI (AJ) and HBS

In [ ]:
%%R

library(data.table)
library(dplyr)
library(glmnet)
library(caret)
library(pROC)
library(ggplot2)

set.seed(1)

# Read in the data
clinical <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt") %>% 
  select(participant_id, case_control_other_at_baseline, sex, age_at_baseline)

coef <- fread("/home/jupyter/multiTRS/results/multi_TRS_w_PRS_SMR_multi_ENET_PDBP_full_model_coefficients.txt")

coef <- coef %>% filter(abs(coefficient) > 0.01)

# Read in the PGS
PRS_PDBP_EUR <- fread("/home/jupyter/multiTRS/geno/PRS_scores_out/PDBP_EUR_PD_PGS_resid.txt")
PRS_PPMI_EUR <- fread("/home/jupyter/multiTRS/geno/PRS_scores_out/PPMI_EUR_PD_PGS_resid.txt")
PRS_PPMI_AJ  <- fread("/home/jupyter/multiTRS/geno/PRS_scores_out/PPMI_AJ_PD_PGS_resid.txt")
PRS_HBS_EUR  <- fread("/home/jupyter/multiTRS/geno/PRS_scores_out/HBS_EUR_PD_PGS_resid.txt")

# Read TRS scores - select columns based on PDBP first
TRS_PDBP_EUR <- fread("/home/jupyter/multiTRS/scores/PDBP_EUR_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt") %>% 
 select(participant_id, any_of(coef$feature))

TRS_PPMI_EUR <- fread("/home/jupyter/multiTRS/scores/PPMI_EUR_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt") %>% 
  select(all_of(colnames(TRS_PDBP_EUR)))

TRS_PPMI_AJ  <- fread("/home/jupyter/multiTRS/scores/PPMI_AJ_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt") %>%  
  select(all_of(colnames(TRS_PDBP_EUR)))

TRS_HBS_EUR  <- fread("/home/jupyter/multiTRS/scores/HBS_EUR_w_PGS_all_TWAS_SMR_FDR_scores_resid.txt") %>% 
  select(all_of(colnames(TRS_PDBP_EUR)))

# -----------------------------------------------
# Create combined datasets
# -----------------------------------------------

# PDBP (training)
combined_data <- clinical %>%
  inner_join(PRS_PDBP_EUR, by = "participant_id") %>%
  inner_join(TRS_PDBP_EUR, by = "participant_id")

# PPMI_EUR (test)
combined_data_PPMI_EUR <- clinical %>%
  inner_join(PRS_PPMI_EUR, by = "participant_id") %>%
  inner_join(TRS_PPMI_EUR, by = "participant_id")

# PPMI_AJ (test)
combined_data_PPMI_AJ <- clinical %>%
  inner_join(PRS_PPMI_AJ, by = "participant_id") %>%
  inner_join(TRS_PPMI_AJ, by = "participant_id")

# HBS_EUR (test)
combined_data_HBS_EUR <- clinical %>%
  inner_join(PRS_HBS_EUR, by = "participant_id") %>%
  inner_join(TRS_HBS_EUR, by = "participant_id")

cat("Testing SMR-multi model...\n")
cat("Rows in combined data - PDBP:", nrow(combined_data), "\n")
cat("Rows in combined data - PPMI_EUR:", nrow(combined_data_PPMI_EUR), "\n")
cat("Rows in combined data - PPMI_AJ:", nrow(combined_data_PPMI_AJ), "\n")
cat("Rows in combined data - HBS_EUR:", nrow(combined_data_HBS_EUR), "\n")

# -----------------------------------------------
# Null model (age + sex + PGS), fit on PDBP, applied to all
# -----------------------------------------------

null_model <- glm(case_control_other_at_baseline ~ scale(age_at_baseline) + sex + scale(PD_PGS_resid),
                  data = combined_data,
                  family = binomial)

# Null model predictions
null_probs_PDBP     <- predict(null_model, newdata = combined_data,          type = "response")
null_probs_PPMI_EUR <- predict(null_model, newdata = combined_data_PPMI_EUR, type = "response")
null_probs_PPMI_AJ  <- predict(null_model, newdata = combined_data_PPMI_AJ,  type = "response")
null_probs_HBS_EUR  <- predict(null_model, newdata = combined_data_HBS_EUR,  type = "response")

auc_null_PDBP     <- auc(roc(combined_data$case_control_other_at_baseline,          null_probs_PDBP))
auc_null_PPMI_EUR <- auc(roc(combined_data_PPMI_EUR$case_control_other_at_baseline, null_probs_PPMI_EUR))
auc_null_PPMI_AJ  <- auc(roc(combined_data_PPMI_AJ$case_control_other_at_baseline,  null_probs_PPMI_AJ))
auc_null_HBS_EUR  <- auc(roc(combined_data_HBS_EUR$case_control_other_at_baseline,  null_probs_HBS_EUR))

cat("Null model AUCs:\n")
cat("  PDBP:", auc_null_PDBP, "\n")
cat("  PPMI_EUR:", auc_null_PPMI_EUR, "\n")
cat("  PPMI_AJ:", auc_null_PPMI_AJ, "\n")
cat("  HBS_EUR:", auc_null_HBS_EUR, "\n")

# -----------------------------------------------
# Prepare predictor matrices
# -----------------------------------------------

X <- combined_data %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y <- as.factor(combined_data$case_control_other_at_baseline)

X_PPMI_EUR <- combined_data_PPMI_EUR %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y_PPMI_EUR <- as.factor(combined_data_PPMI_EUR$case_control_other_at_baseline)

X_PPMI_AJ <- combined_data_PPMI_AJ %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y_PPMI_AJ <- as.factor(combined_data_PPMI_AJ$case_control_other_at_baseline)

X_HBS_EUR <- combined_data_HBS_EUR %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y_HBS_EUR <- as.factor(combined_data_HBS_EUR$case_control_other_at_baseline)

cat("Number of predictors:", ncol(X), "\n")

# -----------------------------------------------
# Scale on PDBP, apply to all
# -----------------------------------------------

penalty <- ifelse(colnames(X) %in% c("PD_PGS_resid", "sex", "age_at_baseline"), 0, 1)
vars_to_scale <- setdiff(colnames(X), "sex")

scaler_full <- preProcess(X[, vars_to_scale], method = c("center", "scale"))

X_scaled          <- X
X_scaled[, vars_to_scale] <- predict(scaler_full, X[, vars_to_scale])

X_PPMI_EUR_scaled <- X_PPMI_EUR
X_PPMI_EUR_scaled[, vars_to_scale] <- predict(scaler_full, X_PPMI_EUR[, vars_to_scale])

X_PPMI_AJ_scaled  <- X_PPMI_AJ
X_PPMI_AJ_scaled[, vars_to_scale]  <- predict(scaler_full, X_PPMI_AJ[, vars_to_scale])

X_HBS_EUR_scaled  <- X_HBS_EUR
X_HBS_EUR_scaled[, vars_to_scale]  <- predict(scaler_full, X_HBS_EUR[, vars_to_scale])

# -----------------------------------------------
# Fit final LASSO model on PDBP
# -----------------------------------------------

final_model <- glmnet(
  x = X_scaled,
  y = y,
  family = "binomial",
  alpha = 0.3,
  lambda = 0.06072183,
  penalty.factor = penalty,
  standardize = FALSE
)

# Extract and print coefficients
coefs <- coef(final_model)
coef_df <- data.frame(
  feature = rownames(coefs),
  coefficient = as.numeric(coefs)
)

coef_df <- coef_df %>% arrange(desc(coefficient))

print(coef_df)

coef_outfile <- "/home/jupyter/multiTRS/results/multi_TRS_w_PRS_SMR_multi_ENET_sparse_PDBP_full_model_coefficients.txt"
write.table(coef_df, coef_outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)
cat("Coefficients saved to:", coef_outfile, "\n")

# -----------------------------------------------
# Predictions and AUCs
# -----------------------------------------------

get_auc <- function(model, X_new, y_new) {
  probs <- predict(model, newx = X_new, type = "response")
  auc(roc(y_new, as.numeric(probs)))
}

auc_PDBP     <- get_auc(final_model, X_scaled,          y)
auc_PPMI_EUR <- get_auc(final_model, X_PPMI_EUR_scaled, y_PPMI_EUR)
auc_PPMI_AJ  <- get_auc(final_model, X_PPMI_AJ_scaled,  y_PPMI_AJ)
auc_HBS_EUR  <- get_auc(final_model, X_HBS_EUR_scaled,  y_HBS_EUR)

cat("Full model AUCs:\n")
cat("  PDBP:", auc_PDBP, "\n")
cat("  PPMI_EUR:", auc_PPMI_EUR, "\n")
cat("  PPMI_AJ:", auc_PPMI_AJ, "\n")
cat("  HBS_EUR:", auc_HBS_EUR, "\n")

# -----------------------------------------------
# ROC objects
# -----------------------------------------------

probs_PPMI_EUR <- predict(final_model, newx = X_PPMI_EUR_scaled, type = "response")
probs_PPMI_AJ  <- predict(final_model, newx = X_PPMI_AJ_scaled,  type = "response")
probs_HBS_EUR  <- predict(final_model, newx = X_HBS_EUR_scaled,  type = "response")

roc_null_PPMI_EUR <- roc(y_PPMI_EUR, null_probs_PPMI_EUR)
roc_full_PPMI_EUR <- roc(y_PPMI_EUR, as.numeric(probs_PPMI_EUR))

roc_null_PPMI_AJ  <- roc(y_PPMI_AJ,  null_probs_PPMI_AJ)
roc_full_PPMI_AJ  <- roc(y_PPMI_AJ,  as.numeric(probs_PPMI_AJ))

roc_null_HBS_EUR  <- roc(y_HBS_EUR,  null_probs_HBS_EUR)
roc_full_HBS_EUR  <- roc(y_HBS_EUR,  as.numeric(probs_HBS_EUR))

# -----------------------------------------------
# CIs for AUCs
# -----------------------------------------------

ci_null_PPMI_EUR <- ci.auc(roc_null_PPMI_EUR)
ci_null_PPMI_AJ  <- ci.auc(roc_null_PPMI_AJ)
ci_null_HBS_EUR  <- ci.auc(roc_null_HBS_EUR)

ci_full_PPMI_EUR <- ci.auc(roc_full_PPMI_EUR)
ci_full_PPMI_AJ  <- ci.auc(roc_full_PPMI_AJ)
ci_full_HBS_EUR  <- ci.auc(roc_full_HBS_EUR)

cat("Null model AUCs with 95% CI:\n")
cat("  PPMI_EUR:", round(auc_null_PPMI_EUR, 3), "(95% CI:", round(ci_null_PPMI_EUR[1], 3), "-", round(ci_null_PPMI_EUR[3], 3), ")\n")
cat("  PPMI_AJ:",  round(auc_null_PPMI_AJ, 3),  "(95% CI:", round(ci_null_PPMI_AJ[1], 3),  "-", round(ci_null_PPMI_AJ[3], 3),  ")\n")
cat("  HBS_EUR:",  round(auc_null_HBS_EUR, 3),   "(95% CI:", round(ci_null_HBS_EUR[1], 3),  "-", round(ci_null_HBS_EUR[3], 3),  ")\n")

cat("Full model AUCs with 95% CI:\n")
cat("  PPMI_EUR:", round(auc_PPMI_EUR, 3), "(95% CI:", round(ci_full_PPMI_EUR[1], 3), "-", round(ci_full_PPMI_EUR[3], 3), ")\n")
cat("  PPMI_AJ:",  round(auc_PPMI_AJ, 3),  "(95% CI:", round(ci_full_PPMI_AJ[1], 3),  "-", round(ci_full_PPMI_AJ[3], 3),  ")\n")
cat("  HBS_EUR:",  round(auc_HBS_EUR, 3),   "(95% CI:", round(ci_full_HBS_EUR[1], 3),  "-", round(ci_full_HBS_EUR[3], 3),  ")\n")

# -----------------------------------------------
# CI bands for plotting
# -----------------------------------------------

ci_se_null_PPMI_EUR <- ci.se(roc_null_PPMI_EUR, specificities = seq(0, 1, 0.01))
ci_se_full_PPMI_EUR <- ci.se(roc_full_PPMI_EUR, specificities = seq(0, 1, 0.01))

ci_se_null_PPMI_AJ  <- ci.se(roc_null_PPMI_AJ,  specificities = seq(0, 1, 0.01))
ci_se_full_PPMI_AJ  <- ci.se(roc_full_PPMI_AJ,  specificities = seq(0, 1, 0.01))

ci_se_null_HBS_EUR  <- ci.se(roc_null_HBS_EUR,  specificities = seq(0, 1, 0.01))
ci_se_full_HBS_EUR  <- ci.se(roc_full_HBS_EUR,  specificities = seq(0, 1, 0.01))

# -----------------------------------------------
# DeLong tests
# -----------------------------------------------

delong_PPMI_EUR <- roc.test(roc_null_PPMI_EUR, roc_full_PPMI_EUR, method = "delong")
delong_PPMI_AJ  <- roc.test(roc_null_PPMI_AJ,  roc_full_PPMI_AJ,  method = "delong")
delong_HBS_EUR  <- roc.test(roc_null_HBS_EUR,  roc_full_HBS_EUR,  method = "delong")

cat("DeLong test p-values:\n")
cat("  PPMI_EUR:", delong_PPMI_EUR$p.value, "\n")
cat("  PPMI_AJ:",  delong_PPMI_AJ$p.value,  "\n")
cat("  HBS_EUR:",  delong_HBS_EUR$p.value,  "\n")

# -----------------------------------------------
# ggplot2 ROC plotting helpers
# -----------------------------------------------

roc_to_df <- function(roc_obj, label) {
  data.frame(
    fpr   = 1 - roc_obj$specificities,
    tpr   = roc_obj$sensitivities,
    model = label
  )
}

ci_se_to_df <- function(ci_se_obj, label) {
  data.frame(
    fpr   = 1 - as.numeric(rownames(ci_se_obj)),
    lower = ci_se_obj[, 1],
    upper = ci_se_obj[, 3],
    model = label
  )
}

make_roc_plot <- function(roc_null, roc_full,
                          ci_se_null, ci_se_full,
                          ci_null, ci_full,
                          title, full_col, outfile) {

  df_lines <- rbind(
    roc_to_df(roc_null, "null"),
    roc_to_df(roc_full, "full")
  )

  df_ribbon <- rbind(
    ci_se_to_df(ci_se_null, "null"),
    ci_se_to_df(ci_se_full, "full")
  )

  label_null <- paste0("Age + Sex + PD-PGS  (AUC = ", round(auc(roc_null), 3),
                       " [", round(ci_null[1], 3), "\u2013", round(ci_null[3], 3), "])")
  label_full <- paste0("Sparse multi-PTS model  (AUC = ", round(auc(roc_full), 3),
                       " [", round(ci_full[1], 3), "\u2013", round(ci_full[3], 3), "])")

  colour_map <- c("null" = "black", "full" = full_col)
  fill_map   <- colour_map
  label_map  <- c("null" = label_null, "full" = label_full)

  p <- ggplot() +
    geom_ribbon(data = df_ribbon,
                aes(x = fpr, ymin = lower, ymax = upper, fill = model),
                alpha = 0.15) +
    geom_line(data = df_lines,
              aes(x = fpr, y = tpr, colour = model),
              linewidth = 0.9) +
    geom_abline(slope = 1, intercept = 0,
                linetype = "dashed", colour = "grey60", linewidth = 0.4) +
    scale_colour_manual(values = colour_map, labels = label_map, name = NULL) +
    scale_fill_manual(  values = fill_map,   labels = label_map, name = NULL) +
    coord_equal(xlim = c(0, 1), ylim = c(0, 1)) +
    labs(title = title, x = "1 - Specificity (FPR)", y = "Sensitivity (TPR)") +
    theme_classic(base_size = 12) +
    theme(
      legend.position      = c(0.97, 0.05),
      legend.justification = c("right", "bottom"),
      legend.background    = element_blank(),
      legend.key           = element_blank(),
      legend.text          = element_text(size = 9),
      plot.title           = element_text(face = "bold", hjust = 0.5)
    )

  print(p)
  ggsave(outfile, plot = p, width = 5, height = 5, dpi = 300, device = "pdf")
  cat("Saved:", outfile, "\n")
  invisible(p)
}

# -----------------------------------------------
# Save all three plots
# -----------------------------------------------

make_roc_plot(roc_null_PPMI_EUR, roc_full_PPMI_EUR,
              ci_se_null_PPMI_EUR, ci_se_full_PPMI_EUR,
              ci_null_PPMI_EUR, ci_full_PPMI_EUR,
              title    = "PPMI (EUR)",
              full_col = "goldenrod",
              outfile  = "/home/jupyter/multiTRS/results/ROC_PPMI_EUR.pdf")

make_roc_plot(roc_null_PPMI_AJ, roc_full_PPMI_AJ,
              ci_se_null_PPMI_AJ, ci_se_full_PPMI_AJ,
              ci_null_PPMI_AJ, ci_full_PPMI_AJ,
              title    = "PPMI (AJ)",
              full_col = "forestgreen",
              outfile  = "/home/jupyter/multiTRS/results/ROC_PPMI_AJ.pdf")

make_roc_plot(roc_null_HBS_EUR, roc_full_HBS_EUR,
              ci_se_null_HBS_EUR, ci_se_full_HBS_EUR,
              ci_null_HBS_EUR, ci_full_HBS_EUR,
              title    = "HBS (EUR)",
              full_col = "dodgerblue",
              outfile  = "/home/jupyter/multiTRS/results/ROC_HBS_EUR.pdf")

# -----------------------------------------------
# Extract sensitivity, specificity, accuracy at
# optimal threshold (closest-to-top-left)
# -----------------------------------------------

coords_null_PPMI_EUR <- coords(roc_null_PPMI_EUR, x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]
coords_full_PPMI_EUR <- coords(roc_full_PPMI_EUR, x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]

coords_null_PPMI_AJ  <- coords(roc_null_PPMI_AJ,  x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]
coords_full_PPMI_AJ  <- coords(roc_full_PPMI_AJ,  x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]

coords_null_HBS_EUR  <- coords(roc_null_HBS_EUR,  x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]
coords_full_HBS_EUR  <- coords(roc_full_HBS_EUR,  x = "best", best.method = "closest.topleft",
                               ret = c("sensitivity", "specificity", "accuracy"))[1, ]

# -----------------------------------------------
# Build results table
# -----------------------------------------------

results <- rbindlist(list(
  data.table(
    cohort         = "PPMI_EUR",
    AUC_null       = as.numeric(auc(roc_null_PPMI_EUR)),
    AUC_lower_null = as.numeric(ci_null_PPMI_EUR[1]),
    AUC_upper_null = as.numeric(ci_null_PPMI_EUR[3]),
    AUC_full       = as.numeric(auc(roc_full_PPMI_EUR)),
    AUC_lower_full = as.numeric(ci_full_PPMI_EUR[1]),
    AUC_upper_full = as.numeric(ci_full_PPMI_EUR[3]),
    AUC_diff       = as.numeric(auc(roc_full_PPMI_EUR)) - as.numeric(auc(roc_null_PPMI_EUR)),
    sens_null      = coords_null_PPMI_EUR$sensitivity,
    sens_full      = coords_full_PPMI_EUR$sensitivity,
    sens_diff      = coords_full_PPMI_EUR$sensitivity - coords_null_PPMI_EUR$sensitivity,
    spec_null      = coords_null_PPMI_EUR$specificity,
    spec_full      = coords_full_PPMI_EUR$specificity,
    spec_diff      = coords_full_PPMI_EUR$specificity - coords_null_PPMI_EUR$specificity,
    acc_null       = coords_null_PPMI_EUR$accuracy,
    acc_full       = coords_full_PPMI_EUR$accuracy,
    acc_diff       = coords_full_PPMI_EUR$accuracy - coords_null_PPMI_EUR$accuracy,
    delong_z       = delong_PPMI_EUR$statistic,
    delong_p       = delong_PPMI_EUR$p.value
  ),
  data.table(
    cohort         = "PPMI_AJ",
    AUC_null       = as.numeric(auc(roc_null_PPMI_AJ)),
    AUC_lower_null = as.numeric(ci_null_PPMI_AJ[1]),
    AUC_upper_null = as.numeric(ci_null_PPMI_AJ[3]),
    AUC_full       = as.numeric(auc(roc_full_PPMI_AJ)),
    AUC_lower_full = as.numeric(ci_full_PPMI_AJ[1]),
    AUC_upper_full = as.numeric(ci_full_PPMI_AJ[3]),
    AUC_diff       = as.numeric(auc(roc_full_PPMI_AJ)) - as.numeric(auc(roc_null_PPMI_AJ)),
    sens_null      = coords_null_PPMI_AJ$sensitivity,
    sens_full      = coords_full_PPMI_AJ$sensitivity,
    sens_diff      = coords_full_PPMI_AJ$sensitivity - coords_null_PPMI_AJ$sensitivity,
    spec_null      = coords_null_PPMI_AJ$specificity,
    spec_full      = coords_full_PPMI_AJ$specificity,
    spec_diff      = coords_full_PPMI_AJ$specificity - coords_null_PPMI_AJ$specificity,
    acc_null       = coords_null_PPMI_AJ$accuracy,
    acc_full       = coords_full_PPMI_AJ$accuracy,
    acc_diff       = coords_full_PPMI_AJ$accuracy - coords_null_PPMI_AJ$accuracy,
    delong_z       = delong_PPMI_AJ$statistic,
    delong_p       = delong_PPMI_AJ$p.value
  ),
  data.table(
    cohort         = "HBS_EUR",
    AUC_null       = as.numeric(auc(roc_null_HBS_EUR)),
    AUC_lower_null = as.numeric(ci_null_HBS_EUR[1]),
    AUC_upper_null = as.numeric(ci_null_HBS_EUR[3]),
    AUC_full       = as.numeric(auc(roc_full_HBS_EUR)),
    AUC_lower_full = as.numeric(ci_full_HBS_EUR[1]),
    AUC_upper_full = as.numeric(ci_full_HBS_EUR[3]),
    AUC_diff       = as.numeric(auc(roc_full_HBS_EUR)) - as.numeric(auc(roc_null_HBS_EUR)),
    sens_null      = coords_null_HBS_EUR$sensitivity,
    sens_full      = coords_full_HBS_EUR$sensitivity,
    sens_diff      = coords_full_HBS_EUR$sensitivity - coords_null_HBS_EUR$sensitivity,
    spec_null      = coords_null_HBS_EUR$specificity,
    spec_full      = coords_full_HBS_EUR$specificity,
    spec_diff      = coords_full_HBS_EUR$specificity - coords_null_HBS_EUR$specificity,
    acc_null       = coords_null_HBS_EUR$accuracy,
    acc_full       = coords_full_HBS_EUR$accuracy,
    acc_diff       = coords_full_HBS_EUR$accuracy - coords_null_HBS_EUR$accuracy,
    delong_z       = delong_HBS_EUR$statistic,
    delong_p       = delong_HBS_EUR$p.value
  )
))

print(results)

results_outfile <- "/home/jupyter/multiTRS/results/multi_TRS_w_PRS_SMR_multi_SPARSE_ENET_external_validation_results_table.txt"
write.table(results, results_outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)
cat("Results table saved to:", results_outfile, "\n")

### Transfer plots

In [ ]:
!ls /home/jupyter/multiTRS/results/
shell_do(f'gsutil -u {BILLING_PROJECT_ID} -m cp -r /home/jupyter/multiTRS/results/ROC_* {WORKSPACE_BUCKET}')